# Harmonization Metrics Analysis

Comprehensive analysis of batch effect metrics and gene sets across all harmonization attempts.

---

# Part 1 — Metrics Analysis
## §1 — Setup, imports, and data loading

In [ ]:
import gzip
import io
import json
import math
import os
import re
import warnings
from pathlib import Path

import boto3
import matplotlib.patches as mpatches
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap as _umap_lib
from matplotlib.colors import Normalize, to_hex, to_rgba
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu, median_abs_deviation, pearsonr, spearmanr, bootstrap
from sklearn.decomposition import PCA
from sklearn.decomposition import PCA as _PCA2
from sklearn.manifold import TSNE as _TSNE
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler as _SS
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
sns.set_style("ticks")

# ── Font size constants (change FONT_BASE to scale all text globally) ──────
FONT_BASE = 7.5
FONT_TITLE = FONT_BASE
FONT_LABEL = FONT_BASE
FONT_TICK = FONT_BASE
FONT_LEGEND = FONT_BASE
FONT_ANNOT = FONT_BASE
FONT_SMALL = FONT_BASE
FONT_TINY = FONT_BASE

plt.rcParams.update(
    {
        "svg.fonttype": "none",
        "figure.dpi": 150,
        "font.size": FONT_BASE,
        "axes.titlesize": FONT_TITLE,
        "axes.labelsize": FONT_LABEL,
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "legend.fontsize": FONT_LEGEND,
        "figure.titlesize": FONT_TITLE,
    }
)

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)
S3_BUCKET = "$FL_S3_BUCKET"
S3_PREFIX = "FL_batch_correction"

In [ ]:
from biomart import BiomartServer

server = BiomartServer("http://www.ensembl.org/biomart")

# Fetch the main genes database
ensembl_db = server.databases.get("ENSEMBL_MART_ENSEMBL")

if ensembl_db:
    print(f"Connected to: {ensembl_db.display_name}")
    # Example output: Connected to: Ensembl Genes 112

### Cell 1.2 — Load metrics CSV

In [ ]:
df = pd.read_csv("./metric_tables/metrics_comprehensive_260609.csv")
df["run_id"] = (
    df["strat"]
    + "__"
    + df["imp"]
    + "__"
    + df["method"]
    + "__post"
    + df["post_rm"].map({False: "0", True: "1"})
)
df = df.set_index("run_id")

META_COLS = ["strat", "imp", "method", "post_rm", "status", "compute_time_s"]
ERROR_COLS = [c for c in df.columns if c.startswith("error")]

HARSHNESS_PAL = {
    "low": "#2ecc71",  # green
    "medium": "#f39c12",  # amber/yellow-orange
    "high": "#e74c3c",  # red
}

df_ok = df[df["status"] == "ok"].copy()
print(
    f"Total rows: {len(df)}  |  OK: {len(df_ok)}  |  Failed: {(df['status']=='failed').sum()}"
)
print(f"Strats: {sorted(df_ok.strat.unique())}")
print(f"Imps: {sorted(df_ok.imp.unique())}")
print(f"Methods ({len(df_ok.method.unique())}): {sorted(df_ok.method.unique())}")
print(f"post_rm values: {sorted(df_ok.post_rm.unique())}")

# ── Strategies actually present in data ──────────────────────────────────────
AVAIL_STRATS = sorted(df_ok["strat"].unique())
for _new_strat in ["I_rare_batches_removed", "J_ff_only", "K_ffpe_only"]:
    if _new_strat not in AVAIL_STRATS:
        print(f"NOTE: {_new_strat} strategy not in CSV — palette added, no data yet.")


# ── Shambhala method metadata ─────────────────────────────────────────────────
def _parse_shambhala_method(method: str) -> tuple[str, str]:
    """Return (p_key, q_key) for shambhala_* methods; ('', '') for others."""
    if not method.startswith("shambhala_"):
        return "", ""
    tail = method[len("shambhala_") :]
    parts = tail.split("_")
    q_idx = next((i for i, p in enumerate(parts) if p.startswith("Q")), None)
    if q_idx is None:
        return tail, ""
    return "_".join(parts[:q_idx]), "_".join(parts[q_idx:])


df_ok["is_shambhala"] = df_ok["method"].str.startswith("shambhala_")
df_ok = df_ok[
    (~df_ok["method"].str.startswith("shambhala_"))
    | (df_ok["method"] == "shambhala_P0std_Q0std")
]
df_ok["method"][(df_ok["method"] == "shambhala_P0std_Q0std")] = "20_shambhala"

df_ok["shambhala_p_key"] = df_ok["method"].apply(
    lambda m: _parse_shambhala_method(m)[0]
)
df_ok["shambhala_q_key"] = df_ok["method"].apply(
    lambda m: _parse_shambhala_method(m)[1]
)
SHAMBHALA_AVAIL = bool(df_ok["is_shambhala"].any())
print(f"Shambhala methods present: {SHAMBHALA_AVAIL}")
if SHAMBHALA_AVAIL:
    print(
        f"  P-keys: {sorted(df_ok.loc[df_ok['is_shambhala'], 'shambhala_p_key'].unique())}"
    )
    print(
        f"  Q-keys: {sorted(df_ok.loc[df_ok['is_shambhala'], 'shambhala_q_key'].unique())}"
    )

# ── Harshness level per method ────────────────────────────────────────────────
HARSHNESS_LEVEL_MAP: dict[str, str] = {
    "01_raw": "low",
    "02_median_scaling": "low",
    "03_limma": "low",
    "04_sva": "low",
    "05_combat": "medium",
    "06_combat_seq": "medium",
    "07_pycombat": "medium",
    "08_inmoose_combatseq": "medium",
    "09_ruv": "medium",
    "10_mnn": "medium",
    "11_harmony": "medium",
    "12_scanorama": "medium",
    "13_fsmvn": "medium",
    "14_qsmooth": "high",
    "15_fsqn_py": "high",
    "16_fsqn_r": "high",
    "17_quantile": "high",
    "18_rank": "high",
    "19_tdm": "high",
    "20_shambhala": "high",
    "21_harmonizr": "medium",
    "22_tmm": "low",
    "23_vst": "low",
    "24_peer_k10": "medium",
    "25_angel": "high",
    "26_xpn": "high",
    "27_dwd": "medium",
    "28_npn": "high",
    "29_combat_ref": "medium",
    "30_recombat": "medium",
    "31_ruv3prps": "medium",
    "32_deepmnn": "medium",
    "33_amdbnorm": "high",
    "34_arsyn": "medium",
    "35_dasc": "medium",
    "36_explobatch": "medium",
    "37_fabatch": "medium",
    "38_harman": "medium",
    "39_procrustes": "high",
}
df_ok["harshness_level"] = df_ok["method"].apply(
    lambda m: HARSHNESS_LEVEL_MAP.get(
        m,
        "high" if m.startswith("shambhala_") else "medium",
    )
)
method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)

try:
    df_ok["pct_samples_allNA"] = df_ok["n_samples_allNA"] * 100 / df_ok["n_samples"]
    df_ok["pct_genes_allNA"] = df_ok["n_genes_allNA"] * 100 / df_ok["n_genes"]
    df_ok = df_ok[df_ok["pct_samples_allNA"] < 5]
except Exception as e:
    print(e)
print(f"Harshness distribution: {df_ok['harshness_level'].value_counts().to_dict()}")

In [ ]:
df_ok

In [ ]:
# columns_to_save = df_ok.columns[~df_ok.columns.isin(['is_shambhala', 'shambhala_p_key', 'shambhala_q_key', 'harshness_level'])]
columns_to_save = [
    "strat",
    "imp",
    "method",
    "post_rm",
    "status",
    "compute_time_s",
] + df_normed.columns.to_list()

df_ok[columns_to_save].to_csv(
    "../figures_for_article/Supplementary File 3.csv.gz",
    index=False,
    compression="gzip",
)

#### Best approaches selection

In [ ]:
# ── Best-approach selection: constants, lookup, filtered DataFrame ────────────
_top_ids = [
    ("10_mnn", "strict", "S0_no_removal"),
    ("10_mnn", "strict", "H_affymetrix_extended"),
    ("10_mnn", "strict", "D_malignant_only"),
    ("04_sva", "knn", "C_rnaseq_only"),
    ("04_sva", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "knn", "C_rnaseq_only"),
    ("16_fsqn_r", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "strict", "C_rnaseq_only"),
    ("10_mnn", "strict", "J_ff_only"),
    ("16_fsqn_r", "strict", "J_ff_only"),
    ("04_sva", "knn", "K_ffpe_only"),
    ("04_sva", "strict", "K_ffpe_only"),
    ("04_sva", "softimpute", "K_ffpe_only"),
    ("13_fsmvn", "strict", "S0_no_removal"),
    ("33_amdbnorm", "strict", "S0_no_removal"),
]
_top_ids_set = set(_top_ids)  # O(1) lookup
_top_methods = sorted({m for m, i, s in _top_ids})
_top_strats = sorted({s for m, i, s in _top_ids})
_top_imps = sorted({i for m, i, s in _top_ids})

BEST_COLOR = "#E63946"  # vivid red — reserved for "★ Best" highlights
BEST_HATCH = "////"  # diagonal stripe
BEST_MARKER = "*"
BEST_MARKERSIZE = 400
BEST_EDGECOLOR = "black"
BEST_ALPHA = 0.85
BEST_LABEL = "Best"

df_ok["is_best"] = df_ok.apply(
    lambda r: (r["method"], r["imp"], r["strat"]) in _top_ids_set and not r["post_rm"],
    axis=1,
)
df_best = df_ok[df_ok["is_best"]].copy()
print(
    f"df_best: {len(df_best)} rows  |  methods: {df_best.method.nunique()}  "
    f"|  strats: {df_best.strat.nunique()}"
)

In [ ]:
df_best

### Cell 1.3 — Column taxonomy

In [ ]:
GROUP_MAP = {
    "A": ["r2_", "pcr_", "dsc_"],
    "B": ["kbet_", "ilisi_", "clisi_", "asw_batch", "asw_bio", "cms_"],
    "C": ["umap_", "tsne_"],
    "D": ["ks_", "per_gene_batch"],
    "E": [
        "n_samples",
        "n_genes",
        "n_batches",
        "n_cohorts",
        "n_diagnosis_groups",
        "n_samples_per_batch",
        "zero_",
        "fraction_",
        "exp_",
        "per_batch_median",
        "bimodal",
        "below_1",
    ],
    "F": ["vp_"],
    "G": ["graph_"],
    "H": ["avg_intra_", "avg_inter_", "dist_ratio_"],
    "I": ["wm_"],
    "J": ["pct_var_pc", "pct_var_cum"],
    "K": [
        "n_genes_noNA",
        "pct_genes_noNA",
        "n_samples_noNA",
        "pct_samples_noNA",
        "n_genes_allNA",
        "n_samples_allNA",
        "n_na_cells",
        "pct_na_cells",
    ],
}

ANNOT_COLS_ORDER = [
    "RNA_BATCH",
    "PLATFORM_RNA",
    "RNASEQ_SOURCE",
    "COHORT_LABEL",
    "Major_group",
    "Diagnosis_cell_type_unified",
    "TUMOR_NORMAL",
    "global/all",
]


def classify_metric_col(col):
    """
    Classify a metric column name into a group letter and annotation column.

    Scans GROUP_MAP in order; the first group whose prefix list contains a
    string that ``col`` starts with (or equals after stripping trailing "_") is
    selected.  If no group matches, returns "?".

    Annotation column is determined by scanning ANNOT_COLS_ORDER and returning
    the first entry that appears as a substring of ``col``; otherwise returns
    "global/all".

    Parameters
    ----------
    col : str
        Metric column name from the metrics CSV.

    Returns
    -------
    tuple of (group, annot_col)
        group     : str — single letter A–H (or "?" if unrecognised)
        annot_col : str — annotation grouping variable (e.g. "RNA_BATCH") or
                    "global/all" if the metric is not tied to a specific covariate
    """
    for g, prefixes in GROUP_MAP.items():
        if any(col.startswith(p) or col == p.rstrip("_") for p in prefixes):
            grp = g
            break
    else:
        grp = "?"
    annot = "global/all"
    for a in ANNOT_COLS_ORDER[:-1]:
        if a in col:
            annot = a
            break
    return grp, annot


_SKIP_EXACT = set(
    [
        "asw_batch_COHORT_LABEL",
        "asw_batch_PLATFORM_RNA",
        "asw_batch_RNASEQ_SOURCE",
        "asw_batch_RNA_BATCH",
        "asw_bio_Diagnosis_cell_type_unified",
        "asw_bio_Major_group",
        "asw_bio_PLATFORM_RNA",
        "asw_bio_TUMOR_NORMAL",
    ]
)

metric_cols = [
    c
    for c in df_ok.columns
    if c not in META_COLS
    and c not in ERROR_COLS
    and c not in _SKIP_EXACT
    and not c.startswith("r2_pc")
    and not c.startswith("dsc_pvalue")
    and not c.startswith("n_cohorts_")
    and not c.startswith("n_samples_per_batch")
    # and not c.startswith("cms_mean")
    and not c.startswith("ilisi_mean") and c != "wm_subsampled"
]

col_meta = pd.DataFrame(
    [classify_metric_col(c) for c in metric_cols],
    index=metric_cols,
    columns=["group", "annot_col"],
)
print(f"Total metric columns: {len(metric_cols)}")
print(col_meta.group.value_counts().sort_index())
# ── Coarse metric type classification ─────────────────────────────────────
METRIC_TYPE_MAP = {
    "local_neighborhood": [
        "kbet_",
        "ilisi_",
        "clisi_",
        "graph_connectivity_",
        "umap_entropy_",
        "tsne_entropy_",
    ],
    "global_distance": [
        "r2_",
        "pcr_",
        "dsc_",
        "umap_centroid_disp_",
        "tsne_centroid_disp_",
        "dist_ratio_",
        "avg_intra_dist_",
        "avg_inter_dist_",
        "asw_batch_",
        "asw_bio_",
        "cms_",
        "pct_var_pc",
        "pct_var_cum",
        "wm_",
    ],
    "distribution_similarity": [
        "ks_mean_D_",
        "ks_frac_sig_",
        "ks_cohort_within_batch_",
        "per_gene_batch_mean_cv_",
    ],
    "other": ["pct_genes_", "pct_samples_", "pct_na_cells"],
}


def classify_metric_type(col: str) -> str:
    for type_name, prefixes in METRIC_TYPE_MAP.items():
        if type_name == "other":
            continue
        if any(col.startswith(p) for p in prefixes):
            return type_name
    return "other"


col_meta["metric_type"] = col_meta.index.map(classify_metric_type)
print(col_meta["metric_type"].value_counts())

In [ ]:
col_meta["metric_type"][col_meta.index.str.startswith("wm")]

### Cell 1.4 — Metric polarity (higher=better +1, lower=better -1, descriptive 0)

In [ ]:
# ── Polarity rationale ────────────────────────────────────────────────────────
# POLARITY[c] = +1  → higher raw value = better batch correction or biology
# POLARITY[c] = -1  → lower raw value = better (e.g. lower R² = less batch)
# POLARITY[c] =  0  → descriptive metric: excluded from composite scoring
#
# Group A (r2_, pcr_, dsc_):
#   R² measures fraction of variance explained by batch. Lower is better (-1).
#   PCR (variance-weighted R²): same direction (-1).
#   DSC (distance between batch centroids): lower = centroids more mixed (-1).
#
# Group B (neighbor-based):
#   asw_batch_norm: ASW of batch labels after normalization; higher = better
#     batch mixing in neighbor graph (+1 regardless of which batch column).
#   asw_bio_norm: ASW of biology labels; higher = biology clusters are tight (+1).
#   kbet, ilisi_norm: acceptance rate / local mixing score; higher = better (+1).
#   clisi_mean: local inverse Simpson index for cell type; higher = better (+1).
#   cms_fraction_mixed: fraction of neighbors from different batches; higher = better (+1).
#   avg_intra_dist, avg_inter_dist: raw distances — direction depends on context,
#     so kept as 0 (excluded from scoring). Use dist_ratio instead.
#
# Group C (embedding metrics):
#   entropy_norm: batch entropy in UMAP/tSNE neighborhood; higher = better mixing (+1).
#   centroid_disp: distance between batch centroids in embedding; lower = better (-1).
#
# Group D (distribution):
#   ks_mean_D: mean KS D-statistic; lower = distributions more aligned (-1).
#   per_gene_batch_mean_cv: per-gene CV across batches; lower = less batch CV (-1).
#
# Group E (data quality):
#   n_genes, n_samples: more is better (+1).
#   fraction_cohorts_bimodal: bimodality is a natural gene-expression property,
#     NOT an artifact. Methods that preserve it are preferred (+1).
#   fraction_cohorts_zero_inflated_bimodal: pathological zero inflation = bad (-1).
#   Other descriptive stats (n_batches, exp_*, per_batch_median): polarity 0.
#
# Group F (variancePartition):
#   vp_ for batch columns: lower variance fraction = better batch removal (-1).
#   vp_ for biology columns: higher variance fraction = biology preserved (+1).
#
# Group G (graph connectivity):
#   graph_connectivity_*: higher = biology groups form connected clusters (+1).
#
#
# Group I (WaterMelon entropy):
#   wm_mean_batch, wm_{batch_col}: lower = better batch mixing (-1).
#   wm_mean_bio, wm_{bio_col}: higher = biology better preserved (+1).
#   wm_ratio_bio_batch: higher ratio = better trade-off (+1).
#
# Group J (PC variance %):
#   pct_var_pc{i}: descriptive; tells how much variance each PC carries (0).
#   Used only as weights in derived wt_r2_pc{i} columns computed later.
#
# Group K (NA retention):
#   pct_genes_noNA, pct_samples_noNA: more non-NA = better (+1).
#   n_genes_allNA, n_samples_allNA, n_na_cells, pct_na_cells: fewer NAs = better (-1).
#   n_genes_noNA, n_samples_noNA: raw counts, not comparable across gene sets (0).
#
# Group H (pairwise distances):
#   dist_ratio_*: intra-group mean distance / inter-group mean distance.
#     For batch columns: lower dist_ratio = batches more mixed = better (-1).
#     For biology columns: lower dist_ratio = biology clusters tighter = better (-1).
#   avg_intra_dist, avg_inter_dist: raw distances, not normalized → polarity 0.

POLARITY = {}
for c in metric_cols:
    g, _ = classify_metric_col(c)
    if g == "A":
        POLARITY[c] = -1
    elif g == "B":
        if "asw_bio_norm" in c:
            POLARITY[c] = +1
        elif "asw_batch_norm" in c:
            POLARITY[c] = +1
        elif any(x in c for x in ("kbet", "ilisi_norm", "cms_fraction_mixed")):
            POLARITY[c] = +1
        elif "clisi_mean" in c:
            POLARITY[c] = +1
        else:
            POLARITY[c] = 0
    elif g == "C":
        if "entropy_norm" in c:
            POLARITY[c] = +1
        elif "centroid_disp" in c:
            POLARITY[c] = -1
        else:
            POLARITY[c] = 0
    elif g == "D":
        if any(x in c for x in ("ks_mean_D", "ks_frac_sig", "ks_cohort")):
            POLARITY[c] = -1
        elif "per_gene_batch_mean_cv" in c:
            POLARITY[c] = -1
        else:
            POLARITY[c] = 0
    elif g == "E":
        POLARITY[c] = 0
    elif g == "F":
        POLARITY[c] = -1 if "RNA_BATCH" in c else +1
    elif g == "G":
        POLARITY[c] = +1
    elif g == "H":
        POLARITY[c] = -1 if "dist_ratio" in c else 0
    elif g == "I":
        if "mean_batch" in c or (
            c.startswith("wm_")
            and any(
                b in c
                for b in ("RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL")
            )
        ):
            POLARITY[c] = -1
        elif (
            "mean_bio" in c
            or "ratio_bio_batch" in c
            or (
                c.startswith("wm_")
                and any(b in c for b in ("Major_group", "Diagnosis", "TUMOR_NORMAL"))
            )
        ):
            POLARITY[c] = +1
        else:
            POLARITY[c] = 0
    elif g == "J":
        POLARITY[c] = 0
    elif g == "K":
        if c in ("pct_genes_noNA", "pct_samples_noNA"):
            POLARITY[c] = +1
        elif c in ("n_genes_allNA", "n_samples_allNA", "n_na_cells", "pct_na_cells"):
            POLARITY[c] = -1
        else:
            POLARITY[c] = 0
    else:
        POLARITY[c] = 0

# Override pcr_: batch covariates lower=better; biology covariates higher=better
for c in metric_cols:
    if c.startswith("pcr_"):
        if any(
            b in c
            for b in ("RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL")
        ):
            POLARITY[c] = -1
        else:
            POLARITY[c] = +1

scoring_cols = [c for c in metric_cols if POLARITY.get(c, 0) != 0]
desc_cols = [c for c in metric_cols if POLARITY.get(c, 0) == 0]
print(f"Scoring columns: {len(scoring_cols)}  |  Descriptive: {len(desc_cols)}")

In [ ]:
pd.Series(POLARITY).value_counts().sort_index().rename("count").to_frame()

In [ ]:
# ── Manual overrides to POLARITY (applied after the auto-assignment above) ─────
# These entries correct cases where the automatic logic assigns the wrong polarity:
#
# asw_batch_norm_* for non-RNA_BATCH columns:
#   The auto-logic sets asw_batch_norm = +1 only when 'asw_batch_norm' prefix is found.
#   Manual check confirmed that ALL asw_batch_norm columns (regardless of batch column)
#   use the same polarity: higher = better mixing (after normalization).
#
# asw_bio_norm_PLATFORM_RNA = 0:
#   PLATFORM_RNA is a batch/technical column, not a biology column. Measuring
#   ASW of a batch grouping as a "biology" metric is meaningless → excluded.
#
# avg_intra_dist_* = 0, avg_inter_dist_* = 0:
#   Raw distances in high-dimensional space are scale-dependent and not directly
#   comparable across methods (different gene sets → different scale). Use
#   dist_ratio (normalized ratio) for scoring instead.
POLARITY_MANUAL = {
    "asw_batch_norm_COHORT_LABEL": 1,
    "asw_batch_norm_PLATFORM_RNA": 1,
    "asw_batch_norm_RNASEQ_SOURCE": 1,
    "asw_batch_norm_RNA_BATCH": 1,
    "asw_bio_norm_Diagnosis_cell_type_unified": 1,
    "asw_bio_norm_Major_group": 1,
    "asw_bio_norm_PLATFORM_RNA": 0,
    "asw_bio_norm_TUMOR_NORMAL": 1,
    "avg_inter_dist_COHORT_LABEL": 0,
    "avg_inter_dist_Diagnosis_cell_type_unified": 0,
    "avg_inter_dist_Major_group": 0,
    "avg_inter_dist_PLATFORM_RNA": 0,
    "avg_inter_dist_RNASEQ_SOURCE": 0,
    "avg_inter_dist_RNA_BATCH": 0,
    "avg_inter_dist_TUMOR_NORMAL": 0,
    "avg_intra_dist_COHORT_LABEL": 0,
    "avg_intra_dist_Diagnosis_cell_type_unified": 0,
    "avg_intra_dist_Major_group": 0,
    "avg_intra_dist_PLATFORM_RNA": 0,
    "avg_intra_dist_RNASEQ_SOURCE": 0,
    "avg_intra_dist_RNA_BATCH": 0,
    "avg_intra_dist_TUMOR_NORMAL": 0,
    "below_1_fraction_by_batch_max": 0,
    "below_1_fraction_by_batch_min": 0,
    "clisi_mean_Diagnosis_cell_type_unified": 1,
    "clisi_mean_Major_group": 1,
    "clisi_mean_PLATFORM_RNA": 0,
    "clisi_mean_TUMOR_NORMAL": 1,
    "cms_fraction_mixed_COHORT_LABEL": 1,
    "cms_fraction_mixed_PLATFORM_RNA": 1,
    "cms_fraction_mixed_RNASEQ_SOURCE": 1,
    "cms_fraction_mixed_RNA_BATCH": 1,
    "cms_mean_COHORT_LABEL": 0,
    "cms_mean_PLATFORM_RNA": 0,
    "cms_mean_RNASEQ_SOURCE": 0,
    "cms_mean_RNA_BATCH": 0,
    "dist_ratio_COHORT_LABEL": 1,
    "dist_ratio_Diagnosis_cell_type_unified": -1,
    "dist_ratio_Major_group": -1,
    "dist_ratio_PLATFORM_RNA": 1,
    "dist_ratio_RNASEQ_SOURCE": 1,
    "dist_ratio_RNA_BATCH": 1,
    "dist_ratio_TUMOR_NORMAL": -1,
    "dsc_COHORT_LABEL": -1,
    "dsc_PLATFORM_RNA": -1,
    "dsc_RNASEQ_SOURCE": -1,
    "dsc_RNA_BATCH": -1,
    "exp_max": 0,
    "exp_median": 0,
    "exp_min": 0,
    "exp_p01": 0,
    "exp_p05": 0,
    "exp_p25": 0,
    "exp_p75": 0,
    "exp_p95": 0,
    "exp_p99": 0,
    "exp_std": 0,
    "fraction_cohorts_bimodal": 1,
    "fraction_cohorts_zero_inflated_bimodal": 1,
    "fraction_genes_below_1": 0,
    "graph_connectivity_Diagnosis_cell_type_unified": 1,
    "graph_connectivity_Major_group": 1,
    "graph_connectivity_PLATFORM_RNA": 0,
    "graph_connectivity_TUMOR_NORMAL": 1,
    "ilisi_norm_COHORT_LABEL": 1,
    "ilisi_norm_PLATFORM_RNA": 1,
    "ilisi_norm_RNASEQ_SOURCE": 1,
    "ilisi_norm_RNA_BATCH": 1,
    "kbet_acceptance_rate_COHORT_LABEL": 1,
    "kbet_acceptance_rate_PLATFORM_RNA": 1,
    "kbet_acceptance_rate_RNASEQ_SOURCE": 1,
    "kbet_acceptance_rate_RNA_BATCH": 1,
    "ks_cohort_within_batch_frac_sig": -1,
    "ks_cohort_within_batch_mean_D": -1,
    "ks_frac_sig_COHORT_LABEL": -1,
    "ks_frac_sig_PLATFORM_RNA": -1,
    "ks_frac_sig_RNA_BATCH": -1,
    "ks_mean_D_COHORT_LABEL": -1,
    "ks_mean_D_PLATFORM_RNA": -1,
    "ks_mean_D_RNA_BATCH": -1,
    "n_batches": 0,
    "n_cohorts": 0,
    "n_diagnosis_groups": 0,
    "n_genes": 0,
    "n_samples": 0,
    "pcr_COHORT_LABEL": 1,
    "pcr_Diagnosis_cell_type_unified": -1,
    "pcr_Major_group": -1,
    "pcr_PLATFORM_RNA": 1,
    "pcr_RNASEQ_SOURCE": 1,
    "pcr_RNA_BATCH": 1,
    "pcr_TUMOR_NORMAL": -1,
    "per_batch_median_cv": 0,
    "pct_samples_allNA": -1,  # calculated not for all harmonization attempts, should uncomment when all of them will be calculated
    "pct_genes_allNA": -1,
    "pct_genes_noNA": 1,
    "pct_na_cells": -1,
    "pct_samples_noNA": 1,
    "n_na_cells": 0,
    "per_gene_batch_mean_cv_COHORT_LABEL": -1,
    "per_gene_batch_mean_cv_PLATFORM_RNA": -1,
    "per_gene_batch_mean_cv_RNA_BATCH": -1,
    "r2_COHORT_LABEL": 0,
    "r2_Major_group": 0,
    "r2_PLATFORM_RNA": 0,
    "r2_RNASEQ_SOURCE": 0,
    "r2_RNA_BATCH": 0,
    "r2_TUMOR_NORMAL": 0,
    "tsne_centroid_disp_COHORT_LABEL": -1,
    "tsne_centroid_disp_PLATFORM_RNA": -1,
    "tsne_centroid_disp_RNASEQ_SOURCE": -1,
    "tsne_centroid_disp_RNA_BATCH": -1,
    "tsne_entropy_mean_COHORT_LABEL": 0,
    "tsne_entropy_mean_PLATFORM_RNA": 0,
    "tsne_entropy_mean_RNASEQ_SOURCE": 0,
    "tsne_entropy_mean_RNA_BATCH": 0,
    "tsne_entropy_norm_COHORT_LABEL": 1,
    "tsne_entropy_norm_PLATFORM_RNA": 1,
    "tsne_entropy_norm_RNASEQ_SOURCE": 1,
    "tsne_entropy_norm_RNA_BATCH": 1,
    "umap_centroid_disp_COHORT_LABEL": -1,
    "umap_centroid_disp_PLATFORM_RNA": -1,
    "umap_centroid_disp_RNASEQ_SOURCE": -1,
    "umap_centroid_disp_RNA_BATCH": -1,
    "umap_entropy_mean_COHORT_LABEL": 0,
    "umap_entropy_mean_PLATFORM_RNA": 0,
    "umap_entropy_mean_RNASEQ_SOURCE": 0,
    "umap_entropy_mean_RNA_BATCH": 0,
    "umap_entropy_norm_COHORT_LABEL": 1,
    "umap_entropy_norm_PLATFORM_RNA": 1,
    "umap_entropy_norm_RNASEQ_SOURCE": 1,
    "umap_entropy_norm_RNA_BATCH": 1,
    "wm_COHORT_LABEL": -1,  # calculated not for all harmonization attempts, should uncomment when all of them will be calculated
    "wm_Diagnosis_cell_type_unified": 1,
    "wm_Major_group": 1,
    "wm_PLATFORM_RNA": -1,
    "wm_RNASEQ_SOURCE": 1,
    "wm_RNA_BATCH": -1,
    "wm_TUMOR_NORMAL": 1,
    "wm_mean_batch": -1,
    "wm_mean_bio": 1,
    "wm_ratio_bio_batch": 1,
    "zero_fraction_by_batch_max": 0,
    "zero_fraction_by_batch_min": 0,
    "zero_fraction_global": 0,
}
POLARITY.update(POLARITY_MANUAL)

# Update scoring/desc cols after manual overrides
scoring_cols = [c for c in metric_cols if POLARITY.get(c, 0) != 0]
desc_cols = [c for c in metric_cols if POLARITY.get(c, 0) == 0]
print(
    f"After manual overrides — scoring: {len(scoring_cols)}, descriptive: {len(desc_cols)}"
)

In [ ]:
metrics_full = ['compute_time_s',
 'asw_batch_COHORT_LABEL',
 'asw_batch_PLATFORM_RNA',
 'asw_batch_RNASEQ_SOURCE',
 'asw_batch_RNA_BATCH',
 'asw_batch_norm_COHORT_LABEL',
 'asw_batch_norm_PLATFORM_RNA',
 'asw_batch_norm_RNASEQ_SOURCE',
 'asw_batch_norm_RNA_BATCH',
 'asw_bio_Diagnosis_cell_type_unified',
 'asw_bio_Major_group',
 'asw_bio_PLATFORM_RNA',
 'asw_bio_TUMOR_NORMAL',
 'asw_bio_norm_Diagnosis_cell_type_unified',
 'asw_bio_norm_Major_group',
 'asw_bio_norm_PLATFORM_RNA',
 'asw_bio_norm_TUMOR_NORMAL',
 'avg_inter_dist_COHORT_LABEL',
 'avg_inter_dist_Diagnosis_cell_type_unified',
 'avg_inter_dist_Major_group',
 'avg_inter_dist_PLATFORM_RNA',
 'avg_inter_dist_RNASEQ_SOURCE',
 'avg_inter_dist_RNA_BATCH',
 'avg_inter_dist_TUMOR_NORMAL',
 'avg_intra_dist_COHORT_LABEL',
 'avg_intra_dist_Diagnosis_cell_type_unified',
 'avg_intra_dist_Major_group',
 'avg_intra_dist_PLATFORM_RNA',
 'avg_intra_dist_RNASEQ_SOURCE',
 'avg_intra_dist_RNA_BATCH',
 'avg_intra_dist_TUMOR_NORMAL',
 'below_1_fraction_by_batch_max',
 'below_1_fraction_by_batch_min',
 'clisi_mean_Diagnosis_cell_type_unified',
 'clisi_mean_Major_group',
 'clisi_mean_PLATFORM_RNA',
 'clisi_mean_TUMOR_NORMAL',
 'cms_fraction_mixed_COHORT_LABEL',
 'cms_fraction_mixed_PLATFORM_RNA',
 'cms_fraction_mixed_RNASEQ_SOURCE',
 'cms_fraction_mixed_RNA_BATCH',
 'cms_mean_COHORT_LABEL',
 'cms_mean_PLATFORM_RNA',
 'cms_mean_RNASEQ_SOURCE',
 'cms_mean_RNA_BATCH',
 'dist_ratio_COHORT_LABEL',
 'dist_ratio_Diagnosis_cell_type_unified',
 'dist_ratio_Major_group',
 'dist_ratio_PLATFORM_RNA',
 'dist_ratio_RNASEQ_SOURCE',
 'dist_ratio_RNA_BATCH',
 'dist_ratio_TUMOR_NORMAL',
 'dsc_COHORT_LABEL',
 'dsc_PLATFORM_RNA',
 'dsc_RNASEQ_SOURCE',
 'dsc_RNA_BATCH',
 'dsc_pvalue_COHORT_LABEL',
 'dsc_pvalue_PLATFORM_RNA',
 'dsc_pvalue_RNASEQ_SOURCE',
 'dsc_pvalue_RNA_BATCH',
 'exp_max',
 'exp_median',
 'exp_min',
 'exp_p01',
 'exp_p05',
 'exp_p25',
 'exp_p75',
 'exp_p95',
 'exp_p99',
 'exp_std',
 'fraction_cohorts_bimodal',
 'fraction_cohorts_zero_inflated_bimodal',
 'fraction_genes_below_1',
 'graph_connectivity_Diagnosis_cell_type_unified',
 'graph_connectivity_Major_group',
 'graph_connectivity_PLATFORM_RNA',
 'graph_connectivity_TUMOR_NORMAL',
 'ilisi_mean_COHORT_LABEL',
 'ilisi_mean_PLATFORM_RNA',
 'ilisi_mean_RNASEQ_SOURCE',
 'ilisi_mean_RNA_BATCH',
 'ilisi_norm_COHORT_LABEL',
 'ilisi_norm_PLATFORM_RNA',
 'ilisi_norm_RNASEQ_SOURCE',
 'ilisi_norm_RNA_BATCH',
 'kbet_acceptance_rate_COHORT_LABEL',
 'kbet_acceptance_rate_PLATFORM_RNA',
 'kbet_acceptance_rate_RNASEQ_SOURCE',
 'kbet_acceptance_rate_RNA_BATCH',
 'ks_cohort_within_batch_frac_sig',
 'ks_cohort_within_batch_mean_D',
 'ks_frac_sig_COHORT_LABEL',
 'ks_frac_sig_PLATFORM_RNA',
 'ks_frac_sig_RNA_BATCH',
 'ks_mean_D_COHORT_LABEL',
 'ks_mean_D_PLATFORM_RNA',
 'ks_mean_D_RNA_BATCH',
 'n_batches',
 'n_cohorts',
 'n_cohorts_bimodal',
 'n_cohorts_zero_inflated_bimodal',
 'n_diagnosis_groups',
 'n_genes',
 'n_genes_allNA',
 'n_genes_noNA',
 'n_na_cells',
 'n_samples',
 'n_samples_allNA',
 'n_samples_noNA',
 'n_samples_per_batch_max',
 'n_samples_per_batch_median',
 'n_samples_per_batch_min',
 'n_samples_per_batch_sd',
 'pcr_COHORT_LABEL',
 'pcr_Diagnosis_cell_type_unified',
 'pcr_Major_group',
 'pcr_PLATFORM_RNA',
 'pcr_RNASEQ_SOURCE',
 'pcr_RNA_BATCH',
 'pcr_TUMOR_NORMAL',
 'pct_genes_noNA',
 'pct_na_cells',
 'pct_samples_noNA',
 'pct_var_cum_top10',
 'pct_var_pc1',
 'pct_var_pc10',
 'pct_var_pc2',
 'pct_var_pc3',
 'pct_var_pc4',
 'pct_var_pc5',
 'pct_var_pc6',
 'pct_var_pc7',
 'pct_var_pc8',
 'pct_var_pc9',
 'per_batch_median_cv',
 'per_gene_batch_mean_cv_COHORT_LABEL',
 'per_gene_batch_mean_cv_PLATFORM_RNA',
 'per_gene_batch_mean_cv_RNA_BATCH',
 'r2_COHORT_LABEL',
 'r2_Major_group',
 'r2_PLATFORM_RNA',
 'r2_RNASEQ_SOURCE',
 'r2_RNA_BATCH',
 'r2_TUMOR_NORMAL',
 'r2_pc10_COHORT_LABEL',
 'r2_pc10_PLATFORM_RNA',
 'r2_pc10_RNASEQ_SOURCE',
 'r2_pc10_RNA_BATCH',
 'r2_pc1_COHORT_LABEL',
 'r2_pc1_PLATFORM_RNA',
 'r2_pc1_RNASEQ_SOURCE',
 'r2_pc1_RNA_BATCH',
 'r2_pc2_COHORT_LABEL',
 'r2_pc2_PLATFORM_RNA',
 'r2_pc2_RNASEQ_SOURCE',
 'r2_pc2_RNA_BATCH',
 'r2_pc3_COHORT_LABEL',
 'r2_pc3_PLATFORM_RNA',
 'r2_pc3_RNASEQ_SOURCE',
 'r2_pc3_RNA_BATCH',
 'r2_pc4_COHORT_LABEL',
 'r2_pc4_PLATFORM_RNA',
 'r2_pc4_RNASEQ_SOURCE',
 'r2_pc4_RNA_BATCH',
 'r2_pc5_COHORT_LABEL',
 'r2_pc5_PLATFORM_RNA',
 'r2_pc5_RNASEQ_SOURCE',
 'r2_pc5_RNA_BATCH',
 'r2_pc6_COHORT_LABEL',
 'r2_pc6_PLATFORM_RNA',
 'r2_pc6_RNASEQ_SOURCE',
 'r2_pc6_RNA_BATCH',
 'r2_pc7_COHORT_LABEL',
 'r2_pc7_PLATFORM_RNA',
 'r2_pc7_RNASEQ_SOURCE',
 'r2_pc7_RNA_BATCH',
 'r2_pc8_COHORT_LABEL',
 'r2_pc8_PLATFORM_RNA',
 'r2_pc8_RNASEQ_SOURCE',
 'r2_pc8_RNA_BATCH',
 'r2_pc9_COHORT_LABEL',
 'r2_pc9_PLATFORM_RNA',
 'r2_pc9_RNASEQ_SOURCE',
 'r2_pc9_RNA_BATCH',
 'tsne_centroid_disp_COHORT_LABEL',
 'tsne_centroid_disp_PLATFORM_RNA',
 'tsne_centroid_disp_RNASEQ_SOURCE',
 'tsne_centroid_disp_RNA_BATCH',
 'tsne_entropy_mean_COHORT_LABEL',
 'tsne_entropy_mean_PLATFORM_RNA',
 'tsne_entropy_mean_RNASEQ_SOURCE',
 'tsne_entropy_mean_RNA_BATCH',
 'tsne_entropy_norm_COHORT_LABEL',
 'tsne_entropy_norm_PLATFORM_RNA',
 'tsne_entropy_norm_RNASEQ_SOURCE',
 'tsne_entropy_norm_RNA_BATCH',
 'umap_centroid_disp_COHORT_LABEL',
 'umap_centroid_disp_PLATFORM_RNA',
 'umap_centroid_disp_RNASEQ_SOURCE',
 'umap_centroid_disp_RNA_BATCH',
 'umap_entropy_mean_COHORT_LABEL',
 'umap_entropy_mean_PLATFORM_RNA',
 'umap_entropy_mean_RNASEQ_SOURCE',
 'umap_entropy_mean_RNA_BATCH',
 'umap_entropy_norm_COHORT_LABEL',
 'umap_entropy_norm_PLATFORM_RNA',
 'umap_entropy_norm_RNASEQ_SOURCE',
 'umap_entropy_norm_RNA_BATCH',
 'wm_COHORT_LABEL',
 'wm_Diagnosis_cell_type_unified',
 'wm_Major_group',
 'wm_PLATFORM_RNA',
 'wm_RNASEQ_SOURCE',
 'wm_RNA_BATCH',
 'wm_TUMOR_NORMAL',
 'wm_mean_batch',
 'wm_mean_bio',
 'wm_ratio_bio_batch',
 'wm_subsampled',
 'zero_fraction_by_batch_max',
 'zero_fraction_by_batch_min',
 'zero_fraction_global',
               'pct_samples_allNA',
 'pct_genes_allNA']

In [ ]:
for metric in metrics_full:
    if metric not in POLARITY.keys():
        POLARITY.update({metric:0})

In [ ]:
len(POLARITY.keys())

In [ ]:
pd.DataFrame(pd.Series(POLARITY), columns=["Polarity"]).to_csv(
    "../figures_for_article/Supplementary File 4.csv"
)

### Cell 1.5 — Normalize metrics to [0,1] where 1=best

In [ ]:
def normalize_for_display(df_vals, cols, polarity_map):
    """
    Normalize metric columns to [0, 1] where 1 always means "best".

    Pipeline: NaN fill → MinMax scaling → polarity flip for lower-is-better metrics.

    Parameters
    ----------
    df_vals : pd.DataFrame
        DataFrame containing raw metric values (samples × metrics).
    cols : list of str
        Metric columns to normalize.  Columns not present in ``df_vals`` are
        silently skipped.
    polarity_map : dict[str, int]
        Maps each column to +1 (higher raw = better) or -1 (lower raw = better).
        Columns with polarity -1 are flipped after scaling so that 1 = best.

    Returns
    -------
    pd.DataFrame
        Same shape as ``df_vals[cols]`` with values in [0, 1] and NaN replaced
        by the column median (neutral position, not best or worst).
    """
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    filled = normed.fillna(normed.median())
    scaler = MinMaxScaler()
    normed[valid_cols] = scaler.fit_transform(filled)
    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1.0 - normed[c]
    return normed


def normalize_for_display_with_dropna(df_vals, cols, polarity_map):
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    # normed = normed.dropna(thresh=50)
    # normed = normed[normed['pct_samples_allNA']<5]
    scaler = MinMaxScaler()
    normed[valid_cols] = scaler.fit_transform(normed)
    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1.0 - normed[c]
    # normed = normed.fillna(-0.1)
    return normed


def dropna_for_display_no_normalization(df_vals, cols, polarity_map):
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    normed = normed.dropna(thresh=36)

    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1 - normed[c]
    normed = normed.fillna(0)
    return normed


df_normed = normalize_for_display_with_dropna(df_ok, scoring_cols, POLARITY)
print("Normalized shape:", df_normed.shape)
print(
    "Value range: [{:.3f}, {:.3f}]".format(df_normed.min().min(), df_normed.max().max())
)

In [ ]:
# Validation: confirm df_normed is correctly shaped and all values in [0, 1]
print(f"Shape: {df_normed[scoring_cols].shape}")
print(
    f"Value range: [{df_normed[scoring_cols].min().min():.4f}, {df_normed[scoring_cols].max().max():.4f}]"
)
nan_count = df_normed[scoring_cols].isna().sum().sum()
print(f"NaN count (should be 0 after fill): {nan_count}")
assert df_normed[scoring_cols].min().min() >= -1e-9, "Values below 0 found"
assert df_normed[scoring_cols].max().max() <= 1 + 1e-9, "Values above 1 found"
assert nan_count == 0, "NaNs remain after normalization"
print("All assertions passed.")

### Cell 1.6 — Plotting helpers

Reusable function used throughout §3. Produces a boxplot + strip (top) and
pairwise FDR-corrected Mann-Whitney p-value heatmap (bottom) for any metric × grouping.

#### Multi-level comparison helper

In [ ]:
def plot_multilevel_comparison(
    df: pd.DataFrame,
    metric_col: str,
    group_col: str,
    figsize: tuple = None,
    title: str = None,
    palette: dict = None,
) -> plt.Figure:
    """
    Two-panel figure: boxplot + FDR-corrected significance heatmap.

    Parameters
    ----------
    df : pd.DataFrame
    metric_col : str
    group_col : str
    figsize : tuple, optional
    title : str, optional
    palette : dict, optional
        Color dict keyed by group value. Auto-detected from group_col when None.
    """
    _auto = {"method": method_pal, "strat": strat_pal, "imp": imp_pal}
    if palette is None:
        palette = _auto.get(group_col, {})

    groups = sorted(df[group_col].dropna().unique())
    data_by_group = {
        g: df.loc[df[group_col] == g, metric_col].dropna().values for g in groups
    }
    n = len(groups)
    if figsize is None:
        figsize = (max(8, n * 0.7 + 2), 8)
    fig, (ax_box, ax_pval) = plt.subplots(
        2, 1, figsize=figsize, gridspec_kw={"height_ratios": [3, 1]}
    )
    bp = ax_box.boxplot(
        [data_by_group[g] for g in groups],
        labels=groups,
        patch_artist=True,
        medianprops={"color": "black", "linewidth": 1.5},
    )
    fallback = sns.color_palette("husl", n)
    colors = [palette.get(g, to_hex(fallback[i])) for i, g in enumerate(groups)]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
    rng = np.random.default_rng(42)
    for i, g in enumerate(groups):
        vals = data_by_group[g]
        jitter = rng.normal(0, 0.08, len(vals))
        ax_box.scatter(
            np.full(len(vals), i + 1) + jitter,
            vals,
            alpha=0.5,
            s=8,
            zorder=5,
            color=colors[i],
        )
    if title:
        ax_box.set_title(title, fontsize=FONT_TITLE)
    ax_box.set_ylabel(metric_col, fontsize=FONT_LABEL)
    ax_box.set_xticklabels(groups, rotation=45, ha="right", fontsize=FONT_SMALL)

    pmat = np.ones((n, n))
    raw_pvals, pairs = [], []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = data_by_group[groups[i]], data_by_group[groups[j]]
            if len(a) >= 2 and len(b) >= 2:
                _, p = mannwhitneyu(a, b, alternative="two-sided")
                raw_pvals.append(p)
                pairs.append((i, j))
    if raw_pvals:
        _, fdr, _, _ = multipletests(raw_pvals, method="fdr_bh")
        for (i, j), p in zip(pairs, fdr):
            pmat[i, j] = p
            pmat[j, i] = p
    log_pmat = -np.log10(np.clip(pmat, 1e-300, 1.0))
    np.fill_diagonal(log_pmat, 0)
    sns.heatmap(
        log_pmat,
        ax=ax_pval,
        xticklabels=groups,
        yticklabels=groups,
        cmap="YlOrRd",
        vmin=0,
        linewidths=0.3,
        cbar_kws={"label": "−log₁₀(FDR)", "shrink": 0.6},
    )
    ax_pval.set_xticklabels(groups, rotation=45, ha="right", fontsize=FONT_TINY)
    ax_pval.set_yticklabels(groups, rotation=0, fontsize=FONT_TINY)
    fig.tight_layout()
    return fig


def clustermap_std(
    data: pd.DataFrame,
    metric_colors_df: pd.DataFrame,
    attempt_colors_df: pd.DataFrame,
    row_cluster: bool = True,
    col_cluster: bool = True,
    figsize: tuple = (60, 20),
    title: str = "",
    fname: str = None,
    **kwargs,
):
    """
    Standard clustermap: rows=metrics, cols=attempts.
    metric_colors_df → seaborn row_colors (left).
    attempt_colors_df → seaborn col_colors (top).

    Parameters
    ----------
    data : pd.DataFrame  shape (n_metrics, n_attempts), values in [0, 1]
    metric_colors_df : pd.DataFrame  row annotations
    attempt_colors_df : pd.DataFrame  column annotations
    fname : str or None  saves to FIGURES_DIR/{fname}.svg and .png if given
    """
    # Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations
    g = sns.clustermap(
        data,
        method="ward",
        metric="euclidean",
        row_cluster=row_cluster,
        col_cluster=col_cluster,
        row_colors=metric_colors_df.reindex(data.index),
        col_colors=attempt_colors_df.reindex(data.columns),
        cmap="RdYlGn",
        center=0.5,
        vmin=0,
        vmax=1,
        linewidths=0,
        xticklabels=False,
        yticklabels=True,
        cbar_pos=(0.02, 0.82, 0.012, 0.12),
        dendrogram_ratio=(0.03, 0.12),
        colors_ratio=(0.015, 0.025),
        **kwargs,
    )
    if title:
        g.fig.suptitle(title, fontsize=FONT_TITLE, y=1.01)
    if fname:
        g.fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
        g.fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    return g


def add_legend_outside(
    fig: plt.Figure,
    ax: plt.Axes,
    palettes: dict,
    x_offset: float = 1.05,
    fontsize: int = None,
) -> None:
    """
    Add one legend block per palette dict to the right of ax.

    Parameters
    ----------
    palettes : dict of str → dict  (legend title → {label: color})
    """
    if fontsize is None:
        fontsize = FONT_LEGEND
    for idx, (leg_title, pal) in enumerate(palettes.items()):
        handles = make_legend_patches(pal)
        legend = fig.legend(
            handles=handles,
            title=leg_title,
            loc="right",
            bbox_to_anchor=(x_offset + idx * 0.07, 0.5),
            bbox_transform=ax.transAxes,
            fontsize=fontsize,
            title_fontsize=fontsize + 1,
            frameon=False,
        )
        fig.add_artist(legend)


def strip_box_plot(
    df: pd.DataFrame,
    x: str,
    y: str,
    ax: plt.Axes,
    palette: dict = None,
    order: list = None,
    box_width: float = 0.5,
    strip_size: float = 4,
    strip_alpha: float = 0.5,
    xlabel: str = None,
    ylabel: str = None,
    title: str = "",
    rotate_x: int = 45,
) -> None:
    """
    Combined boxplot + strip plot. Palette is auto-detected when x is method/strat/imp.
    """
    _auto = {"method": method_pal, "strat": strat_pal, "imp": imp_pal}
    if palette is None:
        palette = _auto.get(x, {})
    if order is None:
        order = sorted(df[x].dropna().unique())
    colors = [palette.get(g, "#AAAAAA") for g in order]
    sns.boxplot(
        data=df,
        x=x,
        y=y,
        ax=ax,
        order=order,
        palette=dict(zip(order, colors)),
        width=box_width,
        fliersize=0,
        boxprops={"alpha": 0.6},
    )
    sns.stripplot(
        data=df,
        x=x,
        y=y,
        ax=ax,
        order=order,
        palette=dict(zip(order, colors)),
        size=strip_size,
        alpha=strip_alpha,
        jitter=True,
        dodge=False,
    )
    ax.set_xlabel(xlabel or x, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel or y, fontsize=FONT_LABEL)
    ax.set_xticklabels(
        ax.get_xticklabels(), rotation=rotate_x, ha="right", fontsize=FONT_SMALL
    )
    ax.tick_params(axis="y", labelsize=FONT_TICK)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)

#### Other reusable plot helpers

In [ ]:
# ── Cell 015b — Reusable plot helpers: scatter_plot, bar_plot, heatmap_plot ──


def scatter_plot(
    data: pd.DataFrame,
    x: str,
    y: str,
    hue: str,
    style: str | None = None,
    size: str | None = None,
    palette: dict | str | None = None,
    hue_order: list | None = None,
    style_order: list | None = None,
    alpha: float = 0.75,
    s: int = 30,
    ax: plt.Axes | None = None,
    figsize: tuple = (8, 6),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    legend_outside: bool = False,
    save_path: str | None = None,
) -> plt.Axes:
    """Scatter plot wrapping sns.scatterplot with standard project styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(
        data=data,
        x=x,
        y=y,
        hue=hue,
        style=style,
        size=size,
        palette=palette,
        hue_order=hue_order,
        style_order=style_order,
        alpha=alpha,
        s=s,
        linewidth=0,
        ax=ax,
    )
    ax.set_xlabel(xlabel if xlabel is not None else x, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel if ylabel is not None else y, fontsize=FONT_LABEL)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if legend_outside and ax.get_legend() is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
        ax.legend(
            handles,
            labels,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            fontsize=FONT_LEGEND,
            frameon=False,
        )
    sns.despine(ax=ax)
    if save_path:
        ax.get_figure().savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        ax.get_figure().savefig(
            FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200
        )
    return ax


def bar_plot(
    data: pd.DataFrame,
    x: str,
    y: str,
    hue: str | None = None,
    palette: dict | str | None = None,
    order: list | None = None,
    hue_order: list | None = None,
    estimator: str = "mean",
    errorbar: tuple | None = ("ci", 95),
    errwidth: float = 0.5,
    orient: str = "v",
    ax: plt.Axes | None = None,
    figsize: tuple = (10, 5),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    xtick_rotation: int = 45,
    legend_outside: bool = False,
    save_path: str | None = None,
) -> plt.Axes:
    """Bar plot wrapping sns.barplot with standard project styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    if orient == "h":
        sns.barplot(
            data=data,
            x=x,
            y=y,
            hue=hue,
            palette=palette,
            order=order,
            hue_order=hue_order,
            estimator=estimator,
            errorbar=errorbar,
            orient="h",
            ax=ax,
            errwidth=errwidth,
        )
    else:
        sns.barplot(
            data=data,
            x=x,
            y=y,
            hue=hue,
            palette=palette,
            order=order,
            hue_order=hue_order,
            estimator=estimator,
            errorbar=errorbar,
            ax=ax,
            errwidth=errwidth,
        )
    ax.set_xlabel(
        xlabel if xlabel is not None else (y if orient == "h" else x),
        fontsize=FONT_LABEL,
    )
    ax.set_ylabel(
        ylabel if ylabel is not None else (x if orient == "h" else y),
        fontsize=FONT_LABEL,
    )
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if orient != "h":
        ax.tick_params(axis="x", rotation=xtick_rotation)
    if legend_outside and ax.get_legend() is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
        ax.legend(
            handles,
            labels,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            fontsize=FONT_LEGEND,
            frameon=False,
        )
    sns.despine(ax=ax)
    if save_path:
        ax.get_figure().savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        ax.get_figure().savefig(
            FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200
        )
    return ax


def heatmap_plot(
    data: pd.DataFrame,
    cmap: str = "RdYlGn",
    center: float | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    annot: bool = False,
    fmt: str = ".2f",
    figsize: tuple = (12, 8),
    xlabel: str | None = None,
    ylabel: str | None = None,
    title: str | None = None,
    save_path: str | None = None,
) -> plt.Axes:
    """Thin wrapper around sns.heatmap with standard project styling."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        data,
        cmap=cmap,
        center=center,
        vmin=vmin,
        vmax=vmax,
        annot=annot,
        fmt=fmt,
        linewidths=0.3,
        ax=ax,
    )
    ax.set_xlabel(xlabel or "", fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel or "", fontsize=FONT_LABEL)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    if save_path:
        fig.savefig(FIGURES_DIR / f"{save_path}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{save_path}.png", bbox_inches="tight", dpi=200)
    return ax

#### Functions for the best bars

In [ ]:
# ── Helpers for "★ Best" group embedding ─────────────────────────────────────


def style_best_bars(ax: plt.Axes, order: list, orient: str = "v") -> None:
    """
    Apply BEST_COLOR, hatching, and star annotation to the BEST_LABEL group.
    Uses ax.containers (one per hue, seaborn >= 0.12).
    orient: "v" for vertical bars (default), "h" for horizontal.
    Uses h * 1.05 for annotation offset — works for both linear and log scale.
    """
    if BEST_LABEL not in order:
        return
    best_idx = list(order).index(BEST_LABEL)
    for container in ax.containers:
        bars = list(container)
        if best_idx < len(bars):
            b = bars[best_idx]
            # b.set_facecolor(BEST_COLOR)
            # b.set_hatch(BEST_HATCH)
            # b.set_edgecolor(BEST_EDGECOLOR)
            b.set_linewidth(1.5)
            if orient == "v":
                h = b.get_height()
                if h > 0:
                    ax.text(
                        b.get_x() + b.get_width() / 2,
                        h * 1.1,
                        "★",
                        ha="center",
                        va="bottom",
                        color=BEST_COLOR,
                        fontsize=FONT_BASE + 2,
                        fontweight="bold",
                        zorder=11,
                    )
            else:  # horizontal bar
                w = b.get_width()
                if w > 0:
                    ax.text(
                        w * 1.05,
                        b.get_y() + b.get_height() / 2,
                        "★",
                        ha="left",
                        va="center",
                        color=BEST_COLOR,
                        fontsize=FONT_BASE + 2,
                        fontweight="bold",
                        zorder=11,
                    )


def style_best_boxes(g: sns.FacetGrid, order: list) -> None:
    """
    Apply BEST_COLOR and hatching to the BEST_LABEL box in each FacetGrid axis.
    Works for catplot kind='box'.
    Identifies patches by x-position (best_idx = last in order).
    """
    if BEST_LABEL not in order:
        return
    best_idx = list(order).index(BEST_LABEL)
    n_groups = len(order)
    for ax_ in g.axes.flat:
        for patch in ax_.patches:
            if not hasattr(patch, "get_x"):
                continue
            x_center = patch.get_x() + patch.get_width() / 2
            # Box centers are at integer x-positions 0, 1, ..., n_groups-1
            if abs(x_center - best_idx) < 0.6:
                # patch.set_facecolor(BEST_COLOR)
                patch.set_hatch(BEST_HATCH)
                patch.set_edgecolor(BEST_EDGECOLOR)
                patch.set_linewidth(1.5)


BEST_MARKER = "*"
BEST_MARKERSIZE = 300
BEST_EDGECOLOR = "black"
BEST_ALPHA = 0.85
BEST_LABEL = "Best"


def scatter_overlay_best(
    ax: plt.Axes,
    df_best: pd.DataFrame,
    x: str,
    y: str,
    label: str = "Best approaches",
    s: float = BEST_MARKERSIZE,
) -> None:
    """Overlay star markers for best approaches on an existing scatter axes."""
    valid = df_best[[x, y]].dropna()
    if valid.empty:
        return
    ax.scatter(
        valid[x],
        valid[y],
        marker=BEST_MARKER,
        s=s,
        color=BEST_COLOR,
        facecolors="none",
        edgecolors=BEST_EDGECOLOR,
        linewidths=0.7,
        alpha=BEST_ALPHA,
        zorder=10,
        label=label,
    )
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="best", framealpha=0.7, fontsize=FONT_LEGEND)

---
## §2 — Full heatmap / clustermap of all metrics

### Build palettes

In [ ]:
import matplotlib.colors as mcolors

ALL_STRATS = [
    "S0_no_removal",
    "A_confirmed_bad",
    "B_extended_bad",
    "C_rnaseq_only",
    "D_malignant_only",
    "E1_iterative_r1",
    "E2_iterative_r2",
    "E3_iterative_r3",
    "F_microarray_only",
    "G_affymetrix_only",
    "H_affymetrix_extended",
    "I_rare_batches_removed",
    "J_ff_only",
    "K_ffpe_only",
]

_pastel_base = sns.color_palette("pastel", n_colors=len(ALL_STRATS))
# strat_pal = dict(zip(ALL_STRATS, _pastel_base))

strat_pal = {
    "S0_no_removal": (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad": (1.0, 0.7058, 0.5098),
    "B_extended_bad": (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only": (1.0, 0.6235, 0.6078),
    "D_malignant_only": (0.8156, 0.7333, 1.0),
    "E1_iterative_r1": (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2": (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3": (0.8117, 0.8117, 0.8117),
    "F_microarray_only": (1.0, 0.9960, 0.6392),
    "G_affymetrix_only": (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",  # previous version was '#7b4f9e',
    "J_ff_only": "#7ecbc4",  # soft teal — fresh frozen
    "K_ffpe_only": "#f2c27f",  # warm amber — FFPE
}

imp_pal = {
    "strict": "#2979ae",
    "knn": "#982d22",
    "softimpute": "#713689",
}

ALL_METHODS = sorted(df_ok.method.unique())
_tab20 = sns.color_palette("tab20", n_colors=20)
_tab20b = sns.color_palette("tab20b", n_colors=20)
_raw_method_colors = (_tab20 + _tab20b)[: max(len(ALL_METHODS), 25)]


def _boost_saturation(rgb, delta: float = 0.15):
    h, s, v = mcolors.rgb_to_hsv(rgb[:3])
    return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))


_vivid_method_colors = [_boost_saturation(c) for c in _raw_method_colors]
method_pal = dict(zip(ALL_METHODS, _vivid_method_colors[: len(ALL_METHODS)]))
method_pal.update({"Best": "red"})

post_rm_pal = {False: "#CCCCCC", True: "#333333"}

group_pal = dict(zip("ABCDEFGHIJK", sns.color_palette("Set3", n_colors=11)))

BATCH_ANNOT_COLS = ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
BIO_ANNOT_COLS = ["Major_group", "Diagnosis_cell_type_unified", "TUMOR_NORMAL"]
GLOBAL_ANNOT_COL = "global/all"
_batch_pal_colors = ["#8B4513", "#CD853F", "#DEB887", "#F5DEB3"]
_bio_pal_colors = ["#2F4F4F", "#5F9EA0", "#B0C4DE"]
annot_pal = (
    dict(zip(BATCH_ANNOT_COLS, _batch_pal_colors))
    | dict(zip(BIO_ANNOT_COLS, _bio_pal_colors))
    | {GLOBAL_ANNOT_COL: "#808080"}
)

# ── Metric large-type palette: dark & vivid, distinct from imp_pal ─────────
metric_large_type_pal = {
    "local_neighborhood": "#1a6b3c",
    "global_distance": "#8b4a0f",
    "distribution_similarity": "#0d5c63",
    "other": "#5c5c1a",
}


def _safe_map(series, pal, default="#AAAAAA"):
    return series.map(pal).fillna(default)


def make_legend_patches(pal_dict):
    return [mpatches.Patch(color=to_hex(v), label=str(k)) for k, v in pal_dict.items()]


# metric_colors: per-metric metadata — indexed by scoring_col name
metric_colors = pd.DataFrame(
    {
        "group": col_meta.loc[scoring_cols, "group"].map(group_pal).fillna("#AAAAAA"),
        "annot_col": col_meta.loc[scoring_cols, "annot_col"]
        .map(annot_pal)
        .fillna("#AAAAAA"),
        "metric_type": col_meta.loc[scoring_cols, "metric_type"]
        .map(metric_large_type_pal)
        .fillna("#AAAAAA"),
    },
    index=scoring_cols,
)

# ── Shambhala palettes ────────────────────────────────────────────────────
SHAMBHALA_P_KEYS = [
    "P0std",
    "ANTE",
    "GTExAffy",
    "NBlegacy",
    "NBGPL570",
    "NBKass",
    "NBRNAseq",
    "NBext",
    "Oncobox",
]
_shamb_p_colors = plt.cm.tab20b(np.linspace(0.0, 0.65, len(SHAMBHALA_P_KEYS)))
shambhala_p_pal = dict(zip(SHAMBHALA_P_KEYS, [to_hex(c) for c in _shamb_p_colors]))

SHAMBHALA_METHOD_COLOR = "#1a1a1a"
for _m in df_ok["method"].unique():
    if _m.startswith("shambhala_"):
        method_pal[_m] = SHAMBHALA_METHOD_COLOR

shambhala_method_pal = {
    _m: (
        shambhala_p_pal.get(_parse_shambhala_method(_m)[0], SHAMBHALA_METHOD_COLOR)
        if _m.startswith("shambhala_")
        else "#ffffff"
    )
    for _m in df_ok["method"].unique()
}

SHAMBHALA_BINARY_PAL = {True: "#3d1a78", False: "#e8e8e8"}

# ── Harshness palette ─────────────────────────────────────────────────────
HARSHNESS_PAL = {
    "low": "#4caf50",
    "medium": "#ff9800",
    "high": "#f44336",
}


attempt_colors = pd.DataFrame(
    {
        "strat": df_ok.strat.map(strat_pal),
        "imp": df_ok.imp.map(imp_pal),
        "method": df_ok.method.map(method_pal),
        "post_rm": df_ok.post_rm.map(post_rm_pal),
    },
    index=df_ok.index,
)

# attempt_colors["is_shambhala"]    = _safe_map(df_ok["is_shambhala"], SHAMBHALA_BINARY_PAL)
attempt_colors["shambhala_detail"] = (
    df_ok["method"].map(shambhala_method_pal).fillna("#ffffff")
)
attempt_colors["harshness"] = _safe_map(df_ok["harshness_level"], HARSHNESS_PAL)


method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)

col_colors_extended = attempt_colors[
    ["strat", "imp", "method", "post_rm", "harshness"]  # "shambhala_detail"
].copy()  # .reindex(df_normed.index)


print("Palettes built.")
print(
    f"attempt_colors shape: {attempt_colors.shape}, metric_colors shape: {metric_colors.shape}"
)
print(f"col_colors_extended columns: {list(col_colors_extended.columns)}")

In [ ]:
attempt_colors.dropna()

### Visualize palettes

In [ ]:
def plot_palette(palette, title=None, filename="palette.svg"):
    """
    Display a color palette as a legend-style swatch figure.

    Parameters
    ----------
    palette : dict
        Mapping of label → color (any matplotlib-compatible color format).
    title : str, optional
        Figure title shown above the swatches.
    """
    fig, ax = plt.subplots(figsize=(4, max(2.5, len(palette) * 0.1)))
    ax.axis("off")
    handles = [
        mpatches.Patch(color=color, label=str(name)) for name, color in palette.items()
    ]
    ax.legend(
        handles=handles,
        loc="center",
        ncol=1,
        fontsize=7.5,
        frameon=False,
        labelspacing=0.1,
        title=title,
    )
    plt.tight_layout()

    plt.savefig(filename, transparent=True)
    plt.show()


plot_palette(strat_pal, title="strat_pal", filename="./figures/strat_pal.svg")

In [ ]:
plot_palette(group_pal, title="group_pal", filename="./figures/group_pal.svg")

In [ ]:
plot_palette(
    metric_large_type_pal,
    title="metric_large_type_pal",
    filename="./figures/metric_large_type_pal.svg",
)

In [ ]:
plot_palette(annot_pal, title="annot_pal", filename="./figures/annot_pal.svg")

In [ ]:
plot_palette(imp_pal, title="imp_pal", filename="./figures/imp_pal.svg")

In [ ]:
plot_palette(method_pal, title="method_pal", filename="./figures/method_pal.svg")

In [ ]:
plot_palette(
    HARSHNESS_PAL, title="HARSHNESS_PAL", filename="./figures/harshness_pal.svg"
)

In [ ]:
# plot_palette(annot_pal, title="annot_pal")
plot_palette(post_rm_pal, title="post_rm_pal", filename="./figures/post_rm_pal.svg")

In [ ]:
plot_palette(metric_large_type_pal, title="metric_large_type_pal")

In [ ]:
# ── Cell 026b — Quality filter: exclude runs with fully-NA samples ────────────
# Attempts where pct_samples_allNA > 0 have at least one completely empty sample,
# indicating a structural failure of the normalization. Set FILTER_ALLNA_SAMPLES = False to disable.

FILTER_ALLNA_SAMPLES = True

if FILTER_ALLNA_SAMPLES and "pct_samples_allNA" in df_ok.columns:
    mask_ok_na = df_ok["pct_samples_allNA"] < 0.5
    df_filtered = df_ok[mask_ok_na].copy()
    n_excluded = len(df_ok) - len(df_filtered)
    print(
        f"Quality filter: {n_excluded} attempts excluded "
        f"(pct_samples_allNA > 0), {len(df_filtered)} remain."
    )
else:
    df_filtered = df_ok.copy()
    if "pct_samples_allNA" not in df_ok.columns:
        print(
            "Quality filter: pct_samples_allNA column not available (Group K not computed)."
        )
    else:
        print("Quality filter: disabled.")

### Cell 2.1 Heatmap of metrics completeness

In [ ]:
metrics_full = ['compute_time_s',
 'asw_batch_COHORT_LABEL',
 'asw_batch_PLATFORM_RNA',
 'asw_batch_RNASEQ_SOURCE',
 'asw_batch_RNA_BATCH',
 'asw_batch_norm_COHORT_LABEL',
 'asw_batch_norm_PLATFORM_RNA',
 'asw_batch_norm_RNASEQ_SOURCE',
 'asw_batch_norm_RNA_BATCH',
 'asw_bio_Diagnosis_cell_type_unified',
 'asw_bio_Major_group',
 'asw_bio_PLATFORM_RNA',
 'asw_bio_TUMOR_NORMAL',
 'asw_bio_norm_Diagnosis_cell_type_unified',
 'asw_bio_norm_Major_group',
 'asw_bio_norm_PLATFORM_RNA',
 'asw_bio_norm_TUMOR_NORMAL',
 'avg_inter_dist_COHORT_LABEL',
 'avg_inter_dist_Diagnosis_cell_type_unified',
 'avg_inter_dist_Major_group',
 'avg_inter_dist_PLATFORM_RNA',
 'avg_inter_dist_RNASEQ_SOURCE',
 'avg_inter_dist_RNA_BATCH',
 'avg_inter_dist_TUMOR_NORMAL',
 'avg_intra_dist_COHORT_LABEL',
 'avg_intra_dist_Diagnosis_cell_type_unified',
 'avg_intra_dist_Major_group',
 'avg_intra_dist_PLATFORM_RNA',
 'avg_intra_dist_RNASEQ_SOURCE',
 'avg_intra_dist_RNA_BATCH',
 'avg_intra_dist_TUMOR_NORMAL',
 'below_1_fraction_by_batch_max',
 'below_1_fraction_by_batch_min',
 'clisi_mean_Diagnosis_cell_type_unified',
 'clisi_mean_Major_group',
 'clisi_mean_PLATFORM_RNA',
 'clisi_mean_TUMOR_NORMAL',
 'cms_fraction_mixed_COHORT_LABEL',
 'cms_fraction_mixed_PLATFORM_RNA',
 'cms_fraction_mixed_RNASEQ_SOURCE',
 'cms_fraction_mixed_RNA_BATCH',
 'cms_mean_COHORT_LABEL',
 'cms_mean_PLATFORM_RNA',
 'cms_mean_RNASEQ_SOURCE',
 'cms_mean_RNA_BATCH',
 'dist_ratio_COHORT_LABEL',
 'dist_ratio_Diagnosis_cell_type_unified',
 'dist_ratio_Major_group',
 'dist_ratio_PLATFORM_RNA',
 'dist_ratio_RNASEQ_SOURCE',
 'dist_ratio_RNA_BATCH',
 'dist_ratio_TUMOR_NORMAL',
 'dsc_COHORT_LABEL',
 'dsc_PLATFORM_RNA',
 'dsc_RNASEQ_SOURCE',
 'dsc_RNA_BATCH',
 'dsc_pvalue_COHORT_LABEL',
 'dsc_pvalue_PLATFORM_RNA',
 'dsc_pvalue_RNASEQ_SOURCE',
 'dsc_pvalue_RNA_BATCH',
 'error_A',
 'error_B',
 'error_C',
 'error_D1',
 'error_F',
 'error_G',
 'error_H',
 'error_I',
 'error_J',
 'error_embeddings',
 'exp_max',
 'exp_median',
 'exp_min',
 'exp_p01',
 'exp_p05',
 'exp_p25',
 'exp_p75',
 'exp_p95',
 'exp_p99',
 'exp_std',
 'fraction_cohorts_bimodal',
 'fraction_cohorts_zero_inflated_bimodal',
 'fraction_genes_below_1',
 'graph_connectivity_Diagnosis_cell_type_unified',
 'graph_connectivity_Major_group',
 'graph_connectivity_PLATFORM_RNA',
 'graph_connectivity_TUMOR_NORMAL',
 'ilisi_mean_COHORT_LABEL',
 'ilisi_mean_PLATFORM_RNA',
 'ilisi_mean_RNASEQ_SOURCE',
 'ilisi_mean_RNA_BATCH',
 'ilisi_norm_COHORT_LABEL',
 'ilisi_norm_PLATFORM_RNA',
 'ilisi_norm_RNASEQ_SOURCE',
 'ilisi_norm_RNA_BATCH',
 'kbet_acceptance_rate_COHORT_LABEL',
 'kbet_acceptance_rate_PLATFORM_RNA',
 'kbet_acceptance_rate_RNASEQ_SOURCE',
 'kbet_acceptance_rate_RNA_BATCH',
 'ks_cohort_within_batch_frac_sig',
 'ks_cohort_within_batch_mean_D',
 'ks_frac_sig_COHORT_LABEL',
 'ks_frac_sig_PLATFORM_RNA',
 'ks_frac_sig_RNA_BATCH',
 'ks_mean_D_COHORT_LABEL',
 'ks_mean_D_PLATFORM_RNA',
 'ks_mean_D_RNA_BATCH',
 'n_batches',
 'n_cohorts',
 'n_cohorts_bimodal',
 'n_cohorts_zero_inflated_bimodal',
 'n_diagnosis_groups',
 'n_genes',
 'n_genes_allNA',
 'n_genes_noNA',
 'n_na_cells',
 'n_samples',
 'n_samples_allNA',
 'n_samples_noNA',
 'n_samples_per_batch_max',
 'n_samples_per_batch_median',
 'n_samples_per_batch_min',
 'n_samples_per_batch_sd',
 'pcr_COHORT_LABEL',
 'pcr_Diagnosis_cell_type_unified',
 'pcr_Major_group',
 'pcr_PLATFORM_RNA',
 'pcr_RNASEQ_SOURCE',
 'pcr_RNA_BATCH',
 'pcr_TUMOR_NORMAL',
 'pct_genes_noNA',
 'pct_na_cells',
 'pct_samples_noNA',
 'pct_var_cum_top10',
 'pct_var_pc1',
 'pct_var_pc10',
 'pct_var_pc2',
 'pct_var_pc3',
 'pct_var_pc4',
 'pct_var_pc5',
 'pct_var_pc6',
 'pct_var_pc7',
 'pct_var_pc8',
 'pct_var_pc9',
 'per_batch_median_cv',
 'per_gene_batch_mean_cv_COHORT_LABEL',
 'per_gene_batch_mean_cv_PLATFORM_RNA',
 'per_gene_batch_mean_cv_RNA_BATCH',
 'r2_COHORT_LABEL',
 'r2_Major_group',
 'r2_PLATFORM_RNA',
 'r2_RNASEQ_SOURCE',
 'r2_RNA_BATCH',
 'r2_TUMOR_NORMAL',
 'r2_pc10_COHORT_LABEL',
 'r2_pc10_PLATFORM_RNA',
 'r2_pc10_RNASEQ_SOURCE',
 'r2_pc10_RNA_BATCH',
 'r2_pc1_COHORT_LABEL',
 'r2_pc1_PLATFORM_RNA',
 'r2_pc1_RNASEQ_SOURCE',
 'r2_pc1_RNA_BATCH',
 'r2_pc2_COHORT_LABEL',
 'r2_pc2_PLATFORM_RNA',
 'r2_pc2_RNASEQ_SOURCE',
 'r2_pc2_RNA_BATCH',
 'r2_pc3_COHORT_LABEL',
 'r2_pc3_PLATFORM_RNA',
 'r2_pc3_RNASEQ_SOURCE',
 'r2_pc3_RNA_BATCH',
 'r2_pc4_COHORT_LABEL',
 'r2_pc4_PLATFORM_RNA',
 'r2_pc4_RNASEQ_SOURCE',
 'r2_pc4_RNA_BATCH',
 'r2_pc5_COHORT_LABEL',
 'r2_pc5_PLATFORM_RNA',
 'r2_pc5_RNASEQ_SOURCE',
 'r2_pc5_RNA_BATCH',
 'r2_pc6_COHORT_LABEL',
 'r2_pc6_PLATFORM_RNA',
 'r2_pc6_RNASEQ_SOURCE',
 'r2_pc6_RNA_BATCH',
 'r2_pc7_COHORT_LABEL',
 'r2_pc7_PLATFORM_RNA',
 'r2_pc7_RNASEQ_SOURCE',
 'r2_pc7_RNA_BATCH',
 'r2_pc8_COHORT_LABEL',
 'r2_pc8_PLATFORM_RNA',
 'r2_pc8_RNASEQ_SOURCE',
 'r2_pc8_RNA_BATCH',
 'r2_pc9_COHORT_LABEL',
 'r2_pc9_PLATFORM_RNA',
 'r2_pc9_RNASEQ_SOURCE',
 'r2_pc9_RNA_BATCH',
 'tsne_centroid_disp_COHORT_LABEL',
 'tsne_centroid_disp_PLATFORM_RNA',
 'tsne_centroid_disp_RNASEQ_SOURCE',
 'tsne_centroid_disp_RNA_BATCH',
 'tsne_entropy_mean_COHORT_LABEL',
 'tsne_entropy_mean_PLATFORM_RNA',
 'tsne_entropy_mean_RNASEQ_SOURCE',
 'tsne_entropy_mean_RNA_BATCH',
 'tsne_entropy_norm_COHORT_LABEL',
 'tsne_entropy_norm_PLATFORM_RNA',
 'tsne_entropy_norm_RNASEQ_SOURCE',
 'tsne_entropy_norm_RNA_BATCH',
 'umap_centroid_disp_COHORT_LABEL',
 'umap_centroid_disp_PLATFORM_RNA',
 'umap_centroid_disp_RNASEQ_SOURCE',
 'umap_centroid_disp_RNA_BATCH',
 'umap_entropy_mean_COHORT_LABEL',
 'umap_entropy_mean_PLATFORM_RNA',
 'umap_entropy_mean_RNASEQ_SOURCE',
 'umap_entropy_mean_RNA_BATCH',
 'umap_entropy_norm_COHORT_LABEL',
 'umap_entropy_norm_PLATFORM_RNA',
 'umap_entropy_norm_RNASEQ_SOURCE',
 'umap_entropy_norm_RNA_BATCH',
 'wm_COHORT_LABEL',
 'wm_Diagnosis_cell_type_unified',
 'wm_Major_group',
 'wm_PLATFORM_RNA',
 'wm_RNASEQ_SOURCE',
 'wm_RNA_BATCH',
 'wm_TUMOR_NORMAL',
 'wm_mean_batch',
 'wm_mean_bio',
 'wm_ratio_bio_batch',
 'wm_subsampled',
 'zero_fraction_by_batch_max',
 'zero_fraction_by_batch_min',
 'zero_fraction_global',
               'pct_samples_allNA',
 'pct_genes_allNA']

In [ ]:
metrics_colums.to_list()

In [ ]:
print(df_ok[(df_ok.strat == "A_confirmed_bad") & (df_ok.method == "01_raw")&(df_ok.imp == "knn")&(df_ok.post_rm == False)].dropna(axis=1).T.to_string())

      

In [ ]:
metrics_colums = df_ok.columns[5:]
metrics_stats = pd.concat(
    [df_ok[df_ok.columns[:5]], (~df_ok[metrics_colums].isna()).sum(axis=1)], axis=1
)
metrics_stats.columns = list(df_ok.columns[:5]) + ["metrics_number"]
metrics_stats_pivot = pd.pivot_table(
    metrics_stats,
    values="metrics_number",
    index=["strat", "imp"],
    columns=["method", "post_rm"],
    aggfunc="sum",
).fillna(0)
index = metrics_stats_pivot.reset_index()[["strat", "imp"]]

metrics_stats_pivot.index = index["strat"] + "__" + index["imp"]
index.index = metrics_stats_pivot.index
columns = metrics_stats_pivot.T.reset_index()[["method", "post_rm"]]
metrics_stats_pivot.columns = columns["method"] + "__" + columns["post_rm"].astype(str)
columns.index = metrics_stats_pivot.columns
metrics_stats_pivot

In [ ]:
metrics_stats_pivot.max().max()

In [ ]:
row_colors_completeness = pd.DataFrame(
    {
        "strat": index.strat.map(strat_pal).fillna("#AAAAAA"),
        "imp": index.imp.map(imp_pal).fillna("#AAAAAA"),
    },
    index=metrics_stats_pivot.index,
)

col_colors_completeness = pd.DataFrame(
    {
        "method": columns.method.map(method_pal).fillna("#AAAAAA"),
        "harshness": columns.method.map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
        # "shambhala": columns.method.map(shambhala_method_pal).fillna("#AAAAAA"),
        "post_rm": columns.post_rm.map(post_rm_pal).fillna("#AAAAAA"),
    },
    index=metrics_stats_pivot.columns,
)
col_colors_completeness

In [ ]:
g2 = sns.clustermap(
    metrics_stats_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors_completeness,
    col_colors=col_colors_completeness,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(9, 8),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.02, 0.02),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)
g2.fig.savefig(
    FIGURES_DIR / "heatmap_metrics_completeness.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / "heatmap_metrics_completeness.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
col_colors_extended.method.value_counts()

### Cell 2.2 — Clustered clustermap (rows and columns clustered)

In [ ]:
# ── §2.2 Clustered clustermap ─────────────────────────────────────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_for_clustermap = (
    df_normed[scoring_cols].dropna().T
)  # shape: (n_metrics, n_attempts)

g = sns.clustermap(
    data_for_clustermap,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,  # metric annotations on the LEFT
    col_colors=col_colors_extended,  # attempt annotations on TOP
    cmap="RdYlGn",
    center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(100, 50),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.007, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=2)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=30)
g.ax_col_colors.tick_params(labelsize=40)
g.ax_row_colors.tick_params(labelsize=40)

legend_specs = [
    ("Strat", {k: v for k, v in strat_pal.items() if k in df_ok.strat.unique()}),
    ("Imp", imp_pal),
    (
        "Method",
        {
            k: v
            for k, v in method_pal.items()
            if k in df_ok.method.unique() and not k.startswith("shambhala_")
        },
    ),
    ("Post-rm", post_rm_pal),
    ("Shambhala", SHAMBHALA_BINARY_PAL),
    ("Shmb P-key", shambhala_p_pal),
    ("Harshness", HARSHNESS_PAL),
    ("Metric group", group_pal),
    ("Annot col", annot_pal),
    ("Metric type", metric_large_type_pal),
]
# add_legend_outside(g.fig, g.ax_heatmap, dict(legend_specs), x_offset=1.07)

g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_dropna_minus_01.pdf",
    bbox_inches="tight",
)
g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_dropna_minus_01.png",
    bbox_inches="tight",
    dpi=200,
)
plt.show()

In [ ]:
# ── §2.2 Clustered clustermap ─────────────────────────────────────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_for_clustermap = (
    df_normed[scoring_cols].T
)  # shape: (n_metrics, n_attempts)

g = sns.clustermap(
    data_for_clustermap,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,    # metric annotations on the LEFT
    col_colors=col_colors_extended,   # attempt annotations on TOP
    cmap="RdYlGn",
    center=0.5,
    #vmin=0,
    #vmax=1,
    linewidths=0,
    figsize=(100, 50),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.007, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=2)
#g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=30)
#g.ax_col_colors.tick_params(labelsize=40) 
#g.ax_row_colors.tick_params(labelsize=40)

]
#add_legend_outside(g.fig, g.ax_heatmap, dict(legend_specs), x_offset=1.07)

g.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_clustered.pdf", bbox_inches="tight")
g.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_clustered.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
df_ok.index == df_normed.index

In [ ]:
df_normed[df_ok.method == "33_amdbnorm"]

In [ ]:
# 1. Isolate the data
data_for_clustermap = df_normed[scoring_cols].T

# 2. Prepare data for clustering and masking
# Fill NAs with 0 for the clustering algorithm
data_filled = data_for_clustermap.fillna(0)
# Create a boolean mask of where the original NAs were (True = NA)
na_mask = data_for_clustermap.isna()

# 3. Generate the clustermap
g = sns.clustermap(
    data=data_filled,  # Use the filled data so scipy clustering succeeds
    mask=na_mask,  # Hide the cells that were originally NA
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,
    col_colors=col_colors_extended,
    cmap="RdYlGn",
    center=0.5,
    linewidths=0,
    figsize=(15, 4),
    dendrogram_ratio=(0.09, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.02, 0.015),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)

# 4. Set the background color to black
# Since the masked cells are transparent, they will show this black background
g.ax_heatmap.set_facecolor("black")

# Format the axes
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=1)
# g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=5)

# Save and show
g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_small.pdf", bbox_inches="tight"
)
g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_small.svg",
    bbox_inches="tight",
    transparent=True,
)
plt.show()

In [ ]:
print(pd.Series(df_normed.index).reindex(g.dendrogram_col.reordered_ind).to_string())

In [ ]:
print(pd.Series(df_normed.columns).reindex(g.dendrogram_row.reordered_ind).to_string())

In [ ]:
g.dendrogram_col.reordered_ind

#### Compare good and bad cross-platform clusters by local metrics


In [ ]:
print(df_ok[df_ok.method == "21_harmonizr"]["pct_genes_noNA"].to_string())

In [ ]:
np.log2(0.730/0.554)

In [ ]:
sns.distplot(df_ok[df_ok.method == "21_harmonizr"]["pct_genes_noNA"], bins=10)

In [ ]:
reordered_indeces = (
    pd.Series(df_normed.index)
    .reindex(g.dendrogram_col.reordered_ind)
    .reset_index(drop=True)
)
good_cross_plat_indices = reordered_indeces[0:560]
bad_cross_plat_indices = reordered_indeces[560:1582]
local_metrics_to_test = col_meta[
    col_meta.metric_type == "local_neighborhood"
].index.intersection(df_normed.columns)
mann_whitney_local_metrics = pd.DataFrame(
    index=local_metrics_to_test,
    columns=[
        "p_value",
        "fold_change",
        "log2_fc",
        "u_stat",
        "median_good",
        "median_bad",
    ],
)
for metric in mann_whitney_local_metrics.index:
    good_values = df_normed.loc[good_cross_plat_indices, [metric]]
    bad_values = df_normed.loc[bad_cross_plat_indices, [metric]]
    u_stat, p_val = mannwhitneyu(good_values, bad_values, alternative="two-sided")

    med_good = np.median(good_values)
    med_bad = np.median(bad_values)

    if med_good == 0:
        fold_change = np.nan
        log2_fc = np.nan
    else:
        fold_change = med_bad / med_good
        log2_fc = np.log2(fold_change) if fold_change > 0 else np.nan
    mann_whitney_local_metrics.loc[[metric], ["p_value"]] = p_val[0]
    mann_whitney_local_metrics.loc[[metric], ["u_stat"]] = u_stat
    mann_whitney_local_metrics.loc[[metric], ["fold_change"]] = fold_change
    mann_whitney_local_metrics.loc[[metric], ["log2_fc"]] = log2_fc
    mann_whitney_local_metrics.loc[[metric], ["median_good"]] = med_good
    mann_whitney_local_metrics.loc[[metric], ["median_bad"]] = med_bad

_, fdr_pvals, _, _ = multipletests(
    mann_whitney_local_metrics["p_value"], method="fdr_bh"
)
mann_whitney_local_metrics["FDR_adjusted_p"] = fdr_pvals
mann_whitney_local_metrics

In [ ]:
reordered_indeces = (
    pd.Series(df_normed.index)
    .reindex(g.dendrogram_col.reordered_ind)
    .reset_index(drop=True)
)
good_cross_plat_indices = reordered_indeces[0:560]
bad_cross_plat_indices = reordered_indeces[560:1582]
metrics_to_test = [
    "pcr_PLATFORM_RNA",
    "tsne_centroid_disp_PLATFORM_RNA",
    "umap_centroid_disp_PLATFORM_RNA",
    "cms_fraction_mixed_PLATFORM_RNA",
    "dist_ratio_PLATFORM_RNA",
    "ks_mean_D_RNA_BATCH",
    "tsne_centroid_disp_RNA_BATCH",
    "umap_centroid_disp_RNA_BATCH",
    "dist_ratio_RNA_BATCH",
    "pcr_RNA_BATCH",
]
mann_whitney_metrics = pd.DataFrame(
    index=metrics_to_test,
    columns=[
        "p_value",
        "fold_change",
        "log2_fc",
        "u_stat",
        "median_good",
        "median_bad",
    ],
)
for metric in mann_whitney_metrics.index:
    good_values = df_normed.loc[good_cross_plat_indices, [metric]]
    bad_values = df_normed.loc[bad_cross_plat_indices, [metric]]
    u_stat, p_val = mannwhitneyu(good_values, bad_values, alternative="two-sided")

    med_good = np.median(good_values)
    med_bad = np.median(bad_values)

    if med_good == 0:
        fold_change = np.nan
        log2_fc = np.nan
    else:
        fold_change = med_bad / med_good
        log2_fc = np.log2(fold_change) if fold_change > 0 else np.nan
    mann_whitney_metrics.loc[[metric], ["p_value"]] = p_val[0]
    mann_whitney_metrics.loc[[metric], ["u_stat"]] = u_stat
    mann_whitney_metrics.loc[[metric], ["fold_change"]] = fold_change
    mann_whitney_metrics.loc[[metric], ["log2_fc"]] = log2_fc
    mann_whitney_metrics.loc[[metric], ["median_good"]] = med_good
    mann_whitney_metrics.loc[[metric], ["median_bad"]] = med_bad

_, fdr_pvals, _, _ = multipletests(mann_whitney_metrics["p_value"], method="fdr_bh")
mann_whitney_metrics["FDR_adjusted_p"] = fdr_pvals
mann_whitney_metrics

#### The rest clustermaps of this subsection

In [ ]:
data_for_clustermap = df_normed[scoring_cols].T
data_filled = data_for_clustermap.fillna(0)

# 2. Generate the clustermap
g = sns.clustermap(
    data=data_filled,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,
    col_colors=col_colors_extended,
    cmap="RdYlGn",
    center=0.5,
    linewidths=0,
    figsize=(7.5, 9),
    dendrogram_ratio=(0.09, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.02, 0.015),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)

# 3. ERASE THE HEATMAP CELLS
# This finds the QuadMesh (the grid of colored cells) and removes it entirely from the plot,
# which drastically reduces the final SVG file size.
for collection in g.ax_heatmap.collections:
    collection.remove()

# 4. REPLACE WITH A SOLID BLACK RECTANGLE
# Since the grid is gone, setting the axis background to black fills the exact heatmap area.
g.ax_heatmap.set_facecolor("black")
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=6.5)

# 5. Format the axes
# Removing the collections does NOT remove the ticks or labels, so these remain perfectly intact.
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=2)

# Save and show
# (Recommended: change the extension to .svg since you need it for Figma)
g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_blank.svg", bbox_inches="tight"
)
g.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_clustered_blank.png",
    bbox_inches="tight",
    dpi=200,
)
plt.show()

In [ ]:
# DIAGNOSTIC FOR COL COLORS

data_for_clustermap = (
    df_normed[scoring_cols[:10]].dropna().T
)  # shape: (n_metrics, n_attempts)

g = sns.clustermap(
    data_for_clustermap,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,  # metric annotations on the LEFT
    col_colors=col_colors_extended,  # attempt annotations on TOP
    cmap="RdYlGn",
    center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(60, 30),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.007, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g.ax_heatmap.set_xlabel("")
# g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=100)
# g.ax_col_colors.tick_params(labelsize=150)
# g.ax_row_colors.tick_params(labelsize=150)
plt.show()

In [ ]:
# ── §2.2b Companion: raw (unnormalized) clustermap ───────────────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_for_clustermap_raw = (
    df_ok[scoring_cols].dropna().T
)  # shape: (n_metrics, n_attempts_with_no_NA)

g_raw = sns.clustermap(
    data_for_clustermap_raw,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors.reindex(data_for_clustermap_raw.index),
    col_colors=col_colors_extended.reindex(data_for_clustermap_raw.columns),
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(80, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g_raw.ax_heatmap.set_xlabel("")

g_raw.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_raw.pdf", bbox_inches="tight")
g_raw.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_raw.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_ok[df_ok.columns[6:]].fillna(0)

In [ ]:
# ALL METRICS including the non-scoring ones
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_all = df_ok[df_ok.columns[6:]].apply(pd.to_numeric, errors="coerce").fillna(0).T

g_all = sns.clustermap(
    data_all,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors.reindex(data_all.index).fillna("#AAAAAA"),
    col_colors=col_colors_extended.reindex(data_all.columns).fillna("#AAAAAA"),
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(80, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g_all.ax_heatmap.set_xlabel("")

g_all.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_all.pdf", bbox_inches="tight")
g_all.fig.savefig(
    FIGURES_DIR / "clustermap_all_metrics_all.png", bbox_inches="tight", dpi=200
)
plt.show()

### Cell 2.3 — Grouped heatmap (columns ordered by metric group then annot_col; no column clustering)

In [ ]:
# ── §2.3 Grouped heatmap: metrics in fixed group order, attempts clustered ────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

metric_order = (
    col_meta.loc[scoring_cols].sort_values(["annot_col", "group"]).index.tolist()
)

g2 = sns.clustermap(
    data_for_clustermap.reindex(metric_order),
    method="ward",
    metric="euclidean",
    row_cluster=False,
    col_cluster=True,
    row_colors=metric_colors.loc[metric_order],
    col_colors=col_colors_extended,
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(60, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
)
g2.fig.savefig(FIGURES_DIR / "clustermap_grouped_annot_col.svg", bbox_inches="tight")
g2.fig.savefig(
    FIGURES_DIR / "clustermap_grouped_metric_order.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
# ── §2.3b Ordered clustermaps: attempts sorted by metadata ───────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

for sort_keys, fname_suffix in [
    (["strat", "imp"], "by_strat"),
    (["method", "imp"], "by_method"),
    (["imp", "strat"], "by_imp"),
]:
    col_order_sorted = (
        df_ok.reindex(data_for_clustermap.columns).sort_values(sort_keys).index.tolist()
    )
    g_sorted = sns.clustermap(
        data_for_clustermap[col_order_sorted],
        method="ward",
        metric="euclidean",
        row_cluster=True,
        col_cluster=False,
        row_colors=metric_colors,
        col_colors=col_colors_extended.reindex(col_order_sorted),
        cmap="RdYlGn",
        center=0.5,
        vmin=0,
        vmax=1,
        linewidths=0,
        figsize=(60, 20),
        dendrogram_ratio=(0.03, 0.12),
        xticklabels=False,
        yticklabels=True,
        colors_ratio=(0.01, 0.02),
    )
    g_sorted.fig.savefig(
        FIGURES_DIR / f"clustermap_sorted_{fname_suffix}.svg", bbox_inches="tight"
    )
    g_sorted.fig.savefig(
        FIGURES_DIR / f"clustermap_sorted_{fname_suffix}.png",
        bbox_inches="tight",
        dpi=200,
    )
    plt.show()

### Cell 2.4 — Ordered heatmap (no clustering — fixed row and column order)

In [ ]:
# ── §2.4 Fully ordered heatmap (no clustering) ────────────────────────────────
# Rows (metrics) ordered: metric group → annotation column.
# Columns (attempts) ordered: strat harshness → imp → method → post_rm.
# This is the most readable structural overview — no dendrograms distort the layout.

METRIC_ORDER = (
    col_meta.loc[scoring_cols].sort_values(["group", "annot_col"]).index.tolist()
)

ATTEMPT_ORDER = (
    df_ok.assign(_si=pd.Categorical(df_ok.strat, categories=ALL_STRATS, ordered=True))
    .sort_values(["_si", "imp", "method", "post_rm"])
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(60, 18))
sns.heatmap(
    df_normed[scoring_cols].dropna().T.reindex(METRIC_ORDER)[ATTEMPT_ORDER],
    ax=ax,
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    xticklabels=False,
    yticklabels=True,
    cbar_kws={"shrink": 0.3, "label": "Normalized score (1=best)"},
)
ax.set_title(
    "Ordered overview — cols: strat\u2192imp\u2192method\u2192post_rm; rows: metric group\u2192annot_col"
)
fig.savefig(FIGURES_DIR / "heatmap_ordered_no_clustering.svg", bbox_inches="tight")
fig.savefig(
    FIGURES_DIR / "heatmap_ordered_no_clustering.png", bbox_inches="tight", dpi=200
)
plt.show()

### §2.5 — Method similarity: clustermaps, embeddings, network


#### Clustermaps

In [ ]:
# ── §2.5.1a Spearman correlation between scoring metrics ─────────────────────
# Orientation: rows=metrics, cols=metrics — same metric_colors on BOTH axes
metric_corr = df_ok[scoring_cols].fillna(-0.1).corr(method="spearman")

g_mcorr = sns.clustermap(
    metric_corr,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,
    col_colors=metric_colors,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0,
    figsize=(10, 10),
    xticklabels=True,
    yticklabels=True,
    annot_kws={"size": FONT_TINY},
)

g_mcorr.ax_heatmap.set_xticklabels(
    g_mcorr.ax_heatmap.get_xmajorticklabels(), fontsize=4
)
g_mcorr.ax_heatmap.set_yticklabels(
    g_mcorr.ax_heatmap.get_ymajorticklabels(), fontsize=4
)

g_mcorr.fig.savefig(
    FIGURES_DIR / "metric_correlation_clustermap.svg", bbox_inches="tight"
)
g_mcorr.fig.savefig(
    FIGURES_DIR / "metric_correlation_clustermap.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
indices_of_interest

In [ ]:
reordered_indeces = (
    pd.Series(metric_corr.index)
    .reindex(g_mcorr.dendrogram_col.reordered_ind)
    .reset_index(drop=True)
)

indices_of_interest = reordered_indeces[:33]
indices_of_interest_2 = reordered_indeces[33:]
# print(reordered_indeces.to_string())
# attempt_corr_interest = attempt_corr.loc[indices_of_interest, attempt_corr.columns[~attempt_corr.columns.isin(indices_of_interest)]]
metric_corr_interest = metric_corr.loc[indices_of_interest, indices_of_interest_2]
metric_corr.melt()["value"].median()

In [ ]:
median_abs_deviation(metric_corr.melt()["value"])

In [ ]:
# ── §2.5.1b Spearman correlation between harmonization attempts ──────────────
# Orientation: rows=attempts, cols=attempts — attempt_colors on BOTH axes
attempt_ids = df_ok.index.tolist()
att_score_mat = df_ok[scoring_cols].fillna(0)
att_score_mat = att_score_mat[att_score_mat.sum(axis=1) != 0]
attempt_corr = att_score_mat.T.corr(method="spearman")

_att_col_idx = attempt_colors.reindex(attempt_corr.index)

g_acorr = sns.clustermap(
    attempt_corr,
    method="ward",
    metric="euclidean",
    row_colors=_att_col_idx,
    col_colors=_att_col_idx,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0,
    figsize=(10, 10),
    xticklabels=False,
    yticklabels=False,
)
g_acorr.fig.suptitle(
    "Attempt–attempt Spearman ρ across all metrics", fontsize=FONT_TITLE, y=1.01
)
g_acorr.fig.savefig(
    FIGURES_DIR / "attempt_correlation_clustermap.pdf", bbox_inches="tight"
)
g_acorr.fig.savefig(
    FIGURES_DIR / "attempt_correlation_clustermap.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
reordered_indeces = (
    pd.Series(attempt_corr.index)
    .reindex(g_acorr.dendrogram_col.reordered_ind)
    .reset_index(drop=True)
)

indices_of_interest = reordered_indeces[781:1109]
indices_of_interest_2 = reordered_indeces[:781]
# print(reordered_indeces.to_string())
# attempt_corr_interest = attempt_corr.loc[indices_of_interest, attempt_corr.columns[~attempt_corr.columns.isin(indices_of_interest)]]
attempt_corr_interest = attempt_corr.loc[indices_of_interest, indices_of_interest_2]
attempt_corr_interest.melt()["value"].median()

In [ ]:
median_abs_deviation(attempt_corr_interest.melt()["value"])

In [ ]:
attempt_corr_out_interest = attempt_corr.loc[
    attempt_corr.columns[~attempt_corr.columns.isin(indices_of_interest)],
    attempt_corr.columns[~attempt_corr.columns.isin(indices_of_interest_2)],
]
attempt_corr_out_interest.melt()["value"].median()

In [ ]:
median_abs_deviation(attempt_corr_out_interest.melt()["value"])

In [ ]:
mannwhitneyu(
    attempt_corr_interest.melt()["value"],
    attempt_corr_out_interest.melt()["value"],
    alternative="two-sided",
)

In [ ]:
reordered_indeces = (
    pd.Series(attempt_corr.index)
    .reindex(g_acorr.dendrogram_col.reordered_ind)
    .reset_index(drop=True)
)
indices_of_interest = reordered_indeces[781:1109]

attempt_corr
correlations =  = col_meta[
    col_meta.metric_type == "local_neighborhood"
].index.intersection(df_normed.columns)
mann_whitney_local_metrics = pd.DataFrame(
    index=local_metrics_to_test,
    columns=[
        "p_value",
        "fold_change",
        "log2_fc",
        "u_stat",
        "median_good",
        "median_bad",
    ],
)
for metric in mann_whitney_local_metrics.index:
    good_values = df_normed.loc[good_cross_plat_indices, [metric]]
    bad_values = df_normed.loc[bad_cross_plat_indices, [metric]]
    u_stat, p_val = mannwhitneyu(good_values, bad_values, alternative="two-sided")

    med_good = np.median(good_values)
    med_bad = np.median(bad_values)

    if med_good == 0:
        fold_change = np.nan
        log2_fc = np.nan
    else:
        fold_change = med_bad / med_good
        log2_fc = np.log2(fold_change) if fold_change > 0 else np.nan
    mann_whitney_local_metrics.loc[[metric], ["p_value"]] = p_val[0]
    mann_whitney_local_metrics.loc[[metric], ["u_stat"]] = u_stat
    mann_whitney_local_metrics.loc[[metric], ["fold_change"]] = fold_change
    mann_whitney_local_metrics.loc[[metric], ["log2_fc"]] = log2_fc
    mann_whitney_local_metrics.loc[[metric], ["median_good"]] = med_good
    mann_whitney_local_metrics.loc[[metric], ["median_bad"]] = med_bad

_, fdr_pvals, _, _ = multipletests(
    mann_whitney_local_metrics["p_value"], method="fdr_bh"
)
mann_whitney_local_metrics["FDR_adjusted_p"] = fdr_pvals
mann_whitney_local_metrics

In [ ]:
attempt_ids = df_ok.index.tolist()
att_score_mat = df_ok[scoring_cols].fillna(0)
att_score_mat = att_score_mat[att_score_mat.sum(axis=1) != 0]
attempt_corr = att_score_mat.T.corr(method="spearman")

_att_col_idx = attempt_colors.reindex(attempt_corr.index)

g_acorr = sns.clustermap(
    attempt_corr,
    method="ward",
    metric="euclidean",
    row_colors=_att_col_idx,
    col_colors=_att_col_idx,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0,
    figsize=(10, 10),
    xticklabels=False,
    yticklabels=False,
)
g_acorr.fig.suptitle(
    "Attempt–attempt Spearman ρ across all metrics", fontsize=FONT_TITLE, y=1.01
)
for collection in g_acorr.ax_heatmap.collections:
    collection.remove()

# 4. REPLACE WITH A SOLID BLACK RECTANGLE
# Since the grid is gone, setting the axis background to black fills the exact heatmap area.
g_acorr.ax_heatmap.set_facecolor("black")

g_acorr.fig.savefig(
    FIGURES_DIR / "attempt_correlation_clustermap.pdf", bbox_inches="tight"
)
g_acorr.fig.savefig(
    FIGURES_DIR / "attempt_correlation_clustermap.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_ok

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("pcr")]

#### PCA/UMAP/tSNE

In [ ]:
# ── §2.5.2 Embeddings of attempts and metrics in each other's space ──────────

method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)


X_att = _SS().fit_transform(df_ok[scoring_cols].fillna(0).values)
X_met = _SS().fit_transform(df_ok[scoring_cols].fillna(0).T.values)

pca_att = _PCA2(n_components=2, random_state=42).fit_transform(X_att)
umap_att = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(
    X_att
)
tsne_att = _TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_att)

_perp_met = min(30, len(scoring_cols) - 1)
pca_met = _PCA2(n_components=2, random_state=42).fit_transform(X_met)
umap_met = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(
    X_met
)
tsne_met = _TSNE(n_components=2, perplexity=_perp_met, random_state=42).fit_transform(
    X_met
)

ATT_COORDS = [pca_att, umap_att, tsne_att]
ATT_LABELS = [("PC1", "PC2"), ("UMAP1", "UMAP2"), ("tSNE1", "tSNE2")]
MET_COORDS = [pca_met, umap_met, tsne_met]
MET_LABELS = [("PC1", "PC2"), ("UMAP1", "UMAP2"), ("tSNE1", "tSNE2")]

fig, axes = plt.subplots(2, 3, figsize=(24, 14))

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[0, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "method": df_ok["method"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="method",
        palette=method_pal,
        hue_order=method_order,
        s=30,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    ax.set_xlabel(xl, fontsize=FONT_LABEL)
    ax.set_ylabel(yl, fontsize=FONT_LABEL)
    ax.set_title(f"Attempts / method  ({xl[:3]})", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    sns.despine(ax=ax)

metric_types = [
    next(
        (
            t
            for t, pfxs in METRIC_TYPE_MAP.items()
            if t != "other" and any(m.startswith(p) for p in pfxs)
        ),
        "other",
    )
    for m in scoring_cols
]
for col, (coords, (xl, yl)) in enumerate(zip(MET_COORDS, MET_LABELS)):
    ax = axes[1, col]
    met_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "metric_type": metric_types}
    )
    sns.scatterplot(
        data=met_df,
        x=xl,
        y=yl,
        hue="metric_type",
        palette=metric_large_type_pal,
        s=60,
        alpha=0.8,
        linewidth=0,
        zorder=3,
        ax=ax,
    )
    handles, labels = ax.get_legend_handles_labels()
    ax.get_legend().remove()
    ax.legend(
        handles,
        labels,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=FONT_LEGEND,
        frameon=False,
    )
    ax.set_xlabel(xl, fontsize=FONT_LABEL)
    ax.set_ylabel(yl, fontsize=FONT_LABEL)
    ax.set_title(f"Metrics / type  ({xl[:3]})", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    sns.despine(ax=ax)

fig.suptitle("Method and metric similarity embeddings", fontsize=FONT_TITLE + 2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "method_metric_embeddings_2x3.pdf", bbox_inches="tight")
fig.savefig(
    FIGURES_DIR / "method_metric_embeddings_2x3.png", bbox_inches="tight", dpi=200
)
plt.show()

#### Harmonization attempts by strategy, imputation, post-removal, harshness

In [ ]:
HARSHNESS_PAL

In [ ]:
method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)


X_att = _SS().fit_transform(df_ok[scoring_cols].fillna(0).values)
X_met = _SS().fit_transform(df_ok[scoring_cols].fillna(0).T.values)

pca_att = _PCA2(n_components=2, random_state=42).fit_transform(X_att)
umap_att = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(
    X_att
)
tsne_att = _TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_att)

ATT_COORDS = [pca_att, umap_att, tsne_att]
ATT_LABELS = [("PC1", "PC2"), ("UMAP1", "UMAP2"), ("tSNE1", "tSNE2")]

fig, axes = plt.subplots(5, 3, figsize=(7, 13))

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[0, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "strat": df_ok["strat"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="strat",
        palette=strat_pal,
        hue_order=df_ok["strat"].values,
        s=10,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[1, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "method": df_ok["method"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="method",
        palette=method_pal,
        hue_order=df_ok["method"].values,
        s=10,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[2, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "method": df_ok["method"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="method",
        palette=method_harshness_pal,
        hue_order=df_ok["method"].values,
        s=10,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[3, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "imp": df_ok["imp"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="imp",
        palette=imp_pal,
        hue_order=df_ok["imp"].values,
        s=10,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[4, col]
    embed_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "post_rm": df_ok["post_rm"].values}
    )
    sns.scatterplot(
        data=embed_df,
        x=xl,
        y=yl,
        hue="post_rm",
        palette=post_rm_pal,
        hue_order=df_ok["post_rm"].values,
        s=10,
        alpha=0.7,
        linewidth=0,
        rasterized=True,
        legend=False,
        ax=ax,
    )
    sns.despine(ax=ax)

fig.savefig(
    FIGURES_DIR / "attempts_PCA_UMAP_TSNE.svg", bbox_inches="tight", transparent=True
)
fig.savefig(FIGURES_DIR / "attempts_PCA_UMAP_TSNE.png", bbox_inches="tight", dpi=400)
plt.show()

In [ ]:
group_pal

#### Metrics similarity

In [ ]:
X_met = _SS().fit_transform(df_normed[scoring_cols].fillna(0).T.values)

_perp_met = min(30, len(scoring_cols) - 1)
pca_met = _PCA2(n_components=2, random_state=42).fit_transform(X_met)
umap_met = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(
    X_met
)
tsne_met = _TSNE(n_components=2, perplexity=_perp_met, random_state=42).fit_transform(
    X_met
)

ATT_COORDS = [pca_att, umap_att, tsne_att]
ATT_LABELS = [("PC1", "PC2"), ("UMAP1", "UMAP2"), ("tSNE1", "tSNE2")]
MET_COORDS = [pca_met, umap_met, tsne_met]
MET_LABELS = [("PC1", "PC2"), ("UMAP1", "UMAP2"), ("tSNE1", "tSNE2")]

fig, axes = plt.subplots(3, 3, figsize=(7, 6))


metric_types = [
    next(
        (
            t
            for t, pfxs in METRIC_TYPE_MAP.items()
            if t != "other" and any(m.startswith(p) for p in pfxs)
        ),
        "other",
    )
    for m in scoring_cols
]
for col, (coords, (xl, yl)) in enumerate(zip(MET_COORDS, MET_LABELS)):
    ax = axes[0, col]
    met_df = pd.DataFrame(
        {xl: coords[:, 0], yl: coords[:, 1], "metric_type": metric_types}
    )

    sns.scatterplot(
        data=met_df,
        x=xl,
        y=yl,
        hue="metric_type",
        palette=metric_large_type_pal,
        s=10,
        alpha=0.8,
        linewidth=0,
        zorder=3,
        ax=ax,
        legend=False,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(MET_COORDS, MET_LABELS)):
    ax = axes[1, col]
    met_df = pd.DataFrame({xl: coords[:, 0], yl: coords[:, 1]}, index=scoring_cols)
    met_df["annot_col"] = col_meta.loc[scoring_cols, "annot_col"]
    sns.scatterplot(
        data=met_df,
        x=xl,
        y=yl,
        hue="annot_col",
        palette=annot_pal,
        s=10,
        alpha=0.8,
        linewidth=0,
        zorder=3,
        ax=ax,
        legend=False,
    )
    sns.despine(ax=ax)

for col, (coords, (xl, yl)) in enumerate(zip(MET_COORDS, MET_LABELS)):
    ax = axes[2, col]
    met_df = pd.DataFrame({xl: coords[:, 0], yl: coords[:, 1]}, index=scoring_cols)
    met_df["group"] = col_meta.loc[scoring_cols, "group"].str.replace("?", "K")

    sns.scatterplot(
        data=met_df,
        x=xl,
        y=yl,
        hue="group",
        palette=group_pal,
        s=10,
        alpha=0.8,
        linewidth=0,
        zorder=3,
        ax=ax,
        legend=False,
    )
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "metric_embeddings_PCA_UMAP_TSNE.svg",
    bbox_inches="tight",
    transparent=True,
)
fig.savefig(
    FIGURES_DIR / "metric_embeddings_PCA_UMAP_TSNE.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
        "group":       col_meta.loc[scoring_cols, "group"].map(group_pal).fillna("#AAAAAA"),
        "annot_col":   col_meta.loc[scoring_cols, "annot_col"].map(annot_pal).fillna("#AAAAAA"),
        "metric_type": col_meta.loc[scoring_cols, "metric_type"].map(metric_large_type_pal).fillna("#AAAAAA"),

#### Method similarity network

In [ ]:
# ── §2.5.3 Method correlation network ────────────────────────────────────────
try:
    import networkx as nx

    method_profiles = df_ok.groupby("method")[scoring_cols].mean()
    method_corr_net = method_profiles.T.corr(method="spearman")

    G = nx.Graph()
    threshold = 0.8
    for i, m1 in enumerate(method_corr_net.index):
        G.add_node(m1)
        for j, m2 in enumerate(method_corr_net.columns):
            if i < j and method_corr_net.loc[m1, m2] > threshold:
                G.add_edge(m1, m2, weight=float(method_corr_net.loc[m1, m2]))

    pos = nx.spring_layout(G, seed=42, weight="weight")
    fig, ax = plt.subplots(figsize=(12, 10))
    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_color=[method_pal.get(n, "#AAAAAA") for n in G.nodes],
        node_size=600,
        alpha=0.9,
    )
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.4, width=1.5)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=FONT_SMALL)
    ax.set_title(
        f"Method similarity network (Spearman ρ > {threshold})", fontsize=FONT_TITLE
    )
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "method_similarity_network.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "method_similarity_network.png", bbox_inches="tight", dpi=200
    )
    plt.show()
except ImportError:
    print("networkx not installed — run: pip install networkx")

### Cell 2.6 - Heatmap of NA and no NA genes in the initial dataset

In [ ]:
def log_transform_by_cohort(
    exp_df: pd.DataFrame,
    ann_df: pd.DataFrame,
    batch_col: str = "COHORT_LABEL",
    threshold: float = 30,
) -> pd.DataFrame:
    """
    Apply log2(x+1) transform per cohort only when max value exceeds threshold.

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    ann_df
        Annotation with batch_col column.
    batch_col
        Column used to group samples (COHORT_LABEL, not RNA_BATCH).
    threshold
        Cohorts with max > threshold are log2(x+1) transformed.

    Returns
    -------
    Log2-transformed expression matrix.
    """
    batches = ann_df[batch_col].value_counts().index
    df_list = []
    for batch in batches:
        samples = ann_df.index[ann_df[batch_col] == batch]
        exp_local = exp_df.reindex(samples)
        if exp_local.max().max() > threshold:
            exp_local = np.log2(exp_local + 1)
        df_list.append(exp_local)
    return pd.concat(df_list, axis=0)


def plot_batch_distributions(
    exp_df, ann_df, batch_col="RNA_BATCH", ncols=8, figsize=(28, 20), title_prefix=""
):
    batches = ann_df[batch_col].value_counts().index
    nrows = int(np.ceil(len(batches) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for ax, batch in zip(axs.flatten(), batches):
        samples = ann_df.index[ann_df[batch_col] == batch]
        vals = exp_df.reindex(samples).values  # .ravel()
        vals = vals[~np.isnan(vals)]
        ax.hist(vals, bins=40, density=True, alpha=0.7, color="steelblue")
        ax.set_title(batch, fontsize=10)
        # ax.set_xlim(-2, 18)
    for ax in axs.flatten()[len(batches) :]:
        ax.set_visible(False)
    fig.suptitle(
        f"{title_prefix} Expression distributions per {batch_col}", fontsize=10
    )
    plt.tight_layout()
    return fig

#### Palettes

In [ ]:
lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

platform_palette = {
    "RNASeq": "#4B0082",
    "RNAseq": "#4B0082",
    "Kassandra": "#6A5ACD",
    "GPL96+97": "#FF4500",
    "GPL96+GPL97": "#FF4500",
    "GPL570": "#FF8C00",
    "GPL96": "#B22222",
    "GPL6244": "#E9967A",
    "GPL13938": "#CD5C5C",
    "A-GEOD-19803": "#A52A2A",
    "GPL1708": "#008080",
    "GPL14951": "#4682B4",
    "GPL17077": "#00CED1",
    "GPL13158": "#1E90FF",
    "GPL17586": "#20B2AA",
    "GPL17047": "#5F9EA0",
    "GPL887": "#B0C4DE",
    "GPL10739": "#2E8B57",
    "GPL10739+GPL19251": "#3CB371",
    "GPL20188": "#6B8E23",
    "GPL23541": "#8FBC8F",
    "GPL8432": "#006400",
    "GPL16686": "#8B4513",
    "GPL26356": "#A0522D",
    float("nan"): "#D3D3D3",
    None: "#D3D3D3",
}

cohort_palette = {
    "PUB_DLBCL_CARTFARAMAND": "#f87d06",
    "PUB_DLBCL_JIANGGSE147986": "#1482b7",
    "PUB_DLBCL_GOYA": "#b72114",
    "PUB_DLBCL_NCICCR": "#b7b014",
    "PUB_DLBCL_PARKSRP100394": "#5bd66f",
    "PUB_DLBCL_SHISRP312586": "#2306f8",
    "PUB_DLBCL_YANCART": "#d67e5b",
    "PUB_FL_DAVE": "#0626f8",
    "PUB_FL_GSE62241": "#b71481",
    "PUB_FL_HUET": "#14b75d",
    "PUB_FL_PARSA": "#146db7",
    "PUB_FL_GUO": "#a3d65b",
    "PUB_FL_GSE148070": "#5b95d6",
    "PUB_FL_BRODTKORB": "#3514b7",
    "PUB_FL_CIBERSORTX": "#9ff806",
    "PUB_FL_HORN": "#5bd6cb",
    "PUB_FL_PASTORE": "#d66f5b",
    "PUB_FL_LEBRUN": "#b71496",
    "PUB_FL_ORICCHIO": "#9d5bd6",
    "PUB_FL_TASKINEN": "#14b772",
    "PUB_FL_DREYLING": "#8d5bd6",
    "PUB_FL_NCIFFPEREBUILT": "#f80625",
    "PUB_FL_NCISTAUDT": "#f80682",
    "PUB_FL_TOBIN": "#069ef8",
    "PUB_DLBCL_GSE11318": "#f80644",
    "PUB_DLBCL_BAGS": "#5bd6ac",
    "PUB_DLBCL_ETABM346": "#5bd660",
    "PUB_DLBCL_DLC1": "#5bd6bb",
    "PUB_DLBCL_NCIACALA": "#06f860",
    "PUB_DLBCL_GSE10846": "#5b85d6",
    "PUB_DLBCL_GSE117556": "#4914b7",
    "PUB_DLBCL_GSE19246": "#9b06f8",
    "PUB_DLBCL_GSE22898": "#c2d65b",
    "PUB_DLBCL_GSE23501": "#bef806",
    "PUB_DLBCL_GSE31312": "#b73614",
    "PUB_DLBCL_GSE32918": "#f706f8",
    "PUB_DLBCL_GSE34171": "#5e14b7",
    "PUB_DLBCL_GSE64555": "#d65bb3",
    "PUB_DLBCL_GSE87371": "#06dbf8",
    "PUB_DLBCL_GSE93984": "#d65ba3",
    "PUB_DLBCL_LYMPHOMICS": "#d65b84",
    "COLL_DLBCL_UMLOSSOSTRANSL": "#5b75d6",
    "CRON3": "#06f8f6",
    "E-MTAB-3974": "#f806da",
    "GSE107683": "#5bd1d6",
    "GSE12195": "#f89c06",
    "GSE12366": "#0607f8",
    "GSE12453": "#b2d65b",
    "GSE12845": "#66d65b",
    "GSE13411": "#14b787",
    "GSE136248": "#4206f8",
    "GSE136249": "#1bb714",
    "GSE141005": "#b906f8",
    "GSE144089": "#f80663",
    "GSE149089": "#f85e06",
    "GSE15271": "#aab714",
    "GSE156751": "#14b748",
    "GSE17186": "#1458b7",
    "GSE19599": "#b78614",
    "GSE22886": "#d806f8",
    "GSE36907": "#bc5bd6",
    "GSE38697": "#b79b14",
    "GSE46261": "#45b714",
    "GSE51528": "#1444b7",
    "GSE55267": "#1497b7",
    "GSE56314": "#b714ab",
    "GSE61912": "#76d65b",
    "GSE64028": "#06f841",
    "GSE68878": "#d68e5b",
    "GSE69033": "#06f822",
    "GSE85543": "#d2d65b",
    "GSE89282": "#06bdf8",
    "GSE99635": "#30b714",
    "Leandro-Normals": "#af14b7",
    "PUB_Beguelin_GSE45982": "#d6ca5b",
    "PUB_Health_B_cells_GSE110669": "#d69e5b",
    "PUB_Health_B_cells_GSE111310": "#b7141c",
    "PUB_Health_B_cells_GSE115655": "#b71431",
    "PUB_Health_B_cells_GSE119234": "#06f87f",
    "PUB_Health_B_cells_GSE139833": "#59b714",
    "PUB_Health_B_cells_GSE154583": "#5b66d6",
    "PUB_Health_B_cells_GSE92387": "#f8bb06",
    "PUB_Health_Bunting_GSE84022": "#d65b94",
    "PUB_Pastore_GSE119103": "#ac5bd6",
    "PUB_Suntsova_GSE120795": "#80f806",
    "Kassandra": "#14b79c",
    "SOM": "#61f806",
    "MDA_Strati_FL": "#2014b7",
}

#### Datasets load

In [ ]:
comb_exp = pd.read_csv(
    "$FL_DATA_ROOT/comb_exp.tsv", sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    "~/FL_harmonization/comb_ann_unified.csv"
).set_index("Unnamed: 0")

comb_ann["Diagnosis_with_coo"] = (
    comb_ann.Diagnosis_cell_type_general + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann["Diagnosis_unified_with_coo"] = (
    comb_ann.Diagnosis_cell_type_unified + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann.loc[comb_ann.index.isin(["PUB_Suntsova_GSE120795"]), "RNA_BATCH"] = (
    "RNASeq_FF_Unknown"
)

rare_groups = [
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
    "Other",
]

comb_ann = comb_ann[~comb_ann.Major_group.isin(rare_groups)]


common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
comb_exp = comb_exp.loc[common_samples]  # .dropna(axis=1)
comb_exp = comb_exp.loc[~comb_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
comb_ann = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]


comb_ann

In [ ]:
comb_exp

In [ ]:
na_statuses = ~comb_exp.isna()
na_statuses = na_statuses.reindex(na_statuses.sum(axis=1).sort_values().index[::-1])
genes_short = na_statuses.sum().sort_values().index[::1][::-1]
na_statuses = na_statuses[genes_short]
na_statuses

col_colors = pd.DataFrame(
    {
        "RNA_BATCH": comb_ann.loc[na_statuses.index, "RNA_BATCH"].map(
            rna_batch_palette
        ),
        # "PLATFORM_RNA":  comb_ann.loc[na_statuses.index, "PLATFORM_RNA"].map(platform_palette),
        # "COHORT_LABEL":  comb_ann.loc[na_statuses.index, "COHORT_LABEL"].map(cohort_palette),
        "Diagnosis_cell_type_unified": comb_ann.loc[
            na_statuses.index, "Diagnosis_cell_type_unified"
        ].map(lymphoma_ontogeny_palette),
    },
    index=na_statuses.index,
)

g2 = sns.clustermap(
    na_statuses.T,
    method="ward",
    metric="euclidean",
    row_cluster=False,
    col_cluster=False,
    col_colors=col_colors,
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(10, 3),
    dendrogram_ratio=(0.06, 0.22),
    xticklabels=False,
    yticklabels=False,
    colors_ratio=(0.02, 0.04),
    rasterized=True,  # <--- THIS IS THE MAGIC BULLET FOR LARGE HEATMAPS
)
g2.fig.savefig(FIGURES_DIR / "genes_na_heatmap.svg", bbox_inches="tight")
g2.fig.savefig(FIGURES_DIR / "genes_na_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()

---
## §3 — Group-by-group comparative analysis

**Repeated pattern throughout §3:** For every quantitative comparison, three flavors of
`plot_multilevel_comparison` are produced: grouped by (1) harmonization method, (2) removal
strategy, (3) imputation approach. Each includes an FDR-corrected Mann-Whitney p-value heatmap.

### §3.1 — Group A: PCA Variance Decomposition

In [ ]:
ANNOT_COLS_ORDER

#### Mean R2 by columns

In [ ]:
r2_cols_main = ["r2_RNA_BATCH", "r2_Major_group", "r2_TUMOR_NORMAL", "r2_PLATFORM_RNA"]
r2_cols_present = [c for c in r2_cols_main if c in df_ok.columns]
method_order_r2 = (
    df_ok.groupby("method")["r2_RNA_BATCH"].mean().sort_values().index.tolist()
)
r2_pal_covar = {col: annot_pal.get(col[3:], "#AAAAAA") for col in r2_cols_main}

df_long_r2 = df_ok[["method"] + r2_cols_present].melt(
    id_vars="method", var_name="covariate", value_name="r2"
)
# v3: append "★ Best" virtual group
df_best_r2 = df_best[["method"] + r2_cols_present].copy()
df_best_r2["method"] = BEST_LABEL
df_long_r2 = pd.concat(
    [
        df_long_r2,
        df_best_r2.melt(id_vars="method", var_name="covariate", value_name="r2"),
    ],
    ignore_index=True,
)
method_order_r2_v3 = method_order_r2 + [BEST_LABEL]

ax = bar_plot(
    df_long_r2,
    x="method",
    y="r2",
    hue="covariate",
    palette=r2_pal_covar,
    order=method_order_r2_v3,
    figsize=(20, 5),
    ylabel="Mean R²",
    title="Mean R² per covariate × method (sorted by r2_RNA_BATCH) — ★ Best = manually selected",
    legend_outside=True,
    xtick_rotation=90,
)
ax.axhline(0.20, color="red", linestyle="--", linewidth=1.0, label="Target r²=0.20")
style_best_bars(ax, method_order_r2_v3)
ax.get_figure().savefig(
    FIGURES_DIR / "a_r2_by_method_covariate.svg", bbox_inches="tight"
)
ax.get_figure().savefig(
    FIGURES_DIR / "a_r2_by_method_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

#### PCR (principal component regression) by columns

In [ ]:
annot_pal

In [ ]:
pcr_cols_main = [
    "pcr_COHORT_LABEL",
    "pcr_Diagnosis_cell_type_unified",
    "pcr_RNA_BATCH",
]
pcr_cols_present = [c for c in pcr_cols_main if c in df_ok.columns]

pcr_pal_covar = {col: annot_pal.get(col[4:], "#AAAAAA") for col in pcr_cols_main}
df_long_pcr = df_ok[["method"] + pcr_cols_present].melt(
    id_vars="method", var_name="covariate", value_name="pcr"
)
# v3: append "★ Best" virtual group
df_best_pcr = df_best[["method"] + pcr_cols_present].copy()
df_best_pcr["method"] = BEST_LABEL
df_long_pcr = pd.concat(
    [
        df_long_pcr,
        df_best_pcr.melt(id_vars="method", var_name="covariate", value_name="pcr"),
    ],
    ignore_index=True,
)
method_order_pcr_v3 = (
    df_long_pcr[df_long_pcr.covariate == "pcr_RNA_BATCH"][["method", "pcr"]]
    .groupby("method")
    .mean()
    .sort_values("pcr")
    .index[::-1]
)


ax = bar_plot(
    df_long_pcr,
    x="method",
    y="pcr",
    hue="covariate",
    palette=pcr_pal_covar,
    order=method_order_pcr_v3,
    figsize=(8, 2),
    ylabel="PCR²",
    title="PCR² per covariate × method (sorted by pcr_RNA_BATCH) — ★ Best = manually selected",
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)

style_best_bars(ax, method_order_pcr_v3)


# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax.get_xaxis_transform()

for i, method in enumerate(method_order_pcr_v3):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

    # Create a rectangle patch
    # x: i - 0.5 (centers the width=1 block directly under the tick marker)
    # y: -0.06 (starts 6% below the bottom spine)
    # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax.tick_params(axis="x", pad=15)


plt.savefig(
    FIGURES_DIR / "pcr_by_method_covariate.svg", bbox_inches="tight", transparent=True
)
plt.savefig(FIGURES_DIR / "pcr_by_method_covariate.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
pcr_cols_main_s = [
    "pcr_COHORT_LABEL",
    "pcr_Diagnosis_cell_type_unified",
    "pcr_RNA_BATCH",
    "pcr_RNASEQ_SOURCE",
]
pcr_cols_present_s = [c for c in pcr_cols_main_s if c in df_ok.columns]

pcr_pal_covar_s = {col: annot_pal.get(col[4:], "#AAAAAA") for col in pcr_cols_main_s}
df_long_pcr_s = df_ok[["strat"] + pcr_cols_present_s].melt(
    id_vars="strat", var_name="covariate", value_name="pcr"
)
# v3: append "★ Best" virtual group
df_best_pcr_s = df_best[["strat"] + pcr_cols_present_s].copy()
df_best_pcr_s["strat"] = BEST_LABEL
df_long_pcr_s = pd.concat(
    [
        df_long_pcr_s,
        df_best_pcr_s.melt(id_vars="strat", var_name="covariate", value_name="pcr"),
    ],
    ignore_index=True,
)

strat_order_pcr_v3 = (
    df_long_pcr_s[df_long_pcr_s.covariate == "pcr_RNA_BATCH"][["strat", "pcr"]]
    .groupby("strat")
    .mean()
    .sort_values("pcr")
    .index[::-1]
)


ax = bar_plot(
    df_long_pcr_s,
    x="strat",
    y="pcr",
    hue="covariate",
    palette=pcr_pal_covar_s,
    order=strat_order_pcr_v3,
    figsize=(4, 2),
    ylabel="Principal component regression",
    # title="PCR² per covariate × strategy (sorted by pcr_RNA_BATCH) — ★ Best = manually selected",
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)
# ax.axhline(0.20, color="red", linestyle="--", linewidth=1.0, label="Target=0.20")
style_best_bars(ax, strat_order_pcr_v3)
ax.get_figure().savefig(
    FIGURES_DIR / "pcr_by_strat_covariate.svg", bbox_inches="tight", transparent=True
)
ax.get_figure().savefig(
    FIGURES_DIR / "pcr_by_strat_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["r2_RNA_BATCH"].mean().sort_values().index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="r2_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.4,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "r² RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "r² RNA_BATCH by method × imputation — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, method_order_v3)
g.figure.savefig(FIGURES_DIR / "a_r2_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_r2_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="pcr_RNA_BATCH",
    col="imp",
    kind="box",
    height=2.5,
    aspect=1.2,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=90, ha="right", fontsize=6)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
# g.figure.suptitle("PCR RNA_BATCH by method × imputation — ★ Best = manually selected",y=1.02, fontsize=FONT_TITLE)
style_best_boxes(g, method_order_v3)
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_imp.svg", bbox_inches="tight", transparent=True
)
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
OUTLIER_SIZE = 2.5
OUTLIER_JITTER = 0.1
OUTLIER_ALPHA = 0.6
# ----------------------------------------------------------


# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

# 1. Draw the boxplot, hiding the default empty-circle outliers
g = sns.catplot(
    data=data_v3,
    x="method",
    y="pcr_RNA_BATCH",
    col="imp",
    kind="box",
    height=2.6,
    aspect=1,
    order=method_order_v3,
    palette=method_pal,
    showfliers=False,  # <--- Hides the default outliers
)

# 2. Overlay the jittered dots
g.map_dataframe(
    sns.stripplot,
    x="method",
    y="pcr_RNA_BATCH",
    hue="method",  # Matches colors to your boxes
    order=method_order_v3,
    hue_order=method_order_v3,
    palette=method_pal,
    size=OUTLIER_SIZE,  # Customizable size
    jitter=OUTLIER_JITTER,  # Customizable jitter width
    alpha=OUTLIER_ALPHA,
    dodge=False,
    zorder=0,  # <--- Pushes dots behind the boxes
)

g.set_xticklabels(rotation=90, ha="right", fontsize=5.5)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
# g.figure.suptitle("PCR RNA_BATCH by method × imputation — ★ Best = manually selected",y=1.02, fontsize=FONT_TITLE)

style_best_boxes(g, method_order_v3)
plt.tight_layout()
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_imp.svg", bbox_inches="tight", transparent=True
)
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
OUTLIER_SIZE = 1.5
OUTLIER_JITTER = 0.1
OUTLIER_ALPHA = 0.6

# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


g = sns.catplot(
    data=data_v3,
    x="method",
    y="pcr_RNA_BATCH",
    col="strat",
    kind="box",
    height=1.5,
    aspect=1.3,
    col_wrap=4,
    order=method_order_v3,
    palette=method_pal,
    showfliers=False,
)

# 2. Overlay the jittered dots
g.map_dataframe(
    sns.stripplot,
    x="method",
    y="pcr_RNA_BATCH",
    hue="method",  # Matches colors to your boxes
    order=method_order_v3,
    hue_order=method_order_v3,
    palette=method_pal,
    size=OUTLIER_SIZE,  # Customizable size
    jitter=OUTLIER_JITTER,  # Customizable jitter width
    alpha=OUTLIER_ALPHA,
    dodge=False,
    #    zorder=0                    # <--- Pushes dots behind the boxes
)

g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
# g.figure.suptitle("PCR RNA_BATCH by method × batch removal strategy — ★ Best = manually selected",y=1.02, fontsize=FONT_TITLE)
style_best_boxes(g, method_order_v3)
plt.tight_layout()
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_strat.svg", bbox_inches="tight", transparent=True
)
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
OUTLIER_SIZE = 1
OUTLIER_JITTER = 0.1
OUTLIER_ALPHA = 0.6


# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


g = sns.catplot(
    data=data_v3,
    x="method",
    y="pcr_RNA_BATCH",
    col="post_rm",
    kind="box",
    height=1.5,
    aspect=1.3,
    order=method_order_v3,
    palette=method_pal,
    showfliers=False,
)

g.map_dataframe(
    sns.stripplot,
    x="method",
    y="pcr_RNA_BATCH",
    hue="method",  # Matches colors to your boxes
    order=method_order_v3,
    hue_order=method_order_v3,
    palette=method_pal,
    size=OUTLIER_SIZE,  # Customizable size
    jitter=OUTLIER_JITTER,  # Customizable jitter width
    alpha=OUTLIER_ALPHA,
    dodge=False,
    #    zorder=0                    # <--- Pushes dots behind the boxes
)


g.set_xticklabels([], rotation=45, ha="right", fontsize=FONT_SMALL)
# axes.xaxis.set_ticklabels([])
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
# g.figure.suptitle("PCR RNA_BATCH by method × post_rm — ★ Best = manually selected",y=1.02, fontsize=FONT_TITLE)
style_best_boxes(g, method_order_v3)
plt.tight_layout()
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_post_rm.svg", bbox_inches="tight", transparent=True
)
g.figure.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_post_rm.png", bbox_inches="tight", dpi=200
)

plt.show()

In [ ]:
# add here a color palette by harshness

for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "r2_RNA_BATCH", group_col, title=f"r\u00b2 RNA_BATCH by {group_col}"
    )
    fname = f"a_r2_multilevel_{group_col}"
    fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "pcr_RNA_BATCH", group_col, title=f"PCR RNA_BATCH by {group_col}"
    )
    fname = f"a_pcr_multilevel_{group_col}"
    fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### PCR negatively correlates with R2 by RNA_BATCH

In [ ]:
plot_multilevel_comparison(
    df_ok, "pcr_RNA_BATCH", "method", title=f"PCR RNA_BATCH by {group_col}"
)

In [ ]:
plot_multilevel_comparison(
    df_ok, "r2_RNA_BATCH", "method", title=f"r\u00b2 RNA_BATCH by {group_col}"
)

In [ ]:
harshn

In [ ]:
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)


fig, axs = plt.subplots(2, 2, figsize=(8, 5), sharex=True)
sns.stripplot(
    data=data_v3,
    x="method",
    y="pcr_RNA_BATCH",
    hue="strat",
    palette=strat_pal,
    order=method_order_v3,
    size=2,
    alpha=0.7,
    ax=axs[0, 0],
)
axs[0, 0].get_legend().remove()
sns.stripplot(
    data=data_v3,
    x="method",
    y="pcr_Diagnosis_cell_type_unified",
    hue="strat",
    palette=strat_pal,
    order=method_order_v3,
    size=2,
    alpha=0.7,
    ax=axs[0, 1],
)
axs[0, 1].get_legend().remove()
sns.stripplot(
    data=data_v3,
    x="method",
    y="pcr_COHORT_LABEL",
    hue="strat",
    palette=strat_pal,
    order=method_order_v3,
    size=2,
    alpha=0.7,
    ax=axs[1, 0],
)
axs[1, 0].get_legend().remove()
sns.stripplot(
    data=data_v3,
    x="method",
    y="pcr_Major_group",
    hue="strat",
    palette=strat_pal,
    order=method_order_v3,
    size=2,
    alpha=0.7,
    ax=axs[1, 1],
)
axs[1, 1].get_legend().remove()
axs[1, 0].set_xticklabels(method_order_v3, rotation=90, ha="right", fontsize=FONT_TINY)
axs[1, 1].set_xticklabels(method_order_v3, rotation=90, ha="right", fontsize=FONT_TINY)

# 3. Add Color Block Annotations
for ax in [axs[1, 0], axs[1, 1]]:
    # get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
    trans = ax.get_xaxis_transform()

    for i, method in enumerate(method_order_v3):
        # Retrieve the color; fallback to black if the method is missing from the palette
        color = method_harshness_pal.get(method, "black")

        # Create a rectangle patch
        # x: i - 0.5 (centers the width=1 block directly under the tick marker)
        # y: -0.06 (starts 6% below the bottom spine)
        # height: 0.04 (thickness of the color bar)
        rect = patches.Rectangle(
            (i - 0.5, -0.06),
            width=1,
            height=0.04,
            transform=trans,
            clip_on=False,
            facecolor=color,
            edgecolor="none",
        )
        ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
    ax.tick_params(axis="x", pad=15)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_method_and_strat.svg",
    bbox_inches="tight",
    transparent=True,
)
fig.savefig(
    FIGURES_DIR / "a_pcr_catplot_by_method_and_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
pcr_cols

In [ ]:
pcr_cols = [c for c in df_ok.columns if c.startswith("pcr_")]

if pcr_cols:
    fig, axes = plt.subplots(1, min(len(pcr_cols), 4), figsize=(16, 4), sharey=False)
    if len(pcr_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, pcr_cols[:4]):
        if col not in df_ok.columns:
            continue
        sns.stripplot(
            data=data_v3,
            x="method",
            y=col,
            hue="strat",
            palette=strat_pal,
            order=method_order_v3,
            size=3,
            alpha=0.7,
            ax=ax,
            legend=(ax is axes[0]),
        )
        ax.set_xticklabels(method_order_v3, rotation=45, ha="right", fontsize=FONT_TINY)
        ax.set_title(col, fontsize=FONT_SMALL)
        ax.set_ylabel("PCR", fontsize=FONT_LABEL)
        ax.set_xlabel("")
        sns.despine(ax=ax)
    if axes[0].get_legend():
        axes[0].legend(
            title="Strat",
            fontsize=FONT_SMALL,
            title_fontsize=FONT_SMALL,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            frameon=False,
        )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_pcr_strip.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_pcr_strip.png", bbox_inches="tight", dpi=200)
    plt.show()

for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "pcr_RNA_BATCH", group_col, title=f"PCR RNA_BATCH by {group_col}"
    )
    fig.savefig(FIGURES_DIR / f"a_pcr_multilevel_{group_col}.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / f"a_pcr_multilevel_{group_col}.png", bbox_inches="tight", dpi=200
    )
    plt.show()

#### Scatter plots

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='pcr_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
print(df_ok["pcr_Diagnosis_cell_type_unified"].sort_values().to_string())

In [ ]:
    x_col='ks_mean_D_RNA_BATCH'
    y_col='ks_cohort_within_batch_mean_D'

In [ ]:
df_ok[df_ok.strat == "J_ff_only"]["pcr_Diagnosis_cell_type_unified"].median()

In [ ]:
df_ok[df_ok.method == "14_qsmooth"]["pcr_RNA_BATCH"].median()

In [ ]:
df_ok[(df_ok.method == "01_raw")&(df_ok.strat == "S0_no_removal")&(df_ok.imp == "strict")&(df_ok.post_rm == False)]["pcr_RNA_BATCH"]

In [ ]:
df_ok[(df_ok.method == "01_raw")&(df_ok.strat == "S0_no_removal")&(df_ok.imp == "strict")&(df_ok.post_rm == False)]["pcr_Diagnosis_cell_type_unified"]

In [ ]:
(1-0.226)/(1-0.923)

In [ ]:
median_abs_deviation(df_ok[df_ok.strat == "C_rnaseq_only"]["ks_mean_D_RNA_BATCH"])

In [ ]:
df_ok[df_ok.strat == "C_rnaseq_only"]["ks_mean_D_RNA_BATCH"].median()

In [ ]:
df_best["ks_cohort_within_batch_mean_D"].median()

In [ ]:
median_abs_deviation(df_best["ks_cohort_within_batch_mean_D"])

In [ ]:
df_best["pcr_Diagnosis_cell_type_unified"].median()

In [ ]:
df_best["clisi_mean_Diagnosis_cell_type_unified"].median()

In [ ]:
df_best["ilisi_mean_RNA_BATCH"].median()

In [ ]:
median_abs_deviation(df_best["pcr_Diagnosis_cell_type_unified"])

In [ ]:
spearmanr(df_ok["pcr_RNA_BATCH"], df_ok["pcr_Diagnosis_cell_type_unified"])

In [ ]:
df_ok[~df_ok["is_best"]]

In [ ]:
mannwhitneyu(
    df_best["pcr_Diagnosis_cell_type_unified"],
    df_ok[~df_ok["is_best"]]["pcr_Diagnosis_cell_type_unified"],
    alternative="two-sided",
)

In [ ]:
mannwhitneyu(
    df_best["pcr_RNA_BATCH"],
    df_ok[~df_ok["is_best"]]["pcr_RNA_BATCH"],
    alternative="two-sided",
)

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='pcr_COHORT_LABEL'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='pcr_PLATFORM_RNA'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="pcr_RNA_BATCH",
    y="pcr_COHORT_LABEL",
    ax=ax,
    hue="method",
    palette=method_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="pcr_RNA_BATCH",
    y="pcr_Diagnosis_cell_type_unified",
    ax=ax,
    hue="method",
    palette=method_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

In [ ]:
spearmanr(
    df_ok["pcr_RNA_BATCH"], df_ok["pcr_Diagnosis_cell_type_unified"]
)

In [ ]:
from scipy.stats import spearmanr, bootstrap

x = df_ok["pcr_RNA_BATCH"].values
y = df_ok["pcr_Diagnosis_cell_type_unified"].values

# 1. Define a wrapper function that returns just the statistic
def spearman_statistic(x, y):
    # Depending on your SciPy version, spearmanr returns a tuple or an object. 
    # [0] safely extracts the correlation coefficient.
    return spearmanr(x, y)[0] 

# 2. Run the bootstrap
res = bootstrap
    data=(x, y), 
    statistic=spearman_statistic, 
    vectorized=False, 
    paired=True, 
    confidence_level=0.95,
    method='BCa' # Bias-corrected and accelerated method
)

print(f"Spearman R: {spearmanr(x, y)[0]:.4f}")
print(f"95% CI: [{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="pcr_RNA_BATCH",
    y="pcr_Diagnosis_cell_type_unified",
    ax=ax,
    hue="strat",
    palette=strat_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="r2_RNA_BATCH",
    y="pcr_RNA_BATCH",
    ax=ax,
    hue="method",
    palette=method_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="r2_RNA_BATCH",
    y="pcr_RNA_BATCH",
    ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
g.legend(loc="right", bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
df_ok

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="r2_RNA_BATCH",
    y="pcr_RNA_BATCH",
    ax=ax,
    hue="method",
    palette=method_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

#### R2 by Principal components analysis in heatmap

In [ ]:
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)

In [ ]:
data_v3.columns[data_v3.columns.str.startswith("r2_pc")]

In [ ]:
df_ok[(df_ok.method == "01_raw") & (df_ok.strat == "S0_no_removal")]["pcr_RNA_BATCH"]

In [ ]:
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in data_v3.columns
]
if pc_r2_cols:
    pivot = data_v3.groupby("method")[pc_r2_cols].mean().reindex(method_order_v3)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        # annot=True,
        # fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "a_per_pc_r2_heatmap.svg", bbox_inches="tight", transparent=True
    )
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
pivot

In [ ]:
pivot.reindex(["33_amdbnorm", "13_fsmvn", "03_limma", "07_pycombat", "28_npn"])

#### Percentage variance explained by PC

In [ ]:
pivot.reindex(["20_shambhala", "18_rank", "17_quantile", "15_fsqn_py", "25_angel"])

In [ ]:
pivot["PC1"].sort_values()

In [ ]:
pivot  # .reindex(['Best', '33_amdbnorm', '28_npn', '26_xpn', '22_tmm', '23_vst', '10_mnn', '09_ruv', '12_scanorama'])

In [ ]:
    pct_var_col_available = [f"pct_var_pc{i}" for i in range(1, 11)]
    pivot = data_v3.groupby("method")[pct_var_col_available].mean().reindex(method_order_v3)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=100,
        #annot=True,
        #fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Percentage of variance explained"},
    )
    ax.set_title("Percentage variance explained by per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_variance_explained_heatmap.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_per_pc_variance_explained_heatmap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in df_ok.columns
]
if pc_r2_cols:
    pivot = df_ok.groupby("method")[pc_r2_cols].mean().reindex(method_order)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
df_best_tagged = df_best.copy()
df_best_tagged["strat"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = (
    data_v3.groupby("strat")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in data_v3.columns
]
if pc_r2_cols:
    pivot = data_v3.groupby("strat")[pc_r2_cols].mean().reindex(method_order_v3)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(4.5, 2.5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        # annot=True,
        # fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC")
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "a_per_pc_r2_heatmap_by_strat.svg",
        bbox_inches="tight",
        transparent=True,
    )
    fig.savefig(
        FIGURES_DIR / "a_per_pc_r2_heatmap_by_strat.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
    pct_var_col_available = [f"pct_var_pc{i}" for i in range(1, 11)]
    pivot = data_v3.groupby("strat")[pct_var_col_available].mean().reindex(method_order_v3)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(4.5, 2.5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=100,
        #annot=True,
        #fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Percentage of variance explained"},
    )
    ax.set_title("Percentage variance explained by per PC")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_variance_explained_heatmap_by_strat.svg", bbox_inches="tight", transparent=True)
    fig.savefig(FIGURES_DIR / "a_per_pc_variance_explained_heatmap_by_strat.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
method_order = (
    df_ok.groupby("strat")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in df_ok.columns
]
if pc_r2_cols:
    pivot = df_ok.groupby("strat")[pc_r2_cols].mean().reindex(method_order)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap_strat.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "a_per_pc_r2_heatmap_strat.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
pct_var_col_available

In [ ]:
df_ok["pct_var_pc2"].dropna()

In [ ]:
# Add harshness color annotation as side plot below:

In [ ]:
# §3.1.x1 — Joint heatmap: r2_pci × pct_var_pci
pct_var_col_available = all(f"pct_var_pc{i}" in df_ok.columns for i in range(1, 11))
r2_pc_available = all(f"r2_pc{i}_RNA_BATCH" in df_ok.columns for i in range(1, 11))

if pct_var_col_available and r2_pc_available:
    for i in range(1, 11):
        df_ok[f"wt_r2_pc{i}"] = (
            df_ok[f"r2_pc{i}_RNA_BATCH"] * df_ok[f"pct_var_pc{i}"] / 100
        )
    df_ok["wt_r2_sum"] = sum(df_ok[f"wt_r2_pc{i}"] for i in range(1, 11))

    r2_mat = (
        df_ok[[f"r2_pc{i}_RNA_BATCH" for i in range(1, 11)]]
        .groupby(df_ok["method"])
        .mean()
    )
    var_mat = (
        df_ok[[f"pct_var_pc{i}" for i in range(1, 11)]].groupby(df_ok["method"]).mean()
    )
    # row_order = df_ok.groupby("method")["wt_r2_sum"].mean().sort_values(ascending=False).index
    row_order = (
        df_ok.groupby("method")["pcr_RNA_BATCH"]
        .mean()
        .sort_values(ascending=False)
        .index
    )

    r2_mat = r2_mat.reindex(row_order)
    var_mat = var_mat.reindex(row_order)
    r2_mat.columns = [f"PC{i}" for i in range(1, 11)]
    var_mat.columns = [f"PC{i}" for i in range(1, 11)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(25, max(6, len(row_order) * 0.25)))
    sns.heatmap(
        r2_mat,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        ax=ax1,
        cbar_kws={"label": "R² RNA_BATCH"},
    )
    sns.heatmap(
        var_mat,
        cmap="Blues",
        vmin=0,
        annot=True,
        fmt=".1f",
        linewidths=0.3,
        ax=ax2,
        cbar_kws={"label": "% variance"},
    )
    ax1.set_title("R² RNA_BATCH per PC (group A)", fontsize=FONT_TITLE)
    ax2.set_title("% variance per PC (group J)", fontsize=FONT_TITLE)
    for ax in (ax1, ax2):
        ax.set_xlabel("Principal component", fontsize=FONT_LABEL)
        ax.set_ylabel("Method", fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
    fig.suptitle(
        "Joint R² × variance per PC (rows sorted by weighted batch R²)",
        fontsize=FONT_TITLE,
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_joint_r2_pct_var_heatmap.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "a_joint_r2_pct_var_heatmap.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("SKIP §3.1.x1: pct_var_pc or r2_pc columns not in df_ok.")

In [ ]:
# §3.1.x1 — Joint heatmap: r2_pci × pct_var_pci
pct_var_col_available = all(f"pct_var_pc{i}" in df_ok.columns for i in range(1, 11))
r2_pc_available = all(f"r2_pc{i}_RNA_BATCH" in df_ok.columns for i in range(1, 11))

if pct_var_col_available and r2_pc_available:
    for i in range(1, 11):
        df_ok[f"wt_r2_pc{i}"] = (
            df_ok[f"r2_pc{i}_RNA_BATCH"] * df_ok[f"pct_var_pc{i}"] / 100
        )
    df_ok["wt_r2_sum"] = sum(df_ok[f"wt_r2_pc{i}"] for i in range(1, 11))

    r2_mat = (
        df_ok[[f"r2_pc{i}_RNA_BATCH" for i in range(1, 11)]]
        .groupby(df_ok["strat"])
        .mean()
    )
    var_mat = (
        df_ok[[f"pct_var_pc{i}" for i in range(1, 11)]].groupby(df_ok["strat"]).mean()
    )
    # row_order = df_ok.groupby("method")["wt_r2_sum"].mean().sort_values(ascending=False).index
    row_order = (
        df_ok.groupby("strat")["pcr_RNA_BATCH"]
        .mean()
        .sort_values(ascending=False)
        .index
    )

    r2_mat = r2_mat.reindex(row_order)
    var_mat = var_mat.reindex(row_order)
    r2_mat.columns = [f"PC{i}" for i in range(1, 11)]
    var_mat.columns = [f"PC{i}" for i in range(1, 11)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(25, max(6, len(row_order) * 0.25)))
    sns.heatmap(
        r2_mat,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        ax=ax1,
        cbar_kws={"label": "R² RNA_BATCH"},
    )
    sns.heatmap(
        var_mat,
        cmap="Blues",
        vmin=0,
        annot=True,
        fmt=".1f",
        linewidths=0.3,
        ax=ax2,
        cbar_kws={"label": "% variance"},
    )
    ax1.set_title("R² RNA_BATCH per PC (group A)", fontsize=FONT_TITLE)
    ax2.set_title("% variance per PC (group J)", fontsize=FONT_TITLE)
    for ax in (ax1, ax2):
        ax.set_xlabel("Principal component", fontsize=FONT_LABEL)
        ax.set_ylabel("strat", fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
    fig.suptitle(
        "Joint R² × variance per PC (rows sorted by weighted batch R²)",
        fontsize=FONT_TITLE,
    )
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "a_joint_r2_pct_var_heatmap_strat.svg", bbox_inches="tight"
    )
    fig.savefig(
        FIGURES_DIR / "a_joint_r2_pct_var_heatmap_strat.png",
        bbox_inches="tight",
        dpi=200,
    )
    plt.show()
else:
    print("SKIP §3.1.x1: pct_var_pc or r2_pc columns not in df_ok.")

In [ ]:
row_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values(ascending=False).index
)

# §3.1.x2 — Weighted batch R² heatmap (wt_r2_pci = r2_pci × pct_var_pci / 100)
if pct_var_col_available and r2_pc_available and "wt_r2_pc1" in df_ok.columns:
    wt_mat = (
        df_ok[[f"wt_r2_pc{i}" for i in range(1, 11)]].groupby(df_ok["method"]).mean()
    )
    wt_mat = wt_mat.reindex(row_order)
    wt_mat.columns = [f"PC{i}" for i in range(1, 11)]

    fig, ax = plt.subplots(figsize=(12, max(6, len(row_order) * 0.4)))
    # mask_small = wt_mat < 0.001
    wt_mat[wt_mat < 0.001] = 0
    sns.heatmap(
        wt_mat,
        cmap="Reds",
        vmin=0,
        annot=True,
        fmt=".3f",
        linewidths=0.3,
        ax=ax,
        cbar_kws={"label": "wt_r2_pci (R² × %var / 100)"},
        # mask=mask_small,
    )
    ax.set_xlabel("Principal component", fontsize=FONT_LABEL)
    ax.set_ylabel("Method", fontsize=FONT_LABEL)
    ax.set_title(
        "Weighted batch R² per PC(rows sorted by sum of wt_r2)", fontsize=FONT_TITLE
    )
    ax.tick_params(labelsize=FONT_TICK)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_weighted_batch_r2_heatmap.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "a_weighted_batch_r2_heatmap.png", bbox_inches="tight", dpi=200
    )
    plt.show()
    print(f"wt_r2_sum added to df_ok: {'wt_r2_sum' in df_ok.columns}")
else:
    print("SKIP §3.1.x2: wt_r2_pc columns not computed.")

In [ ]:
df_ok[
    [
        "wt_r2_pc1",
        "wt_r2_pc2",
        "wt_r2_pc3",
        "wt_r2_pc4",
        "wt_r2_pc5",
        "wt_r2_pc6",
        "wt_r2_pc7",
        "wt_r2_pc8",
        "wt_r2_pc9",
        "wt_r2_pc10",
    ]
].sum(axis=1)

In [ ]:
df_ok["wt_r2_pc_all"] = df_ok[
    [
        "wt_r2_pc1",
        "wt_r2_pc2",
        "wt_r2_pc3",
        "wt_r2_pc4",
        "wt_r2_pc5",
        "wt_r2_pc6",
        "wt_r2_pc7",
        "wt_r2_pc8",
        "wt_r2_pc9",
        "wt_r2_pc10",
    ]
].sum(axis=1)

In [ ]:

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="pcr_RNA_BATCH",
        y="wt_r2_pc_all",
        hue="strat",
        #size="pcr_RNA_BATCH",
        palette=strat_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
    )


    #ax.set_yscale('log')
    #ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

#### DSC: Dispersion Separability Criterion

In [ ]:
### not interesting because it monotonously depends on PCR

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='dsc_RNA_BATCH'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
         figsize=(2.7, 2.7),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.set_yscale('log')
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("dsc")]

In [ ]:
    hue_col='strat'
    x_col='dsc_RNA_BATCH'
    y_col='dsc_PLATFORM_RNA'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
         figsize=(2.7, 2.7),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='dsc_RNA_BATCH'
    y_col='dsc_COHORT_LABEL'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(2.7, 2.7),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="pcr_RNA_BATCH",
        y="dsc_RNA_BATCH",
        hue="method",
        #size="pcr_RNA_BATCH",
        palette=method_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
    )


    ax.set_yscale('log')
    #ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
if "dsc_RNA_BATCH" in df_ok.columns and "dsc_COHORT_LABEL" in df_ok.columns:
    method_order = (
        df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
    )[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="dsc_RNA_BATCH",
        y="dsc_COHORT_LABEL",
        hue="method",
        # size="pcr_RNA_BATCH",
        palette=method_pal,
        # hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
    )

    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.get_legend().remove()
    plt.show()

In [ ]:

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="dsc_RNA_BATCH",
        y="dsc_COHORT_LABEL",
        hue="strat",
        #size="pcr_RNA_BATCH",
        palette=strat_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
    )


    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("pcr")]

In [ ]:
import matplotlib.patches as patches

cols_main = [
    "dsc_COHORT_LABEL",
    "dsc_PLATFORM_RNA",
    "dsc_RNASEQ_SOURCE",
    "dsc_RNA_BATCH",
]
cols_present = [c for c in cols_main if c in df_ok.columns]
method_order_dsc = (
    df_ok.groupby("method")["dsc_RNA_BATCH"].mean().sort_values().index.tolist()
)
pal_covar = {col: annot_pal.get(col[4:], "#AAAAAA") for col in cols_main}

df_long_dsc = df_ok[["method"] + cols_present].melt(
    id_vars="method", var_name="covariate", value_name="dsc"
)
# v3: append "★ Best" virtual group
df_best_dsc = df_best[["method"] + cols_present].copy()
df_best_dsc["method"] = BEST_LABEL
df_long_dsc = pd.concat(
    [
        df_long_dsc,
        df_best_dsc.melt(id_vars="method", var_name="covariate", value_name="dsc"),
    ],
    ignore_index=True,
)
method_order_dsc_v3 = method_order_dsc + [BEST_LABEL]

ax = bar_plot(
    df_long_dsc,
    x="method",
    y="dsc",
    hue="covariate",
    palette=pal_covar,
    order=method_order_dsc_v3,
    figsize=(20, 5),
    ylabel="Mean DSC",
    title="Mean DSC per covariate × method (sorted by dsc_COHORT_LABEL) — ★ Best = manually selected",
    legend_outside=True,
    xtick_rotation=90,
)
ax.set_yscale("log")
# Apply best styling after log scale (h * 1.05 works in log space)
style_best_bars(ax, method_order_dsc_v3)

trans = ax.get_xaxis_transform()
# Harshness rectangles — skip "★ Best" (last entry)
for i, method in enumerate(method_order_dsc_v3[:-1]):
    color = method_harshness_pal.get(method, "black")
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)
ax.tick_params(axis="x", pad=15)

ax.get_figure().savefig(
    FIGURES_DIR / "a_dsc_by_method_covariate.svg", bbox_inches="tight"
)
ax.get_figure().savefig(
    FIGURES_DIR / "a_dsc_by_method_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
cols_main = [
    "dsc_COHORT_LABEL",
    "dsc_PLATFORM_RNA",
    "dsc_RNASEQ_SOURCE",
    "dsc_RNA_BATCH",
]
cols_present = [c for c in cols_main if c in df_ok.columns]
strat_order_dsc = (
    df_ok.groupby("strat")["dsc_RNA_BATCH"].mean().sort_values().index.tolist()
)
pal_covar = {col: annot_pal.get(col[4:], "#AAAAAA") for col in cols_main}

df_long_dsc = df_ok[["strat"] + cols_present].melt(
    id_vars="strat", var_name="covariate", value_name="dsc"
)
# v3: append "★ Best" virtual group
df_best_dsc = df_best[["strat"] + cols_present].copy()
df_best_dsc["strat"] = BEST_LABEL
df_long_dsc = pd.concat(
    [
        df_long_dsc,
        df_best_dsc.melt(id_vars="strat", var_name="covariate", value_name="dsc"),
    ],
    ignore_index=True,
)
strat_order_dsc_v3 = strat_order_dsc + [BEST_LABEL]

ax = bar_plot(
    df_long_dsc,
    x="strat",
    y="dsc",
    hue="covariate",
    palette=pal_covar,
    order=strat_order_dsc_v3,
    figsize=(8, 5),
    ylabel="Mean DSC",
    title="Mean DSC per covariate × strat (sorted by dsc_COHORT_LABEL) — ★ Best = manually selected",
    legend_outside=True,
    xtick_rotation=90,
)
ax.set_yscale("log")
style_best_bars(ax, strat_order_dsc_v3)

ax.get_figure().savefig(
    FIGURES_DIR / "a_dsc_by_strat_covariate.svg", bbox_inches="tight"
)
ax.get_figure().savefig(
    FIGURES_DIR / "a_dsc_by_strat_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

#### Pairplots

### §3.2 — Group B: Neighbor-based Integration

#### Scatterplots

In [ ]:
    hue_col='strat'
    x_col='kbet_acceptance_rate_RNA_BATCH'
    y_col='ilisi_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='kbet_acceptance_rate_RNA_BATCH'
    y_col='ilisi_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='ilisi_mean_RNA_BATCH'
    y_col='clisi_mean_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",x
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
spearmanr(
    df_ok["ilisi_mean_RNA_BATCH"], df_ok["clisi_mean_Diagnosis_cell_type_unified"]
)

In [ ]:
corrs = []
for strat in df_ok.strat.unique():
    df_ok_sub = df_ok[df_ok.strat == strat]
    result = spearmanr(
        df_ok_sub["ilisi_mean_RNA_BATCH"],
        df_ok_sub["clisi_mean_Diagnosis_cell_type_unified"],
    )
    corrs.append(result[0])
    print(strat, result)

In [ ]:
np.median(corrs)

In [ ]:
spearmanr(
    df_ok_sub["ilisi_mean_RNA_BATCH"],
    df_ok_sub["clisi_mean_Diagnosis_cell_type_unified"],
)[1]

In [ ]:
mannwhitneyu(
    df_best["ilisi_mean_RNA_BATCH"],
    df_ok[~df_ok["is_best"]]["ilisi_mean_RNA_BATCH"],
    alternative="two-sided",
)

In [ ]:
df_ok[df_ok["strat"] == "H_affymetrix_extended"][
    "clisi_mean_Diagnosis_cell_type_unified"
].median()

In [ ]:
df_ok[df_ok["method"] == "14_qsmooth"]["tsne_entropy_norm_RNA_BATCH"].median()

In [ ]:
median_abs_deviation(
    df_ok[df_ok["method"] == "14_qsmooth"]["tsne_entropy_norm_RNA_BATCH"]
)

In [ ]:
df_ok["tsne_entropy_norm_RNA_BATCH"].median()

In [ ]:
median_abs_deviation(df_ok["ilisi_mean_RNA_BATCH"])

In [ ]:
mannwhitneyu(
    df_best["clisi_mean_Diagnosis_cell_type_unified"],
    df_ok[~df_ok["is_best"]]["clisi_mean_Diagnosis_cell_type_unified"],
    alternative="two-sided",
)

In [ ]:
df_ok[["method", "ilisi_mean_RNA_BATCH"]].groupby("method").median().sort_values(
    by="ilisi_mean_RNA_BATCH"
)

In [ ]:
    hue_col='method'
    x_col='ilisi_mean_RNA_BATCH'
    y_col='clisi_mean_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="kbet_acceptance_rate_RNA_BATCH",
        y="ilisi_norm_RNA_BATCH",
        hue="method",
        #size="pcr_RNA_BATCH",
        palette=method_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="kbet_acceptance_rate_RNA_BATCH",
        y="ilisi_norm_RNA_BATCH",
        hue="strat",
        #size="pcr_RNA_BATCH",
        palette=strat_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="kbet_acceptance_rate_RNA_BATCH",
        y="clisi_mean_Diagnosis_cell_type_unified",
        hue="method",
        #size="pcr_RNA_BATCH",
        palette=method_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="kbet_acceptance_rate_RNA_BATCH",
        y="clisi_mean_Diagnosis_cell_type_unified",
        hue="strat",
        #size="pcr_RNA_BATCH",
        palette=strat_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
kbet_col = "kbet_acceptance_rate_RNA_BATCH"
ilisi_col = "ilisi_norm_RNA_BATCH"
if kbet_col in df_ok.columns and ilisi_col in df_ok.columns:
    method_order_b = (
        df_ok.groupby("method")[kbet_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=kbet_col,
        y=ilisi_col,
        hue="method",
        style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=35,
        figsize=(10, 7),
        xlabel="kBET acceptance rate RNA_BATCH (↑ better)",
        ylabel="iLISI norm RNA_BATCH (↑ better)",
        title="kBET vs iLISI — color=method, style=imputation — ★ = best approaches",
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=kbet_col, y=ilisi_col)
    ax.get_figure().savefig(
        FIGURES_DIR / "b_kbet_ilisi_scatter.svg", bbox_inches="tight"
    )
    ax.get_figure().savefig(
        FIGURES_DIR / "b_kbet_ilisi_scatter.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
kbet_col = "kbet_acceptance_rate_RNA_BATCH"
if kbet_col in df_ok.columns:
    strat_order_kbet = (
        df_ok.groupby("strat")[kbet_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x="strat",
        y=kbet_col,
        hue="strat",
        palette=strat_pal,
        hue_order=strat_order_kbet,
        alpha=0.5,
        s=20,
        figsize=(10, 5),
        xlabel="Strategy",
        ylabel="kBET acceptance rate RNA_BATCH (↑ better)",
        title="kBET acceptance rate by strategy",
        legend_outside=True,
        save_path="kbet_by_strat",
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("kbet")]

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("clisi")]

#### Pairplots

In [ ]:
Index(['asw_batch_COHORT_LABEL', 'asw_batch_PLATFORM_RNA',
       'asw_batch_RNASEQ_SOURCE', 'asw_batch_RNA_BATCH',
       'asw_batch_norm_COHORT_LABEL', 'asw_batch_norm_PLATFORM_RNA',
       'asw_batch_norm_RNASEQ_SOURCE', 'asw_batch_norm_RNA_BATCH',
       'asw_bio_Diagnosis_cell_type_unified', 'asw_bio_Major_group',
       'asw_bio_PLATFORM_RNA', 'asw_bio_TUMOR_NORMAL',
       'asw_bio_norm_Diagnosis_cell_type_unified', 'asw_bio_norm_Major_group',
       'asw_bio_norm_PLATFORM_RNA', 'asw_bio_norm_TUMOR_NORMAL'],

In [ ]:
data_to_plot = df_ok[
    [
        "method",
        "asw_batch_COHORT_LABEL",
        "asw_batch_PLATFORM_RNA",
        "asw_batch_RNASEQ_SOURCE",
        "asw_batch_RNA_BATCH",
        "asw_bio_Diagnosis_cell_type_unified",
        "asw_bio_Major_group",
        "asw_bio_TUMOR_NORMAL",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="method",
    palette=method_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "asw_batch_COHORT_LABEL",
        "asw_batch_PLATFORM_RNA",
        "asw_batch_RNASEQ_SOURCE",
        "asw_batch_RNA_BATCH",
        "asw_bio_Diagnosis_cell_type_unified",
        "asw_bio_Major_group",
        "asw_bio_TUMOR_NORMAL",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)

In [ ]:
df_ok.columns[df_ok.columns.str.contains("RNA_BATCH")]

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "asw_batch_RNA_BATCH",
        "ilisi_mean_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
        "pcr_RNA_BATCH",
        "dist_ratio_RNA_BATCH",
        "dsc_RNA_BATCH",
        "tsne_centroid_disp_RNA_BATCH",
        "tsne_entropy_mean_RNA_BATCH",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)

In [ ]:
data_to_plot = df_ok[
    [
        "method",
        "kbet_acceptance_rate_COHORT_LABEL",
        "kbet_acceptance_rate_PLATFORM_RNA",
        "kbet_acceptance_rate_RNASEQ_SOURCE",
        "kbet_acceptance_rate_RNA_BATCH",
        "clisi_mean_Diagnosis_cell_type_unified",
        "clisi_mean_Major_group",
        "clisi_mean_TUMOR_NORMAL",
        "ilisi_mean_COHORT_LABEL",
        "ilisi_mean_PLATFORM_RNA",
        "ilisi_mean_RNASEQ_SOURCE",
        "ilisi_mean_RNA_BATCH",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="method",
    palette=method_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
data_to_plot = df_ok[
    [
        "method",
        "kbet_acceptance_rate_RNA_BATCH",
        "clisi_mean_Diagnosis_cell_type_unified",
        "clisi_mean_Major_group",
        "ilisi_mean_RNA_BATCH",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="method",
    palette=method_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="ilisi_mean_RNA_BATCH",
        y="clisi_mean_Diagnosis_cell_type_unified",
        hue="method",
        #size="pcr_RNA_BATCH",
        palette=method_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

In [ ]:
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        df_ok,
        x="ilisi_mean_RNA_BATCH",
        y="clisi_mean_Diagnosis_cell_type_unified",
        hue="strat",
        #size="pcr_RNA_BATCH",
        palette=strat_pal,
        #hue_order=method_order,
        alpha=0.7,
        s=40,
        ax=ax,
        style='imp'
    )


#    ax.set_yscale('log')
#    ax.set_xscale('log')
    ax.get_legend().remove()
    plt.show()

#### Barplots by local metrics

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("kbet")]

In [ ]:
kbet_cols_main = [
    "kbet_acceptance_rate_RNA_BATCH",
    "kbet_acceptance_rate_COHORT_LABEL",
    "kbet_acceptance_rate_PLATFORM_RNA",
    #'kbet_acceptance_rate_RNASEQ_SOURCE'
]
kbet_cols_present = [c for c in kbet_cols_main if c in df_ok.columns]

kbet_pal_covar = {col: annot_pal.get(col[21:], "#AAAAAA") for col in kbet_cols_main}
df_long_kbet = df_ok[["method"] + kbet_cols_present].melt(
    id_vars="method", var_name="covariate", value_name="kbet"
)
# v3: append "★ Best" virtual group
df_best_kbet = df_best[["method"] + kbet_cols_present].copy()
df_best_kbet["method"] = BEST_LABEL
df_long_kbet = pd.concat(
    [
        df_long_kbet,
        df_best_kbet.melt(id_vars="method", var_name="covariate", value_name="kbet"),
    ],
    ignore_index=True,
)
method_order_kbet_v3 = (
    df_long_kbet[df_long_kbet.covariate == "kbet_acceptance_rate_RNA_BATCH"][
        ["method", "kbet"]
    ]
    .groupby("method")
    .mean()
    .sort_values("kbet")
    .index[::-1]
)


ax = bar_plot(
    df_long_kbet,
    x="method",
    y="kbet",
    hue="covariate",
    palette=kbet_pal_covar,
    order=method_order_kbet_v3,
    figsize=(8, 2),
    ylabel="kBET acceptance rate",
    title="kBET acceptance rate per covariate × method",
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)

# style_best_bars(ax, method_order_kbet_v3)


# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax.get_xaxis_transform()

for i, method in enumerate(method_order_kbet_v3):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

    # Create a rectangle patch
    # x: i - 0.5 (centers the width=1 block directly under the tick marker)
    # y: -0.06 (starts 6% below the bottom spine)
    # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax.tick_params(axis="x", pad=15)
ax.set_yscale("log")

plt.savefig(
    FIGURES_DIR / "kBET_by_method_barplot.svg", bbox_inches="tight", transparent=True
)
plt.savefig(FIGURES_DIR / "kBET_by_method_barplot.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
kbet_cols_main = [
    "kbet_acceptance_rate_RNA_BATCH",
    "kbet_acceptance_rate_COHORT_LABEL",
    "kbet_acceptance_rate_PLATFORM_RNA",
    #'kbet_acceptance_rate_RNASEQ_SOURCE'
]
kbet_cols_present = [c for c in kbet_cols_main if c in df_ok.columns]

kbet_pal_covar = {col: annot_pal.get(col[21:], "#AAAAAA") for col in kbet_cols_main}
df_long_kbet = df_ok[["strat"] + kbet_cols_present].melt(
    id_vars="strat", var_name="covariate", value_name="kbet"
)
# v3: append "★ Best" virtual group
df_best_kbet = df_best[["strat"] + kbet_cols_present].copy()
df_best_kbet["strat"] = BEST_LABEL
df_long_kbet = pd.concat(
    [
        df_long_kbet,
        df_best_kbet.melt(id_vars="strat", var_name="covariate", value_name="kbet"),
    ],
    ignore_index=True,
)
strat_order_kbet_v3 = (
    df_long_kbet[df_long_kbet.covariate == "kbet_acceptance_rate_RNA_BATCH"][
        ["strat", "kbet"]
    ]
    .groupby("strat")
    .mean()
    .sort_values("kbet")
    .index[::-1]
)


ax = bar_plot(
    df_long_kbet,
    x="strat",
    y="kbet",
    hue="covariate",
    palette=kbet_pal_covar,
    order=strat_order_kbet_v3,
    figsize=(8, 2),
    ylabel="kBET acceptance rate",
    title="kBET acceptance rate per covariate × strat",
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)

# style_best_bars(ax, method_order_kbet_v3)
ax.set_yscale("log")


plt.savefig(
    FIGURES_DIR / "kBET_by_strat_barplot.svg", bbox_inches="tight", transparent=True
)
plt.savefig(FIGURES_DIR / "kBET_by_strat_barplot.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
kbet_cols_main = [
    "kbet_acceptance_rate_RNA_BATCH",
    "kbet_acceptance_rate_COHORT_LABEL",
    "kbet_acceptance_rate_PLATFORM_RNA",
]
kbet_cols_present = [c for c in kbet_cols_main if c in df_ok.columns]
method_order_kbet = (
    df_ok.groupby("method")["kbet_acceptance_rate_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
    if "kbet_acceptance_rate_RNA_BATCH" in df_ok.columns
    else method_order
)
kbet_pal_covar = {col: annot_pal.get(col[21:], "#AAAAAA") for col in kbet_cols_main}
df_long_kbet = df_ok[["method"] + kbet_cols_present].melt(
    id_vars="method", var_name="covariate", value_name="kbet"
)
# v3: append "★ Best" virtual group
df_best_kbet = df_best[["method"] + kbet_cols_present].copy()
df_best_kbet["method"] = BEST_LABEL
df_long_kbet = pd.concat(
    [
        df_long_kbet,
        df_best_kbet.melt(id_vars="method", var_name="covariate", value_name="kbet"),
    ],
    ignore_index=True,
)
method_order_kbet_v3 = method_order_kbet + [BEST_LABEL]

ax = bar_plot(
    df_long_kbet,
    x="method",
    y="kbet",
    hue="covariate",
    palette=kbet_pal_covar,
    order=method_order_kbet_v3,
    figsize=(14, 5),
    ylabel="kBET acceptance rate",
    title="kBET acceptance rate per covariate × method — ★ Best = manually selected",
    legend_outside=True,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
style_best_bars(ax, method_order_kbet_v3)
ax.get_figure().savefig(
    FIGURES_DIR / "kbet_by_method_covariate.svg", bbox_inches="tight"
)
ax.get_figure().savefig(
    FIGURES_DIR / "kbet_by_method_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_long_kbet

In [ ]:
kbet_cols_main = [
    "kbet_acceptance_rate_RNA_BATCH",
    "kbet_acceptance_rate_COHORT_LABEL",
    "kbet_acceptance_rate_PLATFORM_RNA",
]
kbet_cols_present = [c for c in kbet_cols_main if c in df_ok.columns]
strat_order_kbet = (
    df_ok.groupby("strat")["kbet_acceptance_rate_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
kbet_pal_covar = {col: annot_pal.get(col[21:], "#AAAAAA") for col in kbet_cols_main}
df_long_kbet = df_ok[["strat"] + kbet_cols_present].melt(
    id_vars="strat", var_name="covariate", value_name="kbet"
)
# v3: append "★ Best" virtual group
df_best_kbet = df_best[["strat"] + kbet_cols_present].copy()
df_best_kbet["strat"] = BEST_LABEL
df_long_kbet = pd.concat(
    [
        df_long_kbet,
        df_best_kbet.melt(id_vars="strat", var_name="covariate", value_name="kbet"),
    ],
    ignore_index=True,
)
strat_order_kbet_v3 = strat_order_kbet + [BEST_LABEL]

ax = bar_plot(
    df_long_kbet,
    x="strat",
    y="kbet",
    hue="covariate",
    palette=kbet_pal_covar,
    order=strat_order_kbet_v3,
    figsize=(14, 5),
    ylabel="kBET acceptance rate",
    title="kBET acceptance rate per covariate × strat — ★ Best = manually selected",
    legend_outside=True,
)
ax.set_yscale("log")
style_best_bars(ax, strat_order_kbet_v3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.get_figure().savefig(
    FIGURES_DIR / "kbet_by_strat_covariate.svg", bbox_inches="tight"
)
ax.get_figure().savefig(
    FIGURES_DIR / "kbet_by_strat_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_long_lisi

In [ ]:
df_long_lisi

In [ ]:
df_ok.columns[df_ok.columns.str.contains("lisi")]

In [ ]:
column = "method"


# lisi_cols_main = ["clisi_mean_Diagnosis_cell_type_unified","clisi_mean_TUMOR_NORMAL", "ilisi_mean_PLATFORM_RNA", "ilisi_mean_RNA_BATCH",]

lisi_cols_main = [
    "clisi_mean_Diagnosis_cell_type_unified",
    "ilisi_mean_COHORT_LABEL",
    "ilisi_mean_RNA_BATCH",
    #    "ilisi_mean_RNASEQ_SOURCE",
]
lisi_cols_present = [c for c in lisi_cols_main if c in df_ok.columns]

lisi_pal_covar = {col: annot_pal.get(col[11:], "#AAAAAA") for col in lisi_cols_main}
df_long_lisi = df_ok[[column] + lisi_cols_present].melt(
    id_vars=column, var_name="covariate", value_name="LISI"
)
# v3: append "★ Best" virtual group
df_best_lisi = df_best[[column] + lisi_cols_present].copy()
df_best_lisi[column] = BEST_LABEL
df_long_lisi = pd.concat(
    [
        df_long_lisi,
        df_best_lisi.melt(id_vars=column, var_name="covariate", value_name="LISI"),
    ],
    ignore_index=True,
)
method_order_lisi_v3 = (
    df_long_lisi[df_long_lisi["covariate"] == "ilisi_mean_RNA_BATCH"][[column, "LISI"]]
    .groupby(column)
    .mean()
    .sort_values(by="LISI")
    .index.tolist()[::-1]
)


ax = bar_plot(
    df_long_lisi,
    x=column,
    y="LISI",
    hue="covariate",
    palette=lisi_pal_covar,
    order=method_order_lisi_v3,
    figsize=(8, 2),
    ylabel="LISI",
    title="LISI per covariate × method — ★ Best = manually selected",
    legend_outside=True,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
style_best_bars(ax, method_order_lisi_v3)


trans = ax.get_xaxis_transform()

for i, method in enumerate(method_order_lisi_v3):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

    # Create a rectangle patch
    # x: i - 0.5 (centers the width=1 block directly under the tick marker)
    # y: -0.06 (starts 6% below the bottom spine)
    # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax.tick_params(axis="x", pad=15)

ax.get_figure().savefig(
    FIGURES_DIR / "lisi_by_method_covariate.svg", bbox_inches="tight", transparent=True
)
ax.get_figure().savefig(
    FIGURES_DIR / "lisi_by_method_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
column = "strat"


# lisi_cols_main = ["clisi_mean_Diagnosis_cell_type_unified","clisi_mean_TUMOR_NORMAL", "ilisi_mean_PLATFORM_RNA", "ilisi_mean_RNA_BATCH",]

lisi_cols_main = [
    "clisi_mean_Diagnosis_cell_type_unified",
    "ilisi_mean_COHORT_LABEL",
    "ilisi_mean_RNA_BATCH",
    "ilisi_mean_RNASEQ_SOURCE",
]
lisi_cols_present = [c for c in lisi_cols_main if c in df_ok.columns]

lisi_pal_covar = {col: annot_pal.get(col[11:], "#AAAAAA") for col in lisi_cols_main}
df_long_lisi = df_ok[[column] + lisi_cols_present].melt(
    id_vars=column, var_name="covariate", value_name="LISI"
)
# v3: append "★ Best" virtual group
df_best_lisi = df_best[[column] + lisi_cols_present].copy()
df_best_lisi[column] = BEST_LABEL
df_long_lisi = pd.concat(
    [
        df_long_lisi,
        df_best_lisi.melt(id_vars=column, var_name="covariate", value_name="LISI"),
    ],
    ignore_index=True,
)
method_order_lisi_v3 = (
    df_long_lisi[df_long_lisi["covariate"] == "ilisi_mean_RNA_BATCH"][[column, "LISI"]]
    .groupby(column)
    .mean()
    .sort_values(by="LISI")
    .index.tolist()[::-1]
)


ax = bar_plot(
    df_long_lisi,
    x=column,
    y="LISI",
    hue="covariate",
    palette=lisi_pal_covar,
    order=method_order_lisi_v3,
    figsize=(6, 2),
    ylabel="LISI",
    # title="LISI per covariate × method — ★ Best = manually selected",
    legend_outside=True,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
style_best_bars(ax, method_order_lisi_v3)


ax.get_figure().savefig(
    FIGURES_DIR / "lisi_by_strat_covariate.svg", bbox_inches="tight", transparent=True
)
ax.get_figure().savefig(
    FIGURES_DIR / "lisi_by_strat_covariate.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("LISI")]

In [ ]:
method_order = (
    df_ok.groupby("method")["kbet_acceptance_rate_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="kbet_acceptance_rate_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "kBET RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "kBET RNA_BATCH by method × imputation — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, method_order_v3)
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "a_kbet_catplot_by_imp.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["kbet_acceptance_rate_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="kbet_acceptance_rate_RNA_BATCH",
    col="strat",
    kind="box",
    height=4,
    aspect=1.5,
    col_wrap=4,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "kBET RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "kBET RNA_BATCH by method × strat — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, method_order_v3)
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_strat.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "a_kbet_catplot_by_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
strat_order = (
    df_ok.groupby("strat")["kbet_acceptance_rate_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
# v3: append "★ Best" virtual group (group_col = strat)
df_best_tagged = df_best.copy()
df_best_tagged["strat"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
strat_order_v3 = strat_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="strat",
    y="kbet_acceptance_rate_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=strat_order_v3,
    palette=strat_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Strategy", "kBET RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "kBET RNA_BATCH by strat × imputation — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, strat_order_v3)
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp_strat.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "a_kbet_catplot_by_imp_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("lisi")]

In [ ]:
data_to_show = df_ok[~df_ok.method.isin(["34_arsyn", "38_harman"])]
# df_best does not contain 34_arsyn or 38_harman, so no extra filter needed
method_order = (
    data_to_show.groupby("method")["ilisi_mean_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([data_to_show, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="ilisi_mean_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "ilisi RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "ilisi RNA_BATCH by method × imputation — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, method_order_v3)
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "a_ilisi_catplot_by_imp.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["ilisi_mean_RNA_BATCH"].mean().sort_values().index.tolist()
)
# v3: append "★ Best" virtual group
df_best_tagged = df_best.copy()
df_best_tagged["method"] = BEST_LABEL
data_v3 = pd.concat([df_ok, df_best_tagged], ignore_index=True)
method_order_v3 = method_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="method",
    y="ilisi_mean_RNA_BATCH",
    col="strat",
    kind="box",
    height=4,
    aspect=1.5,
    col_wrap=4,
    order=method_order_v3,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "iLISI RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "iLISI RNA_BATCH by method × strat — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, method_order_v3)
g.figure.savefig(FIGURES_DIR / "b_isisi_catplot_by_strat.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "b_lisi_catplot_by_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
data_to_show = df_ok[~df_ok.method.isin(["34_arsyn", "38_harman"])]
strat_order = (
    data_to_show.groupby("strat")["ilisi_mean_RNA_BATCH"]
    .mean()
    .sort_values()
    .index.tolist()
)
# v3: append "★ Best" virtual group (group_col = strat)
df_best_tagged = df_best.copy()
df_best_tagged["strat"] = BEST_LABEL
data_v3 = pd.concat([data_to_show, df_best_tagged], ignore_index=True)
strat_order_v3 = strat_order + [BEST_LABEL]

g = sns.catplot(
    data=data_v3,
    x="strat",
    y="ilisi_mean_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=strat_order_v3,
    palette=strat_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Strategy", "ilisi RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle(
    "ilisi RNA_BATCH by strat × imputation — ★ Best = manually selected",
    y=1.02,
    fontsize=FONT_TITLE,
)
style_best_boxes(g, strat_order_v3)
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp_strat.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "a_ilisi_catplot_by_imp_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

#### Correlations of local and global metrics

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='ilisi_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='kbet_acceptance_rate_RNA_BATCH'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.set_yscale('log')
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='pcr_RNA_BATCH'
    y_col='kbet_acceptance_rate_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.set_yscale('log')
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='ilisi_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="pcr_RNA_BATCH",
    y="kbet_acceptance_rate_RNA_BATCH",
    ax=ax,
    hue="strat",
    palette=strat_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
    data=df_ok,
    x="pcr_RNA_BATCH",
    y="kbet_acceptance_rate_RNA_BATCH",
    ax=ax,
    hue="method",
    palette=method_pal,
    legend=False,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
# fig, ax = plt.subplots(figsize=(7.5, 7.5))


data_to_plot = df_ok[
    [
        "clisi_mean_Diagnosis_cell_type_unified",
        "clisi_mean_Major_group",
        "clisi_mean_TUMOR_NORMAL",
        "ilisi_mean_COHORT_LABEL",
        "ilisi_mean_PLATFORM_RNA",
        "ilisi_mean_RNASEQ_SOURCE",
        "ilisi_mean_RNA_BATCH",
        "method",
    ]
]
data_to_plot = data_to_plot[~data_to_plot.method.isin(["34_arsyn", "38_harman"])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="method",
    palette=method_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
# g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

#### The rest plots

In [ ]:
df_ok.columns[(df_ok.columns.str.startswith("asw"))]

In [ ]:
bio_asw_cols = [c for c in df_ok.columns if c.startswith("asw_bio_norm_")]
clisi_bio_cols = [c for c in df_ok.columns if c.startswith("clisi_mean_")]

for group_col in ["method", "strat", "imp"]:
    for col in bio_asw_cols[:2] + clisi_bio_cols[:2]:
        if col in df_ok.columns:
            fig = plot_multilevel_comparison(
                df_ok, col, group_col, title=f"{col} by {group_col}"
            )
            safe_name = col.replace("/", "_").replace(" ", "_")
            fig.savefig(
                FIGURES_DIR / f"b_bio_{safe_name}_{group_col}.svg", bbox_inches="tight"
            )
            fig.savefig(
                FIGURES_DIR / f"b_bio_{safe_name}_{group_col}.png",
                bbox_inches="tight",
                dpi=200,
            )
            plt.show()

In [ ]:
batch_asw = "asw_batch_norm_RNA_BATCH"
bio_panels = [
    ("asw_bio_norm_Major_group", "Major_group"),
    ("asw_bio_norm_Diagnosis_cell_type_unified", "Diagnosis_cell_type_unified"),
]

for bio_col, bio_label in bio_panels:
    if batch_asw not in df_ok.columns or bio_col not in df_ok.columns:
        continue
    ax = scatter_plot(
        df_ok,
        x=batch_asw,
        y=bio_col,
        hue="imp",
        palette=imp_pal,
        alpha=0.75,
        s=40,
        figsize=(8, 6),
        xlabel="ASW batch norm RNA_BATCH (↑ better)",
        ylabel=f"ASW bio norm {bio_label} (↑ better)",
        title=f"Batch–biology trade-off ({bio_label})",
        legend_outside=True,
        save_path=f"b_batch_biology_tradeoff_{bio_label[:8]}",
    )
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
    combined = df_ok[batch_asw] + df_ok[bio_col]
    for ridx in combined.nlargest(3).index:
        row = df_ok.loc[ridx]
        ax.annotate(
            row.method, (row[batch_asw], row[bio_col]), fontsize=FONT_ANNOT, alpha=0.9
        )
    plt.show()

In [ ]:
cms_present = {
    b: f"cms_fraction_mixed_{b}"
    for b in ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
    if f"cms_fraction_mixed_{b}" in df_ok.columns
}

if cms_present:
    fig, axes = plt.subplots(1, len(cms_present), figsize=(5 * len(cms_present), 5))
    if len(cms_present) == 1:
        axes = [axes]
    for ax, (batch_name, col) in zip(axes, cms_present.items()):
        bp = ax.boxplot(
            [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
            labels=method_order,
            patch_artist=True,
        )
        ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8)
        ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"CMS fraction mixed\n{batch_name}", fontsize=9)
        ax.set_ylabel("CMS fraction mixed")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "b_cms_grid.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "b_cms_grid.png", bbox_inches="tight", dpi=200)
    plt.show()

    for group_col in ["method", "strat", "imp"]:
        for col in list(cms_present.values())[:2]:
            fig = plot_multilevel_comparison(
                df_ok, col, group_col, title=f"{col} by {group_col}"
            )
            safe = col.replace("/", "_")
            fig.savefig(
                FIGURES_DIR / f"b_cms_multilevel_{safe}_{group_col}.svg",
                bbox_inches="tight",
            )
            fig.savefig(
                FIGURES_DIR / f"b_cms_multilevel_{safe}_{group_col}.png",
                bbox_inches="tight",
                dpi=200,
            )
            plt.show()

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='ilisi_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby("strat")[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

### §3.3 — Group C: Embedding Metrics

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

In [ ]:
batch_embed_cols = ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
embed_types = [("umap", "UMAP"), ("tsne", "tSNE")]

fig, axes = plt.subplots(
    2,
    len(batch_embed_cols),
    figsize=(4 * len(batch_embed_cols), 7),
    sharey=True,
)
for row_idx, (prefix, embed_name) in enumerate(embed_types):
    for col_idx, bc in enumerate(batch_embed_cols):
        ax = axes[row_idx, col_idx]
        col = f"{prefix}_centroid_disp_{bc}"
        if col not in df_ok.columns:
            ax.set_visible(False)
            continue
        vals = [df_ok[df_ok.method == m][col].mean() for m in method_order]
        ax.bar(
            range(len(method_order)),
            vals,
            color=[method_pal.get(m, "#AAAAAA") for m in method_order],
        )
        ax.set_xticks(range(len(method_order)))
        ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
        ax.set_title(f"{embed_name} disp — {bc}", fontsize=FONT_SMALL)
        if col_idx == 0:
            ax.set_ylabel("Centroid dispersion (↓)", fontsize=FONT_LABEL)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "c_centroid_disp_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "c_centroid_disp_grid.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
batch_embed_cols_present = [
    bc
    for bc in batch_embed_cols
    if f"umap_entropy_norm_{bc}" in df_ok.columns
    and f"tsne_entropy_norm_{bc}" in df_ok.columns
]

if batch_embed_cols_present:
    fig, axes = plt.subplots(
        1, len(batch_embed_cols_present), figsize=(5 * len(batch_embed_cols_present), 4)
    )
    if len(batch_embed_cols_present) == 1:
        axes = [axes]
    for ax, bc in zip(axes, batch_embed_cols_present):
        uc = f"umap_entropy_norm_{bc}"
        tc = f"tsne_entropy_norm_{bc}"
        plot_df = df_ok[[uc, tc, "method"]].dropna()
        sns.scatterplot(
            data=plot_df,
            x=uc,
            y=tc,
            hue="method",
            palette=method_pal,
            alpha=0.6,
            s=20,
            linewidth=0,
            legend=False,
            ax=ax,
        )
        if len(plot_df) >= 3:
            r, p = pearsonr(plot_df[uc], plot_df[tc])
            ax.set_title(f"{bc}\nr={r:.2f}, p={p:.3f}", fontsize=FONT_SMALL)
        else:
            ax.set_title(bc, fontsize=FONT_SMALL)
        ax.plot([0, 1], [0, 1], "k--", linewidth=0.7, alpha=0.5)
        ax.set_xlabel("UMAP entropy norm", fontsize=FONT_LABEL)
        ax.set_ylabel("tSNE entropy norm", fontsize=FONT_LABEL)
        sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "c_umap_tsne_entropy_corr.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "c_umap_tsne_entropy_corr.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("No UMAP/tSNE entropy columns found.")

In [ ]:
sns.distplot(df_ok["tsne_centroid_disp_RNA_BATCH"], bins=122)

#### Scatterplots of embeddings and local metrics

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='tsne_centroid_disp_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("tsne_entropy")]

In [ ]:
    hue_col='strat'
    x_col='tsne_entropy_mean_RNA_BATCH'
    y_col='tsne_centroid_disp_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='umap_entropy_mean_RNA_BATCH'
    y_col='umap_centroid_disp_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='umap_entropy_mean_RNA_BATCH'
    y_col='tsne_entropy_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("r2")]

In [ ]:
    hue_col='method'
    x_col='pcr_RNA_BATCH'
    y_col='r2_pc1_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
batch_asw = "asw_batch_norm_RNA_BATCH"
bio_panels = [
    ("asw_bio_norm_Major_group", "Major_group"),
    ("asw_bio_norm_Diagnosis_cell_type_unified", "Diagnosis_cell_type_unified"),
]

In [ ]:
    hue_col='method'
    x_col='asw_batch_norm_RNA_BATCH'
    y_col='asw_bio_norm_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='asw_batch_RNA_BATCH'
    y_col='asw_bio_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("entropy")]

In [ ]:
    hue_col='method'
    x_col='tsne_entropy_mean_RNA_BATCH'
    y_col='umap_entropy_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='ilisi_mean_RNA_BATCH'
    y_col='tsne_entropy_mean_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='pcr_RNA_BATCH'
    y_col='asw_batch_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='pcr_RNA_BATCH'
    y_col='asw_batch_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='pcr_RNA_BATCH'
    y_col='tsne_centroid_disp_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:


    hue_col='post_rm'
    x_col='pcr_RNA_BATCH'
    y_col='tsne_centroid_disp_RNA_BATCH'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=post_rm_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=20,
        figsize=(4.5, 4.5),
        legend_outside=True,
    )
    # ------------------------------------------------------------------
    # NEW: Connect paired False -> True points 
    # ------------------------------------------------------------------
    import numpy as np
    from matplotlib.collections import LineCollection

    group_keys = ['strat', 'method', 'imp']

    # Isolate coordinates for both states mapped to their group multi-index
    df_f = df_ok[df_ok[hue_col] == False].set_index(group_keys)[[x_col, y_col]]
    df_t = df_ok[df_ok[hue_col] == True].set_index(group_keys)[[x_col, y_col]]

    # Inner join safely drops any orphaned points that don't have a pair
    paired = df_f.join(df_t, lsuffix='_f', rsuffix='_t', how='inner')

    if not paired.empty:
        pts_f = paired[[f"{x_col}_f", f"{y_col}_f"]].to_numpy()
        pts_t = paired[[f"{x_col}_t", f"{y_col}_t"]].to_numpy()
        
        # Creates a (N, 2, 2) array of [[x0, y0], [x1, y1]] coordinate pairs
        segments = np.stack([pts_f, pts_t], axis=1)

        lc = LineCollection(
            segments,
            colors='gray',
            linewidths=0.4,
            alpha=0.35,
            zorder=0.8  # <--- Forces lines to render behind the scatter dots
        )
        ax.add_collection(lc)
    # ------------------------------------------------------------------
    
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### Ordered heatmaps by iLISI, cLISI, tSNE entropy - ordered by strategy and method

In [ ]:
# THINK HOW TO ADD THE BEST APPROACHES, ASK GEMINI

In [ ]:
metric_pivot = (
    df_ok.groupby(["strat", "method"])["ilisi_mean_RNA_BATCH"].mean().unstack()
)
# Sort strategies by total NA and methods by total NA
metric_pivot = metric_pivot.loc[
    metric_pivot.mean(axis=1).sort_values(ascending=False).index,
    metric_pivot.mean(axis=0).sort_values(ascending=False).index,
]
metric_pivot

In [ ]:
df_best[["strat", "method"]]

In [ ]:
row_colors = pd.DataFrame(
    {
        "strat": metric_pivot.index.map(strat_pal).fillna("#AAAAAA"),
    },
    index=metric_pivot.index,
)

col_colors = pd.DataFrame(
    {
        "method": metric_pivot.columns.get_level_values(0).map(method_pal),
        "harshness": metric_pivot.columns.get_level_values(0)
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=metric_pivot.columns,
)

In [ ]:
g2 = sns.clustermap(
    metric_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    col_colors=col_colors,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(5, 4),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.03, 0.04),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)
g2.fig.savefig(
    FIGURES_DIR / "heatmap_ilisi_mean_RNA_BATCH.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / "heatmap_ilisi_mean_RNA_BATCH.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
# 1. Generate the clustermap
g2 = sns.clustermap(
    metric_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    col_colors=col_colors,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(5, 4),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=False,
    colors_ratio=(0.03, 0.04),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)

# 2. Add black stars for the best combinations
# We drop duplicates just in case there are multiple runs for the same strat/method combo
best_pairs = df_best[["strat", "method"]].drop_duplicates()

for _, row in best_pairs.iterrows():
    strat_val = row["strat"]
    method_val = row["method"]

    # Check if they exist in the pivot table to avoid KeyErrors
    if strat_val in metric_pivot.index and method_val in metric_pivot.columns:
        # Get integer indices for the row (y) and column (x)
        y_idx = metric_pivot.index.get_loc(strat_val)
        x_idx = metric_pivot.columns.get_loc(method_val)

        # Plot the star on the heatmap axes.
        # Heatmap coordinates are offset by 0.5 to land perfectly in the cell center.
        g2.ax_heatmap.scatter(
            x_idx + 0.5,
            y_idx + 0.5,
            marker="*",
            color="black",
            s=40,  # Size of the star (adjust as needed)
            zorder=10,  # Ensures the star is drawn on top of the heatmap colors
        )

# 3. Save and display the figure
g2.fig.savefig(
    FIGURES_DIR / "heatmap_ilisi_mean_RNA_BATCH.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / "heatmap_ilisi_mean_RNA_BATCH.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("asw")]

In [ ]:
# column_to_show = "ilisi_mean_RNA_BATCH"
# column_to_show = "clisi_mean_Diagnosis_cell_type_unified"
# column_to_show = "tsne_centroid_disp_RNA_BATCH"
column_to_show = "asw_batch_RNA_BATCH"
# column_to_show='asw_bio_norm_Diagnosis_cell_type_unified'

metric_pivot = df_ok.groupby(["strat", "method"])[column_to_show].mean().unstack()
# Sort strategies by total NA and methods by total NA
metric_pivot = metric_pivot.loc[
    metric_pivot.mean(axis=1).sort_values(ascending=False).index,
    metric_pivot.mean(axis=0).sort_values(ascending=False).index,
]

row_colors = pd.DataFrame(
    {
        "strat": [strat_pal[val] for val in metric_pivot.index],
    },
    index=metric_pivot.index,
)

col_colors = pd.DataFrame(
    {
        "method": metric_pivot.columns.get_level_values(0).map(method_pal),
        "harshness": metric_pivot.columns.get_level_values(0)
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=metric_pivot.columns,
)
# 1. Generate the clustermap
g2 = sns.clustermap(
    metric_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    col_colors=col_colors,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(3, 2),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=False,
    colors_ratio=(0.03, 0.06),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)

# 2. Add black stars for the best combinations
# We drop duplicates just in case there are multiple runs for the same strat/method combo
best_pairs = df_best[["strat", "method"]].drop_duplicates()

for _, row in best_pairs.iterrows():
    strat_val = row["strat"]
    method_val = row["method"]

    # Check if they exist in the pivot table to avoid KeyErrors
    if strat_val in metric_pivot.index and method_val in metric_pivot.columns:
        # Get integer indices for the row (y) and column (x)
        y_idx = metric_pivot.index.get_loc(strat_val)
        x_idx = metric_pivot.columns.get_loc(method_val)

        # Plot the star on the heatmap axes.
        # Heatmap coordinates are offset by 0.5 to land perfectly in the cell center.
        g2.ax_heatmap.scatter(
            x_idx + 0.5,
            y_idx + 0.5,
            marker="*",
            color="black",
            s=10,  # Size of the star (adjust as needed)
            zorder=10,  # Ensures the star is drawn on top of the heatmap colors
        )

# 3. Save and display the figure
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
metric_pivot = (
    df_ok.groupby(["strat", "method"])["ilisi_mean_RNA_BATCH"].mean().unstack()
)
# Sort strategies by total NA and methods by total NA
metric_pivot = metric_pivot.loc[
    metric_pivot.mean(axis=1).sort_values(ascending=False).index,
    metric_pivot.mean(axis=0).sort_values(ascending=False).index,
]
fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(
    metric_pivot,
    ax=ax,
    cmap="Blues",
    annot=False,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"shrink": 0.5, "label": "iLISI RNA_BATCH"},
)

ax.set_xlabel("Method")
ax.set_ylabel("Strategy")

### §3.4 — Group D: Distribution Metrics

In [ ]:
 "ks_mean_D_",
        "ks_frac_sig_",
        "ks_cohort_within_batch_",
        "per_gene_batch_mean_cv_",

df_ok.columns[df_ok.columns.str.contains('ks_frac_sig_')]

In [ ]:
df_ok.columns[df_ok.columns.str.contains("ks_frac_sig_")]

In [ ]:
"ks_frac_sig_RNA_BATCH"[12:]

#### Barplots

In [ ]:
cols_main = [
    "ks_frac_sig_COHORT_LABEL",
    "ks_frac_sig_PLATFORM_RNA",
    "ks_frac_sig_RNA_BATCH",
]

x_col = "method"
metric_name_all = "KS_frac_sig"
cols_present = [c for c in cols_main if c in df_ok.columns]

pal_covar = {col: annot_pal.get(col[12:], "#AAAAAA") for col in cols_main}
df_long = df_ok[[x_col] + cols_present].melt(
    id_vars=x_col, var_name="covariate", value_name=metric_name_all
)
# v3: append "★ Best" virtual group
df_best = df_ok[df_ok["is_best"]].copy()
df_best = df_best[[x_col] + cols_present].copy()
df_best[x_col] = BEST_LABEL
df_long = pd.concat(
    [
        df_long,
        df_best.melt(id_vars=x_col, var_name="covariate", value_name=metric_name_all),
    ],
    ignore_index=True,
)
order_v3 = (
    df_long[df_long.covariate == "ks_frac_sig_RNA_BATCH"][[x_col, metric_name_all]]
    .groupby(x_col)
    .mean()
    .sort_values(metric_name_all)
    .index
)  # [::-1]


ax = bar_plot(
    df_long,
    x=x_col,
    y=metric_name_all,
    hue="covariate",
    palette=pal_covar,
    order=order_v3,
    figsize=(6, 2),
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)


# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax.get_xaxis_transform()

for i, method in enumerate(order_v3):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

    # Create a rectangle patch
    # x: i - 0.5 (centers the width=1 block directly under the tick marker)
    # y: -0.06 (starts 6% below the bottom spine)
    # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax.tick_params(axis="x", pad=15)
# ax.set_yscale('log')

plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.svg",
    bbox_inches="tight",
    transparent=True,
)
plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.png",
    bbox_inches="tight",
    dpi=200,
)
plt.show()

In [ ]:
cols_main = [
    "ks_frac_sig_COHORT_LABEL",
    "ks_frac_sig_PLATFORM_RNA",
    "ks_frac_sig_RNA_BATCH",
]

x_col = "strat"
metric_name_all = "KS_frac_sig"
cols_present = [c for c in cols_main if c in df_ok.columns]

pal_covar = {col: annot_pal.get(col[12:], "#AAAAAA") for col in cols_main}
df_long = df_ok[[x_col] + cols_present].melt(
    id_vars=x_col, var_name="covariate", value_name=metric_name_all
)
# v3: append "★ Best" virtual group
df_best = df_ok[df_ok["is_best"]].copy()
df_best = df_best[[x_col] + cols_present].copy()
df_best[x_col] = BEST_LABEL
df_long = pd.concat(
    [
        df_long,
        df_best.melt(id_vars=x_col, var_name="covariate", value_name=metric_name_all),
    ],
    ignore_index=True,
)
order_v3 = (
    df_long[df_long.covariate == "ks_frac_sig_RNA_BATCH"][[x_col, metric_name_all]]
    .groupby(x_col)
    .mean()
    .sort_values(metric_name_all)
    .index
)  # [::-1]


ax = bar_plot(
    df_long,
    x=x_col,
    y=metric_name_all,
    hue="covariate",
    palette=pal_covar,
    order=order_v3,
    figsize=(4, 2),
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)

# ax.set_yscale('log')

plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.svg",
    bbox_inches="tight",
    transparent=True,
)
plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.png",
    bbox_inches="tight",
    dpi=200,
)
plt.show()

#### Scatterplots

In [ ]:
    hue_col='method'
    x_col='ks_mean_D_RNA_BATCH'
    y_col='ks_cohort_within_batch_mean_D'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='ks_mean_D_RNA_BATCH'
    y_col='ks_cohort_within_batch_mean_D'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### Pairplots

In [ ]:
 "ks_mean_D_",
        "ks_frac_sig_",
        "ks_cohort_within_batch_",
        "per_gene_batch_mean_cv_",

In [ ]:
df_ok.columns[df_ok.columns.str.contains("per_gene_batch_mean_cv_")]

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "ks_frac_sig_RNA_BATCH",
        "ks_mean_D_RNA_BATCH",
        "ks_cohort_within_batch_frac_sig",
        "ks_cohort_within_batch_mean_D",
        #'per_gene_batch_mean_cv_RNA_BATCH'
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "per_gene_batch_mean_cv_COHORT_LABEL",
        "per_gene_batch_mean_cv_PLATFORM_RNA",
        "per_gene_batch_mean_cv_RNA_BATCH",
        #'per_gene_batch_mean_cv_RNA_BATCH'
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)
for ax in g.axes.flat:
    ax.set(xscale="log")
    ax.set(yscale="log")

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "ks_frac_sig_COHORT_LABEL",
        "ks_frac_sig_PLATFORM_RNA",
        "ks_frac_sig_RNA_BATCH",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

In [ ]:
data_to_plot = df_ok[
    [
        "strat",
        "ks_frac_sig_COHORT_LABEL",
        "ks_frac_sig_PLATFORM_RNA",
        "ks_frac_sig_RNA_BATCH",
        "ks_mean_D_COHORT_LABEL",
        "ks_mean_D_PLATFORM_RNA",
        "ks_mean_D_RNA_BATCH",
        "ks_cohort_within_batch_frac_sig",
        "ks_cohort_within_batch_mean_D",
        "per_gene_batch_mean_cv_COHORT_LABEL",
        "per_gene_batch_mean_cv_PLATFORM_RNA",
        "per_gene_batch_mean_cv_RNA_BATCH",
    ]
]


# data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
    data=data_to_plot,  # ax=ax,
    hue="strat",
    palette=strat_pal,
    # palette=dict(zip(order, colors)),
    # size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
)

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_ylabel(ax.get_ylabel(), rotation=0)
        ax.set_xlabel(ax.get_xlabel(), rotation=90)

#### The rest plots

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

In [ ]:
ks_batch_cols = ["RNA_BATCH", "PLATFORM_RNA", "COHORT_LABEL"]
ks_present = {
    b: f"ks_mean_D_{b}" for b in ks_batch_cols if f"ks_mean_D_{b}" in df_ok.columns
}
ks_frac_map = {b: f"ks_frac_sig_{b}" for b in ks_batch_cols}

fig, axes = plt.subplots(1, len(ks_present), figsize=(5 * len(ks_present), 5))
if len(ks_present) == 1:
    axes = [axes]
for ax, (batch_name, col) in zip(axes, ks_present.items()):
    frac_col = ks_frac_map.get(batch_name)
    bp = ax.boxplot(
        [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
        labels=method_order,
        patch_artist=True,
    )
    for patch, m in zip(bp["boxes"], method_order):
        mean_frac = (
            df_ok[df_ok.method == m][frac_col].mean()
            if (frac_col and frac_col in df_ok.columns)
            else 0
        )
        patch.set_facecolor("#FF4444" if mean_frac > 0.5 else "#4499FF")
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"KS mean D\n{batch_name}", fontsize=9)
    ax.set_ylabel("Mean KS D-statistic (\u2193 better)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "d_ks_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "d_ks_grid.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["method", "strat", "imp"]:
    if "ks_mean_D_RNA_BATCH" in df_ok.columns:
        fig = plot_multilevel_comparison(
            df_ok,
            "ks_mean_D_RNA_BATCH",
            group_col,
            title=f"KS D RNA_BATCH by {group_col}",
        )
        fig.savefig(
            FIGURES_DIR / f"d_ks_multilevel_{group_col}.svg", bbox_inches="tight"
        )
        fig.savefig(
            FIGURES_DIR / f"d_ks_multilevel_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

In [ ]:
if (
    "ks_mean_D_RNA_BATCH" in df_ok.columns
    and "ks_cohort_within_batch_mean_D" in df_ok.columns
):
    ax = scatter_plot(
        df_ok,
        x="ks_mean_D_RNA_BATCH",
        y="ks_cohort_within_batch_mean_D",
        hue="method",
        palette=method_pal,
        alpha=0.7,
        s=50,
        figsize=(8, 6),
        xlabel="KS mean D RNA_BATCH (platform-level)",
        ylabel="KS cohort within-batch mean D",
        title="Platform vs within-platform cohort KS effects",
        legend_outside=True,
        save_path="d_ks_platform_vs_cohort",
    )
    plt.show()

In [ ]:
cv_present = {
    b: f"per_gene_batch_mean_cv_{b}"
    for b in ["RNA_BATCH", "PLATFORM_RNA", "COHORT_LABEL"]
    if f"per_gene_batch_mean_cv_{b}" in df_ok.columns
}

for batch_name, col in cv_present.items():
    means = df_ok.groupby("method")[col].mean().reindex(method_order)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(
        range(len(method_order)),
        means.values,
        color=[method_pal.get(m, "#AAAAAA") for m in method_order],
    )
    ax.set_xticks(range(len(method_order)))
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean per-gene batch CV (\u2193 better)")
    ax.set_title(f"Per-gene batch mean CV \u2014 {batch_name}")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"d_gene_cv_{batch_name}.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / f"d_gene_cv_{batch_name}.png", bbox_inches="tight", dpi=200
    )
    plt.show()

    for group_col in ["method", "strat", "imp"]:
        fig = plot_multilevel_comparison(
            df_ok, col, group_col, title=f"{col} by {group_col}"
        )
        fig.savefig(
            FIGURES_DIR / f"d_cv_multilevel_{batch_name}_{group_col}.svg",
            bbox_inches="tight",
        )
        fig.savefig(
            FIGURES_DIR / f"d_cv_multilevel_{batch_name}_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

In [ ]:
len(scoring_cols)

### §3.5 — Group E: Data Quality

#### Ordered heatmaps of NA/noNA cells/genes/samples

In [ ]:
df_ok.columns[df_ok.columns.str.contains("na")]

In [ ]:
df_ok.columns[df_ok.columns.str.contains("noNA")]

In [ ]:
metric_pivot

In [ ]:
column_to_show = "pct_genes_noNA"
# column_to_show='pct_samples_noNA'
# column_to_show = "pct_na_cells"

metric_pivot = df_ok.groupby(["strat", "method"])[column_to_show].mean().unstack()
# Sort strategies by total NA and methods by total NA
metric_pivot = metric_pivot.loc[
    metric_pivot.mean(axis=1).sort_values(ascending=False).index,
    metric_pivot.mean(axis=0).sort_values(ascending=False).index,
]

row_colors = pd.DataFrame(
    {
        "strat": [strat_pal[val] for val in metric_pivot.index],
    },
    index=metric_pivot.index,
)

col_colors = pd.DataFrame(
    {
        "method": metric_pivot.columns.get_level_values(0).map(method_pal),
        "harshness": metric_pivot.columns.get_level_values(0)
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=metric_pivot.columns,
)
# 1. Generate the clustermap
g2 = sns.clustermap(
    metric_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    col_colors=col_colors,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(3, 2),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=False,
    colors_ratio=(0.03, 0.06),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)

# 2. Add black stars for the best combinations
# We drop duplicates just in case there are multiple runs for the same strat/method combo
df_best = df_ok[df_ok["is_best"]].copy()
best_pairs = df_best[["strat", "method"]].drop_duplicates()

for _, row in best_pairs.iterrows():
    strat_val = row["strat"]
    method_val = row["method"]

    # Check if they exist in the pivot table to avoid KeyErrors
    if strat_val in metric_pivot.index and method_val in metric_pivot.columns:
        # Get integer indices for the row (y) and column (x)
        y_idx = metric_pivot.index.get_loc(strat_val)
        x_idx = metric_pivot.columns.get_loc(method_val)

        # Plot the star on the heatmap axes.
        # Heatmap coordinates are offset by 0.5 to land perfectly in the cell center.
        g2.ax_heatmap.scatter(
            x_idx + 0.5,
            y_idx + 0.5,
            marker="*",
            color="black",
            s=10,  # Size of the star (adjust as needed)
            zorder=10,  # Ensures the star is drawn on top of the heatmap colors
        )

# 3. Save and display the figure
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.png", bbox_inches="tight", dpi=200
)
plt.show()

#### Scatterplots

In [ ]:
df_ok.columns[df_ok.columns.str.contains("fraction")]

In [ ]:
    hue_col='method'
    x_col='fraction_cohorts_bimodal'
    y_col='fraction_cohorts_zero_inflated_bimodal'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='method'
    x_col='zero_fraction_global'
    y_col='zero_fraction_by_batch_min'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### The rest plots

In [ ]:
si_df = df_ok.groupby(["strat", "imp"]).first()[["n_genes", "n_samples"]].reset_index()
si_df["label"] = si_df["strat"] + "\n" + si_df["imp"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, label in [
    (axes[0], "n_genes", "Number of genes"),
    (axes[1], "n_samples", "Number of samples"),
]:
    vals = [
        df_ok[(df_ok.strat == r.strat) & (df_ok.imp == r.imp)][col].mean()
        for _, r in si_df.iterrows()
    ]
    bar_colors = [strat_pal.get(r.strat, "#AAAAAA") for _, r in si_df.iterrows()]
    bars = ax.bar(range(len(si_df)), vals, color=bar_colors)
    ax.set_xticks(range(len(si_df)))
    ax.set_xticklabels(si_df["label"], rotation=45, ha="right", fontsize=FONT_SMALL)
    ax.set_ylabel(label, fontsize=FONT_LABEL)
    ax.set_title(f"{label} by (strat, imp)", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            f"{int(val)}",
            ha="center",
            va="bottom",
            fontsize=FONT_ANNOT,
        )
fig.tight_layout()
fig.savefig(FIGURES_DIR / "e_gene_sample_counts.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "e_gene_sample_counts.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["strat", "imp"]:
    for col in ["n_genes", "n_samples"]:
        fig = plot_multilevel_comparison(
            df_ok, col, group_col, title=f"{col} by {group_col}"
        )
        fig.savefig(
            FIGURES_DIR / f"e_{col}_multilevel_{group_col}.svg", bbox_inches="tight"
        )
        fig.savefig(
            FIGURES_DIR / f"e_{col}_multilevel_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

In [ ]:
bimod_col = "fraction_cohorts_bimodal"
zi_col = "fraction_cohorts_zero_inflated_bimodal"
if bimod_col in df_ok.columns and zi_col in df_ok.columns:
    ax = scatter_plot(
        df_ok,
        x=bimod_col,
        y=zi_col,
        hue="method",
        palette=method_pal,
        alpha=0.75,
        s=50,
        figsize=(9, 6),
        xlabel="Fraction cohorts bimodal (↑ good)",
        ylabel="Fraction cohorts zero-inflated bimodal (↑ bad)",
        title="Bimodality landscape — color=method",
        legend_outside=True,
        save_path="e_bimodality_landscape",
    )
    plt.show()

    for group_col in ["imp", "strat"]:
        for col in [bimod_col, zi_col]:
            if col not in df_ok.columns:
                continue
            fig = plot_multilevel_comparison(
                df_ok, col, group_col, title=f"{col} by {group_col}"
            )
            safe = col.replace("/", "_")
            fig.savefig(
                FIGURES_DIR / f"e_bimodal_{safe}_{group_col}.svg", bbox_inches="tight"
            )
            fig.savefig(
                FIGURES_DIR / f"e_bimodal_{safe}_{group_col}.png",
                bbox_inches="tight",
                dpi=200,
            )
            plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains("exp")]

#### Heatmaps by expression properties

In [ ]:
pivot

In [ ]:
from matplotlib.colors import LogNorm, SymLogNorm

method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)
exp_stat_cols = [
    c for c in ["exp_median", "exp_std", "exp_p01", "exp_p99"] if c in df_ok.columns
]


exp_stat_cols = [
    "exp_min",
    "exp_p01",
    "exp_p05",
    "exp_p25",
    "exp_median",
    "exp_p75",
    "exp_p95",
    "exp_p99",
    "exp_max",
    "exp_std",
]

pivot = df_ok.groupby("method")[exp_stat_cols].mean().reindex(method_order)


row_colors = pd.DataFrame(
    {
        "method": pivot.index.get_level_values(0).map(method_pal),
        "harshness": pivot.index.get_level_values(0)
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=pivot.index,
)

g_exp = sns.clustermap(
    pivot,  # .fillna(pivot.median()),
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    cmap="RdYlBu_r",
    # standard_scale=1,
    figsize=(3.5, 6),
    linewidths=0.3,
    # annot=True,
    # fmt=".2f",
    norm=SymLogNorm(linthresh=1.0, vmin=-200, vmax=1000000, base=10),
    colors_ratio=(0.05, 0.06),
)

# g_exp.ax_heatmap.set_title("Expression statistics by method")
g_exp.fig.savefig(FIGURES_DIR / "e_exp_stats_heatmap.svg", bbox_inches="tight")
g_exp.fig.savefig(FIGURES_DIR / "e_exp_stats_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
pivot["exp_median"].median()

In [ ]:
pivot

In [ ]:
# SAME FOR THE df_best attempts

In [ ]:
df_best[exp_stat_cols].reindex(method_order)

In [ ]:
df_best

In [ ]:
from matplotlib.colors import LogNorm, SymLogNorm

method_order = df_best["pcr_RNA_BATCH"].sort_values().index.tolist()

exp_stat_cols = [
    "exp_min",
    "exp_p01",
    "exp_p05",
    "exp_p25",
    "exp_median",
    "exp_p75",
    "exp_p95",
    "exp_p99",
    "exp_max",
    "exp_std",
]

row_colors = pd.DataFrame(
    {
        "strat": df_best["strat"].map(strat_pal),
        "imp": df_best["imp"].map(imp_pal),
        "method": df_best["method"].map(method_pal),
        "harshness": df_best["method"]
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=df_best.index,
)

g_exp = sns.clustermap(
    df_best[exp_stat_cols].reindex(method_order),  # .fillna(pivot.median()),
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    cmap="RdYlBu_r",
    # standard_scale=1,
    figsize=(5, 3.5),
    linewidths=0.3,
    # annot=True,
    # fmt=".2f",
    norm=SymLogNorm(linthresh=1.0, vmin=-200, vmax=1000000, base=10),
    colors_ratio=(0.05, 0.06),
)

# g_exp.ax_heatmap.set_title("Expression statistics by method")
g_exp.fig.savefig(
    FIGURES_DIR / "e_exp_stats_heatmap_best.svg", bbox_inches="tight", transparent=True
)
g_exp.fig.savefig(
    FIGURES_DIR / "e_exp_stats_heatmap_best.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
from matplotlib.colors import LogNorm, SymLogNorm

method_order = (
    df_ok.groupby("strat")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)
exp_stat_cols = [
    c for c in ["exp_median", "exp_std", "exp_p01", "exp_p99"] if c in df_ok.columns
]


exp_stat_cols = [
    "exp_min",
    "exp_p01",
    "exp_p05",
    "exp_p25",
    "exp_median",
    "exp_p75",
    "exp_p95",
    "exp_p99",
    "exp_max",
    "exp_std",
]

pivot = df_ok.groupby("strat")[exp_stat_cols].mean().reindex(method_order)


row_colors = pd.DataFrame(
    {
        "strat": pivot.index.get_level_values(0).map(strat_pal),
    },
    index=pivot.index,
)

g_exp = sns.clustermap(
    pivot,  # .fillna(pivot.median()),
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    cmap="RdYlBu_r",
    # standard_scale=1,
    figsize=(3.5, 3),
    linewidths=0.3,
    # annot=True,
    # fmt=".2f",
    norm=SymLogNorm(linthresh=1.0, vmin=-200, vmax=1000000, base=10),
    colors_ratio=(0.05, 0.06),
)

# g_exp.ax_heatmap.set_title("Expression statistics by method")
g_exp.fig.savefig(FIGURES_DIR / "e_exp_stats_heatmap_by_strat.svg", bbox_inches="tight")
g_exp.fig.savefig(
    FIGURES_DIR / "e_exp_stats_heatmap_by_strat.png", bbox_inches="tight", dpi=200
)
plt.show()

### §3.6 — Groups G and H: Graph Connectivity and Euclidean Distances

In [ ]:
    "G": ["graph_"],
    "H": ["avg_intra_", "avg_inter_", "dist_ratio_"],

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("graph_")]

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("dist_")]

#### Scatterplots

In [ ]:
    hue_col='method'
    x_col='dist_ratio_RNA_BATCH'
    y_col='dist_ratio_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### The rest plots

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

graph_cols = [c for c in df_ok.columns if c.startswith("graph_connectivity_")]

fig, axes = plt.subplots(
    1, len(graph_cols), figsize=(5 * len(graph_cols), 5), sharey=True
)
if len(graph_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, graph_cols):
    bio_label = col.replace("graph_connectivity_", "")
    bp = ax.boxplot(
        [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
        labels=method_order,
        patch_artist=True,
    )
    for patch, m in zip(bp["boxes"], method_order):
        patch.set_facecolor(method_pal.get(m, "#AAAAAA"))
        patch.set_alpha(0.7)
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
    ax.set_title(f"Graph connectivity — {bio_label}", fontsize=FONT_SMALL)
    if ax == axes[0]:
        ax.set_ylabel("Graph connectivity (↑ better)", fontsize=FONT_LABEL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "g_graph_connectivity_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "g_graph_connectivity_grid.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["method", "strat", "imp"]:
    for col in graph_cols[:2]:
        fig = plot_multilevel_comparison(
            df_ok, col, group_col, title=f"{col} by {group_col}"
        )
        safe = col.replace("/", "_")
        fig.savefig(
            FIGURES_DIR / f"g_graph_multilevel_{safe}_{group_col}.svg",
            bbox_inches="tight",
        )
        fig.savefig(
            FIGURES_DIR / f"g_graph_multilevel_{safe}_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

In [ ]:
dr_batch = ["dist_ratio_RNA_BATCH", "dist_ratio_PLATFORM_RNA"]
dr_bio = ["dist_ratio_Major_group", "dist_ratio_Diagnosis_cell_type_unified"]
dr_all = [c for c in (dr_batch + dr_bio) if c in df_ok.columns]

fig, axes = plt.subplots(1, len(dr_all), figsize=(4 * len(dr_all), 5), sharey=True)
if len(dr_all) == 1:
    axes = [axes]
for ax, col in zip(axes, dr_all):
    label = col.replace("dist_ratio_", "")
    vals = df_ok.groupby("method")[col].mean().reindex(method_order)
    ax.bar(
        range(len(method_order)),
        vals.values,
        color=[method_pal.get(m, "#AAAAAA") for m in method_order],
    )
    ax.set_xticks(range(len(method_order)))
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
    ax.set_title(f"Distance ratio — {label}", fontsize=FONT_SMALL)
    if ax == axes[0]:
        ax.set_ylabel("Distance ratio (↓ better)", fontsize=FONT_LABEL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "h_dist_ratio_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "h_dist_ratio_grid.png", bbox_inches="tight", dpi=200)
plt.show()

### §3.7 — Parallel coordinates for top-N methods


In [ ]:
[method_pal.get(m, "#AAAAAA") for m in _pc_df.method.unique()]

In [ ]:
_pc_df

In [ ]:
# ── §3.7.1 Parallel coordinates — top-N methods on key metrics ───────────────
from pandas.plotting import parallel_coordinates

display_metrics = [
    c
    for c in [
        "r2_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
        "umap_entropy_norm_RNA_BATCH",
        "ks_mean_D_RNA_BATCH",
        "n_genes",
        "graph_connectivity_Diagnosis_cell_type_unified",
        "dist_ratio_Diagnosis_cell_type_unified",
        "composite_score",
    ]
    if c in df_normed.columns or c in df_ok.columns
]

# top_methods = df_ok.nlargest(N_TOP, "composite_score")["method"].unique()
top_methods = df_best
_pc_df = (
    df_normed.reindex(df_ok[df_ok.method.isin(top_methods.method)].index)[
        [c for c in display_metrics if c in df_normed.columns]
    ]
    .assign(method=df_ok.method)
    .dropna()
)

fig, ax = plt.subplots(figsize=(14, 6))
parallel_coordinates(
    _pc_df,
    "method",
    color=[method_pal.get(m, "#AAAAAA") for m in _pc_df.method.unique()],
    ax=ax,
    linewidth=1.5,
    alpha=0.7,
)
ax.set_ylabel("Normalized score (1 = best)", fontsize=FONT_LABEL)
ax.tick_params(axis="x", labelsize=FONT_SMALL, rotation=30)
ax.tick_params(axis="y", labelsize=FONT_TICK)
ax.set_title(
    "Parallel coordinates: top-N methods across key metrics", fontsize=FONT_TITLE
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "parallel_coords_top_methods.svg", bbox_inches="tight")
fig.savefig(
    FIGURES_DIR / "parallel_coords_top_methods.png", bbox_inches="tight", dpi=200
)
plt.show()

### §3.8 — Ranking stability heatmap


In [ ]:
df_ok.strat.unique()

In [ ]:
rank_df

In [ ]:
# ── §3.8.1 Rank stability across metric subsets ───────────────────────────────
rank_variants: dict = {}
full_scores = df_ok[scoring_cols].fillna(0)
full_composite = df_normed[scoring_cols].mean(axis=1)
rank_variants["full"] = full_composite.rank(ascending=False)

for excluded_group in "ABCDEFGHIJK?":
    excl_cols = [c for c in scoring_cols if col_meta.loc[c, "group"] == excluded_group]
    remaining = [c for c in scoring_cols if c not in excl_cols]
    if not remaining:
        continue
    sub_score = df_normed[remaining].mean(axis=1)
    rank_variants[f"no_{excluded_group}"] = sub_score.rank(ascending=False)

rank_df = pd.DataFrame(rank_variants)
top20_idx = full_composite.nlargest(50).index
rank_df = rank_df.loc[rank_df.index.intersection(top20_idx)]
rank_labels = (
    df_ok.loc[rank_df.index, "method"] + " / " + df_ok.loc[rank_df.index, "strat"]
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    rank_df.set_index(rank_labels),
    annot=True,
    fmt=".0f",
    cmap="YlOrRd_r",
    ax=ax,
    annot_kws={"size": FONT_TINY},
    linewidths=0.3,
    linecolor="#DDDDDD",
    cbar_kws={"label": "Rank (lower = better)"},
)
ax.set_title("Ranking stability: removing each metric group", fontsize=FONT_TITLE)
ax.tick_params(axis="x", labelsize=FONT_SMALL)
ax.tick_params(axis="y", labelsize=FONT_SMALL)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "rank_stability_heatmap.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rank_stability_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()

### §3.9 — Bubble chart: method × metric group summary


In [ ]:
# ── §3.9.1 Bubble chart: method × metric group ────────────────────────────────

df_ok["composite_score"] = df_normed[scoring_cols].mean(axis=1)
method_order_comp = (
    df_ok.groupby("method")["composite_score"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bubble_df = pd.DataFrame(
    {
        grp: df_normed.assign(method=df_ok.method)
        .groupby("method")[[c for c in scoring_cols if col_meta.loc[c, "group"] == grp]]
        .mean()
        .mean(axis=1)
        for grp in "ABCDEFGH"
        if any(col_meta.loc[c, "group"] == grp for c in scoring_cols)
    }
)

fig, ax = plt.subplots(figsize=(4, 7))
for col_idx, grp in enumerate(bubble_df.columns):
    for row_idx, method_name in enumerate(method_order_comp):
        if method_name not in bubble_df.index:
            continue
        score = bubble_df.loc[method_name, grp]
        ax.scatter(
            col_idx,
            -row_idx,
            s=max(score * 500, 5),
            c=group_pal.get(grp, "#AAAAAA"),
            alpha=0.8,
            linewidths=0,
        )

ax.set_xticks(range(len(bubble_df.columns)))
ax.set_xticklabels([f"Group {g}" for g in bubble_df.columns], fontsize=FONT_LABEL)
ax.set_yticks(-np.arange(len(method_order_comp)))
ax.set_yticklabels(method_order_comp, fontsize=FONT_SMALL)
ax.set_title(
    "Normalized score by method × metric group (bubble size = mean score)",
    fontsize=FONT_TITLE,
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bubble_method_x_group.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "bubble_method_x_group.png", bbox_inches="tight", dpi=200)
plt.show()

--- 
## §3.10 — Group K: NA Retention and Data Completeness

Group K tracks how many genes and samples survive without NA after harmonization. Methods that produce complete expression matrices (`pct_genes_noNA = 100%`, `n_samples_allNA = 0`) are preferred. High-harshness distribution-matching methods often produce NAs when a sample has no overlap with the reference quantile range.

In [ ]:
df_ok["pct_genes_noNA"]

In [ ]:
df_ok["n_genes_noNA"] * 100 / (df_ok["pct_genes_noNA"])

#### Barplots

In [ ]:
df_long

In [ ]:
cols_main = [
    "pct_genes_noNA",
    "pct_samples_noNA",
]

x_col = "method"
metric_name_all = "pct"
cols_present = [c for c in cols_main if c in df_ok.columns]

pal_covar = dict(zip(cols_main, sns.color_palette("Set3", n_colors=2)))
# {col: annot_pal.get(col[12:], "#AAAAAA") for col in cols_main}
# group_pal = dict(zip("ABCDEFGHIJK", sns.color_palette("Set3", n_colors=11)))
df_long = df_ok[[x_col] + cols_present].melt(
    id_vars=x_col, var_name="covariate", value_name=metric_name_all
)
# v3: append "★ Best" virtual group
df_best = df_ok[df_ok["is_best"]].copy()
df_best = df_best[[x_col] + cols_present].copy()
df_best[x_col] = BEST_LABEL
df_long = pd.concat(
    [
        df_long,
        df_best.melt(id_vars=x_col, var_name="covariate", value_name=metric_name_all),
    ],
    ignore_index=True,
)
order_v3 = (
    df_long[df_long.covariate == "pct_genes_noNA"][[x_col, metric_name_all]]
    .groupby(x_col)
    .mean()
    .sort_values(metric_name_all)
    .index
)  # [::-1]


ax = bar_plot(
    df_long,
    x=x_col,
    y=metric_name_all,
    hue="covariate",
    palette=pal_covar,
    order=order_v3,
    figsize=(6, 2),
    legend_outside=True,
    xtick_rotation=90,
    errwidth=0.7,
)


# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax.get_xaxis_transform()

for i, method in enumerate(order_v3):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

    # Create a rectangle patch
    # x: i - 0.5 (centers the width=1 block directly under the tick marker)
    # y: -0.06 (starts 6% below the bottom spine)
    # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
        (i - 0.5, -0.06),
        width=1,
        height=0.04,
        transform=trans,
        clip_on=False,
        facecolor=color,
        edgecolor="none",
    )
    ax.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax.tick_params(axis="x", pad=15)
# ax.set_yscale('log')

plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.svg",
    bbox_inches="tight",
    transparent=True,
)
plt.savefig(
    FIGURES_DIR / f"{metric_name_all}_by_{x_col}_barplot.png",
    bbox_inches="tight",
    dpi=200,
)
plt.show()

#### Ordered heatmaps

In [ ]:
# column_to_show = "ilisi_mean_RNA_BATCH"
# column_to_show = "clisi_mean_Diagnosis_cell_type_unified"
# column_to_show = "tsne_centroid_disp_RNA_BATCH"
column_to_show = "pct_genes_noNA"
# column_to_show='asw_bio_norm_Diagnosis_cell_type_unified'

metric_pivot = df_ok.groupby(["strat", "method"])[column_to_show].mean().unstack()
# Sort strategies by total NA and methods by total NA
metric_pivot = metric_pivot.loc[
    metric_pivot.mean(axis=1).sort_values(ascending=False).index,
    metric_pivot.mean(axis=0).sort_values(ascending=False).index,
]

row_colors = pd.DataFrame(
    {
        "strat": [strat_pal[val] for val in metric_pivot.index],
    },
    index=metric_pivot.index,
)

col_colors = pd.DataFrame(
    {
        "method": metric_pivot.columns.get_level_values(0).map(method_pal),
        "harshness": metric_pivot.columns.get_level_values(0)
        .map(HARSHNESS_LEVEL_MAP)
        .map(HARSHNESS_PAL)
        .fillna("#AAAAAA"),
    },
    index=metric_pivot.columns,
)
# 1. Generate the clustermap
g2 = sns.clustermap(
    metric_pivot,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    col_colors=col_colors,
    cmap="RdYlGn",
    # center=0.5,
    # vmin=0,
    # vmax=1,
    linewidths=0,
    figsize=(3, 2),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=False,
    colors_ratio=(0.03, 0.06),
    # cbar=False
)
# g2.ax_cbar.set_visible(False)

# 2. Add black stars for the best combinations
# We drop duplicates just in case there are multiple runs for the same strat/method combo
df_best = df_ok[df_ok["is_best"]].copy()
best_pairs = df_best[["strat", "method"]].drop_duplicates()

for _, row in best_pairs.iterrows():
    strat_val = row["strat"]
    method_val = row["method"]

    # Check if they exist in the pivot table to avoid KeyErrors
    if strat_val in metric_pivot.index and method_val in metric_pivot.columns:
        # Get integer indices for the row (y) and column (x)
        y_idx = metric_pivot.index.get_loc(strat_val)
        x_idx = metric_pivot.columns.get_loc(method_val)

        # Plot the star on the heatmap axes.
        # Heatmap coordinates are offset by 0.5 to land perfectly in the cell center.
        g2.ax_heatmap.scatter(
            x_idx + 0.5,
            y_idx + 0.5,
            marker="*",
            color="black",
            s=10,  # Size of the star (adjust as needed)
            zorder=10,  # Ensures the star is drawn on top of the heatmap colors
        )

# 3. Save and display the figure
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.svg",
    bbox_inches="tight",
    transparent=True,
)
g2.fig.savefig(
    FIGURES_DIR / f"heatmap_{column_to_show}.png", bbox_inches="tight", dpi=200
)
plt.show()

#### Scatterplots

In [ ]:
    hue_col='method'
    x_col='pct_genes_allNA'
    y_col='pct_samples_allNA'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=method_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### The rest plots

In [ ]:
K_COLS = [
    c
    for c in [
        "n_genes_noNA",
        "pct_genes_noNA",
        "n_samples_noNA",
        "pct_samples_noNA",
        "n_genes_allNA",
        "n_samples_allNA",
        "n_na_cells",
        "pct_na_cells",
    ]
    if c in df_ok.columns
]
K_AVAIL = len(K_COLS) > 0
if not K_AVAIL:
    print("SKIP §3.10: Group K (NA retention) columns not in CSV.")
else:
    print(f"Group K columns present: {K_COLS}")
    print(
        f"Attempts with any fully-NA samples: {(df_ok.get('n_samples_allNA',pd.Series(0,index=df_ok.index)) > 0).sum()}"
    )

In [ ]:
if K_AVAIL and "pct_genes_noNA" in df_ok.columns:
    # §3.10.2 — Horizontal bar chart: pct_genes_noNA by method
    method_order_k = (
        df_ok.groupby("method")["pct_genes_noNA"]
        .mean()
        .sort_values(ascending=True)
        .index.tolist()
    )
    df_long_k = df_ok[["method", "imp", "pct_genes_noNA"]].copy()
    # v3: append "★ Best" virtual group
    df_best_k = df_best[["method", "imp", "pct_genes_noNA"]].copy()
    df_best_k["method"] = BEST_LABEL
    df_long_k = pd.concat([df_long_k, df_best_k], ignore_index=True)
    method_order_k_v3 = method_order_k + [BEST_LABEL]

    ax = bar_plot(
        df_long_k,
        x="pct_genes_noNA",
        y="method",
        hue="imp",
        palette=imp_pal,
        order=method_order_k_v3,
        orient="h",
        figsize=(9, max(7, len(method_order_k_v3) * 0.35)),
        xlabel="% genes with no NA",
        ylabel="Method",
        title="Gene completeness by method (pct_genes_noNA) — ★ Best = manually selected",
        legend_outside=True,
    )
    ax.axvline(
        80, color="red", linestyle="--", linewidth=1, alpha=0.7, label="80% threshold"
    )
    style_best_bars(ax, method_order_k_v3, orient="h")
    ax.get_figure().savefig(
        FIGURES_DIR / "k_pct_genes_nona_by_method.svg", bbox_inches="tight"
    )
    ax.get_figure().savefig(
        FIGURES_DIR / "k_pct_genes_nona_by_method.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
if K_AVAIL and "pct_na_cells" in df_ok.columns and "pct_genes_noNA" in df_ok.columns:
    # §3.10.3 — Heatmap: pct_na_cells per (method × strategy)
    pivot_na = (
        df_ok.groupby(["method", "strat"])["pct_na_cells"].mean().unstack(fill_value=0)
    )
    if not pivot_na.empty:
        fig, ax = plt.subplots(
            figsize=(max(8, pivot_na.shape[1] * 0.9), max(6, pivot_na.shape[0] * 0.4))
        )
        sns.heatmap(
            pivot_na,
            cmap="YlOrRd",
            annot=True,
            fmt=".1f",
            linewidths=0.5,
            ax=ax,
            cbar_kws={"label": "% NA cells"},
        )
        ax.set_xlabel("Strategy", fontsize=FONT_LABEL)
        ax.set_ylabel("Method", fontsize=FONT_LABEL)
        ax.set_title("Mean pct_na_cells per method × strategy", fontsize=FONT_TITLE)
        ax.tick_params(labelsize=FONT_TICK)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "k_na_cells_heatmap.svg", bbox_inches="tight")
        fig.savefig(
            FIGURES_DIR / "k_na_cells_heatmap.png", bbox_inches="tight", dpi=200
        )
        plt.show()

In [ ]:
if K_AVAIL and "pct_genes_noNA" in df_ok.columns:
    # §3.10.4 — Boxplot: pct_genes_noNA by harshness
    harshness_order = ["low", "medium", "high"]
    harshness_order_v3 = harshness_order + [BEST_LABEL]
    HARSHNESS_PAL_V3 = {**HARSHNESS_PAL, BEST_LABEL: BEST_COLOR}

    # v3: append "★ Best" virtual group
    df_best_k2 = df_best.copy()
    df_best_k2["harshness_level"] = BEST_LABEL
    data_k2_v3 = pd.concat([df_ok, df_best_k2], ignore_index=True)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(
        data=data_k2_v3,
        x="harshness_level",
        y="pct_genes_noNA",
        order=harshness_order_v3,
        palette=HARSHNESS_PAL_V3,
        width=0.6,
        showfliers=False,
        ax=ax,
    )
    sns.stripplot(
        data=data_k2_v3,
        x="harshness_level",
        y="pct_genes_noNA",
        order=harshness_order_v3,
        hue="strat",
        palette=strat_pal,
        alpha=0.5,
        s=4,
        ax=ax,
        legend=False,
    )
    # Apply hatch and thick edge to the "★ Best" box (at x-position 3)
    best_x_pos = len(harshness_order)
    for patch in ax.patches:
        if not hasattr(patch, "get_x"):
            continue
        x_center = patch.get_x() + patch.get_width() / 2
        if abs(x_center - best_x_pos) < 0.6:
            patch.set_hatch(BEST_HATCH)
            patch.set_edgecolor(BEST_EDGECOLOR)
            patch.set_linewidth(1.5)

    ax.axhline(
        80, color="red", linestyle="--", linewidth=1, alpha=0.7, label="80% threshold"
    )
    ax.set_xlabel("Harshness level", fontsize=FONT_LABEL)
    ax.set_ylabel("% genes with no NA", fontsize=FONT_LABEL)
    ax.set_title(
        "Gene completeness by normalization harshness — ★ Best = manually selected",
        fontsize=FONT_TITLE,
    )
    ax.tick_params(labelsize=FONT_TICK)
    ax.legend(fontsize=FONT_SMALL)
    sns.despine()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "k_pct_genes_nona_by_harshness.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "k_pct_genes_nona_by_harshness.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
if K_AVAIL and "n_samples_allNA" in df_ok.columns:
    # §3.10.5 — Failed-sample summary table
    fail_pivot = (
        df_ok.assign(has_allna=df_ok["n_samples_allNA"] > 0)
        .groupby(["method", "strat"])["has_allna"]
        .sum()
        .unstack(fill_value=0)
    )
    if not fail_pivot.empty:
        fail_pivot.to_csv(FIGURES_DIR / "k_failed_samples_table.csv")
        display(
            fail_pivot.style.background_gradient(
                cmap="Reds", vmin=0, vmax=fail_pivot.max().max()
            ).set_caption("Attempts with n_samples_allNA > 0 (structural NA failure)")
        )
        print(f"Total structural NA failures: {fail_pivot.values.sum()}")

#### Gene number analysis in raw datasets

In [ ]:
df_ok["gene_number"] = df_ok["n_genes_noNA"] * 100 / (df_ok["pct_genes_noNA"])
df_ok["gene_number"]

In [ ]:
df_ok["gene_number"]

In [ ]:
??sns.catplot

In [ ]:
data_to_plot = df_ok[df_ok.method == "01_raw"]


ax = bar_plot(
    data_to_plot,
    x="imp",
    y="gene_number",
    hue="method",
    palette=method_pal,
    order=[],
    figsize=(14, 5),
    ylabel="gene number",
    # title="kBET acceptance rate per covariate × method",
    legend_outside=True,
    save_path="gene_number_by_method_imp",
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
plt.show()

In [ ]:
data_to_show["gene_number"]

In [ ]:
data_to_show = df_ok[df_ok.method == "01_raw"]


strat_order = (
    data_to_show.groupby("strat")["gene_number"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=data_to_show,
    x="strat",
    y="gene_number",
    col="imp",
    kind="bar",
    height=2,
    aspect=1.2,
    order=strat_order,
    palette=strat_pal,
)
g.set_xticklabels(rotation=90, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "gene total number", fontsize=FONT_LABEL)
g.figure.savefig(FIGURES_DIR / "gene_number_by_method_imp.svg", bbox_inches="tight")
g.figure.savefig(
    FIGURES_DIR / "gene_number_by_method_imp.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
ax = bar_plot(
    df_long_kbet,
    x="method",
    y="kbet",
    hue="covariate",
    palette=kbet_pal_covar,
    order=method_order_kbet,
    figsize=(14, 5),
    ylabel="kBET acceptance rate",
    title="kBET acceptance rate per covariate × method",
    legend_outside=True,
    save_path="kbet_by_method_covariate",
)

--- 
## §3.11 — Group I: WaterMelon Score Analysis

WaterMelon score (Zolotovskaya et al. 2020, PMC7084891) measures hierarchical clustering quality using entropy. **Lower** `wm_mean_batch` = better batch mixing (batches don't cluster separately). **Higher** `wm_mean_bio` = biology clusters are well-separated. `wm_ratio_bio_batch` combines both into a single trade-off index.

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("wm")]

#### Scatterplots

In [ ]:
    hue_col='strat'
    x_col='wm_RNA_BATCH'
    y_col='wm_Diagnosis_cell_type_unified'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
    hue_col='strat'
    x_col='wm_mean_batch'
    y_col='wm_mean_bio'
    method_order_b = (
        df_ok.groupby(hue_col)[x_col]
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ax = scatter_plot(
        df_ok,
        x=x_col,
        y=y_col,
        hue=hue_col,
        #style="imp",
        palette=strat_pal,
        hue_order=method_order_b,
        alpha=0.75,
        s=10,
        figsize=(2.2, 2.2),
        legend_outside=True,
    )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x=x_col, y=y_col, s=35)
    ax.legend().set_visible(False)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.svg", bbox_inches="tight", transparent=True)
    ax.get_figure().savefig(FIGURES_DIR / f"a_scatterplot_{x_col}_vs_{y_col}_by_{hue_col}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### The rest plots

In [ ]:
WM_COLS = [c for c in df_ok.columns if c.startswith("wm_") and c != "wm_subsampled"]
WM_AVAIL = len(WM_COLS) > 0
if not WM_AVAIL:
    print(
        "SKIP §3.11: Group I (WaterMelon) columns not in CSV. "
        "Run: python run_metrics_parallel.py --skip-wm False"
    )
else:
    WM_BATCH_COLS = [
        c
        for c in WM_COLS
        if any(
            b in c
            for b in ("RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL")
        )
    ]
    WM_BIO_COLS = [
        c
        for c in WM_COLS
        if any(b in c for b in ("Major_group", "Diagnosis", "TUMOR_NORMAL"))
    ]
    WM_SUMMARY_COLS = [
        c
        for c in ["wm_mean_batch", "wm_mean_bio", "wm_ratio_bio_batch"]
        if c in df_ok.columns
    ]
    print(
        f"WM columns: {len(WM_COLS)} total, {len(WM_BATCH_COLS)} batch, {len(WM_BIO_COLS)} bio"
    )
    print(f"WM summary: {WM_SUMMARY_COLS}")

In [ ]:
if WM_AVAIL and WM_COLS:
    # §3.11.2 — WM correlation matrix
    wm_present = [c for c in WM_COLS if df_ok[c].notna().any()]
    if len(wm_present) >= 2:
        wm_corr = df_ok[wm_present].dropna(how="all").corr(method="spearman")
        wm_row_colors = pd.DataFrame(
            {
                "type": pd.Series(wm_present)
                .apply(
                    lambda c: (
                        "#8B4513"
                        if any(
                            b in c
                            for b in (
                                "RNA_BATCH",
                                "PLATFORM_RNA",
                                "RNASEQ_SOURCE",
                                "COHORT_LABEL",
                            )
                        )
                        else "#2F4F4F"
                    )
                )
                .values
            },
            index=wm_present,
        )
        g_wm = sns.clustermap(
            wm_corr,
            cmap="coolwarm",
            center=0,
            vmin=-1,
            vmax=1,
            annot=len(wm_present) <= 12,
            fmt=".2f",
            row_colors=wm_row_colors,
            col_colors=wm_row_colors,
            figsize=(max(8, len(wm_present) * 0.6), max(8, len(wm_present) * 0.6)),
        )
        g_wm.fig.suptitle(
            "WaterMelon score Spearman correlation", y=1.01, fontsize=FONT_TITLE
        )
        g_wm.fig.savefig(
            FIGURES_DIR / "i_wm_correlation_matrix.svg", bbox_inches="tight"
        )
        g_wm.fig.savefig(
            FIGURES_DIR / "i_wm_correlation_matrix.png", bbox_inches="tight", dpi=200
        )
        plt.show()

In [ ]:
if WM_AVAIL and "wm_mean_batch" in df_ok.columns:
    # §3.11.3 — WM vs local/global metrics
    wm_x = "wm_mean_batch"
    pairs = [
        ("kbet_acceptance_rate_RNA_BATCH", "kBET acceptance rate (local)"),
        ("r2_RNA_BATCH", "r² RNA_BATCH (global)"),
    ]
    valid_pairs = [(col, lbl) for col, lbl in pairs if col in df_ok.columns]
    if valid_pairs:
        fig, axes = plt.subplots(1, len(valid_pairs), figsize=(6 * len(valid_pairs), 5))
        if len(valid_pairs) == 1:
            axes = [axes]
        for ax, (y_col, y_label) in zip(axes, valid_pairs):
            plot_df = df_ok[[wm_x, y_col, "method"]].dropna()
            sns.scatterplot(
                data=plot_df,
                x=wm_x,
                y=y_col,
                hue="method",
                palette=method_pal,
                alpha=0.6,
                s=20,
                linewidth=0,
                legend=False,
                ax=ax,
            )
            if len(plot_df) >= 3:
                r, p = pearsonr(plot_df[wm_x], plot_df[y_col])
                ax.set_title(f"r={r:.2f}, p={p:.3f}", fontsize=FONT_SMALL)
            ax.set_xlabel("WM mean batch (↓ better mixing)", fontsize=FONT_LABEL)
            ax.set_ylabel(y_label, fontsize=FONT_LABEL)
            sns.despine(ax=ax)
        fig.suptitle("WaterMelon vs local/global batch metrics", fontsize=FONT_TITLE)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "i_wm_vs_local_global.svg", bbox_inches="tight")
        fig.savefig(
            FIGURES_DIR / "i_wm_vs_local_global.png", bbox_inches="tight", dpi=200
        )
        plt.show()

In [ ]:
if WM_AVAIL and "wm_mean_batch" in df_ok.columns and "wm_mean_bio" in df_ok.columns:
    # §3.11.4 — WM batch-biology tradeoff
    ax = scatter_plot(
        df_ok,
        x="wm_mean_batch",
        y="wm_mean_bio",
        hue="method",
        style="strat" if df_ok["strat"].nunique() <= 12 else None,
        palette=method_pal,
        alpha=0.65,
        s=30,
        figsize=(9, 6),
        xlabel="WM batch score (↓ better mixing)",
        ylabel="WM biology score (↑ biology preserved)",
        title="WaterMelon: batch-biology tradeoff — ★ = best approaches",
        legend_outside=True,
    )
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1]),
    ]
    ax.plot(lims, lims, "k--", linewidth=0.7, alpha=0.5, label="diagonal")
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x="wm_mean_batch", y="wm_mean_bio")
    ax.get_figure().savefig(
        FIGURES_DIR / "i_wm_batch_bio_tradeoff.svg", bbox_inches="tight"
    )
    ax.get_figure().savefig(
        FIGURES_DIR / "i_wm_batch_bio_tradeoff.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
if (
    WM_AVAIL
    and "wm_ratio_bio_batch" in df_ok.columns
    and "composite_score" in df_ok.columns
):
    # §3.11.5 — wm_ratio vs composite score
    ax = scatter_plot(
        df_ok,
        x="composite_score",
        y="wm_ratio_bio_batch",
        hue="method",
        palette=method_pal,
        alpha=0.65,
        s=25,
        figsize=(8, 6),
        xlabel="Composite score (Groups A–H, K)",
        ylabel="WM ratio bio/batch",
        title="WaterMelon ratio vs composite score — ★ = best approaches",
        legend_outside=True,
    )
    plot_df = df_ok[["composite_score", "wm_ratio_bio_batch"]].dropna()
    if len(plot_df) >= 3:
        r, p = pearsonr(plot_df["composite_score"], plot_df["wm_ratio_bio_batch"])
        ax.set_title(
            f"WM ratio vs composite — r={r:.2f}, p={p:.3f} — ★ = best approaches",
            fontsize=FONT_TITLE,
        )
    # v3: overlay best approaches as stars on same axes
    scatter_overlay_best(ax, df_best, x="composite_score", y="wm_ratio_bio_batch")
    ax.get_figure().savefig(
        FIGURES_DIR / "i_wm_ratio_vs_composite.svg", bbox_inches="tight"
    )
    ax.get_figure().savefig(
        FIGURES_DIR / "i_wm_ratio_vs_composite.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
# §3.11.6 — Optional: add WM to composite scoring
WM_SCORING_ENABLED = False  # set True to include WM in composite score

WM_SCORE_ADDITIONS = {
    "batch_mixing": {
        "wm_mean_batch": 1.5,
    },
    "bio_preservation": {
        "wm_mean_bio": 1.0,
        "wm_ratio_bio_batch": 1.0,
    },
}

if WM_SCORING_ENABLED and WM_AVAIL:
    sw_with_wm = {k: dict(v) for k, v in SCORE_WEIGHTS.items()}
    for comp, additions in WM_SCORE_ADDITIONS.items():
        sw_with_wm[comp].update(additions)
    component_df_wm, composite_wm = compute_composite_score(
        df_normed, sw_with_wm, COMPONENT_WEIGHTS
    )
    df_ok["composite_score_wm"] = composite_wm
    print("Top 10 by composite score (WITH WM):")
    print(
        df_ok.nlargest(10, "composite_score_wm")[
            ["method", "strat", "composite_score_wm", "composite_score"]
        ].to_string()
    )
else:
    print("WM scoring disabled (WM_SCORING_ENABLED=False) or WM columns absent.")

--- 
## §3.12 — Shambhala Methods: Calibration Dataset Analysis

Shambhala uses paired P/Q calibration datasets. **P-datasets** encode the reference expression signature (9 variants: P0std, ANTE, GTExAffy, NBlegacy, NBGPL570, NBKass, NBRNAseq, NBext, Oncobox). **Q-datasets** encode the target quantile mapping (2 variants: Q0std, QNBKass). This section is gated by `SHAMBHALA_AVAIL`.

In [ ]:
if not SHAMBHALA_AVAIL:
    print(
        "SKIP §3.12: no Shambhala methods in CSV. Run Shambhala benchmark and re-aggregate."
    )
else:
    df_shambhala = df_ok[df_ok["is_shambhala"]].copy()
    print(f"Shambhala runs: {len(df_shambhala)}")
    print(f"P-keys: {sorted(df_shambhala['shambhala_p_key'].unique())}")
    print(f"Q-keys: {sorted(df_shambhala['shambhala_q_key'].unique())}")

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.2 — Shambhala mini-clustermap
    shamb_sc = [c for c in scoring_cols if c in df_shambhala.columns]
    if len(shamb_sc) > 0 and len(df_shambhala) > 0:
        shamb_normed = df_shambhala[shamb_sc].fillna(df_shambhala[shamb_sc].median())
        shamb_col_colors = pd.DataFrame(
            {
                "p_key": df_shambhala["shambhala_p_key"]
                .map(shambhala_p_pal)
                .fillna("#aaaaaa"),
                "q_key": df_shambhala["shambhala_q_key"]
                .map({"Q0std": "#3d7ab5", "QNBKass": "#b03d3d"})
                .fillna("#aaaaaa"),
            },
            index=df_shambhala.index,
        )
        g_sh = sns.clustermap(
            shamb_normed.T,
            method="ward",
            metric="euclidean",
            row_colors=metric_colors.reindex(shamb_normed.columns),
            col_colors=shamb_col_colors,
            cmap="RdYlGn",
            center=0.5,
            linewidths=0,
            figsize=(max(12, len(df_shambhala) * 0.6), 20),
        )
        g_sh.fig.savefig(FIGURES_DIR / "shambhala_clustermap.svg", bbox_inches="tight")
        g_sh.fig.savefig(
            FIGURES_DIR / "shambhala_clustermap.png", bbox_inches="tight", dpi=200
        )
        plt.show()

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.3 — Ranking table
    rank_cols = [
        "composite_score",
        "score_batch_mixing",
        "score_bio_preservation",
        "r2_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
    ]
    avail_rank = ["shambhala_p_key", "shambhala_q_key"] + [
        c for c in rank_cols if c in df_shambhala.columns
    ]
    display(
        df_shambhala[avail_rank]
        .sort_values(
            (
                "composite_score"
                if "composite_score" in df_shambhala.columns
                else avail_rank[-1]
            ),
            ascending=False,
        )
        .style.background_gradient(
            cmap="RdYlGn", subset=[c for c in rank_cols if c in df_shambhala.columns]
        )
        .format({c: "{:.3f}" for c in rank_cols if c in df_shambhala.columns})
    )

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.4 — P-key impact on batch metrics
    for metric, label in [
        ("r2_RNA_BATCH", "r² RNA_BATCH (↓ better)"),
        ("kbet_acceptance_rate_RNA_BATCH", "kBET acceptance rate (↑ better)"),
    ]:
        if metric not in df_shambhala.columns:
            continue
        ax = bar_plot(
            df_shambhala,
            x="shambhala_p_key",
            y=metric,
            hue="shambhala_q_key",
            palette={"Q0std": "#3d7ab5", "QNBKass": "#b03d3d"},
            order=[
                k
                for k in SHAMBHALA_P_KEYS
                if k in df_shambhala["shambhala_p_key"].unique()
            ],
            figsize=(11, 5),
            xlabel="Shambhala P-calibration dataset",
            ylabel=label,
            title=f"Shambhala: batch removal by P-key — {metric}",
            legend_outside=True,
            save_path=f"shambhala_pkey_{metric[:15]}",
        )
        plt.show()

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.5 — P-key impact on biology
    for metric, label in [
        ("asw_bio_norm_Diagnosis_cell_type_unified", "ASW bio Diagnosis (↑ better)"),
        ("clisi_mean_Diagnosis_cell_type_unified", "cLISI Diagnosis (↑ better)"),
    ]:
        if metric not in df_shambhala.columns:
            continue
        ax = bar_plot(
            df_shambhala,
            x="shambhala_p_key",
            y=metric,
            hue="shambhala_q_key",
            palette={"Q0std": "#3d7ab5", "QNBKass": "#b03d3d"},
            order=[
                k
                for k in SHAMBHALA_P_KEYS
                if k in df_shambhala["shambhala_p_key"].unique()
            ],
            figsize=(11, 5),
            xlabel="Shambhala P-calibration dataset",
            ylabel=label,
            title=f"Shambhala: biology preservation by P-key — {metric[:35]}",
            legend_outside=True,
            save_path=f"shambhala_pkey_bio_{metric[:12]}",
        )
        plt.show()

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.6 — Q-key comparison scatter (WM or batch/bio tradeoff)
    x_col = (
        "score_batch_mixing"
        if "score_batch_mixing" in df_shambhala.columns
        else "r2_RNA_BATCH"
    )
    y_col = (
        "score_bio_preservation"
        if "score_bio_preservation" in df_shambhala.columns
        else "asw_bio_norm_Major_group"
    )
    if x_col in df_shambhala.columns and y_col in df_shambhala.columns:
        ax = scatter_plot(
            df_shambhala,
            x=x_col,
            y=y_col,
            hue="shambhala_p_key",
            style="shambhala_q_key",
            palette=shambhala_p_pal,
            hue_order=[
                k
                for k in SHAMBHALA_P_KEYS
                if k in df_shambhala["shambhala_p_key"].unique()
            ],
            s=80,
            alpha=0.85,
            figsize=(10, 7),
            xlabel=x_col,
            ylabel=y_col,
            title="Shambhala: batch-biology tradeoff by P-key and Q-key",
            legend_outside=True,
            save_path="shambhala_pq_tradeoff",
        )
        plt.show()

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.7 — Shambhala vs best non-Shambhala
    df_nonshambhala = df_ok[~df_ok["is_shambhala"]].copy()
    top5_nonshamb = (
        df_nonshambhala.groupby("method")["composite_score"]
        .mean()
        .nlargest(5)
        .index.tolist()
        if "composite_score" in df_nonshambhala.columns
        else []
    )
    compare_methods = (
        list(df_shambhala["method"].unique())
        + top5_nonshamb
        + ["10_mnn", "16_fsqn_r", "01_raw"]
    )
    df_compare = df_ok[df_ok["method"].isin(compare_methods)].copy()
    df_compare["group_label"] = df_compare["method"].apply(
        lambda m: "Shambhala" if m.startswith("shambhala_") else m
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, metric, label in [
        (axes[0], "composite_score", "Composite score"),
        (axes[1], "r2_RNA_BATCH", "r² RNA_BATCH"),
    ]:
        if metric not in df_compare.columns:
            continue
        order = (
            df_compare.groupby("group_label")[metric]
            .median()
            .sort_values(ascending=False if metric != "r2_RNA_BATCH" else True)
            .index.tolist()
        )
        sns.boxplot(
            data=df_compare,
            x="group_label",
            y=metric,
            order=order,
            palette=method_pal | {"Shambhala": SHAMBHALA_METHOD_COLOR},
            width=0.6,
            ax=ax,
        )
        sns.stripplot(
            data=df_compare,
            x="group_label",
            y=metric,
            order=order,
            color="black",
            alpha=0.4,
            s=3,
            ax=ax,
        )
        ax.set_xlabel("Method group", fontsize=FONT_LABEL)
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.tick_params(axis="x", rotation=45, labelsize=FONT_SMALL)
        sns.despine(ax=ax)
    fig.suptitle("Shambhala vs top non-Shambhala methods", fontsize=FONT_TITLE)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "shambhala_vs_best.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "shambhala_vs_best.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
if SHAMBHALA_AVAIL:
    # §3.12.8 — Shambhala batch-biology tradeoff vs all non-Shambhala
    x_col = (
        "score_batch_mixing"
        if "score_batch_mixing" in df_ok.columns
        else "r2_RNA_BATCH"
    )
    y_col = (
        "score_bio_preservation"
        if "score_bio_preservation" in df_ok.columns
        else "asw_bio_norm_Major_group"
    )
    if x_col in df_ok.columns and y_col in df_ok.columns:
        fig, ax = plt.subplots(figsize=(10, 7))
        sns.scatterplot(
            data=df_ok[~df_ok["is_shambhala"]],
            x=x_col,
            y=y_col,
            color="#aaaaaa",
            s=15,
            alpha=0.3,
            ax=ax,
            label="Other methods",
        )
        sns.scatterplot(
            data=df_shambhala,
            x=x_col,
            y=y_col,
            hue="shambhala_p_key",
            style="shambhala_q_key",
            palette=shambhala_p_pal,
            hue_order=[
                k
                for k in SHAMBHALA_P_KEYS
                if k in df_shambhala["shambhala_p_key"].unique()
            ],
            s=70,
            alpha=0.85,
            ax=ax,
        )
        ax.set_xlabel(x_col, fontsize=FONT_LABEL)
        ax.set_ylabel(y_col, fontsize=FONT_LABEL)
        ax.set_title(
            "Shambhala vs all methods: batch-biology tradeoff", fontsize=FONT_TITLE
        )
        ax.legend(
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            fontsize=FONT_LEGEND,
            frameon=False,
        )
        sns.despine()
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "shambhala_tradeoff_vs_all.svg", bbox_inches="tight")
        fig.savefig(
            FIGURES_DIR / "shambhala_tradeoff_vs_all.png", bbox_inches="tight", dpi=200
        )
        plt.show()

--- 
## §3.13 — Harshness Level Impact Analysis

**Harshness** describes how aggressively a method transforms expression values. Low-harshness methods apply a gentle global shift/scale; high-harshness methods reshape the distribution.

| Level | Color | Methods |
|---|---|---|
| Low | Green | 01_raw, 02_median_scaling, 03_limma, 04_sva, 22_tmm, 23_vst |
| Medium | Orange | 05-13, 21, 24, 27, 29-38 (ComBat, Harmony, MNN, etc.) |
| High | Red | 14-20, 25-26, 28, 33, 39, all shambhala_* variants |

**Key questions:** Does harshness level predict batch mixing vs biology preservation? Does high harshness help in RNA-seq-only strategies but hurt in mixed-platform strategies?

In [ ]:
# §3.13.2 — Harshness vs all metric groups: grouped boxplots
for group_name, group_pfxs in GROUP_MAP.items():
    group_cols_avail = [
        c
        for c in df_ok.columns
        if any(c.startswith(p) for p in group_pfxs) and c in scoring_cols
    ]
    if not group_cols_avail:
        continue
    df_long_g = df_ok[["harshness_level"] + group_cols_avail].melt(
        id_vars="harshness_level", var_name="metric", value_name="value"
    )
    g = sns.FacetGrid(
        df_long_g, col="metric", col_wrap=4, height=3, sharex=True, sharey=False
    )
    g.map_dataframe(
        sns.boxplot,
        x="harshness_level",
        y="value",
        palette=HARSHNESS_PAL,
        order=["low", "medium", "high"],
    )
    g.set_axis_labels("harshness level", "metric value")
    for axes_row in g.axes:
        if hasattr(axes_row, "__iter__"):
            for a in axes_row:
                a.tick_params(labelsize=FONT_TICK)
                a.set_xlabel(a.get_xlabel(), fontsize=FONT_LABEL)
                a.set_ylabel(a.get_ylabel(), fontsize=FONT_LABEL)
        else:
            axes_row.tick_params(labelsize=FONT_TICK)
    g.fig.suptitle(
        f"Group {group_name}: metric distributions by harshness level",
        y=1.02,
        fontsize=FONT_TITLE,
    )
    g.savefig(
        FIGURES_DIR / f"harshness_group_{group_name}_boxplots.svg", bbox_inches="tight"
    )
    g.savefig(
        FIGURES_DIR / f"harshness_group_{group_name}_boxplots.png",
        bbox_inches="tight",
        dpi=200,
    )
    plt.show()

In [ ]:
# §3.13.3 — Harshness × metric-type interaction (local vs global)
from scipy.stats import pearsonr as _pearsonr_h

local_pfxs = METRIC_TYPE_MAP.get("local_neighborhood", [])
global_pfxs = METRIC_TYPE_MAP.get("global_distance", [])

local_cols_h = [c for c in scoring_cols if any(c.startswith(p) for p in local_pfxs)]
global_cols_h = [c for c in scoring_cols if any(c.startswith(p) for p in global_pfxs)]

if local_cols_h and global_cols_h:
    df_ok["mean_local"] = df_ok[local_cols_h].mean(axis=1)
    df_ok["mean_global"] = df_ok[global_cols_h].mean(axis=1)

    ax = scatter_plot(
        df_ok,
        x="mean_global",
        y="mean_local",
        hue="harshness_level",
        style="strat",
        palette=HARSHNESS_PAL,
        hue_order=["low", "medium", "high"],
        alpha=0.6,
        s=30,
        figsize=(9, 6),
        xlabel="Mean global batch score (↓ better)",
        ylabel="Mean local batch score (↑ better)",
        title="Local vs global performance by harshness level",
        legend_outside=True,
        save_path="harshness_local_vs_global",
    )
    for lvl, grp in df_ok.groupby("harshness_level"):
        valid = grp[["mean_local", "mean_global"]].dropna()
        if len(valid) > 5:
            r, p = _pearsonr_h(valid["mean_local"], valid["mean_global"])
            print(f"  {lvl}: r={r:.3f}, p={p:.3g}")
    plt.show()
else:
    print("Skipping §3.13.3 — insufficient local/global metric columns.")

#### Barplots

In [ ]:
cols_main = [
    "ks_frac_sig_COHORT_LABEL",
    "ks_frac_sig_PLATFORM_RNA",
    "ks_frac_sig_RNA_BATCH",
]

x_col = "method"
metric_name_all = "KS_frac_sig"
cols_present = [c for c in cols_main if c in df_ok.columns]

pal_covar = {col: annot_pal.get(col[12:], "#AAAAAA") for col in cols_main}
df_long = df_ok[[x_col] + cols_present].melt(
    id_vars=x_col, var_name="covariate", value_name=metric_name_all
)
# v3: append "★ Best" virtual group
df_best = df_ok[df_ok["is_best"]].copy()
df_best = df_best[[x_col] + cols_present].copy()
df_best[x_col] = BEST_LABEL
df_long = pd.concat(
    [
        df_long,
        df_best.melt(id_vars=x_col, var_name="covariate", value_name=metric_name_all),
    ],
    ignore_index=True,
)
order_v3 = (
    df_long[df_long.covariate == "ks_frac_sig_RNA_BATCH"][[x_col, metric_name_all]]
    .groupby(x_col)
    .mean()
    .sort_values(metric_name_all)
    .index
)  # [::-1]

In [ ]:
harshness_strat_df[["strat", "pcr_RNA_BATCH"]].groupby("strat").mean().sort_values(
    by="pcr_RNA_BATCH"
)

In [ ]:
df_ok.columns[df_ok.columns.str.startswith("ks")]

In [ ]:
y_column = "ks_frac_sig_RNA_BATCH"

df_best = df_ok[df_ok["is_best"]].copy()
df_best["strat"] = "best"

df_ok_and_best = pd.concat([df_best, df_ok])
harshness_strat_df = (
    df_ok_and_best.groupby(["strat", "harshness_level"])[y_column].mean().reset_index()
)
order = (
    harshness_strat_df[["strat", y_column]]
    .groupby("strat")
    .mean()
    .sort_values(by=y_column)
    .index[::-1]
)
ax = bar_plot(
    harshness_strat_df,
    x="strat",
    y=y_column,
    hue="harshness_level",
    palette=HARSHNESS_PAL,
    hue_order=["low", "medium", "high"],
    order=order,
    estimator="mean",
    figsize=(2.5, 2),
    xlabel="Strategy",
    xtick_rotation=90,
    legend_outside=True,
    save_path=f"bar_plot_harshness_by_strategy_{y_column}",
)
plt.show()

In [ ]:
# §3.13.4 — Harshness impact by strategy (batch mixing)
harshness_strat_r2 = (
    df_ok.groupby(["strat", "harshness_level"])["r2_RNA_BATCH"].mean().reset_index()
)
if not harshness_strat_r2.empty:
    ax = bar_plot(
        harshness_strat_r2,
        x="strat",
        y="r2_RNA_BATCH",
        hue="harshness_level",
        palette=HARSHNESS_PAL,
        hue_order=["low", "medium", "high"],
        order=[s for s in ALL_STRATS if s in harshness_strat_r2["strat"].unique()],
        estimator="mean",
        figsize=(14, 5),
        xlabel="Strategy",
        ylabel="Mean r² RNA_BATCH (↓ better)",
        title="Harshness impact on batch removal by strategy",
        xtick_rotation=45,
        legend_outside=True,
        save_path="harshness_by_strategy",
    )
    plt.show()

In [ ]:
# §3.13.5 — Harshness vs batch score broken down by removal strategy
_h_batch_metrics = [
    c
    for c in [
        "r2_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
        "asw_batch_norm_RNA_BATCH",
        "ilisi_norm_RNA_BATCH",
    ]
    if c in df_ok.columns
]
if _h_batch_metrics:
    _pivot_rows = df_ok.groupby(["harshness_level", "strat"])[_h_batch_metrics].median()
    _pivot_counts = df_ok.groupby(["harshness_level", "strat"]).size().rename("n")
    _pivot_rows = _pivot_rows.join(_pivot_counts)

    for metric in _h_batch_metrics:
        _piv = _pivot_rows[metric].unstack("strat")
        _piv_n = _pivot_rows["n"].unstack("strat")
        _piv = _piv.reindex(["low", "medium", "high"])

        fig, ax = plt.subplots(figsize=(max(10, len(_piv.columns) * 1.4), 4))
        _cmap = "RdYlGn_r" if metric.startswith("r2_") else "RdYlGn"
        sns.heatmap(
            _piv,
            annot=_piv_n.reindex(_piv.index).values,
            fmt="g",
            cmap=_cmap,
            ax=ax,
            linewidths=0.5,
            linecolor="#cccccc",
            annot_kws={"size": FONT_TINY},
        )
        ax.set_title(
            f"Median {metric} by harshness × strategy  (cells = n attempts)",
            fontsize=FONT_TITLE,
        )
        ax.set_xlabel("Strategy", fontsize=FONT_LABEL)
        ax.set_ylabel("Harshness level", fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
        plt.xticks(rotation=35, ha="right")
        _fname = f"harshness_by_strategy_{metric[:20]}_heatmap"
        fig.savefig(FIGURES_DIR / f"{_fname}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{_fname}.png", bbox_inches="tight", dpi=200)
        plt.show()
else:
    print("Skipping §3.13.5 — no batch metric columns available.")

In [ ]:
# §3.13.6 — Harshness vs data quality (NA retention and gene count)
k_cols = [
    c
    for c in ["pct_genes_noNA", "pct_samples_noNA", "pct_na_cells"]
    if c in df_ok.columns
]
if k_cols:
    fig, axes = plt.subplots(1, len(k_cols), figsize=(5 * len(k_cols), 5))
    if len(k_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, k_cols):
        sns.boxplot(
            data=df_ok,
            x="harshness_level",
            y=col,
            order=["low", "medium", "high"],
            palette=HARSHNESS_PAL,
            width=0.6,
            ax=ax,
        )
        sns.stripplot(
            data=df_ok,
            x="harshness_level",
            y=col,
            order=["low", "medium", "high"],
            color="black",
            alpha=0.3,
            s=3,
            ax=ax,
        )
        ax.set_xlabel("Harshness level", fontsize=FONT_LABEL)
        ax.set_ylabel(col, fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
        sns.despine(ax=ax)
    fig.suptitle("Harshness vs data quality (NA retention)", fontsize=FONT_TITLE)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "harshness_data_quality.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "harshness_data_quality.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
# §3.13.7 — Harshness summary: composite score vs harshness level
if "composite_score" in df_ok.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metric, label in [
        (ax1, "composite_score", "Composite score (↑ better)"),
        (ax2, "r2_RNA_BATCH", "r² RNA_BATCH (↓ better)"),
    ]:
        if metric not in df_ok.columns:
            continue
        sns.violinplot(
            data=df_ok,
            x="harshness_level",
            y=metric,
            order=["low", "medium", "high"],
            palette=HARSHNESS_PAL,
            inner="box",
            ax=ax,
        )
        ax.set_xlabel("Harshness level", fontsize=FONT_LABEL)
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
        sns.despine(ax=ax)
    fig.suptitle(
        "Harshness level: composite score and batch removal", fontsize=FONT_TITLE
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "harshness_summary.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "harshness_summary.png", bbox_inches="tight", dpi=200)
    plt.show()

---
## §4 — Composite Scoring and Top-N Selection

### §4.1 — Literature review: metric weighting approaches

**Batch mixing metrics (60% weight):**
PCA R\u00b2 is the most widely reported metric in harmonization benchmarks.
Tran et al. 2020 (*Nature Methods*) used it as the primary outcome in a benchmark of 14 single-cell integration methods.
Luecken et al. 2022 (scIB benchmark, *Nature Methods*) report kBET (B\u00fcttner et al. 2019) as the primary and
ASW\_batch as the secondary batch correction score.
The LISI metric (Korsunsky et al. 2019, Harmony paper) is complementary \u2014 it operates in reduced-dimension space
and measures local neighborhood mixing.
The scIB benchmark assigns 60% of the overall score to batch correction, motivating the 0.60 component weight here.

**Biology preservation metrics (35% weight):**
ASW\_bio and graph connectivity are the standard biology preservation metrics in scIB (Luecken et al. 2022).
R\u00b2 of biology covariates (Major\_group, Diagnosis\_cell\_type\_unified) on PCA axes should remain high
after batch correction \u2014 if biology R\u00b2 collapses, the correction is over-aggressive.
The 40% biology weight in scIB directly motivates the 0.35 component weight here
(slightly down-weighted because batch correction is the primary objective).

**Data quality metrics (5% weight):**
Gene retention and bimodality are upstream properties of data preparation (not the normalization itself)
and are rarely used in published benchmarks.
The 0.05 weight reflects their role as a sanity check.
Bimodality coefficient is **positive** because bimodal gene expression distributions are biologically natural
(reflecting two cell-state populations) and methods that preserve this structure are preferred.

In [ ]:
# ── Weight rationale ──────────────────────────────────────────────────────────
# All weights are positive because df_normed already applies polarity (1=best for all).
# Weight magnitude encodes relative importance within each component.
#
# Component split (60 / 35 / 5):
#   Mirrors the scIB benchmark (Luecken et al. 2022, Nature Methods) which assigns
#   60% to batch correction and 40% to biology preservation. We use 35/5 for biology
#   vs. data quality because gene count is upstream of normalization (not a correction
#   metric) and bimodality is a sanity check rather than a primary outcome.
#
# Within batch_mixing:
#   r2_RNA_BATCH weight=3.0: primary metric in virtually all published benchmarks;
#     platform-level PCA variance is the most interpretable summary.
#   pcr_RNA_BATCH weight=2.0: variance-weighted version; more sensitive to PCs 2–10.
#   kbet, ilisi, asw_batch weight=1.5: three independent neighbor-graph metrics;
#     collectively cover local mixing at different scales.
#   cms, umap_entropy, dsc, ks weight=1.0: supporting metrics with lower reliability
#     or narrower scope.
#   PLATFORM_RNA, RNASEQ_SOURCE variants weight=0.5: secondary batch groupings;
#     correction at RNA_BATCH level subsumes platform-level correction.
#
# Within bio_preservation:
#   Diagnosis_cell_type_unified metrics > Major_group because Diagnosis is the
#   fine-grained biology relevant to FL subtyping (C1/C2/C3 × GCB/ABC).
#   r2_TUMOR_NORMAL weight=1.5: tumor vs normal is a fundamental biology axis
#     that must be preserved; loss here = complete analysis failure.
#   clisi entries are polarity-corrected by POLARITY dict (1=better); weight
#   magnitude matches the corresponding ASW entry.
#
# Within data_quality:
#   n_genes weight=1.0: log-normalized (see compute_composite_score) so
#     differences in the 3,000–25,000 range matter more than raw count.
#   fraction_cohorts_bimodal weight=1.0: bimodal distributions are expected
#     in gene expression and indicate the normalization does not over-smooth.
#   per_gene_batch_mean_cv weight=0.5: secondary check; already captured by
#     ks and r2 metrics.
SCORE_WEIGHTS = {
    "batch_mixing": {
        "r2_RNA_BATCH": 3.0,
        "pcr_RNA_BATCH": 2.0,
        "kbet_acceptance_rate_RNA_BATCH": 1.5,
        "ilisi_norm_RNA_BATCH": 1.5,
        "asw_batch_norm_RNA_BATCH": 1.5,
        "cms_fraction_mixed_RNA_BATCH": 1.0,
        "umap_entropy_norm_RNA_BATCH": 1.0,
        "dsc_RNA_BATCH": 1.0,
        "ks_mean_D_RNA_BATCH": 1.0,
        "r2_PLATFORM_RNA": 0.5,
        "r2_RNASEQ_SOURCE": 0.5,
    },
    "bio_preservation": {
        "r2_Major_group": 2.0,
        "r2_TUMOR_NORMAL": 2.0,
        "asw_bio_norm_Diagnosis_cell_type_unified": 1.5,
        "asw_bio_norm_Major_group": 1.0,
        "clisi_mean_Diagnosis_cell_type_unified": 1.5,
        "clisi_mean_Major_group": 0.5,
        "graph_connectivity_Diagnosis_cell_type_unified": 1.0,
        "graph_connectivity_Major_group": 0.5,
        "dist_ratio_Diagnosis_cell_type_unified": 1.0,
        "dist_ratio_Major_group": 0.5,
    },
    "data_quality": {
        "n_genes": 0.5,
        "fraction_cohorts_bimodal": 1.0,
        "per_gene_batch_mean_cv_RNA_BATCH": 0.5,
        "pct_genes_noNA": 0.8,
        "pct_samples_noNA": 0.5,
        "n_samples_allNA": -1.0,
    },
}

COMPONENT_WEIGHTS = {
    "batch_mixing": 0.60,
    "bio_preservation": 0.35,
    "data_quality": 0.05,
}


def compute_composite_score(df_normed_in, score_weights, component_weights):
    """
    Compute a weighted composite harmonization score from normalized metrics.

    Aggregates normalized metric columns (all in [0, 1] where 1=best) into
    component scores (batch_mixing, bio_preservation, data_quality), then
    combines them using component-level weights.

    Parameters
    ----------
    df_normed_in : pd.DataFrame
        Normalized metrics DataFrame (output of ``normalize_for_display``).
        Index = run IDs; columns include all metric columns.
    score_weights : dict[str, dict[str, float]]
        Nested dict: component name → {metric_col: weight}.
        Metrics absent from ``df_normed_in`` are silently skipped.
    component_weights : dict[str, float]
        Weight for each component in the final composite.  Need not sum to 1;
        weights are applied as-is (unnormalized).

    Returns
    -------
    component_df : pd.DataFrame
        Per-run component scores (one column per component key).
    composite : pd.Series
        Final composite score per run (weighted sum of components).
    """
    """
    Compute a weighted composite harmonization score from normalized metrics.

    Aggregates normalized metric columns (all in [0, 1] where 1=best) into
    component scores (batch_mixing, bio_preservation, data_quality), then
    combines them using component-level weights.

    Parameters
    ----------
    df_normed_in : pd.DataFrame
        Normalized metrics DataFrame (output of ``normalize_for_display``).
        Index = run IDs; columns include all metric columns.
    score_weights : dict[str, dict[str, float]]
        Nested dict: component name → {metric_col: weight}.
        Metrics absent from ``df_normed_in`` are silently skipped.
    component_weights : dict[str, float]
        Weight for each component in the final composite.  Need not sum to 1;
        weights are applied as-is (unnormalized).

    Returns
    -------
    component_df : pd.DataFrame
        Per-run component scores (one column per component key).
    composite : pd.Series
        Final composite score per run (weighted sum of components).
    """
    scores = {}
    for comp, metrics_dict in score_weights.items():
        present = {c: w for c, w in metrics_dict.items() if c in df_normed_in.columns}
        total_w = sum(present.values()) or 1.0
        comp_score = sum(df_normed_in[c] * w / total_w for c, w in present.items())
        scores[comp] = comp_score
    final = sum(
        component_weights[k] * scores[k] for k in component_weights if k in scores
    )
    return pd.DataFrame(scores), final


component_df, composite_score = compute_composite_score(
    df_normed, SCORE_WEIGHTS, COMPONENT_WEIGHTS
)
df_ok = df_ok.copy()
df_ok["composite_score"] = composite_score
df_ok["score_batch_mixing"] = component_df["batch_mixing"]
df_ok["score_bio_preservation"] = component_df["bio_preservation"]
df_ok["score_data_quality"] = component_df.get(
    "data_quality", pd.Series(0.0, index=df_ok.index)
)

print("Top 10 by composite score:")
print(
    df_ok.nlargest(10, "composite_score")[
        [
            "strat",
            "imp",
            "method",
            "composite_score",
            "score_batch_mixing",
            "score_bio_preservation",
            "r2_RNA_BATCH",
            "r2_Major_group",
        ]
    ].to_string()
)

In [ ]:
for c in scoring_cols:
    asc = POLARITY[c] == -1
    df_ok[f"rank_{c}"] = df_ok[c].rank(ascending=asc, na_option="bottom")

rank_cols = [f"rank_{c}" for c in scoring_cols if f"rank_{c}" in df_ok.columns]
df_ok["borda_score"] = df_ok[rank_cols].mean(axis=1)
print("Top 10 by Borda count (lower rank sum = better):")
print(
    df_ok.nsmallest(10, "borda_score")[
        ["strat", "imp", "method", "borda_score", "composite_score"]
    ].to_string()
)

In [ ]:
# === USER PARAMETERS ===
N_TOP = 10
SCORE_METHOD = "composite"  # "composite" | "borda" | "batch_only" | "bio_only"
# =======================

sort_map = {
    "composite": ("composite_score", False),
    "borda": ("borda_score", True),
    "batch_only": ("score_batch_mixing", False),
    "bio_only": ("score_bio_preservation", False),
}
sort_col, ascending = sort_map[SCORE_METHOD]
top_n = (
    df_ok.nsmallest(N_TOP, sort_col) if ascending else df_ok.nlargest(N_TOP, sort_col)
)

display_cols = [
    "strat",
    "imp",
    "method",
    "composite_score",
    "score_batch_mixing",
    "score_bio_preservation",
    "r2_RNA_BATCH",
    "r2_Major_group",
]
print(f"Top {N_TOP} by {SCORE_METHOD}:")
print(top_n[display_cols].to_string())

In [ ]:
rank_display = df_ok.sort_values("composite_score", ascending=False)[
    [
        "strat",
        "imp",
        "method",
        "composite_score",
        "score_batch_mixing",
        "score_bio_preservation",
        "r2_RNA_BATCH",
        "r2_Major_group",
        "kbet_acceptance_rate_RNA_BATCH",
        "n_genes",
    ]
].copy()
rank_display.insert(0, "Rank", range(1, len(rank_display) + 1))

kbet_col_name = "kbet_acceptance_rate_RNA_BATCH"
fmt_dict = {
    "composite_score": "{:.3f}",
    "score_batch_mixing": "{:.3f}",
    "score_bio_preservation": "{:.3f}",
    "r2_RNA_BATCH": "{:.3f}",
    "r2_Major_group": "{:.3f}",
    kbet_col_name: "{:.3f}",
    "n_genes": "{:.0f}",
}

styled = (
    rank_display.style.background_gradient(subset=["composite_score"], cmap="RdYlGn")
    .background_gradient(subset=["score_batch_mixing"], cmap="RdYlGn")
    .background_gradient(subset=["score_bio_preservation"], cmap="RdYlGn")
    .background_gradient(subset=["r2_RNA_BATCH"], cmap="RdYlGn_r")
    .format(fmt_dict)
    .set_caption("All harmonization attempts ranked by composite score")
)
styled

In [ ]:
def _safe_get(row, col, default=0.0):
    """
    NaN-safe scalar accessor for radar chart value extraction.

    Radar chart axes require float values in [0, 1]; pd.NA or np.nan would cause
    matplotlib to skip the polygon vertex, producing a degenerate shape. This
    helper returns ``default`` whenever the value is missing or not a number.

    Parameters
    ----------
    row : pd.Series
        One row of the metrics DataFrame.
    col : str
        Column name to access.
    default : float
        Value to return when the column is absent or NaN (default 0.0).

    Returns
    -------
    float
    """
    val = row.get(col, default)
    return float(val) if pd.notna(val) else default


radar_metrics = {
    "Batch mixing\n(R\u00b2 inv)": lambda r: 1 - _safe_get(r, "r2_RNA_BATCH"),
    "kBET": lambda r: _safe_get(r, "kbet_acceptance_rate_RNA_BATCH"),
    "KS align\n(inv)": lambda r: 1 - _safe_get(r, "ks_mean_D_RNA_BATCH"),
    "Bio preserv\n(ASW)": lambda r: _safe_get(r, "asw_bio_norm_Major_group"),
    "Graph\nconnect": lambda r: _safe_get(r, "graph_connectivity_Major_group"),
    "Data quality": lambda r: _safe_get(r, "score_data_quality"),
}

top5 = df_ok.nlargest(5, "composite_score")
labels = list(radar_metrics.keys())
N_axes = len(labels)
angles = np.linspace(0, 2 * np.pi, N_axes, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
colors = sns.color_palette("husl", 5)

for i, (run_id, row) in enumerate(top5.iterrows()):
    vals = [max(0.0, min(1.0, fn(row))) for fn in radar_metrics.values()]
    vals += vals[:1]
    ax.plot(
        angles,
        vals,
        "o-",
        linewidth=1.5,
        color=colors[i],
        label=f"{row.method} ({row.strat}/{row.imp})",
    )
    ax.fill(angles, vals, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, 1)
ax.set_title("Radar chart \u2014 Top 5 methods", size=11, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radar_top5.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "radar_top5.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
top15["clisi_mean_Diagnosis_cell_type_unified"]

In [ ]:
df_normed["ilisi_norm_RNA_BATCH"]

In [ ]:
row.name

In [ ]:
top15.strat.map(strat_pal)

In [ ]:
df_ok["composite_score_equal"] = df_normed[scoring_cols].mean(axis=1)
df_normed["composite_score_equal"] = df_normed[scoring_cols].mean(axis=1)

radar_metrics = {
    "Global batch mixing": lambda r: r.get("pcr_RNA_BATCH"),
    "Local batch mixing LISI": lambda r: r.get("ilisi_norm_RNA_BATCH"),
    "Local batch mixing kBET": lambda r: r.get("kbet_acceptance_rate_RNA_BATCH"),
    "Global biology separation": lambda r: r.get("pcr_Diagnosis_cell_type_unified"),
    "Local biology separation": lambda r: r.get(
        "clisi_mean_Diagnosis_cell_type_unified"
    ),
    "Composite score": lambda r: r.get("composite_score_equal"),
}

top15 = df_ok[df_ok["is_best"]].copy()

labels = list(radar_metrics.keys())
N_axes = len(labels)
angles = np.linspace(0, 2 * np.pi, N_axes, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"polar": True})
# colors = sns.color_palette("husl", 15)
colors = top15.strat.map(strat_pal)

top_15_normed = df_normed.reindex(top15.index)
for i, (run_id, row) in enumerate(top_15_normed.iterrows()):
    vals = [max(0.0, min(1.0, fn(row))) for fn in radar_metrics.values()]
    vals += vals[:1]
    ax.plot(
        angles,
        vals,
        "o-",
        linewidth=1.5,
        color=colors[i],
        label=row.name,
    )
    ax.fill(angles, vals, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, 1)
ax.set_title("Radar chart \u2014 Best 15 methods", size=11, pad=20)
# ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radar_best.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "radar_best.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), subplot_kw={"polar": True})
# colors = sns.color_palette("husl", 15)
colors = top15.method.map(method_pal)

top_15_normed = df_normed.reindex(top15.index)
for i, (run_id, row) in enumerate(top_15_normed.iterrows()):
    vals = [max(0.0, min(1.0, fn(row))) for fn in radar_metrics.values()]
    vals += vals[:1]
    ax.plot(
        angles,
        vals,
        "o-",
        linewidth=1.5,
        color=colors[i],
        # label=row.name,
    )
    ax.fill(angles, vals, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, 1)
# ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radar_best_by_method.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "radar_best_by_method.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
def is_pareto(x_vals, y_vals):
    """
    Identify Pareto-optimal points in a 2-D objective space.

    A point i is Pareto-optimal (non-dominated) if no other point j simultaneously
    achieves x_vals[j] >= x_vals[i] AND y_vals[j] >= y_vals[i] with at least one
    strict inequality.  Both objectives are assumed to be "higher is better".

    Parameters
    ----------
    x_vals : array-like of float
        Values of the first objective (e.g. batch mixing score).
    y_vals : array-like of float
        Values of the second objective (e.g. biology preservation score).

    Returns
    -------
    np.ndarray of bool, shape (n,)
        True at index i if point i is Pareto-optimal.
    """
    n = len(x_vals)
    pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j and x_vals[j] >= x_vals[i] and y_vals[j] >= y_vals[i]:
                if x_vals[j] > x_vals[i] or y_vals[j] > y_vals[i]:
                    pareto[i] = False
                    break
    return pareto


x_arr = df_ok["score_batch_mixing"].values
y_arr = df_ok["score_bio_preservation"].values
m_arr = df_ok["method"].values
pareto_mask = is_pareto(x_arr, y_arr)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    x_arr[~pareto_mask],
    y_arr[~pareto_mask],
    c=[method_pal.get(m, "#AAAAAA") for m in m_arr[~pareto_mask]],
    alpha=0.45,
    s=40,
)
ax.scatter(
    x_arr[pareto_mask],
    y_arr[pareto_mask],
    c=[method_pal.get(m, "#AAAAAA") for m in m_arr[pareto_mask]],
    alpha=1.0,
    s=120,
    edgecolors="black",
    linewidths=1.5,
    zorder=5,
)
for xi, yi, mi in zip(x_arr[pareto_mask], y_arr[pareto_mask], m_arr[pareto_mask]):
    ax.annotate(mi, (xi, yi), fontsize=8, xytext=(5, 3), textcoords="offset points")

ax.set_xlabel("Score \u2014 batch mixing (\u2191 better)")
ax.set_ylabel("Score \u2014 biology preservation (\u2191 better)")
ax.set_title(
    "Pareto frontier: batch correction vs biology preservation\n(larger markers = Pareto-optimal)"
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pareto_frontier.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "pareto_frontier.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Pareto-optimal methods: {sorted(set(m_arr[pareto_mask]))}")

### §4.9 — Rank correlation between key metrics


In [ ]:
top_metric_corr

In [ ]:
# ── §4.9.1 Spearman rank correlation — top scoring metrics ───────────────────
key_metrics = (
    [c for c in SCORE_WEIGHTS if c in scoring_cols]
    if "SCORE_WEIGHTS" in dir()
    else scoring_cols[:20]
)
top_metric_corr = df_ok[scoring_cols].corr(method="spearman")

g_km = sns.clustermap(
    top_metric_corr,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors.reindex(key_metrics).fillna("#AAAAAA"),
    col_colors=metric_colors.reindex(key_metrics).fillna("#AAAAAA"),
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    figsize=(16, 14),
    xticklabels=True,
    yticklabels=True,
    annot_kws={"size": FONT_TINY},
)
g_km.fig.suptitle(
    "Rank correlation between top scoring metrics", fontsize=FONT_TITLE, y=1.01
)
g_km.fig.savefig(FIGURES_DIR / "key_metric_rank_corr.svg", bbox_inches="tight")
g_km.fig.savefig(FIGURES_DIR / "key_metric_rank_corr.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# ── §4.3 Radar chart: top-N methods on key metrics ───────────────────────────
# Axes = representative metrics from each group A–H.
# Each polygon = one method from top_n, colored by method_pal.

_RADAR_METRICS = [
    m
    for m in [
        "r2_RNA_BATCH",
        "pcr_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
        "ilisi_norm_RNA_BATCH",
        "asw_batch_norm_RNA_BATCH",
        "asw_bio_norm_Major_group",
        "umap_centroid_disp_RNA_BATCH",
        "ks_mean_D_RNA_BATCH",
        "n_genes",
        "fraction_cohorts_bimodal",
        "graph_connectivity_Major_group",
        "dist_ratio_RNA_BATCH",
    ]
    if m in df_normed.columns
]

N_AXES = len(_RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N_AXES, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    [m.replace("_RNA_BATCH", "").replace("_Major_group", "") for m in _RADAR_METRICS],
    size=8,
)
ax.set_ylim(0, 1)

for _, row in top_n.iterrows():
    vals = [
        _safe_get(df_normed.loc[row.name] if row.name in df_normed.index else row, m)
        for m in _RADAR_METRICS
    ]
    vals += vals[:1]
    color = method_pal.get(row["method"], "#AAAAAA")
    ax.plot(angles, vals, linewidth=1.5, color=color, alpha=0.8)
    ax.fill(angles, vals, alpha=0.08, color=color)

handles = [
    mpatches.Patch(
        color=method_pal.get(r["method"], "#AAAAAA"),
        label=f'{r["method"]} ({r["composite_score"]:.3f})',
    )
    for _, r in top_n.iterrows()
]
ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.4, 1.15), fontsize=7)
ax.set_title(f"Top-{N_TOP} methods — radar chart (normalized scores, 1=best)", pad=20)
fig.savefig(FIGURES_DIR / "score_radar_top_n.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "score_radar_top_n.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# ── §4.4 Pareto frontier: batch mixing vs biology preservation ──────────────────────
x = df_ok["score_batch_mixing"].values
y = df_ok["score_bio_preservation"].values
pareto_mask = is_pareto(x, y)
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)
ax = scatter_plot(
    df_ok,
    x="score_batch_mixing",
    y="score_bio_preservation",
    hue="method",
    palette=method_pal,
    hue_order=method_order,
    alpha=0.4,
    s=20,
    figsize=(9, 6),
    xlabel="Batch mixing score (higher=better)",
    ylabel="Biology preservation score (higher=better)",
    title="Pareto frontier: batch mixing vs biology preservation",
    legend_outside=True,
)

ax.scatter(
    x[pareto_mask],
    y[pareto_mask],
    alpha=0.9,
    s=50,
    color="#d62728",
    label="Pareto-optimal",
    zorder=5,
)

top_mask = df_ok.index.isin(top_n.index)
ax.scatter(
    x[top_mask],
    y[top_mask],
    alpha=1.0,
    s=80,
    color="#1f77b4",
    marker="*",
    label="top-N",
    zorder=6,
)

pareto_sorted = np.argsort(x[pareto_mask])
ax.plot(
    x[pareto_mask][pareto_sorted],
    y[pareto_mask][pareto_sorted],
    color="#d62728",
    linewidth=1,
    linestyle="--",
    alpha=0.6,
)

for idx in np.where(pareto_mask)[0]:
    ax.annotate(
        df_ok.iloc[idx]["method"],
        (x[idx], y[idx]),
        fontsize=5,
        alpha=0.7,
        xytext=(2, 2),
        textcoords="offset points",
    )

ax.legend(fontsize=FONT_LEGEND)
fig = ax.get_figure()
fig.savefig(FIGURES_DIR / "score_pareto_frontier.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "score_pareto_frontier.png", bbox_inches="tight", dpi=200)
plt.show()

---
## §5 — Visual Inspection of Top Approaches - in a Separate Notebook

Download harmonized expression matrices from S3 for the top-N methods and the raw baseline, then compute PCA/UMAP/tSNE embeddings to visually verify that batch mixing and biology separation match what the metrics indicate.

# Part 2 - Gene Sets Analysis

## Gene Set Sources and Citations

| Gene set | Source | DOI / URL |
|---|---|---|
| FL transcriptomic subtypes C1/C2/C3 (FL_2025_MARKERS) | Xochelli et al. 2025, *Leukemia* | https://doi.org/10.1038/s41375-025-02603-9 |
| legacy internal GC B-cell markers (Centroblast/Centrocyte) | unpublished internal panel | superseded by the Dybkaer et al. 2015 panels |
| FL PFS prognostic signature (FL_PROG_SIGNATURES) | Pastore et al. 2019, *J Clin Oncol* | https://doi.org/10.1200/JCO.18.01545 (PMID 29475724) |
| POD24 mutation markers | Huet et al. 2018, *Blood* | https://doi.org/10.1182/blood-2018-03-837443 |
| PROGENy pathway activating genes | Schubert et al. 2018, *Nature Communications* | https://doi.org/10.1038/s41467-017-02391-6 |
| Housekeeping genes (stability reference) | Eisenberg & Levanon 2013, *Trends Genet* | https://doi.org/10.1016/j.tig.2013.05.010 |
| kBET batch metric | Büttner et al. 2019, *Nature Methods* | https://doi.org/10.1038/s41592-018-0254-1 |
| LISI / Harmony metric | Korsunsky et al. 2019, *Nature Methods* | https://doi.org/10.1038/s41592-019-0619-0 |
| scIB benchmark + ASW/kBET weighting | Luecken et al. 2022, *Nature Methods* | https://doi.org/10.1038/s41592-021-01336-8 |
| FSQN normalization | Franks et al. 2018, *Biostatistics* | https://doi.org/10.1093/biostatistics/kxx053 |

## Helper functions

In [ ]:
S3_BUCKET = "$FL_S3_BUCKET"
S3_PREFIX = "FL_batch_correction"


def load_harmonized_exp(
    strat: str, imp: str, method: str, post_rm: bool = False
) -> pd.DataFrame:
    """
    Download a harmonized expression matrix from S3.

    Parameters
    ----------
    strat, imp, method : str
    post_rm : bool

    Returns
    -------
    pd.DataFrame  samples × genes
    """
    s3 = boto3.client("s3")
    pm = "1" if post_rm else "0"
    key = f"{S3_PREFIX}/exp/{strat}__{imp}__{method}__post{pm}.tsv.gz"
    body = s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)


def load_annotation(strat: str, imp: str) -> pd.DataFrame:
    """
    Download the annotation file aligned to a given (strat, imp) preparation.

    Parameters
    ----------
    strat, imp : str

    Returns
    -------
    pd.DataFrame  samples × annotation columns
    """
    s3 = boto3.client("s3")
    key = f"{S3_PREFIX}/prepared/{strat}__{imp}__ann.tsv.gz"
    body = s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)


def align_exp_ann(
    exp: pd.DataFrame, ann: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (exp, ann) restricted to their shared sample index."""
    common = exp.index.intersection(ann.index)
    return exp.loc[common], ann.loc[common]


def compute_pca_coords(
    exp: pd.DataFrame, n_components: int = 50
) -> tuple[np.ndarray, PCA]:
    """Standardise then PCA; returns (coords, pca_obj)."""
    from sklearn.preprocessing import StandardScaler

    X = StandardScaler().fit_transform(exp.dropna(axis=1).values)
    pca = PCA(n_components=min(n_components, min(X.shape) - 1))
    coords = pca.fit_transform(X)
    return coords, pca


def compute_umap_coords(
    exp: pd.DataFrame,
    n_pcs: int = 50,
    n_neighbors: int = 30,
    min_dist: float = 0.3,
    random_state: int = 42,
) -> np.ndarray:
    """UMAP on top-N PCA coordinates."""
    import umap as umap_lib

    pca_coords, _ = compute_pca_coords(exp.dropna(axis=1), n_components=n_pcs)
    reducer = umap_lib.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=2,
        random_state=random_state,
    )
    return reducer.fit_transform(pca_coords)


def compute_tsne_coords(
    exp: pd.DataFrame,
    n_pcs: int = 50,
    perplexity: float = 30.0,
    random_state: int = 42,
) -> np.ndarray:
    """t-SNE on top-N PCA coordinates."""
    from sklearn.manifold import TSNE

    pca_coords, _ = compute_pca_coords(exp.dropna(axis=1), n_components=n_pcs)
    return TSNE(
        n_components=2, perplexity=perplexity, random_state=random_state
    ).fit_transform(pca_coords)


def _get_diag_color(label: str) -> str:
    for prefix, color in lymphoma_ontogeny_palette.items():
        if str(label).startswith(prefix):
            return color
    return "#AAAAAA"


# ── Expression / annotation cache ─────────────────────────────────────────────
# Keys: (strat, imp, method, post_rm) → dict with "exp", "ann", and embedding arrays.
# Clear with: _EXP_CACHE.clear()
_EXP_CACHE: dict = {}


def load_and_cache(
    strat: str, imp: str, method: str, post_rm: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return aligned (exp, ann), downloading from S3 only on first call.

    Parameters
    ----------
    strat, imp, method : str
    post_rm : bool

    Returns
    -------
    tuple of (exp_df, ann_df) indexed by sample ID
    """
    key = (strat, imp, method, post_rm)
    if key not in _EXP_CACHE:
        exp = load_harmonized_exp(strat, imp, method, post_rm=post_rm)
        ann = load_annotation(strat, imp)
        exp, ann = align_exp_ann(exp, ann)
        _EXP_CACHE[key] = {"exp": exp, "ann": ann}
    entry = _EXP_CACHE[key]
    return entry["exp"], entry["ann"]


def get_or_compute_pca(
    strat: str,
    imp: str,
    method: str,
    post_rm: bool = False,
    n_components: int = 50,
) -> tuple[np.ndarray, PCA]:
    """Return cached (pca_coords, pca_obj) or compute and cache them."""
    key = (strat, imp, method, post_rm)
    cache_key = f"pca_{n_components}"
    if key not in _EXP_CACHE or cache_key not in _EXP_CACHE.get(key, {}):
        exp, _ = load_and_cache(strat, imp, method, post_rm)
        result = compute_pca_coords(exp, n_components=n_components)
        _EXP_CACHE.setdefault(key, {})[cache_key] = result
    return _EXP_CACHE[key][cache_key]


def get_or_compute_umap(
    strat: str,
    imp: str,
    method: str,
    post_rm: bool = False,
    n_pcs: int = 50,
    n_neighbors: int = 30,
    min_dist: float = 0.3,
) -> np.ndarray:
    """Return cached UMAP coords or compute and cache them."""
    key = (strat, imp, method, post_rm)
    cache_key = f"umap_{n_pcs}_{n_neighbors}_{min_dist}"
    if key not in _EXP_CACHE or cache_key not in _EXP_CACHE.get(key, {}):
        exp, _ = load_and_cache(strat, imp, method, post_rm)
        coords = compute_umap_coords(
            exp, n_pcs=n_pcs, n_neighbors=n_neighbors, min_dist=min_dist
        )
        _EXP_CACHE.setdefault(key, {})[cache_key] = coords
    return _EXP_CACHE[key][cache_key]


def get_or_compute_tsne(
    strat: str,
    imp: str,
    method: str,
    post_rm: bool = False,
    n_pcs: int = 50,
    perplexity: float = 30.0,
) -> np.ndarray:
    """Return cached tSNE coords or compute and cache them."""
    key = (strat, imp, method, post_rm)
    cache_key = f"tsne_{n_pcs}_{perplexity}"
    if key not in _EXP_CACHE or cache_key not in _EXP_CACHE.get(key, {}):
        exp, _ = load_and_cache(strat, imp, method, post_rm)
        coords = compute_tsne_coords(exp, n_pcs=n_pcs, perplexity=perplexity)
        _EXP_CACHE.setdefault(key, {})[cache_key] = coords
    return _EXP_CACHE[key][cache_key]


def _color_by_col(
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    prefix_match: bool = False,
) -> pd.Series:
    """Map an annotation column to hex colors via a palette dict."""
    if not prefix_match:
        return ann_df[color_col].map(palette).fillna("#AAAAAA")

    def _match(label: str) -> str:
        if pd.isna(label):
            return "#AAAAAA"
        for prefix, color in palette.items():
            if str(label).startswith(str(prefix)):
                return color
        return "#AAAAAA"

    return ann_df[color_col].apply(_match)


def pca_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    xlabel: str = "PC1",
    ylabel: str = "PC2",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """
    Draw a single scatter plot (PCA / UMAP / tSNE) on ax.

    Parameters
    ----------
    coords : np.ndarray  shape (n_samples, 2+)
    ann_df : pd.DataFrame  annotation aligned to coords rows
    color_col : str  annotation column used for colour mapping
    palette : dict  colour dictionary
    ax : plt.Axes
    pca_obj : PCA or None  if given, overrides xlabel/ylabel with variance %
    """
    if pca_obj is not None:
        v1 = pca_obj.explained_variance_ratio_[0] * 100
        v2 = pca_obj.explained_variance_ratio_[1] * 100
        xlabel = f"PC1 ({v1:.1f}%)"
        ylabel = f"PC2 ({v2:.1f}%)"
    colors = _color_by_col(ann_df, color_col, palette, prefix_match=prefix_match)
    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=colors.values,
        s=s,
        alpha=alpha,
        linewidths=0,
        rasterized=rasterized,
    )
    ax.set_xlabel(xlabel, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_LABEL)
    ax.tick_params(labelsize=FONT_TICK)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)


def umap_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """Draw a single UMAP scatter (delegates to pca_plot with UMAP labels)."""
    pca_plot(
        coords,
        ann_df,
        color_col,
        palette,
        ax,
        title=title,
        xlabel="UMAP1",
        ylabel="UMAP2",
        s=s,
        alpha=alpha,
        prefix_match=prefix_match,
        rasterized=rasterized,
    )


def tsne_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """Draw a single tSNE scatter (delegates to pca_plot with tSNE labels)."""
    pca_plot(
        coords,
        ann_df,
        color_col,
        palette,
        ax,
        title=title,
        xlabel="tSNE1",
        ylabel="tSNE2",
        s=s,
        alpha=alpha,
        prefix_match=prefix_match,
        rasterized=rasterized,
    )


def plot_embedding_grid(
    top_ids: list[tuple],
    embedding_fn,
    plot_fn,
    color_rows: list[tuple],
    title_prefix: str = "",
    fname: str = None,
    figsize_per_col: float = 5.0,
    figsize_per_row: float = 5.0,
    post_rm: bool = False,
    **embed_kwargs,
) -> plt.Figure:
    """
    Build a (n_color_rows × n_methods) grid of embedding scatter plots.

    Parameters
    ----------
    top_ids : list of (strat, imp, method)
    embedding_fn : get_or_compute_pca | get_or_compute_umap | get_or_compute_tsne
    plot_fn : pca_plot | umap_plot | tsne_plot
    color_rows : list of (color_col, palette_dict, prefix_match)
    title_prefix : suptitle
    fname : if given, save to FIGURES_DIR/{fname}.svg and .png
    figsize_per_col, figsize_per_row : inches per panel
    post_rm : use post-rm expression variant
    """
    n_rows = len(color_rows)
    n_cols = len(top_ids)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_col * n_cols, figsize_per_row * n_rows),
        sharex="col",
        sharey="col",
        squeeze=False,
    )
    for col_idx, (strat, imp, method) in enumerate(top_ids):
        try:
            result = embedding_fn(strat, imp, method, post_rm=post_rm, **embed_kwargs)
            if isinstance(result, tuple):
                coords, pca_obj = result
            else:
                coords, pca_obj = result, None
            _, ann = load_and_cache(strat, imp, method, post_rm=post_rm)
            r2_val = df_ok.loc[
                df_ok.method.eq(method) & df_ok.strat.eq(strat) & df_ok.imp.eq(imp),
                "r2_RNA_BATCH",
            ]
            # r2_str = f" R²={r2_val.iloc[0]:.3f}" if len(r2_val) else ""
            for row_idx, (color_col, palette, prefix_match) in enumerate(color_rows):
                ax = axes[row_idx, col_idx]
                col_title = f"{method}\n{strat} {imp}" if row_idx == 0 else ""
                plot_fn(
                    coords,
                    ann,
                    color_col,
                    palette,
                    ax,
                    title=col_title,
                    s=3,
                    alpha=0.5,
                    prefix_match=prefix_match,
                    pca_obj=pca_obj,
                )
        except Exception as exc:
            for row_idx in range(n_rows):
                axes[row_idx, col_idx].text(
                    0.5,
                    0.5,
                    f"Error:\n{exc}",
                    ha="center",
                    va="center",
                    transform=axes[row_idx, col_idx].transAxes,
                    fontsize=FONT_SMALL,
                )
    # Suppress redundant tick labels for shared axes
    for ax in axes.flat:
        spec = ax.get_subplotspec()
        if not spec.is_last_row():
            ax.tick_params(labelbottom=False)
            # ax.set_xlabel("")
        if not spec.is_first_col():
            ax.tick_params(labelleft=False)
            # ax.set_ylabel("")
    if title_prefix:
        fig.suptitle(title_prefix, fontsize=FONT_TITLE + 2)
    fig.tight_layout()
    if fname:
        fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    return fig


print(f"S3 helpers + cache + plot functions defined. Cache: {len(_EXP_CACHE)} entries")

### Universal best approaches

In [ ]:
_top_ids = [
    ("10_mnn", "strict", "S0_no_removal"),
    ("10_mnn", "strict", "H_affymetrix_extended"),
    ("10_mnn", "strict", "D_malignant_only"),
    ("04_sva", "knn", "C_rnaseq_only"),
    ("04_sva", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "knn", "C_rnaseq_only"),
    ("16_fsqn_r", "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r", "strict", "C_rnaseq_only"),
    ("10_mnn", "strict", "J_ff_only"),
    ("16_fsqn_r", "strict", "J_ff_only"),
    ("04_sva", "knn", "K_ffpe_only"),
    ("04_sva", "strict", "K_ffpe_only"),
    ("04_sva", "softimpute", "K_ffpe_only"),
    ("13_fsmvn", "strict", "S0_no_removal"),
    ("33_amdbnorm", "strict", "S0_no_removal"),
]

top_n_list = [
    df_ok[
        (df_ok.method == method)
        & (df_ok.imp == imp)
        & (df_ok.strat == strat)
        & (df_ok.post_rm == False)
    ]
    for method, imp, strat in _top_ids
]
top_n = pd.concat(top_n_list)
top_n

In [ ]:
# ── §5.6 FL/GC marker gene correlation heatmap + housekeeping gene stability ──

_HK_GENES = [
    "ACTB",
    "GAPDH",
    "B2M",
    "HMBS",
    "HPRT1",
    "PPIA",
    "RPL13A",
    "RPLP0",
    "TBP",
    "YWHAZ",
    "UBC",
    "VIM",
    "LDHA",
    "PGK1",
    "SDHA",
]

_fl_genes_for_corr: list[str] = []

try:
    _method_for_corr = top_n.iloc[0]
    exp_corr, _ = load_and_cache(
        _method_for_corr.strat, _method_for_corr.imp, _method_for_corr.method
    )
    if _fl_genes_for_corr:
        _genes_avail = [g for g in _fl_genes_for_corr if g in exp_corr.columns]
        if len(_genes_avail) >= 5:
            corr_mat = exp_corr[_genes_avail].corr(method="pearson")
            fig_corr = sns.clustermap(
                corr_mat,
                cmap="coolwarm",
                vmin=-1,
                vmax=1,
                figsize=(12, 12),
                linewidths=0.3,
            )
            fig_corr.fig.suptitle(
                f"FL marker gene Pearson correlation — {_method_for_corr.method}",
                fontsize=FONT_TITLE,
            )
            fig_corr.savefig(
                FIGURES_DIR / "fl_marker_gene_corr.svg", bbox_inches="tight"
            )
            fig_corr.savefig(
                FIGURES_DIR / "fl_marker_gene_corr.png", bbox_inches="tight", dpi=200
            )
            plt.show()
        else:
            print(
                f"Too few FL genes in expression matrix ({len(_genes_avail)}); skipping Panel A."
            )
    else:
        print("FL_GENES_ALL not yet defined — run §11 first, then re-run this cell.")
except Exception as exc:
    print(f"Panel A skipped: {exc}")

_hk_rows = []
for method, imp, strat in _top_ids:
    try:
        exp_hk, ann_hk = load_and_cache(strat, imp, method)
        hk_avail = [g for g in _HK_GENES if g in exp_hk.columns]
        for g in hk_avail:
            vals = exp_hk[g]
            batch_cvs = ann_hk.groupby("RNA_BATCH").apply(
                lambda grp: vals.loc[grp.index].std()
                / (vals.loc[grp.index].mean() or np.nan)
            )
            bio_cvs = ann_hk.groupby("Major_group").apply(
                lambda grp: vals.loc[grp.index].std()
                / (vals.loc[grp.index].mean() or np.nan)
            )
            _hk_rows.append(
                {
                    "method": method,
                    "gene": g,
                    "batch_cv": batch_cvs.mean(),
                    "bio_cv": bio_cvs.mean(),
                }
            )
    except Exception:
        pass

if _hk_rows:
    hk_df = pd.DataFrame(_hk_rows)
    hk_pivot_batch = hk_df.pivot(index="gene", columns="method", values="batch_cv")
    hk_pivot_bio = hk_df.pivot(index="gene", columns="method", values="bio_cv")

    fig_hk, (ax_b, ax_bio) = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(
        hk_pivot_batch,
        ax=ax_b,
        cmap="YlOrRd",
        cbar_kws={"label": "Mean CV across RNA_BATCH"},
        annot_kws={"size": FONT_TINY},
    )
    ax_b.set_title(
        "Housekeeping gene CV — batch axis (lower=better)", fontsize=FONT_TITLE
    )
    ax_b.tick_params(labelsize=FONT_TICK)
    sns.heatmap(
        hk_pivot_bio,
        ax=ax_bio,
        cmap="YlOrRd",
        cbar_kws={"label": "Mean CV across Major_group"},
        annot_kws={"size": FONT_TINY},
    )
    ax_bio.set_title("Housekeeping gene CV — biology axis", fontsize=FONT_TITLE)
    ax_bio.tick_params(labelsize=FONT_TICK)
    fig_hk.tight_layout()
    fig_hk.savefig(FIGURES_DIR / "fl_hk_gene_cv.svg", bbox_inches="tight")
    fig_hk.savefig(FIGURES_DIR / "fl_hk_gene_cv.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("Housekeeping CV data unavailable — check S3 access.")

### §5.7 — Ridgeline plots: expression distribution per RNA_BATCH


In [ ]:
# ── §5.7.1 Ridgeline (joy) plots: stacked density curves per RNA_BATCH ───────
from scipy.stats import gaussian_kde

methods_to_plot = [
    t
    for t in [
        ("A_confirmed_bad", "strict", "16_fsqn_r"),
        ("A_confirmed_bad", "strict", "17_quantile"),
        ("A_confirmed_bad", "strict", "01_raw"),
    ]
    if any(df_ok.method.eq(t[2]) & df_ok.strat.eq(t[0]) & df_ok.imp.eq(t[1]))
]

methods_to_plot = _top_ids

if methods_to_plot:
    fig, axes = plt.subplots(
        1, len(methods_to_plot), figsize=(6 * len(methods_to_plot), 8)
    )
    if len(methods_to_plot) == 1:
        axes = [axes]
    for ax, (strat, imp, method) in zip(axes, methods_to_plot):
        try:
            exp, ann = load_and_cache(strat, imp, method)
            batches = ann["RNA_BATCH"].dropna().unique()
            flat = exp.values.ravel()
            x_range = np.linspace(
                np.nanpercentile(flat, 1), np.nanpercentile(flat, 99), 300
            )
            for i, batch in enumerate(sorted(batches)):
                sub = exp.loc[ann.RNA_BATCH == batch].values.ravel()
                sub = sub[np.isfinite(sub)]
                if len(sub) < 20:
                    continue
                kde = gaussian_kde(sub, bw_method=0.1)
                y = kde(x_range)
                offset = -i * (y.max() * 0.8)
                color = rna_batch_palette.get(batch, "#AAAAAA")
                ax.fill_between(x_range, offset, offset + y, color=color, alpha=0.6)
                ax.plot(x_range, offset + y, color="white", linewidth=0.5)
                ax.text(
                    x_range[-1],
                    offset + y.max() * 0.5,
                    batch,
                    fontsize=FONT_TINY,
                    ha="left",
                    va="center",
                )
            ax.set_title(method, fontsize=FONT_TITLE)
            ax.set_xlabel("Expression (log2)", fontsize=FONT_LABEL)
            ax.set_yticks([])
        except Exception as exc:
            ax.text(
                0.5,
                0.5,
                str(exc),
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=FONT_SMALL,
            )
    fig.suptitle(
        "Expression distribution ridgelines per RNA_BATCH", fontsize=FONT_TITLE + 2
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "ridgeline_rna_batch.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "ridgeline_rna_batch.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("No matching methods found in df_ok for ridgeline plot.")

---
## §6 — Download Gene Lists from S3

Download per-harmonization gene list JSON files and build a structured dictionary for gene-level analysis (§7–§11).

In [ ]:
import botocore.exceptions


def list_gene_files(s3_bucket: str, s3_prefix: str) -> list[str]:
    """
    List all per-harmonization gene list JSON files on S3.

    Parameters
    ----------
    s3_bucket : str
        S3 bucket name.
    s3_prefix : str
        S3 path prefix.

    Returns
    -------
    list of str
        Full S3 keys of all gene list files found.
    """
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=f"{s3_prefix}/genes/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("_genes.json"):
                keys.append(obj["Key"])
    return keys


try:
    gene_file_keys = list_gene_files(S3_BUCKET, S3_PREFIX)
    print(f"Gene list files found: {len(gene_file_keys)}")
except botocore.exceptions.ClientError as exc:
    gene_file_keys = []
    print(f"S3 access error: {exc}")

In [ ]:
def parse_gene_key(s3_key: str) -> tuple[str, str, str, bool] | None:
    """
    Parse a gene list S3 key into its (strat, imp, method, post_rm) components.

    Parameters
    ----------
    s3_key : str
        Full S3 key string.

    Returns
    -------
    tuple (strat, imp, method, post_rm) or None if the key cannot be parsed.
    """
    fname = s3_key.split("/")[-1].replace("_genes.json", "")
    parts = fname.split("__")
    if len(parts) != 4:
        return None
    strat, imp, method, pm_tag = parts
    return strat, imp, method, (pm_tag == "post1")


gene_sets: dict[tuple[str, str, str, bool], set[str]] = {}
_s3_client = boto3.client("s3")

for key in gene_file_keys:
    parsed = parse_gene_key(key)
    if parsed is None:
        continue
    try:
        body = _s3_client.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()
        genes = set(json.loads(body))
        gene_sets[parsed] = genes
    except botocore.exceptions.ClientError:
        pass

print(f"Gene sets loaded: {len(gene_sets)}")

In [ ]:
gene_df = pd.DataFrame(
    [
        {"strat": k[0], "imp": k[1], "method": k[2], "post_rm": k[3], "n_genes": len(v)}
        for k, v in gene_sets.items()
    ]
)

if len(gene_df):
    print(gene_df.groupby(["strat", "imp", "post_rm"])["n_genes"].describe())
    print("\nGene count by method:")
    print(gene_df.groupby("method")["n_genes"].mean().sort_values())
else:
    print("No gene sets loaded — check S3 access.")

In [ ]:
gene_sets.keys()

In [ ]:
# Aggregate gene sets by (strat, imp) — union across all methods except 25_angel.
# 25_angel applies its own internal gene filtering on top of the preparation step
# and is tracked separately.
gene_sets_by_strat_imp: dict[tuple[str, str], set[str]] = {}
gene_sets_by_strat_imp_raw: dict[tuple[str, str], set[str]] = {}
gene_sets_angel: dict[tuple[str, str], set[str]] = {}

for (strat, imp, method, post_rm), genes in gene_sets.items():
    if not post_rm:
        key = (strat, imp)
        if method == "25_angel":
            gene_sets_angel.setdefault(key, set()).update(genes)
        else:
            gene_sets_by_strat_imp.setdefault(key, set()).update(genes)
    if method == "01_raw":
        gene_sets_by_strat_imp_raw.setdefault(key, set()).update(genes)

print(f"(strat, imp) pairs with gene sets: {len(gene_sets_by_strat_imp)}")
print(f"Angel-specific gene sets: {len(gene_sets_angel)}")

---
## §7 — Gene Count and Overlap Analysis

How does the number of genes available for analysis change as sample removal strategy becomes more aggressive? What is the pairwise overlap between strategies and imputation methods?

In [ ]:
# ── §7.1 Gene count per (strat, imp) ─────────────────────────────────────────

if gene_sets_by_strat_imp:
    _gene_counts = pd.DataFrame(
        [
            {"strat": s, "imp": i, "n_genes": len(g)}
            for (s, i), g in gene_sets_by_strat_imp.items()
        ]
    )
    _gene_counts_a = pd.DataFrame(
        [
            {"strat": s, "imp": i, "n_genes": len(g), "method": "25_angel"}
            for (s, i), g in gene_sets_angel.items()
        ]
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    _ordered = [
        (s, i)
        for s in ALL_STRATS
        for i in ["strict", "knn", "softimpute"]
        if (s, i) in gene_sets_by_strat_imp
    ]
    _labels = [f"{s}\n{i}" for s, i in _ordered]
    _counts = [len(gene_sets_by_strat_imp[(s, i)]) for s, i in _ordered]
    _colors = [strat_pal.get(s, "#AAAAAA") for s, i in _ordered]
    ax1.bar(range(len(_ordered)), _counts, color=_colors, edgecolor="white")
    ax1.set_xticks(range(len(_ordered)))
    ax1.set_xticklabels(_labels, rotation=90, fontsize=7)
    ax1.set_ylabel("Number of genes")
    ax1.set_title("Gene count per (strat, imp) — all methods union")

    if len(_gene_counts_a):
        _a_ordered = [s for s in ALL_STRATS if (s, "strict") in gene_sets_angel]
        _a_counts = [len(gene_sets_angel.get((s, "strict"), set())) for s in _a_ordered]
        ax2.bar(
            range(len(_a_ordered)),
            _a_counts,
            color=[strat_pal.get(s, "#AAAAAA") for s in _a_ordered],
        )
        ax2.set_xticks(range(len(_a_ordered)))
        ax2.set_xticklabels(_a_ordered, rotation=45, ha="right", fontsize=8)
        ax2.set_ylabel("Number of genes")
        ax2.set_title("Gene count for 25_angel (strict imp)")

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_count_by_strat_imp.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "gene_count_by_strat_imp.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("gene_sets_by_strat_imp is empty — run §6 first.")

In [ ]:
# ── §7.2 Pairwise Jaccard overlap heatmap ─────────────────────────────────────


def jaccard_matrix(sets: list[set]) -> np.ndarray:
    """
    Compute the pairwise Jaccard similarity matrix for a list of gene sets.

    Parameters
    ----------
    sets : list of set

    Returns
    -------
    np.ndarray, shape (n, n) — symmetric Jaccard matrix, values in [0, 1].
    """
    n = len(sets)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            inter = len(sets[i] & sets[j])
            union = len(sets[i] | sets[j])
            mat[i, j] = mat[j, i] = inter / union if union > 0 else 0.0
    return mat


if gene_sets_by_strat_imp:
    _ordered_keys = [
        (s, i)
        for s in ALL_STRATS
        for i in ["strict", "knn", "softimpute"]
        if (s, i) in gene_sets_by_strat_imp
    ]
    _set_list = [gene_sets_by_strat_imp[k] for k in _ordered_keys]
    _jmat = jaccard_matrix(_set_list)
    _jlabels = [f"{s}\n{i}" for s, i in _ordered_keys]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(_jmat, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(_jlabels)))
    ax.set_yticks(range(len(_jlabels)))
    ax.set_xticklabels(_jlabels, rotation=90, fontsize=7)
    ax.set_yticklabels(_jlabels, fontsize=7)
    for i in range(len(_jlabels)):
        for j in range(len(_jlabels)):
            ax.text(j, i, f"{_jmat[i, j]:.2f}", ha="center", va="center", fontsize=5)
    plt.colorbar(im, ax=ax, label="Jaccard similarity")
    ax.set_title("Pairwise Jaccard gene set overlap by (strat, imp)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_jaccard_overlap.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "gene_jaccard_overlap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
gene_sets_by_strat_imp_raw.keys()

In [ ]:
def sensitivity_matrix(sets: list[set]) -> np.ndarray:
    """
    Compute the pairwise sensitivity matrix for a list of gene sets.

    Parameters
    ----------
    sets : list of set

    Returns
    -------
    np.ndarray, shape (n, n) — symmetric Jaccard matrix, values in [0, 1].
    """
    n = len(sets)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            inter = len(sets[i] & sets[j])
            # union = len(sets[i] | sets[j])
            mat[i, j] = mat[j, i] = inter / len(sets[i]) if len(sets[i]) > 0 else 0.0
    return mat


ordered_keys = [
    (s, i)
    for s in ALL_STRATS
    for i in ["strict", "knn", "softimpute"]
    if (s, i) in gene_sets_by_strat_imp_raw
]

set_list = [gene_sets_by_strat_imp_raw[k] for k in ordered_keys]
jmat = jaccard_matrix(set_list)
jlabels = [f"{s}__{i}" for s, i in ordered_keys]

counts_df = pd.DataFrame(
    [len(list(set_local)) for set_local in set_list],
    index=jlabels,
    columns=["gene number"],
)
counts_df["strat"] = counts_df.index.str.split("__").str[0]
counts_df["imp"] = counts_df.index.str.split("__").str[1]
jmat_df = pd.DataFrame(jmat, columns=jlabels, index=jlabels)
jmat_df

In [ ]:
counts_df

#### Gene counts barplot

In [ ]:
data_to_show = counts_df.copy()


strat_order = (
    counts_df.groupby("strat")["gene number"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=data_to_show,
    x="strat",
    y="gene number",
    col="imp",
    kind="bar",
    height=2,
    aspect=1.2,
    order=strat_order,
    palette=strat_pal,
)
g.set_xticklabels(rotation=90, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "gene total number", fontsize=FONT_LABEL)
g.figure.savefig(
    FIGURES_DIR / "gene_number_by_method_imp_unions.svg", bbox_inches="tight"
)
g.figure.savefig(
    FIGURES_DIR / "gene_number_by_method_imp_unions.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
approach_colors = pd.DataFrame(
    {
        "strat": pd.Series(
            jmat_df.index.str.split("__").str[0], index=jmat_df.index
        ).map(strat_pal),
        "imp": pd.Series(jmat_df.index.str.split("__").str[1], index=jmat_df.index).map(
            imp_pal
        ),
    },
    index=jmat_df.index,
)
approach_colors

#### Jaccard index and sensitivity clustermap

In [ ]:
g_mcorr = sns.clustermap(
    jmat_df,
    method="ward",
    metric="euclidean",
    row_colors=approach_colors,
    col_colors=approach_colors,
    cmap="RdBu_r",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(8.5, 8.5),
    xticklabels=True,
    yticklabels=True,
    # annot_kws={"size": FONT_TINY},
)

# g_mcorr.ax_heatmap.set_xticklabels(g_mcorr.ax_heatmap.get_xmajorticklabels(), fontsize=4)
# g_mcorr.ax_heatmap.set_yticklabels(g_mcorr.ax_heatmap.get_ymajorticklabels(), fontsize=4)

g_mcorr.fig.savefig(FIGURES_DIR / "jaccard_index_clustermap.svg", bbox_inches="tight")
g_mcorr.fig.savefig(
    FIGURES_DIR / "jaccard_index_clustermap.png", bbox_inches="tight", dpi=200
)
plt.show()

In [ ]:
ordered_keys

#### Supervenn

In [ ]:
# ── §7.3 Set intersection: supervenn plots ────────────────────────────────────

try:
    from supervenn import supervenn

    # Figure 1: fixed imp="strict", varying strat (how harshness shrinks gene universe)
    _sv_strats = [s for s in ALL_STRATS if (s, "strict") in gene_sets_by_strat_imp_raw]
    if len(_sv_strats) >= 2:
        _sv_sets = [gene_sets_by_strat_imp_raw[(s, "strict")] for s in _sv_strats]
        fig_sv1, ax_sv1 = plt.subplots(figsize=(14, 5))
        supervenn(_sv_sets, _sv_strats, ax=ax_sv1)
        ax_sv1.set_title("Gene overlap across strategies (imp=strict)")
        fig_sv1.tight_layout()
        fig_sv1.savefig(FIGURES_DIR / "supervenn_by_strat.svg", bbox_inches="tight")
        fig_sv1.savefig(
            FIGURES_DIR / "supervenn_by_strat.png", bbox_inches="tight", dpi=200
        )
        plt.show()
        core_genes = set.intersection(*_sv_sets)
        print(f"Core genes (all strategies): {len(core_genes)}")

    # Figure 2: fixed strat="A_confirmed_bad", varying imp
    _sv_imps = ["strict", "knn", "softimpute"]
    _sv_imp_sets = [
        gene_sets_by_strat_imp_raw.get(("A_confirmed_bad", i), set()) for i in _sv_imps
    ]
    _sv_imp_sets_nonempty = [
        (i, s) for i, s in zip(_sv_imps, _sv_imp_sets) if len(s) > 0
    ]
    if len(_sv_imp_sets_nonempty) >= 2:
        _imp_labels, _imp_sets = zip(*_sv_imp_sets_nonempty)
        fig_sv2, ax_sv2 = plt.subplots(figsize=(10, 4))
        supervenn(list(_imp_sets), list(_imp_labels), ax=ax_sv2)
        ax_sv2.set_title(
            "Gene overlap across imputation methods (strat=A_confirmed_bad)"
        )
        fig_sv2.tight_layout()
        fig_sv2.savefig(FIGURES_DIR / "supervenn_by_imp.svg", bbox_inches="tight")
        fig_sv2.savefig(
            FIGURES_DIR / "supervenn_by_imp.png", bbox_inches="tight", dpi=200
        )
        plt.show()

except ImportError:
    print("supervenn not installed — run: pip install supervenn")

#### Circos sankey plot for genes (no success)

In [ ]:
import math
import os

import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from matplotlib.path import Path


def transform(x, y, theta_deg):
    """Converts local ribbon coordinates into global radial (Cartesian) space."""
    theta = math.radians(theta_deg)
    X = y * math.cos(theta) - x * math.sin(theta)
    Y = y * math.sin(theta) + x * math.cos(theta)
    return X, Y


def draw_curved_ribbon(
    ax, y1, theta1, cx1, y2, theta2, cx2, width, color, R_OUT, alpha=0.65
):
    """Draws a fluid cubic Bezier ribbon in radial space with precise local offsets."""
    # Ensure curvature ignores the hidden base segment underneath the ring
    d = (y2 - max(y1, R_OUT)) * 0.5

    p0L = transform(cx1 - width / 2.0, y1, theta1)
    p1L = transform(cx1 - width / 2.0, max(y1, R_OUT) + d, theta1)
    p2L = transform(cx2 - width / 2.0, y2 - d, theta2)
    p3L = transform(cx2 - width / 2.0, y2, theta2)

    p0R = transform(cx1 + width / 2.0, y1, theta1)
    p1R = transform(cx1 + width / 2.0, max(y1, R_OUT) + d, theta1)
    p2R = transform(cx2 + width / 2.0, y2 - d, theta2)
    p3R = transform(cx2 + width / 2.0, y2, theta2)

    verts = [p0L, p1L, p2L, p3L, p3R, p2R, p1R, p0R, p0L]
    codes = [
        Path.MOVETO,
        Path.CURVE4,
        Path.CURVE4,
        Path.CURVE4,
        Path.LINETO,
        Path.CURVE4,
        Path.CURVE4,
        Path.CURVE4,
        Path.CLOSEPOLY,
    ]

    # zorder=2 ensures ribbons are drawn underneath the central ring
    patch = patches.PathPatch(
        Path(verts, codes), facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(patch)


def darken_color(color, factor=0.75):
    """Returns a darker version of the palette color for the terminal boxes."""
    try:
        rgb = mcolors.to_rgb(color)
        return tuple(c * factor for c in rgb)
    except:
        return color


def get_span(w, R):
    """Calculates the exact angular span (in degrees) of a ribbon at a given radius."""
    val = (w / 2.0) / R
    val = max(-1.0, min(1.0, val))  # Clamp to prevent domain errors
    return math.degrees(math.asin(val)) * 2.0


# =============================================================================
# MAIN PLOTTING FUNCTION
# =============================================================================
def create_circos_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(10, 10),
    fontsize=7.5,
    strat_pal=None,
    imp_pal=None,
):

    # Default imputation palette if not provided
    if imp_pal is None:
        imp_pal = {
            "strict": "#2979ae",
            "knn": "#982d22",
            "softimpute": "#713689",
        }

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Spatial Parameters ---
    SCALE = 0.0013  # Mapping count to visual width (adjust if gene N is much larger)
    R_IN = 5.5  # Inner radius of the ring
    R_OUT = 7.5  # Outer radius of the ring
    L1, L2, L3, L4 = 16, 26, 36, 46  # Scaled radial distances for 10x10 figsize
    RECT_H = 1.0  # Height of the strategy rectangles

    # --- Define Node Layout (Theta angles & Radii) ---
    nodes = {
        "Ring_J": {"y": R_OUT, "theta": 12},
        "Ring_K": {"y": R_OUT, "theta": 48},
        "Ring_C": {"y": R_OUT, "theta": 80},
        "Ring_F": {"y": R_OUT, "theta": 124},
        "Ring_D": {"y": R_OUT, "theta": 186},
        "Ring_I": {"y": R_OUT, "theta": 258},
        "Ring_A": {"y": R_OUT, "theta": 328},
        "J_ff_only": {"y": L1, "theta": 12},
        "K_ffpe_only": {"y": L1, "theta": 48},
        "C_rnaseq_only": {"y": L1, "theta": 80},
        "F_microarray_only": {"y": L1, "theta": 124},
        "D_malignant_only": {"y": L1, "theta": 186},
        "I_rare_batches_removed": {"y": L1, "theta": 258},
        "A_confirmed_bad": {"y": L1, "theta": 328},
        "H_affymetrix_extended": {"y": L2, "theta": 124},
        "G_affymetrix_only": {"y": L3, "theta": 124},
        "B_extended_bad": {"y": L2, "theta": 305},
        "E1_iterative_r1": {"y": L2, "theta": 350},
        "E2_iterative_r2": {"y": L3, "theta": 350},
        "E3_iterative_r3": {"y": L4, "theta": 350},
    }

    # --- Helper Functions for Sets ---
    def get_node_sets(node_name):
        """Returns the strictly extracted sets and their combined union for a given node."""
        if node_name.startswith("Ring_"):
            return set(), set(), set()
        s_strict = gene_sets_by_strat_imp_raw.get((node_name, "strict"), set())
        s_knn = gene_sets_by_strat_imp_raw.get((node_name, "knn"), set())
        s_union = s_strict | s_knn
        return s_strict, s_knn, s_union

    # Calculate union sizes to represent total node logic mapping
    counts = {name: len(get_node_sets(name)[2]) for name in nodes.keys()}

    # --- Dynamic Arcs (Perfectly Embracing the Ribbons) ---
    arcs = [
        {
            "label": "By biomaterial",
            "start": 12 - get_span(counts["J_ff_only"] * SCALE, R_OUT) / 2 - 1.5,
            "end": 48 + get_span(counts["K_ffpe_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 30,
        },
        {
            "label": "By platform",
            "start": 80 - get_span(counts["C_rnaseq_only"] * SCALE, R_OUT) / 2 - 1.5,
            "end": 124 + get_span(counts["F_microarray_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 95,
        },
        {
            "label": "By biology",
            "start": 186
            - get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2
            - 1.5,
            "end": 186 + get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 186,
        },
        {
            "label": "Bad batches\nremoval",
            "start": 258
            - get_span(counts["I_rare_batches_removed"] * SCALE, R_OUT) / 2
            - 1.5,
            "end": 328 + get_span(counts["A_confirmed_bad"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 297,
        },
    ]

    # 1. Draw Arcs and Arc Text
    ring_color = strat_pal.get("S0_no_removal", "#808080")
    for arc in arcs:
        wedge = patches.Wedge(
            (0, 0),
            R_OUT,
            arc["start"],
            arc["end"],
            width=(R_OUT - R_IN),
            facecolor=ring_color,
            edgecolor="white",
            linewidth=1.5,
            alpha=1.0,
            zorder=3,
        )
        ax.add_patch(wedge)

        th = arc["text_angle"] % 360
        ha, rot = ("right", th - 180) if 90 < th < 270 else ("left", th)

        txt_x, txt_y = transform(0, R_OUT + 0.5, th)
        ax.text(
            txt_x,
            txt_y,
            arc["label"],
            ha=ha,
            va="center",
            rotation=rot,
            fontsize=fontsize + 2,
            weight="bold",
            color="#333333",
            zorder=15,
        )

    # --- Define Flows (Parent -> Child) ---
    flows = [
        ("Ring_J", "J_ff_only"),
        ("Ring_K", "K_ffpe_only"),
        ("Ring_C", "C_rnaseq_only"),
        ("Ring_F", "F_microarray_only"),
        ("F_microarray_only", "H_affymetrix_extended"),
        ("H_affymetrix_extended", "G_affymetrix_only"),
        ("Ring_D", "D_malignant_only"),
        ("Ring_I", "I_rare_batches_removed"),
        ("Ring_A", "A_confirmed_bad"),
        ("A_confirmed_bad", "B_extended_bad"),
        ("A_confirmed_bad", "E1_iterative_r1"),
        ("E1_iterative_r1", "E2_iterative_r2"),
        ("E2_iterative_r2", "E3_iterative_r3"),
    ]

    # 2. Draw Branch Ribbons
    for parent, child in flows:
        n1, n2 = nodes[parent], nodes[child]

        # Child sets
        c_s_strict, c_s_knn, c_s_union = get_node_sets(child)
        C_str_len, C_knn_len, C_tot_len = len(c_s_strict), len(c_s_knn), len(c_s_union)

        # Center points mapping the child branches
        # (Strict is anchored to the left, KNN is anchored to the right)
        cx_C_strict = -(C_tot_len * SCALE) / 2 + (C_str_len * SCALE) / 2
        cx_C_knn = (C_tot_len * SCALE) / 2 - (C_knn_len * SCALE) / 2

        y1_draw = n1["y"] + (RECT_H / 2)
        y2_draw = n2["y"] - (RECT_H / 2)

        # Ribbons start beneath the ring for a seamless flush look
        if parent.startswith("Ring_"):
            y1_draw = R_IN
            shared_strict, shared_knn = C_str_len, C_knn_len
            cx_P_strict, cx_P_knn = cx_C_strict, cx_C_knn
        else:
            p_s_strict, p_s_knn, p_s_union = get_node_sets(parent)
            P_str_len, P_knn_len, P_tot_len = (
                len(p_s_strict),
                len(p_s_knn),
                len(p_s_union),
            )

            # Intersection logic ensuring ribbons only map to strictly shared genes
            shared_strict = len(p_s_strict & c_s_strict)
            shared_knn = len(p_s_knn & c_s_knn)

            cx_P_strict = -(P_tot_len * SCALE) / 2 + (P_str_len * SCALE) / 2
            cx_P_knn = (P_tot_len * SCALE) / 2 - (P_knn_len * SCALE) / 2

        # Draw specific branches using identical coordinates from overlaps
        if shared_strict > 0:
            draw_curved_ribbon(
                ax,
                y1_draw,
                n1["theta"],
                cx_P_strict,
                y2_draw,
                n2["theta"],
                cx_C_strict,
                width=shared_strict * SCALE,
                color=imp_pal["strict"],
                R_OUT=R_OUT,
                alpha=0.65,
            )
        if shared_knn > 0:
            draw_curved_ribbon(
                ax,
                y1_draw,
                n1["theta"],
                cx_P_knn,
                y2_draw,
                n2["theta"],
                cx_C_knn,
                width=shared_knn * SCALE,
                color=imp_pal["knn"],
                R_OUT=R_OUT,
                alpha=0.65,
            )

    # 3. Draw Strategy Rectangles & Internal Method Blocks
    for name, n in nodes.items():
        if name.startswith("Ring_"):
            continue

        s_strict, s_knn, s_union = get_node_sets(name)
        N_tot, N_str, N_knn = len(s_union), len(s_strict), len(s_knn)
        if N_tot == 0:
            continue

        w = N_tot * SCALE
        theta = n["theta"]

        # --- A. Outer Strategy Rectangle ---
        corners_local = [
            (-w / 2, n["y"] - RECT_H / 2),
            (w / 2, n["y"] - RECT_H / 2),
            (w / 2, n["y"] + RECT_H / 2),
            (-w / 2, n["y"] + RECT_H / 2),
        ]
        corners_global = [transform(lx, ly, theta) for lx, ly in corners_local]

        rect_color = darken_color(strat_pal.get(name, "#CCCCCC"), factor=0.70)
        rect = patches.Polygon(
            corners_global, facecolor=rect_color, edgecolor="none", zorder=4
        )
        ax.add_patch(rect)

        h_inner = RECT_H * 0.6  # Inner blocks are slightly smaller

        # --- B. Inner 'strict' Branch Rectangle ---
        if N_str > 0:
            w_s = N_str * SCALE
            cl_strict = [
                (-w / 2, n["y"] - h_inner / 2),
                (-w / 2 + w_s, n["y"] - h_inner / 2),
                (-w / 2 + w_s, n["y"] + h_inner / 2),
                (-w / 2, n["y"] + h_inner / 2),
            ]
            cg_strict = [transform(lx, ly, theta) for lx, ly in cl_strict]
            # alpha=0.75 enables visual mixing at the overlapping center
            rect_s = patches.Polygon(
                cg_strict,
                facecolor=imp_pal["strict"],
                edgecolor="none",
                alpha=0.75,
                zorder=5,
            )
            ax.add_patch(rect_s)

        # --- C. Inner 'knn' Branch Rectangle ---
        if N_knn > 0:
            w_k = N_knn * SCALE
            cl_knn = [
                (w / 2 - w_k, n["y"] - h_inner / 2),
                (w / 2, n["y"] - h_inner / 2),
                (w / 2, n["y"] + h_inner / 2),
                (w / 2 - w_k, n["y"] + h_inner / 2),
            ]
            cg_knn = [transform(lx, ly, theta) for lx, ly in cl_knn]
            rect_k = patches.Polygon(
                cg_knn, facecolor=imp_pal["knn"], edgecolor="none", alpha=0.75, zorder=5
            )
            ax.add_patch(rect_k)

        # Smart Horizontal Text Placement by Quadrant
        th = theta % 360
        X, Y = transform(0, n["y"], theta)

        dx = (w / 2) * abs(math.sin(math.radians(th)))
        dy = (w / 2) * abs(math.cos(math.radians(th)))
        pad = 0.8

        if 45 <= th <= 135:  # Top Quadrant
            tx, ty = X, Y + dy + pad
            ha, va = "center", "bottom"
        elif 225 <= th <= 315:  # Bottom Quadrant
            tx, ty = X, Y - dy - pad
            ha, va = "center", "top"
        elif th < 45 or th > 315:  # Right Quadrant
            tx, ty = X + dx + pad, Y
            ha, va = "left", "center"
        else:  # Left Quadrant
            tx, ty = X - dx - pad, Y
            ha, va = "right", "center"

        label = f"{name.replace('_', ' ')}\n(N={N_tot})"
        ax.text(
            tx,
            ty,
            label,
            ha=ha,
            va=va,
            rotation=0,
            fontsize=fontsize,
            color="#111111",
            zorder=10,
        )

    # --- Finalize and Save ---
    max_radius = L4 + 14
    ax.set_xlim(-max_radius, max_radius)
    ax.set_ylim(-max_radius, max_radius)

    os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/circos_sankey_strategies.svg"
    out_png = "./figures/circos_sankey_strategies.png"

    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    fig.savefig(out_png, format="png", dpi=300, bbox_inches="tight")
    print(f"Successfully saved to {out_svg}")
    plt.show()

In [ ]:
create_circos_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(10, 10),
    fontsize=7.5,
    strat_pal=strat_pal,
    imp_pal=imp_pal,
)

#### Concentric sankey plot

In [ ]:
# =============================================================================
# Helper Geometry Functions
# =============================================================================
def polar_to_cartesian(r, theta_deg):
    """Converts polar coordinates (radius, angle in degrees) to Cartesian (x, y)."""
    theta_rad = math.radians(theta_deg)
    return r * math.cos(theta_rad), r * math.sin(theta_rad)


def get_arc_points(r, th1, th2, n=30):
    """Returns a list of Cartesian coordinates along an arc."""
    angles = np.linspace(th1, th2, n)
    return [polar_to_cartesian(r, a) for a in angles]


def bezier_curve(p0, p1, p2, p3, n=50):
    """Returns points along a cubic Bezier curve."""
    t = np.linspace(0, 1, n)[:, None]
    pts = (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )
    return pts


def draw_radial_ribbon(ax, r_in, th1_s, th1_e, r_out, th2_s, th2_e, color, alpha=0.6):
    """Draws a smooth, radially curved ribbon between two concentric arcs."""
    # Control point offset relative to the radial distance between rings
    d = (r_out - r_in) * 0.4

    # Left edge Bezier (from inner start to outer start)
    p0_L = np.array(polar_to_cartesian(r_in, th1_s))
    p1_L = np.array(polar_to_cartesian(r_in + d, th1_s))
    p2_L = np.array(polar_to_cartesian(r_out - d, th2_s))
    p3_L = np.array(polar_to_cartesian(r_out, th2_s))
    curve_L = bezier_curve(p0_L, p1_L, p2_L, p3_L)

    # Outer Arc (from outer start to outer end)
    arc_outer = get_arc_points(r_out, th2_s, th2_e)

    # Right edge Bezier (from outer end to inner end)
    p0_R = np.array(polar_to_cartesian(r_out, th2_e))
    p1_R = np.array(polar_to_cartesian(r_out - d, th2_e))
    p2_R = np.array(polar_to_cartesian(r_in + d, th1_e))
    p3_R = np.array(polar_to_cartesian(r_in, th1_e))
    curve_R = bezier_curve(p0_R, p1_R, p2_R, p3_R)

    # Inner Arc (from inner end back to inner start)
    arc_inner = get_arc_points(r_in, th1_e, th1_s)

    # Stitch everything together into a polygon
    verts = np.vstack([curve_L, arc_outer, curve_R, arc_inner])
    poly = patches.Polygon(
        verts, facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(poly)


def get_text_rotation(theta):
    """Keeps text upright regardless of its position on the circle."""
    th = theta % 360
    if 90 < th < 270:
        return th - 180, "right"
    return th, "left"


# =============================================================================
# MAIN PLOTTING FUNCTION
# =============================================================================
def create_concentric_sankey(
    gene_sets_by_strat_imp_raw, figsize=(14, 14), strat_pal=None, imp_pal=None
):

    if imp_pal is None:
        imp_pal = {"strict": "#2979ae", "knn": "#982d22"}

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Structural Parameters ---
    SCALE = 0.0055  # Degrees per gene. Adjust if arcs overlap globally.
    R_S0 = 2.5  # Radius of the central blue hole
    R_WIDTH = 2.0  # Thickness of the strategy arcs
    R_GAP = 1.8  # Gap between concentric levels

    # --- Define Node Layout ---
    # Levels dictate radial distance. Theta defines the center angle of the arc.
    nodes = {
        "J_ff_only": {"level": 1, "theta": 30},
        "K_ffpe_only": {"level": 1, "theta": 70},
        "C_rnaseq_only": {"level": 1, "theta": 120},
        "F_microarray_only": {"level": 1, "theta": 160},
        "H_affymetrix_extended": {"level": 2, "theta": 160},
        "G_affymetrix_only": {"level": 3, "theta": 160},
        "D_malignant_only": {"level": 1, "theta": 225},
        "I_rare_batches_removed": {"level": 1, "theta": 295},
        "A_confirmed_bad": {"level": 1, "theta": 345},
        "B_extended_bad": {"level": 2, "theta": 325},
        "E1_iterative_r1": {"level": 2, "theta": 365},  # Overlaps zero gracefully
        "E2_iterative_r2": {"level": 3, "theta": 365},
        "E3_iterative_r3": {"level": 4, "theta": 365},
    }

    flows = [
        ("S0", "J_ff_only"),
        ("S0", "K_ffpe_only"),
        ("S0", "C_rnaseq_only"),
        ("S0", "F_microarray_only"),
        ("F_microarray_only", "H_affymetrix_extended"),
        ("H_affymetrix_extended", "G_affymetrix_only"),
        ("S0", "D_malignant_only"),
        ("S0", "I_rare_batches_removed"),
        ("S0", "A_confirmed_bad"),
        ("A_confirmed_bad", "B_extended_bad"),
        ("A_confirmed_bad", "E1_iterative_r1"),
        ("E1_iterative_r1", "E2_iterative_r2"),
        ("E2_iterative_r2", "E3_iterative_r3"),
    ]

    # --- Helper: Extract Sets ---
    def get_node_sets(node_name):
        if node_name == "S0":
            return set(), set(), set()
        s_strict = gene_sets_by_strat_imp_raw.get((node_name, "strict"), set())
        s_knn = gene_sets_by_strat_imp_raw.get((node_name, "knn"), set())
        return s_strict, s_knn, s_strict | s_knn

    # Calculate bounding angles for all nodes
    node_angles = {}
    for name, config in nodes.items():
        _, _, s_union = get_node_sets(name)
        span = len(s_union) * SCALE
        node_angles[name] = {
            "start": config["theta"] - span / 2,
            "end": config["theta"] + span / 2,
            "span": span,
            "r_in": R_S0 + (config["level"] - 1) * (R_WIDTH + R_GAP) + R_GAP,
            "r_out": R_S0 + (config["level"] - 1) * (R_WIDTH + R_GAP) + R_GAP + R_WIDTH,
        }

    # --- 1. Draw Target Background (Concentric Rings) ---
    max_level = max(c["level"] for c in nodes.values())
    for lvl in range(1, max_level + 1):
        r_target_in = R_S0 + (lvl - 1) * (R_WIDTH + R_GAP) + R_GAP
        r_target_out = r_target_in + R_WIDTH
        # Draw faint dashed circles to enhance the "target" look
        circle_in = patches.Circle(
            (0, 0),
            r_target_in,
            fill=False,
            edgecolor="#DDDDDD",
            linestyle="--",
            linewidth=1,
            zorder=0,
        )
        circle_out = patches.Circle(
            (0, 0),
            r_target_out,
            fill=False,
            edgecolor="#DDDDDD",
            linestyle="--",
            linewidth=1,
            zorder=0,
        )
        ax.add_patch(circle_in)
        ax.add_patch(circle_out)

    # --- 2. Draw Central S0 Ring and Major Sector Labels ---
    s0_color = strat_pal.get("S0_no_removal", "#abcdef")
    s0_ring = patches.Wedge(
        (0, 0),
        R_S0,
        0,
        360,
        width=0.8,
        facecolor=s0_color,
        edgecolor="white",
        lw=1.5,
        zorder=5,
    )
    ax.add_patch(s0_ring)

    sectors = [
        ("By biomaterial", 50),
        ("By platform", 140),
        ("By biology", 225),
        ("Bad batches\nremoval", 320),
    ]
    for label, th in sectors:
        lx, ly = polar_to_cartesian(R_S0 - 1.2, th)
        rot, ha = get_text_rotation(th)
        ax.text(
            lx,
            ly,
            label,
            ha="center",
            va="center",
            rotation=rot,
            weight="bold",
            fontsize=9,
            color="#333",
        )

    # --- 3. Draw Connecting Ribbons ---
    for parent, child in flows:
        c_ang = node_angles[child]

        if parent == "S0":
            # Flow straight out of S0 matching the child's exact angular footprint
            draw_radial_ribbon(
                ax,
                R_S0,
                c_ang["start"],
                c_ang["end"],
                c_ang["r_in"],
                c_ang["start"],
                c_ang["end"],
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )
        else:
            p_ang = node_angles[parent]
            p_strict, p_knn, _ = get_node_sets(parent)
            c_strict, c_knn, _ = get_node_sets(child)

            # Ribbon width is strictly proportional to inherited genes
            shared = len((p_strict | p_knn) & (c_strict | c_knn))
            shared_span = shared * SCALE

            # Center the ribbon out of the parent and into the child
            p_mid = (p_ang["start"] + p_ang["end"]) / 2
            c_mid = (c_ang["start"] + c_ang["end"]) / 2

            draw_radial_ribbon(
                ax,
                p_ang["r_out"],
                p_mid - shared_span / 2,
                p_mid + shared_span / 2,
                c_ang["r_in"],
                c_mid - shared_span / 2,
                c_mid + shared_span / 2,
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 4. Draw Strategy Arcs and Nested Imputation Arcs ---
    for name, config in nodes.items():
        s_strict, s_knn, s_union = get_node_sets(name)
        if not s_union:
            continue

        ang = node_angles[name]
        strat_c = strat_pal.get(name, "#CCCCCC")

        # Outer Base Rectangle (The Strategy Background)
        base_wedge = patches.Wedge(
            (0, 0),
            ang["r_out"],
            ang["start"],
            ang["end"],
            width=R_WIDTH,
            facecolor=strat_c,
            edgecolor="white",
            lw=0.5,
            zorder=6,
        )
        ax.add_patch(base_wedge)

        # Strategy Label (Removed N counts per instructions)
        lx, ly = polar_to_cartesian(ang["r_out"] + 0.3, config["theta"])
        rot, ha = get_text_rotation(config["theta"])
        ax.text(
            lx,
            ly,
            name.replace("_", " "),
            ha=ha,
            va="center",
            rotation=rot,
            fontsize=8,
            weight="bold",
            color="#222",
        )

        # --- Nested Imputation Arcs ---
        # Draw STRICT strictly on the inner half of the arc
        if s_strict:
            strict_span = len(s_strict) * SCALE
            w_strict = patches.Wedge(
                (0, 0),
                ang["r_in"] + R_WIDTH * 0.45,
                ang["start"],
                ang["start"] + strict_span,
                width=R_WIDTH * 0.4,
                facecolor=imp_pal["strict"],
                zorder=7,
            )
            ax.add_patch(w_strict)

            # Text inside strict
            if len(s_strict) > 500:  # Only draw text if arc is large enough
                tx, ty = polar_to_cartesian(
                    ang["r_in"] + R_WIDTH * 0.25, ang["start"] + strict_span / 2
                )
                t_rot, _ = get_text_rotation(ang["start"] + strict_span / 2)
                ax.text(
                    tx,
                    ty,
                    f"{len(s_strict)}",
                    ha="center",
                    va="center",
                    rotation=t_rot,
                    fontsize=6,
                    color="white",
                )

        # Draw KNN strictly on the outer half of the arc, anchoring to the right side
        if s_knn:
            knn_span = len(s_knn) * SCALE
            w_knn = patches.Wedge(
                (0, 0),
                ang["r_out"] - R_WIDTH * 0.05,
                ang["end"] - knn_span,
                ang["end"],
                width=R_WIDTH * 0.4,
                facecolor=imp_pal["knn"],
                zorder=7,
            )
            ax.add_patch(w_knn)

            # Text inside knn
            if len(s_knn) > 500:
                tx, ty = polar_to_cartesian(
                    ang["r_out"] - R_WIDTH * 0.25, ang["end"] - knn_span / 2
                )
                t_rot, _ = get_text_rotation(ang["end"] - knn_span / 2)
                ax.text(
                    tx,
                    ty,
                    f"{len(s_knn)}",
                    ha="center",
                    va="center",
                    rotation=t_rot,
                    fontsize=6,
                    color="white",
                )

    # --- Finalize ---
    ax.set_xlim(-18, 18)
    ax.set_ylim(-18, 18)

    # os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/concentric_sankey_strategies.svg"
    out_png = "./figures/concentric_sankey_strategies.png"

    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    fig.savefig(out_png, format="png", dpi=300, bbox_inches="tight")
    print(f"Successfully saved to {out_svg}")
    plt.show()

In [ ]:
create_concentric_sankey(
    gene_sets_by_strat_imp_raw, figsize=(14, 14), strat_pal=strat_pal, imp_pal=imp_pal
)

In [ ]:
# =============================================================================
# Helper Geometry & Drawing Functions
# =============================================================================
def polar_to_cartesian(r, theta_deg):
    """Converts polar coordinates (radius, angle in degrees) to Cartesian."""
    theta_rad = math.radians(theta_deg)
    return r * math.cos(theta_rad), r * math.sin(theta_rad)


def get_arc_points(r, th1, th2, n=30):
    """Returns a list of Cartesian coordinates along an arc."""
    angles = np.linspace(th1, th2, n)
    return [polar_to_cartesian(r, a) for a in angles]


def bezier_curve(p0, p1, p2, p3, n=50):
    """Returns points along a cubic Bezier curve."""
    t = np.linspace(0, 1, n)[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def draw_radial_ribbon(ax, r_in, th1_s, th1_e, r_out, th2_s, th2_e, color, alpha=0.6):
    """Draws a smooth, radially curved ribbon between two concentric arcs."""
    d = (r_out - r_in) * 0.4

    p0_L = np.array(polar_to_cartesian(r_in, th1_s))
    p1_L = np.array(polar_to_cartesian(r_in + d, th1_s))
    p2_L = np.array(polar_to_cartesian(r_out - d, th2_s))
    p3_L = np.array(polar_to_cartesian(r_out, th2_s))
    curve_L = bezier_curve(p0_L, p1_L, p2_L, p3_L)

    arc_outer = get_arc_points(r_out, th2_s, th2_e)

    p0_R = np.array(polar_to_cartesian(r_out, th2_e))
    p1_R = np.array(polar_to_cartesian(r_out - d, th2_e))
    p2_R = np.array(polar_to_cartesian(r_in + d, th1_e))
    p3_R = np.array(polar_to_cartesian(r_in, th1_e))
    curve_R = bezier_curve(p0_R, p1_R, p2_R, p3_R)

    arc_inner = get_arc_points(r_in, th1_e, th1_s)

    verts = np.vstack([curve_L, arc_outer, curve_R, arc_inner])
    poly = patches.Polygon(
        verts, facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(poly)


def get_text_rotation(theta):
    """Keeps text upright regardless of its position on the circle."""
    th = theta % 360
    if 90 < th < 270:
        return th - 180, "right"
    return th, "left"


def draw_curved_text(ax, text, r, theta_center, fontsize, color, weight="normal"):
    """Places text along a curved path at a specific radius."""
    char_width_data = fontsize * 0.035
    char_angular_width = math.degrees(char_width_data / r)
    total_angle = len(text) * char_angular_width

    is_bottom = 90 < (theta_center % 360) < 270

    if not is_bottom:
        start_theta = theta_center + total_angle / 2 - char_angular_width / 2
        step = -char_angular_width
        rot_offset = -90
    else:
        start_theta = theta_center - total_angle / 2 + char_angular_width / 2
        step = char_angular_width
        rot_offset = 90

    curr_theta = start_theta
    for char in text:
        x, y = polar_to_cartesian(r, curr_theta)
        ax.text(
            x,
            y,
            char,
            ha="center",
            va="center",
            rotation=curr_theta + rot_offset,
            fontsize=fontsize,
            color=color,
            weight=weight,
            zorder=10,
        )
        curr_theta += step


# =============================================================================
# MAIN PLOTTING FUNCTION
# =============================================================================
def create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(14, 14),
    strat_pal=None,
    imp_pal=None,
    linear_scale=0.022,  # <-- NEW: Scales physical length of all arcs
    arc_spacing=2.5,
):  # <-- NEW: Physical gap between adjacent arcs on same level

    if imp_pal is None:
        imp_pal = {"strict": "#2979ae", "knn": "#982d22"}

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Structural Radii Parameters ---
    R_S0 = 2.5
    R_WIDTH = 2.0
    R_GAP = 1.8
    IMP_WIDTH = R_WIDTH * 0.28

    # --- 1. Define Hierarchy & Auto-Layout Engine ---
    # Instead of hardcoding coordinates, we define the dependency tree.
    # The layout engine will dynamically calculate angles to avoid overlaps.
    node_levels = {
        "J_ff_only": 1,
        "K_ffpe_only": 1,
        "C_rnaseq_only": 1,
        "F_microarray_only": 1,
        "H_affymetrix_extended": 2,
        "G_affymetrix_only": 3,
        "D_malignant_only": 1,
        "I_rare_batches_removed": 1,
        "A_confirmed_bad": 1,
        "B_extended_bad": 2,
        "E1_iterative_r1": 2,
        "E2_iterative_r2": 3,
        "E3_iterative_r3": 4,
    }

    # (Anchor Parent, Absolute Theta if anchored to S0, [Ordered Children])
    layout_groups = [
        ("S0", 50, ["J_ff_only", "K_ffpe_only"]),
        ("S0", 140, ["C_rnaseq_only", "F_microarray_only"]),
        ("S0", 225, ["D_malignant_only"]),
        ("S0", 320, ["I_rare_batches_removed", "A_confirmed_bad"]),
        ("F_microarray_only", None, ["H_affymetrix_extended"]),
        ("A_confirmed_bad", None, ["B_extended_bad", "E1_iterative_r1"]),
        ("H_affymetrix_extended", None, ["G_affymetrix_only"]),
        ("E1_iterative_r1", None, ["E2_iterative_r2"]),
        ("E2_iterative_r2", None, ["E3_iterative_r3"]),
    ]

    flows = [
        ("S0", "J_ff_only"),
        ("S0", "K_ffpe_only"),
        ("S0", "C_rnaseq_only"),
        ("S0", "F_microarray_only"),
        ("F_microarray_only", "H_affymetrix_extended"),
        ("H_affymetrix_extended", "G_affymetrix_only"),
        ("S0", "D_malignant_only"),
        ("S0", "I_rare_batches_removed"),
        ("S0", "A_confirmed_bad"),
        ("A_confirmed_bad", "B_extended_bad"),
        ("A_confirmed_bad", "E1_iterative_r1"),
        ("E1_iterative_r1", "E2_iterative_r2"),
        ("E2_iterative_r2", "E3_iterative_r3"),
    ]

    def get_node_sets(node_name):
        if node_name == "S0":
            return set(), set(), set()
        s_strict = gene_sets_by_strat_imp_raw.get((node_name, "strict"), set())
        s_knn = gene_sets_by_strat_imp_raw.get((node_name, "knn"), set())
        return s_strict, s_knn, s_strict | s_knn

    def get_angular_span(N, radius):
        """Calculates angular span (theta) to strictly enforce linear physical length."""
        arc_length = N * linear_scale
        return math.degrees(arc_length / radius)

    # --- 2. Dynamically compute all geometry to prevent overlaps ---
    node_angles = {}

    # Pre-calculate base span and radii for every node
    for name, lvl in node_levels.items():
        _, _, s_union = get_node_sets(name)
        r_in = R_S0 + (lvl - 1) * (R_WIDTH + R_GAP) + R_GAP
        r_out = r_in + R_WIDTH
        r_mid = (r_in + r_out) / 2

        node_angles[name] = {
            "span": get_angular_span(len(s_union), r_mid),
            "r_in": r_in,
            "r_out": r_out,
            "r_mid": r_mid,
        }

    # Execute Layout Engine: Pack nodes symmetrically around their parent's center
    for parent, anchor_theta, children in layout_groups:
        center_theta = anchor_theta if parent == "S0" else node_angles[parent]["theta"]

        # Convert the physical gap parameter into angular degrees at this specific radius
        r_mid = node_angles[children[0]]["r_mid"]
        gap_deg = math.degrees(arc_spacing / r_mid)

        total_span = sum(node_angles[c]["span"] for c in children) + gap_deg * (
            len(children) - 1
        )
        curr_theta = center_theta - (total_span / 2)

        for c in children:
            span = node_angles[c]["span"]
            node_angles[c]["start"] = curr_theta
            node_angles[c]["end"] = curr_theta + span
            node_angles[c]["theta"] = curr_theta + span / 2
            curr_theta += span + gap_deg

    # --- 3. Draw Target Background ---
    max_level = max(node_levels.values())
    for lvl in range(1, max_level + 1):
        r_target_in = R_S0 + (lvl - 1) * (R_WIDTH + R_GAP) + R_GAP
        r_target_out = r_target_in + R_WIDTH
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_in,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_out,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )

    # --- 4. Draw Central S0 Ring & Sector Labels ---
    s0_color = strat_pal.get("S0_no_removal", "#abcdef")
    ax.add_patch(
        patches.Wedge(
            (0, 0),
            R_S0,
            0,
            360,
            width=0.8,
            facecolor=s0_color,
            edgecolor="white",
            lw=1.5,
            zorder=5,
        )
    )

    sectors = [
        ("By biomaterial", 50),
        ("By platform", 140),
        ("By biology", 225),
        ("Bad batches\nremoval", 320),
    ]
    for label, th in sectors:
        lx, ly = polar_to_cartesian(R_S0 - 1.2, th)
        rot, ha = get_text_rotation(th)
        ax.text(
            lx,
            ly,
            label,
            ha="center",
            va="center",
            rotation=rot,
            weight="bold",
            fontsize=9,
            color="#333",
        )

    # --- 5. Draw Connecting Ribbons ---
    for parent, child in flows:
        c_ang = node_angles[child]

        if parent == "S0":
            s0_span = get_angular_span(len(get_node_sets(child)[2]), R_S0)
            c_mid = c_ang["theta"]

            draw_radial_ribbon(
                ax,
                R_S0,
                c_mid - s0_span / 2,
                c_mid + s0_span / 2,
                c_ang["r_in"],
                c_ang["start"],
                c_ang["end"],
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )
        else:
            p_ang = node_angles[parent]
            p_strict, p_knn, _ = get_node_sets(parent)
            c_strict, c_knn, _ = get_node_sets(child)

            shared = len((p_strict | p_knn) & (c_strict | c_knn))

            p_shared_span = get_angular_span(shared, p_ang["r_out"])
            c_shared_span = get_angular_span(shared, c_ang["r_in"])

            p_mid = p_ang["theta"]
            c_mid = c_ang["theta"]

            draw_radial_ribbon(
                ax,
                p_ang["r_out"],
                p_mid - p_shared_span / 2,
                p_mid + p_shared_span / 2,
                c_ang["r_in"],
                c_mid - c_shared_span / 2,
                c_mid + c_shared_span / 2,
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 6. Draw Strategy Arcs and Nested Imputation Arcs ---
    for name, lvl in node_levels.items():
        s_strict, s_knn, s_union = get_node_sets(name)
        if not s_union:
            continue

        ang = node_angles[name]
        strat_c = strat_pal.get(name, "#CCCCCC")

        # Strategy Base Arc
        base_wedge = patches.Wedge(
            (0, 0),
            ang["r_out"],
            ang["start"],
            ang["end"],
            width=R_WIDTH,
            facecolor=strat_c,
            edgecolor="white",
            lw=0.5,
            zorder=6,
        )
        ax.add_patch(base_wedge)

        # Curved Black Text for Strategy
        clean_name = name.replace("_", " ")
        draw_curved_text(
            ax,
            clean_name,
            ang["r_mid"],
            ang["theta"],
            fontsize=8,
            color="black",
            weight="bold",
        )

        # --- Nested Imputations with True Linear Constraints ---

        # STRICT (Inner Track) - Anchored exactly to the start
        if s_strict:
            strict_r_out = ang["r_in"] + R_WIDTH * 0.1 + IMP_WIDTH
            strict_r_mid = strict_r_out - IMP_WIDTH / 2
            strict_span = get_angular_span(len(s_strict), strict_r_mid)

            w_strict = patches.Wedge(
                (0, 0),
                strict_r_out,
                ang["start"],
                ang["start"] + strict_span,
                width=IMP_WIDTH,
                facecolor=imp_pal["strict"],
                zorder=7,
            )
            ax.add_patch(w_strict)

            if len(s_strict) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_strict)),
                    strict_r_mid,
                    ang["start"] + strict_span / 2,
                    fontsize=6,
                    color="white",
                    weight="bold",
                )

        # KNN (Outer Track) - Anchored exactly to the end
        if s_knn:
            knn_r_out = ang["r_out"] - R_WIDTH * 0.1
            knn_r_mid = knn_r_out - IMP_WIDTH / 2
            knn_span = get_angular_span(len(s_knn), knn_r_mid)

            w_knn = patches.Wedge(
                (0, 0),
                knn_r_out,
                ang["end"] - knn_span,
                ang["end"],
                width=IMP_WIDTH,
                facecolor=imp_pal["knn"],
                zorder=7,
            )
            ax.add_patch(w_knn)

            if len(s_knn) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_knn)),
                    knn_r_mid,
                    ang["end"] - knn_span / 2,
                    fontsize=6,
                    color="white",
                    weight="bold",
                )

    # --- Finalize ---
    ax.set_xlim(-18, 18)
    ax.set_ylim(-18, 18)

    os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/concentric_sankey_strategies.svg"
    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    plt.show()

In [ ]:
def polar_to_cartesian(r, theta_deg):
    """Converts polar coordinates (radius, angle in degrees) to Cartesian."""
    theta_rad = math.radians(theta_deg)
    return r * math.cos(theta_rad), r * math.sin(theta_rad)


def get_arc_points(r, th1, th2, n=30):
    """Returns a list of Cartesian coordinates along an arc."""
    angles = np.linspace(th1, th2, n)
    return [polar_to_cartesian(r, a) for a in angles]


def bezier_curve(p0, p1, p2, p3, n=50):
    """Returns points along a cubic Bezier curve."""
    t = np.linspace(0, 1, n)[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def draw_radial_ribbon(ax, r_in, th1_s, th1_e, r_out, th2_s, th2_e, color, alpha=0.6):
    """Draws a smooth, radially curved ribbon between two concentric arcs."""
    d = (r_out - r_in) * 0.4

    p0_L = np.array(polar_to_cartesian(r_in, th1_s))
    p1_L = np.array(polar_to_cartesian(r_in + d, th1_s))
    p2_L = np.array(polar_to_cartesian(r_out - d, th2_s))
    p3_L = np.array(polar_to_cartesian(r_out, th2_s))
    curve_L = bezier_curve(p0_L, p1_L, p2_L, p3_L)

    arc_outer = get_arc_points(r_out, th2_s, th2_e)

    p0_R = np.array(polar_to_cartesian(r_out, th2_e))
    p1_R = np.array(polar_to_cartesian(r_out - d, th2_e))
    p2_R = np.array(polar_to_cartesian(r_in + d, th1_e))
    p3_R = np.array(polar_to_cartesian(r_in, th1_e))
    curve_R = bezier_curve(p0_R, p1_R, p2_R, p3_R)

    arc_inner = get_arc_points(r_in, th1_e, th1_s)

    verts = np.vstack([curve_L, arc_outer, curve_R, arc_inner])
    poly = patches.Polygon(
        verts, facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(poly)


def draw_curved_text(ax, text, r, theta_center, fontsize, color, weight="normal"):
    """Places text along a curved path at a specific radius."""
    char_width_data = fontsize * 0.035
    char_angular_width = math.degrees(char_width_data / r)
    total_angle = len(text) * char_angular_width

    is_bottom = 90 < (theta_center % 360) < 270

    if not is_bottom:
        start_theta = theta_center + total_angle / 2 - char_angular_width / 2
        step = -char_angular_width
        rot_offset = -90
    else:
        start_theta = theta_center - total_angle / 2 + char_angular_width / 2
        step = char_angular_width
        rot_offset = 90

    curr_theta = start_theta
    for char in text:
        x, y = polar_to_cartesian(r, curr_theta)
        ax.text(
            x,
            y,
            char,
            ha="center",
            va="center",
            rotation=curr_theta + rot_offset,
            fontsize=fontsize,
            color=color,
            weight=weight,
            zorder=10,
        )
        curr_theta += step


# =============================================================================
# MAIN PLOTTING FUNCTION
# =============================================================================
def create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(14, 14),
    strat_pal=None,
    imp_pal=None,
    linear_scale=0.00022,
    arc_spacing=2.5,
    r_s0=3.75,  # <-- NEW: Radius of the inner blue circle (1.5x larger)
    r_first_level=6.45,  # <-- NEW: Radius of the 1st level circle (1.5x larger)
    level_gap=0.9,
):  # <-- NEW: Gap between outer rings (2x tighter)

    if imp_pal is None:
        imp_pal = {"strict": "#2979ae", "knn": "#982d22"}

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Structural Width Parameters ---
    R_WIDTH = 2.0
    IMP_WIDTH = R_WIDTH * 0.28
    R_S0_WIDTH = 1.2  # Scaled up slightly to match the larger S0 radius

    # --- 1. Define Hierarchy & Layout Groups ---
    node_levels = {
        "J_ff_only": 1,
        "K_ffpe_only": 1,
        "C_rnaseq_only": 1,
        "F_microarray_only": 1,
        "H_affymetrix_extended": 2,
        "G_affymetrix_only": 3,
        "D_malignant_only": 1,
        "I_rare_batches_removed": 1,
        "A_confirmed_bad": 1,
        "B_extended_bad": 2,
        "E1_iterative_r1": 2,
        "E2_iterative_r2": 3,
        "E3_iterative_r3": 4,
    }

    # S0 Macro Groups
    s0_groups = {
        "Biomaterial": {
            "theta": 50,
            "children": ["J_ff_only", "K_ffpe_only"],
            "label": "By biomaterial",
        },
        "Platform": {
            "theta": 140,
            "children": ["C_rnaseq_only", "F_microarray_only"],
            "label": "By platform",
        },
        "Biology": {
            "theta": 225,
            "children": ["D_malignant_only"],
            "label": "By biology",
        },
        "Batch": {
            "theta": 320,
            "children": ["I_rare_batches_removed", "A_confirmed_bad"],
            "label": "Bad batches removal",
        },
    }

    # Outer Flow Dependencies
    outer_flows = [
        ("F_microarray_only", ["H_affymetrix_extended"]),
        ("A_confirmed_bad", ["B_extended_bad", "E1_iterative_r1"]),
        ("H_affymetrix_extended", ["G_affymetrix_only"]),
        ("E1_iterative_r1", ["E2_iterative_r2"]),
        ("E2_iterative_r2", ["E3_iterative_r3"]),
    ]

    def get_node_sets(node_name):
        if node_name == "S0":
            return set(), set(), set()
        s_strict = gene_sets_by_strat_imp_raw.get((node_name, "strict"), set())
        s_knn = gene_sets_by_strat_imp_raw.get((node_name, "knn"), set())
        return s_strict, s_knn, s_strict | s_knn

    def get_angular_span(N, radius):
        arc_length = N * linear_scale
        return math.degrees(arc_length / radius)

    def get_radii(level):
        r_in = (
            r_first_level
            if level == 1
            else r_first_level + (level - 1) * (R_WIDTH + level_gap)
        )
        r_out = r_in + R_WIDTH
        return r_in, r_out, (r_in + r_out) / 2

    # --- 2. Dynamically compute all geometry ---
    node_angles = {}

    # Pre-calculate base span and radii for every node
    for name, lvl in node_levels.items():
        _, _, s_union = get_node_sets(name)
        r_in, r_out, r_mid = get_radii(lvl)
        node_angles[name] = {
            "span": get_angular_span(len(s_union), r_mid),
            "r_in": r_in,
            "r_out": r_out,
            "r_mid": r_mid,
        }

    # Layout Engine: Pack Level 1 nodes around their S0 macro group
    for g_name, g_info in s0_groups.items():
        children = g_info["children"]
        r_mid = get_radii(1)[2]
        gap_deg = math.degrees(arc_spacing / r_mid)

        total_span = sum(node_angles[c]["span"] for c in children) + gap_deg * (
            len(children) - 1
        )
        curr_theta = g_info["theta"] - (total_span / 2)

        for c in children:
            span = node_angles[c]["span"]
            node_angles[c]["start"] = curr_theta
            node_angles[c]["end"] = curr_theta + span
            node_angles[c]["theta"] = curr_theta + span / 2
            curr_theta += span + gap_deg

    # Layout Engine: Pack Level 2+ nodes symmetrically around their parent's center
    for parent, children in outer_flows:
        center_theta = node_angles[parent]["theta"]
        r_mid = node_angles[children[0]]["r_mid"]
        gap_deg = math.degrees(arc_spacing / r_mid)

        total_span = sum(node_angles[c]["span"] for c in children) + gap_deg * (
            len(children) - 1
        )
        curr_theta = center_theta - (total_span / 2)

        for c in children:
            span = node_angles[c]["span"]
            node_angles[c]["start"] = curr_theta
            node_angles[c]["end"] = curr_theta + span
            node_angles[c]["theta"] = curr_theta + span / 2
            curr_theta += span + gap_deg

    # --- 3. Draw Target Background ---
    max_level = max(node_levels.values())
    for lvl in range(1, max_level + 1):
        r_target_in, r_target_out, _ = get_radii(lvl)
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_in,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_out,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )

    # --- 4. Draw Split S0 Ring & Overlapping Ribbons ---
    s0_color = strat_pal.get("S0_no_removal", "#abcdef")

    for g_name, g_info in s0_groups.items():
        # Get total unique genes for the macro group
        union_genes = set()
        for child in g_info["children"]:
            _, _, c_union = get_node_sets(child)
            union_genes |= c_union

        span_deg = get_angular_span(len(union_genes), r_s0)

        # Draw the isolated S0 arc
        wedge = patches.Wedge(
            (0, 0),
            r_s0,
            g_info["theta"] - span_deg / 2,
            g_info["theta"] + span_deg / 2,
            width=R_S0_WIDTH,
            facecolor=s0_color,
            edgecolor="white",
            lw=1.5,
            zorder=5,
        )
        ax.add_patch(wedge)

        # Curved text inside the arc
        draw_curved_text(
            ax,
            g_info["label"],
            r_s0 - (R_S0_WIDTH / 2),
            g_info["theta"],
            fontsize=9,
            color="black",
            weight="bold",
        )

        # Draw Ribbons extending from the EXACT same anchor point (overlapping at base)
        for child in g_info["children"]:
            c_ang = node_angles[child]
            _, _, c_union = get_node_sets(child)

            child_s0_span = get_angular_span(len(c_union), r_s0)
            c_mid = g_info[
                "theta"
            ]  # All siblings anchor to the center of the macro arc

            draw_radial_ribbon(
                ax,
                r_s0,
                c_mid - child_s0_span / 2,
                c_mid + child_s0_span / 2,
                c_ang["r_in"],
                c_ang["start"],
                c_ang["end"],
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 5. Draw Outer Connecting Ribbons ---
    for parent, children in outer_flows:
        for child in children:
            p_ang = node_angles[parent]
            c_ang = node_angles[child]

            p_strict, p_knn, _ = get_node_sets(parent)
            c_strict, c_knn, _ = get_node_sets(child)

            shared = len((p_strict | p_knn) & (c_strict | c_knn))

            p_shared_span = get_angular_span(shared, p_ang["r_out"])
            c_shared_span = get_angular_span(shared, c_ang["r_in"])

            p_mid = p_ang["theta"]
            c_mid = c_ang["theta"]

            draw_radial_ribbon(
                ax,
                p_ang["r_out"],
                p_mid - p_shared_span / 2,
                p_mid + p_shared_span / 2,
                c_ang["r_in"],
                c_mid - c_shared_span / 2,
                c_mid + c_shared_span / 2,
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 6. Draw Strategy Arcs and Nested Imputation Arcs ---
    for name, lvl in node_levels.items():
        s_strict, s_knn, s_union = get_node_sets(name)
        if not s_union:
            continue

        ang = node_angles[name]
        strat_c = strat_pal.get(name, "#CCCCCC")

        # Strategy Base Arc
        base_wedge = patches.Wedge(
            (0, 0),
            ang["r_out"],
            ang["start"],
            ang["end"],
            width=R_WIDTH,
            facecolor=strat_c,
            edgecolor="white",
            lw=0.5,
            zorder=6,
        )
        ax.add_patch(base_wedge)

        # Curved Black Text for Strategy
        clean_name = name.replace("_", " ")
        draw_curved_text(
            ax,
            clean_name,
            ang["r_mid"],
            ang["theta"],
            fontsize=8,
            color="black",
            weight="bold",
        )

        # --- Nested Imputations with True Linear Constraints ---

        # STRICT (Inner Track) - Anchored exactly to the start
        if s_strict:
            strict_r_out = ang["r_in"] + R_WIDTH * 0.1 + IMP_WIDTH
            strict_r_mid = strict_r_out - IMP_WIDTH / 2
            strict_span = get_angular_span(len(s_strict), strict_r_mid)

            w_strict = patches.Wedge(
                (0, 0),
                strict_r_out,
                ang["start"],
                ang["start"] + strict_span,
                width=IMP_WIDTH,
                facecolor=imp_pal["strict"],
                zorder=7,
            )
            ax.add_patch(w_strict)

            if len(s_strict) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_strict)),
                    strict_r_mid,
                    ang["start"] + strict_span / 2,
                    fontsize=6,
                    color="white",
                    weight="bold",
                )

        # KNN (Outer Track) - Anchored exactly to the end
        if s_knn:
            knn_r_out = ang["r_out"] - R_WIDTH * 0.1
            knn_r_mid = knn_r_out - IMP_WIDTH / 2
            knn_span = get_angular_span(len(s_knn), knn_r_mid)

            w_knn = patches.Wedge(
                (0, 0),
                knn_r_out,
                ang["end"] - knn_span,
                ang["end"],
                width=IMP_WIDTH,
                facecolor=imp_pal["knn"],
                zorder=7,
            )
            ax.add_patch(w_knn)

            if len(s_knn) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_knn)),
                    knn_r_mid,
                    ang["end"] - knn_span / 2,
                    fontsize=6,
                    color="white",
                    weight="bold",
                )

    # --- Finalize ---
    ax.set_xlim(-18, 18)
    ax.set_ylim(-18, 18)

    os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/concentric_sankey_strategies.svg"
    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    plt.show()

In [ ]:
create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(14, 14),
    strat_pal=strat_pal,
    imp_pal=imp_pal,
    linear_scale=0.00022,
    arc_spacing=2.5,
    r_s0=3.75,  # <-- NEW: Radius of the inner blue circle (1.5x larger)
    r_first_level=6.45,  # <-- NEW: Radius of the 1st level circle (1.5x larger)
    level_gap=0.9,  # <-- NEW: Gap between outer rings (2x tighter)
)

In [ ]:
create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(14, 14),
    strat_pal=strat_pal,
    imp_pal=imp_pal,
    linear_scale=0.00022,
    arc_spacing=2.5,
    r_s0=3.75,  # <-- NEW: Radius of the inner blue circle (1.5x larger)
    r_first_level=6.45,  # <-- NEW: Radius of the 1st level circle (1.5x larger)
    level_gap=0.9,  # <-- NEW: Gap between outer rings (2x tighter)
)

In [ ]:
def polar_to_cartesian(r, theta_deg):
    """Converts polar coordinates (radius, angle in degrees) to Cartesian."""
    theta_rad = math.radians(theta_deg)
    return r * math.cos(theta_rad), r * math.sin(theta_rad)


def get_arc_points(r, th1, th2, n=30):
    """Returns a list of Cartesian coordinates along an arc."""
    angles = np.linspace(th1, th2, n)
    return [polar_to_cartesian(r, a) for a in angles]


def bezier_curve(p0, p1, p2, p3, n=50):
    """Returns points along a cubic Bezier curve."""
    t = np.linspace(0, 1, n)[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def draw_radial_ribbon(ax, r_in, th1_s, th1_e, r_out, th2_s, th2_e, color, alpha=0.6):
    """Draws a smooth, radially curved ribbon between two concentric arcs."""
    d = (r_out - r_in) * 0.4

    p0_L = np.array(polar_to_cartesian(r_in, th1_s))
    p1_L = np.array(polar_to_cartesian(r_in + d, th1_s))
    p2_L = np.array(polar_to_cartesian(r_out - d, th2_s))
    p3_L = np.array(polar_to_cartesian(r_out, th2_s))
    curve_L = bezier_curve(p0_L, p1_L, p2_L, p3_L)

    arc_outer = get_arc_points(r_out, th2_s, th2_e)

    p0_R = np.array(polar_to_cartesian(r_out, th2_e))
    p1_R = np.array(polar_to_cartesian(r_out - d, th2_e))
    p2_R = np.array(polar_to_cartesian(r_in + d, th1_e))
    p3_R = np.array(polar_to_cartesian(r_in, th1_e))
    curve_R = bezier_curve(p0_R, p1_R, p2_R, p3_R)

    arc_inner = get_arc_points(r_in, th1_e, th1_s)

    verts = np.vstack([curve_L, arc_outer, curve_R, arc_inner])
    poly = patches.Polygon(
        verts, facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(poly)


def draw_curved_text(
    ax, text, r, theta_center, fontsize, color, weight="normal", letter_spacing=1.0
):
    """Places text along a curved path with dynamic kerning for natural spacing."""
    rel_widths = {
        "i": 0.3,
        "l": 0.3,
        "I": 0.3,
        "1": 0.4,
        "j": 0.4,
        "f": 0.5,
        "t": 0.5,
        "r": 0.5,
        " ": 0.6,
        "m": 1.3,
        "w": 1.3,
        "M": 1.3,
        "W": 1.3,
    }

    base_char_width = fontsize * 0.030 * letter_spacing
    char_angles_deg = [
        math.degrees((rel_widths.get(c, 0.8) * base_char_width) / r) for c in text
    ]
    total_angle = sum(char_angles_deg)

    theta_center = theta_center % 360

    is_bottom = 180 < theta_center < 360

    if not is_bottom:
        curr_theta = theta_center + total_angle / 2
        rot_offset = -90
        direction = -1
    else:
        curr_theta = theta_center - total_angle / 2
        rot_offset = 90
        direction = 1

    for i, char in enumerate(text):
        curr_theta += direction * (char_angles_deg[i] / 2)
        x, y = polar_to_cartesian(r, curr_theta)
        ax.text(
            x,
            y,
            char,
            ha="center",
            va="center",
            rotation=curr_theta + rot_offset,
            fontsize=fontsize,
            color=color,
            weight=weight,
            zorder=10,
        )
        curr_theta += direction * (char_angles_deg[i] / 2)


# =============================================================================
# MAIN PLOTTING FUNCTION
# =============================================================================
def create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(14, 14),
    strat_pal=None,
    imp_pal=None,
    linear_scale=0.00022,
    arc_spacing=2.5,
    s0_arc_spacing=5.0,
    r_s0=3.75,
    r_first_level=6.45,
    level_gap=0.9,
    s0_arc_length_factor=1.5,
    letter_spacing=1.0,
):

    if imp_pal is None:
        imp_pal = {"strict": "#2979ae", "knn": "#982d22"}

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    R_WIDTH = 2.0
    IMP_WIDTH = R_WIDTH * 0.28
    R_S0_WIDTH = 1.2

    # --- 1. Define Hierarchy & Layout Groups ---
    node_levels = {
        "J_ff_only": 1,
        "K_ffpe_only": 1,
        "C_rnaseq_only": 1,
        "F_microarray_only": 1,
        "H_affymetrix_extended": 2,
        "G_affymetrix_only": 3,
        "D_malignant_only": 1,
        "I_rare_batches_removed": 1,
        "A_confirmed_bad": 1,
        "B_extended_bad": 2,
        "E1_iterative_r1": 2,
        "E2_iterative_r2": 3,
        "E3_iterative_r3": 4,
    }

    s0_groups = {
        "Biomaterial": {
            "children": ["J_ff_only", "K_ffpe_only"],
            "label": "Biomaterial",
        },
        "Platform": {
            "children": ["C_rnaseq_only", "F_microarray_only"],
            "label": "Platform",
        },
        "Biology": {"children": ["D_malignant_only"], "label": "Biology"},
        "Batch": {
            "children": ["I_rare_batches_removed", "A_confirmed_bad"],
            "label": "Bad batches",
        },
    }

    outer_flows = [
        ("F_microarray_only", ["H_affymetrix_extended"]),
        ("A_confirmed_bad", ["B_extended_bad", "E1_iterative_r1"]),
        ("H_affymetrix_extended", ["G_affymetrix_only"]),
        ("E1_iterative_r1", ["E2_iterative_r2"]),
        ("E2_iterative_r2", ["E3_iterative_r3"]),
    ]

    def get_node_sets(node_name):
        if node_name == "S0":
            node_name = "S0_no_removal"
        s_strict = gene_sets_by_strat_imp_raw.get((node_name, "strict"), set())
        s_knn = gene_sets_by_strat_imp_raw.get((node_name, "knn"), set())
        return s_strict, s_knn, s_strict | s_knn

    def get_angular_span(N, radius):
        arc_length = N * linear_scale
        return math.degrees(arc_length / radius)

    def get_radii(level):
        r_in = (
            r_first_level
            if level == 1
            else r_first_level + (level - 1) * (R_WIDTH + level_gap)
        )
        r_out = r_in + R_WIDTH
        return r_in, r_out, (r_in + r_out) / 2

    # --- 2. Dynamically compute all geometry ---
    node_angles = {}
    for name, lvl in node_levels.items():
        _, _, s_union = get_node_sets(name)
        r_in, r_out, r_mid = get_radii(lvl)
        node_angles[name] = {
            "span": get_angular_span(len(s_union), r_mid),
            "r_in": r_in,
            "r_out": r_out,
            "r_mid": r_mid,
        }

    # Layout Engine: Pack S0 Arcs
    curr_theta = 45
    s0_gap_deg = math.degrees(s0_arc_spacing / r_s0)
    for g_name, g_info in s0_groups.items():
        union_genes = set()
        for child in g_info["children"]:
            _, _, c_union = get_node_sets(child)
            union_genes |= c_union

        g_info["span"] = get_angular_span(len(union_genes), r_s0) * s0_arc_length_factor
        g_info["theta"] = curr_theta + g_info["span"] / 2
        curr_theta += g_info["span"] + s0_gap_deg

    # Layout Engine: Pack Level 1 nodes
    for g_name, g_info in s0_groups.items():
        children = g_info["children"]
        r_mid = get_radii(1)[2]
        gap_deg = math.degrees(arc_spacing / r_mid)

        total_span = sum(node_angles[c]["span"] for c in children) + gap_deg * (
            len(children) - 1
        )
        curr_theta = g_info["theta"] - (total_span / 2)

        for c in children:
            span = node_angles[c]["span"]
            node_angles[c]["start"] = curr_theta
            node_angles[c]["end"] = curr_theta + span
            node_angles[c]["theta"] = curr_theta + span / 2
            curr_theta += span + gap_deg

    # Layout Engine: Pack Level 2+ nodes
    for parent, children in outer_flows:
        center_theta = node_angles[parent]["theta"]
        r_mid = node_angles[children[0]]["r_mid"]
        gap_deg = math.degrees(arc_spacing / r_mid)

        total_span = sum(node_angles[c]["span"] for c in children) + gap_deg * (
            len(children) - 1
        )
        curr_theta = center_theta - (total_span / 2)

        for c in children:
            span = node_angles[c]["span"]
            node_angles[c]["start"] = curr_theta
            node_angles[c]["end"] = curr_theta + span
            node_angles[c]["theta"] = curr_theta + span / 2
            curr_theta += span + gap_deg

    # --- 3. Draw Target Background ---
    max_level = max(node_levels.values())
    for lvl in range(1, max_level + 1):
        r_target_in, r_target_out, _ = get_radii(lvl)
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_in,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )
        ax.add_patch(
            patches.Circle(
                (0, 0),
                r_target_out,
                fill=False,
                edgecolor="#DDDDDD",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )
        )

    # --- 4. Draw Inner KNN/Strict Circles ---
    s0_strict, s0_knn, _ = get_node_sets("S0")
    knn_radius = r_s0 - R_S0_WIDTH - 0.25

    # KNN Background Circle
    ax.add_patch(
        patches.Circle(
            (0, 0),
            knn_radius,
            facecolor=imp_pal["knn"],
            edgecolor="white",
            lw=1.0,
            zorder=1,
        )
    )

    # Strict Inner Arc & Text
    if len(s0_knn) > 0 and len(s0_strict) > 0:
        strict_ratio = len(s0_strict) / len(s0_knn)
        strict_span = 360 * strict_ratio

        ax.add_patch(
            patches.Wedge(
                (0, 0),
                knn_radius,
                90 - strict_span / 2,
                90 + strict_span / 2,
                width=0.4,
                facecolor=imp_pal["strict"],
                edgecolor="none",
                zorder=2,
            )
        )

        # Draw the gene count text inside the strict arc
        draw_curved_text(
            ax,
            str(len(s0_strict)),
            knn_radius - 0.2,
            90,
            fontsize=6,
            color="white",
            weight="bold",
            letter_spacing=letter_spacing,
        )

    ax.text(
        0,
        0,
        f"S0 KNN\nN={len(s0_knn)}",
        ha="center",
        va="center",
        color="white",
        weight="bold",
        fontsize=8,
        zorder=3,
    )

    # --- 5. Draw Split S0 Ring & Overlapping Ribbons ---
    s0_color = strat_pal.get("S0_no_removal", "#abcdef")
    for g_name, g_info in s0_groups.items():
        wedge = patches.Wedge(
            (0, 0),
            r_s0,
            g_info["theta"] - g_info["span"] / 2,
            g_info["theta"] + g_info["span"] / 2,
            width=R_S0_WIDTH,
            facecolor=s0_color,
            edgecolor="white",
            lw=1.5,
            zorder=5,
        )
        ax.add_patch(wedge)

        draw_curved_text(
            ax,
            g_info["label"],
            r_s0 - (R_S0_WIDTH / 2),
            g_info["theta"],
            fontsize=9,
            color="black",
            weight="bold",
            letter_spacing=letter_spacing,
        )

        for child in g_info["children"]:
            c_ang = node_angles[child]
            th1_s = g_info["theta"] - g_info["span"] / 2
            th1_e = g_info["theta"] + g_info["span"] / 2
            draw_radial_ribbon(
                ax,
                r_s0,
                th1_s,
                th1_e,
                c_ang["r_in"],
                c_ang["start"],
                c_ang["end"],
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 6. Draw Outer Connecting Ribbons ---
    for parent, children in outer_flows:
        for child in children:
            p_ang = node_angles[parent]
            c_ang = node_angles[child]

            p_strict, p_knn, _ = get_node_sets(parent)
            c_strict, c_knn, _ = get_node_sets(child)

            shared = len((p_strict | p_knn) & (c_strict | c_knn))

            p_shared_span = get_angular_span(shared, p_ang["r_out"])
            c_shared_span = get_angular_span(shared, c_ang["r_in"])

            p_mid = p_ang["theta"]
            c_mid = c_ang["theta"]

            draw_radial_ribbon(
                ax,
                p_ang["r_out"],
                p_mid - p_shared_span / 2,
                p_mid + p_shared_span / 2,
                c_ang["r_in"],
                c_mid - c_shared_span / 2,
                c_mid + c_shared_span / 2,
                color=strat_pal.get(child, "#ccc"),
                alpha=0.5,
            )

    # --- 7. Draw Strategy Arcs and Nested Imputation Arcs ---
    for name, lvl in node_levels.items():
        s_strict, s_knn, s_union = get_node_sets(name)
        if not s_union:
            continue

        ang = node_angles[name]
        strat_c = strat_pal.get(name, "#CCCCCC")

        base_wedge = patches.Wedge(
            (0, 0),
            ang["r_out"],
            ang["start"],
            ang["end"],
            width=R_WIDTH,
            facecolor=strat_c,
            edgecolor="white",
            lw=0.5,
            zorder=6,
        )
        ax.add_patch(base_wedge)

        clean_name = name.replace("_", " ")
        draw_curved_text(
            ax,
            clean_name,
            ang["r_mid"],
            ang["theta"],
            fontsize=8,
            color="black",
            weight="bold",
            letter_spacing=letter_spacing,
        )

        # --- Nested Imputations (Fully Centered) ---

        # STRICT (Inner Track) - Centered inside the strategy arc
        if s_strict:
            strict_r_out = ang["r_in"] + R_WIDTH * 0.1 + IMP_WIDTH
            strict_r_mid = strict_r_out - IMP_WIDTH / 2
            strict_span = get_angular_span(len(s_strict), strict_r_mid)

            strict_start = ang["theta"] - strict_span / 2
            w_strict = patches.Wedge(
                (0, 0),
                strict_r_out,
                strict_start,
                strict_start + strict_span,
                width=IMP_WIDTH,
                facecolor=imp_pal["strict"],
                zorder=7,
            )
            ax.add_patch(w_strict)

            if len(s_strict) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_strict)),
                    strict_r_mid,
                    ang["theta"],
                    fontsize=6,
                    color="white",
                    weight="bold",
                    letter_spacing=letter_spacing,
                )

        # KNN (Outer Track) - Centered inside the strategy arc
        if s_knn:
            knn_r_out = ang["r_out"] - R_WIDTH * 0.1
            knn_r_mid = knn_r_out - IMP_WIDTH / 2
            knn_span = get_angular_span(len(s_knn), knn_r_mid)

            knn_start = ang["theta"] - knn_span / 2
            w_knn = patches.Wedge(
                (0, 0),
                knn_r_out,
                knn_start,
                knn_start + knn_span,
                width=IMP_WIDTH,
                facecolor=imp_pal["knn"],
                zorder=7,
            )
            ax.add_patch(w_knn)

            if len(s_knn) > 500:
                draw_curved_text(
                    ax,
                    str(len(s_knn)),
                    knn_r_mid,
                    ang["theta"],
                    fontsize=6,
                    color="white",
                    weight="bold",
                    letter_spacing=letter_spacing,
                )

    # --- Finalize ---
    ax.set_xlim(-18, 18)
    ax.set_ylim(-18, 18)

    os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/concentric_sankey_strategies.svg"
    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    plt.show()

In [ ]:
gene_sets_by_strat_imp_raw.keys()

In [ ]:
create_concentric_sankey(
    gene_sets_by_strat_imp_raw,
    figsize=(10, 10),
    strat_pal=strat_pal,
    imp_pal=imp_pal,
    linear_scale=0.00022,
    arc_spacing=2.5,
    s0_arc_spacing=1.0,  # <-- NEW: Spacing between the 4 inner blue arcs
    r_s0=3.75,  # <-- NEW: Radius of the inner blue circle (1.5x larger)
    r_first_level=6.45,  # <-- NEW: Radius of the 1st level circle (1.5x larger)
    level_gap=0.9,  # <-- NEW: Gap between outer rings (2x tighter)
    s0_arc_length_factor=1.5,  # <-- NEW: Multiplier for inner blue arcs
    letter_spacing=2,
)

---
## §8 — Progressive Gene Accumulation / Depletion Curves

How does gene count change as sample removal becomes more aggressive? Track the progressive loss of genes across the harshness axis.

In [ ]:
# ── §8.1 Harshness ordering ───────────────────────────────────────────────────
# S0=permissive → G=aggressive. Within each strat: strict < knn < softimpute
# (strict loses genes; softimpute recovers the most).

HARSHNESS_ORDER = [
    ("S0_no_removal", "strict"),
    ("S0_no_removal", "knn"),
    ("S0_no_removal", "softimpute"),
    ("A_confirmed_bad", "strict"),
    ("A_confirmed_bad", "knn"),
    ("A_confirmed_bad", "softimpute"),
    ("B_extended_bad", "strict"),
    ("B_extended_bad", "knn"),
    ("B_extended_bad", "softimpute"),
    ("C_rnaseq_only", "strict"),
    ("C_rnaseq_only", "knn"),
    ("C_rnaseq_only", "softimpute"),
    ("D_malignant_only", "strict"),
    ("D_malignant_only", "knn"),
    ("D_malignant_only", "softimpute"),
    ("E1_iterative_r1", "strict"),
    ("E1_iterative_r1", "knn"),
    ("E1_iterative_r1", "softimpute"),
    ("E2_iterative_r2", "strict"),
    ("E2_iterative_r2", "knn"),
    ("E2_iterative_r2", "softimpute"),
    ("E3_iterative_r3", "strict"),
    ("E3_iterative_r3", "knn"),
    ("E3_iterative_r3", "softimpute"),
    ("F_microarray_only", "strict"),
    ("F_microarray_only", "knn"),
    ("F_microarray_only", "softimpute"),
    ("G_affymetrix_only", "strict"),
    ("G_affymetrix_only", "knn"),
    ("G_affymetrix_only", "softimpute"),
    ("I_rare_batches_removed", "strict"),
    ("I_rare_batches_removed", "knn"),
    ("I_rare_batches_removed", "softimpute"),
    ("H_affymetrix_extended", "strict"),
    ("H_affymetrix_extended", "knn"),
    ("H_affymetrix_extended", "softimpute"),
    ("J_ff_only", "strict"),
    ("J_ff_only", "knn"),
    ("J_ff_only", "softimpute"),
    ("K_ffpe_only", "strict"),
    ("K_ffpe_only", "knn"),
    ("K_ffpe_only", "softimpute"),
]
# Filter to only combinations that exist in gene_sets_by_strat_imp.
HARSHNESS_ORDER = [si for si in HARSHNESS_ORDER if si in gene_sets_by_strat_imp]

angel_representative_genes = gene_sets_angel.get(("S0_no_removal", "strict"), set())
print(f"HARSHNESS_ORDER steps: {len(HARSHNESS_ORDER)}")
print(f"Angel gene set size (reference): {len(angel_representative_genes)}")

In [ ]:
# ── §8.2 Gene retention curve ─────────────────────────────────────────────────

if HARSHNESS_ORDER:
    _step_counts = [len(gene_sets_by_strat_imp[k]) for k in HARSHNESS_ORDER]
    _cumulative_union: list[int] = []
    _running_union: set[str] = set()
    for k in HARSHNESS_ORDER:
        _running_union |= gene_sets_by_strat_imp[k]
        _cumulative_union.append(len(_running_union))

    _x = range(len(HARSHNESS_ORDER))
    _labels = [f"{s}\n{i}" for s, i in HARSHNESS_ORDER]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

    ax1.plot(_x, _step_counts, marker="o", markersize=4, label="n_genes (per step)")
    ax1.plot(
        _x,
        _cumulative_union,
        marker="s",
        markersize=4,
        linestyle="--",
        label="cumulative union",
    )
    if angel_representative_genes:
        ax1.axhline(
            len(angel_representative_genes),
            color="red",
            linestyle=":",
            label=f"25_angel ({len(angel_representative_genes)} genes)",
        )
    ax1.set_xticks(list(_x))
    ax1.set_xticklabels(_labels, rotation=45, ha="right", fontsize=7)
    ax1.set_ylabel("Number of genes")
    ax1.set_title("Gene retention along harshness axis")
    ax1.legend(fontsize=8)

    _prev_counts = [_step_counts[0]] + _step_counts[:-1]
    _lost = [max(0, p - c) for p, c in zip(_prev_counts, _step_counts)]
    _retained = _step_counts
    ax2.bar(_x, _retained, label="retained", color="#4292c6")
    ax2.bar(_x, _lost, bottom=_retained, label="lost vs prev", color="#d62728")
    ax2.set_xticks(list(_x))
    ax2.set_xticklabels(_labels, rotation=45, ha="right", fontsize=7)
    ax2.set_ylabel("Genes")
    ax2.set_title("Genes retained vs lost per step")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_retention_curve.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "gene_retention_curve.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("HARSHNESS_ORDER is empty — run §6 first.")

In [ ]:
# ── §8.3 Imputation recovery supervenn (strat=A_confirmed_bad) ────────────────

try:
    from supervenn import supervenn

    _imps = ["strict", "knn", "softimpute"]
    _imp_sets_abc = {
        i: gene_sets_by_strat_imp.get(("A_confirmed_bad", i), set()) for i in _imps
    }
    _nonempty = {i: s for i, s in _imp_sets_abc.items() if len(s) > 0}

    if len(_nonempty) >= 2:
        _labels_n = list(_nonempty.keys())
        _sets_n = list(_nonempty.values())
        fig_imp, ax_imp = plt.subplots(figsize=(10, 4))
        supervenn(_sets_n, _labels_n, ax=ax_imp)
        ax_imp.set_title("Gene recovery by imputation method (strat=A_confirmed_bad)")
        fig_imp.tight_layout()
        fig_imp.savefig(FIGURES_DIR / "supervenn_imp_recovery.svg", bbox_inches="tight")
        fig_imp.savefig(
            FIGURES_DIR / "supervenn_imp_recovery.png", bbox_inches="tight", dpi=200
        )
        plt.show()

        if "strict" in _nonempty and "knn" in _nonempty:
            recovered_knn = _nonempty["knn"] - _nonempty["strict"]
            print(f"Genes recovered by KNN vs strict: {len(recovered_knn)}")
        if "strict" in _nonempty and "softimpute" in _nonempty:
            recovered_soft = _nonempty["softimpute"] - _nonempty["strict"]
            print(f"Genes recovered by softimpute vs strict: {len(recovered_soft)}")
    else:
        print("Not enough non-empty imputation sets for supervenn.")

except ImportError:
    print("supervenn not installed — run: pip install supervenn")

---
## §9 — GO Enrichment Analysis Setup

Load the Gene Ontology database and define the enrichment function used in §10 and §11.

In [ ]:
import os

GO_OBO_PATH = "go-basic.obo"
GO_GAF_PATH = "goa_human.gaf"
GO_GAF_GZ = "goa_human.gaf.gz"

# Check for existing files in current dir or BG project cache
_BG_CACHE = "../../Retroelements/T2T_genes_article/T2T_transposons_genes/"


def _find_or_download(fname: str, url: str, gz_name: str | None = None) -> str:
    for candidate in [fname, os.path.join(_BG_CACHE, fname)]:
        if os.path.exists(candidate):
            print(f"Found: {candidate}")
            return candidate
    import urllib.request

    print(f"Downloading {url} ...")
    tmp = gz_name or fname
    urllib.request.urlretrieve(url, tmp)
    if gz_name:
        import gzip as _gz
        import shutil

        with _gz.open(gz_name, "rb") as f_in, open(fname, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_name)
    print(f"Saved: {fname}")
    return fname


GO_OBO_PATH = _find_or_download(
    GO_OBO_PATH,
    "http://purl.obolibrary.org/obo/go/go-basic.obo",
)
GO_GAF_PATH = _find_or_download(
    GO_GAF_PATH,
    "http://geneontology.org/gene-associations/goa_human.gaf.gz",
    gz_name=GO_GAF_GZ,
)
print("GO database files ready.")

In [ ]:
def load_go_database(
    obo_path: str, gaf_path: str
) -> tuple[object, dict[str, set[str]], list[str]]:
    """
    Load the Gene Ontology DAG and human gene → GO term associations.

    The background gene list is fixed to ALL genes with GO annotations in the GAF
    file, not a dataset-specific subset. A fixed background ensures that enrichment
    strength correctly increases when more biologically relevant genes are retained.

    Parameters
    ----------
    obo_path : str
        Path to go-basic.obo file.
    gaf_path : str
        Path to goa_human.gaf file (not compressed).

    Returns
    -------
    godag : GODag
    full_assoc : dict[str, set[str]]
        Mapping gene symbol → set of GO IDs.
    GO_BACKGROUND : list[str]
        All gene symbols present in the GAF file.
    """
    from goatools.anno.gaf_reader import GafReader
    from goatools.obo_parser import GODag

    godag = GODag(obo_path, optional_attrs={"relationship"})
    gaf = GafReader(gaf_path)
    ns2assoc = gaf.get_ns2assc()
    full_assoc: dict[str, set[str]] = {}
    for ns_assoc in ns2assoc.values():
        for gene, go_ids in ns_assoc.items():
            full_assoc.setdefault(gene, set()).update(go_ids)
    GO_BACKGROUND = sorted(full_assoc.keys())
    print(f"GO DAG loaded: {len(godag)} terms")
    print(f"Annotated genes: {len(GO_BACKGROUND)}")
    return godag, full_assoc, GO_BACKGROUND


try:
    godag, full_assoc, GO_BACKGROUND = load_go_database(GO_OBO_PATH, GO_GAF_PATH)
except Exception as exc:
    godag, full_assoc, GO_BACKGROUND = None, {}, []
    print(f"GO database load failed: {exc}")

In [ ]:
def run_goatools_enrichment(
    study_genes: list[str],
    background_genes: list[str],
    go_assoc: dict[str, set[str]],
    go_dag: object,
    alpha: float = 0.05,
    methods: list[str] | None = None,
    min_study_count: int = 3,
) -> pd.DataFrame:
    """
    Run GO term enrichment using goatools Fisher's exact test.

    Parameters
    ----------
    study_genes : list of str
        Gene symbols to test for enrichment.
    background_genes : list of str
        Full background gene universe.
    go_assoc : dict[str, set[str]]
        Gene → GO term mapping.
    go_dag : GODag
        Loaded GO DAG.
    alpha : float
        FDR threshold for significance.
    methods : list of str or None
        Multiple testing correction methods (default: ["fdr_bh"]).
    min_study_count : int
        Minimum number of study genes per GO term.

    Returns
    -------
    pd.DataFrame
        Enrichment results with columns: GO_ID, name, namespace, p_uncorrected,
        p_fdr_bh, ratio_in_study, ratio_in_pop, study_genes_str.
    """
    from goatools.go_enrichment import GOEnrichmentStudy

    if methods is None:
        methods = ["fdr_bh"]
    if go_dag is None or not background_genes:
        return pd.DataFrame()

    goeaobj = GOEnrichmentStudy(
        background_genes,
        go_assoc,
        go_dag,
        propagate_counts=True,
        alpha=alpha,
        methods=methods,
    )
    results = goeaobj.run_study(study_genes, prt=None)
    rows = []
    for r in results:
        if r.study_count < min_study_count:
            continue
        p_fdr = getattr(r, "p_fdr_bh", 1.0)
        rows.append(
            {
                "GO_ID": r.GO,
                "name": r.name,
                "namespace": r.NS,
                "p_uncorrected": r.p_uncorrected,
                "p_fdr_bh": p_fdr,
                "ratio_in_study": f"{r.study_count}/{r.study_n}",
                "ratio_in_pop": f"{r.pop_count}/{r.pop_n}",
                "study_genes_str": ",".join(sorted(r.study_items)),
            }
        )
    df_go = pd.DataFrame(rows)
    if len(df_go):
        df_go = df_go.sort_values("p_fdr_bh")
    return df_go


print("run_goatools_enrichment defined.")

---
## §10 — Progressive GO Term Accumulation Curves

As sample removal strategy becomes more aggressive, which GO terms are gained or lost? Track GO enrichment at each step of the harshness axis.

In [ ]:
# ── §10.1 Harshness-ordered gene sets for GO analysis ─────────────────────────
# For each step in HARSHNESS_ORDER, get the union of gene sets across all methods
# (except 25_angel), intersected with GO-annotated genes.

if GO_BACKGROUND and HARSHNESS_ORDER:
    _go_bg_set = set(GO_BACKGROUND)
    _go_step_genes = {
        si: gene_sets_by_strat_imp[si] & _go_bg_set for si in HARSHNESS_ORDER
    }
    _angel_go_genes = angel_representative_genes & _go_bg_set
    print(f"GO-annotated gene counts per step:")
    for si, gs in list(_go_step_genes.items())[:5]:
        print(f"  {si}: {len(gs)} genes")
    print(f"  ... (total {len(_go_step_genes)} steps)")
    print(f"Angel GO-annotated genes: {len(_angel_go_genes)}")
else:
    _go_step_genes = {}
    _angel_go_genes = set()
    print("Skipped: GO database or HARSHNESS_ORDER not available.")

In [ ]:
GO_CACHE_DIR = Path("go_cache")
GO_CACHE_DIR.mkdir(exist_ok=True)

go_results: dict[tuple[str, str], pd.DataFrame] = {}

if godag is not None and _go_step_genes:
    for strat, imp in HARSHNESS_ORDER:
        cache_path = GO_CACHE_DIR / f"{strat}_{imp}.parquet"
        if cache_path.exists():
            go_results[(strat, imp)] = pd.read_parquet(cache_path)
            continue
        study = list(_go_step_genes[(strat, imp)])
        df_enr = run_goatools_enrichment(study, GO_BACKGROUND, full_assoc, godag)
        df_enr.to_parquet(cache_path, index=False)
        go_results[(strat, imp)] = df_enr
        print(f"  {strat} / {imp}: {len(df_enr)} enriched terms (FDR<0.05)")

    _angel_cache = GO_CACHE_DIR / "angel_reference.parquet"
    if _angel_cache.exists():
        go_results_angel = pd.read_parquet(_angel_cache)
    elif _angel_go_genes:
        go_results_angel = run_goatools_enrichment(
            list(_angel_go_genes), GO_BACKGROUND, full_assoc, godag
        )
        go_results_angel.to_parquet(_angel_cache, index=False)
    else:
        go_results_angel = pd.DataFrame()
    print(f"Angel reference: {len(go_results_angel)} enriched terms")
else:
    go_results_angel = pd.DataFrame()
    print("Skipped: GO database not available.")

In [ ]:
# ── §10.3 GO accumulation curve ───────────────────────────────────────────────

if go_results:
    _sig_term_sets = {
        si: set(go_results[si].loc[go_results[si].p_fdr_bh < 0.05, "GO_ID"])
        for si in HARSHNESS_ORDER
        if si in go_results
    }
    _n_sig = [len(_sig_term_sets.get(si, set())) for si in HARSHNESS_ORDER]
    _cumul_terms: list[int] = []
    _seen_terms: set[str] = set()
    _new_per_step: list[int] = []
    _n_genes_list = [len(_go_step_genes.get(si, set())) for si in HARSHNESS_ORDER]
    for si in HARSHNESS_ORDER:
        new_terms = _sig_term_sets.get(si, set()) - _seen_terms
        _seen_terms |= _sig_term_sets.get(si, set())
        _cumul_terms.append(len(_seen_terms))
        _new_per_step.append(len(new_terms))

    _labels_10 = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER]
    _x10 = list(range(len(HARSHNESS_ORDER)))

    fig10, (ax10a, ax10b, ax10c) = plt.subplots(1, 3, figsize=(21, 5))

    ax10a.plot(
        _x10, _n_genes_list, marker="o", markersize=4, label="GO-annotated genes"
    )
    ax10a.plot(
        _x10,
        _cumul_terms,
        marker="s",
        markersize=4,
        linestyle="--",
        label="cumulative unique GO terms",
    )
    if _angel_go_genes:
        ax10a.axhline(
            len(_angel_go_genes),
            color="red",
            linestyle=":",
            label=f"Angel genes ({len(_angel_go_genes)})",
        )
    if len(go_results_angel):
        _n_angel_terms = len(go_results_angel.loc[go_results_angel.p_fdr_bh < 0.05])
        ax10a.axhline(
            _n_angel_terms,
            color="darkred",
            linestyle="-.",
            label=f"Angel GO terms ({_n_angel_terms})",
        )
    ax10a.set_xticks(_x10)
    ax10a.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10a.set_ylabel("Count")
    ax10a.set_title("Gene count and GO term accumulation")
    ax10a.legend(fontsize=7)

    ax10b.bar(_x10, _new_per_step, color="#4292c6")
    ax10b.set_xticks(_x10)
    ax10b.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10b.set_ylabel("New unique GO terms")
    ax10b.set_title("New GO terms per step")

    _density = [
        n / max(1, g - (_n_genes_list[i - 1] if i > 0 else 0))
        for i, (n, g) in enumerate(zip(_new_per_step, _n_genes_list))
    ]
    ax10c.bar(_x10, _density, color="#41ab5d")
    ax10c.set_xticks(_x10)
    ax10c.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10c.set_ylabel("New GO terms / genes added")
    ax10c.set_title("Biological information density per gene added")

    fig10.tight_layout()
    fig10.savefig(FIGURES_DIR / "go_accumulation_curve.svg", bbox_inches="tight")
    fig10.savefig(
        FIGURES_DIR / "go_accumulation_curve.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("GO results not available — run §10.2 first.")

In [ ]:
# ── §10.4 Terms lost between adjacent strategy pairs ─────────────────────────

if len(HARSHNESS_ORDER) >= 2 and go_results:
    _loss_data = []
    for i in range(1, len(HARSHNESS_ORDER)):
        prev = HARSHNESS_ORDER[i - 1]
        curr = HARSHNESS_ORDER[i]
        prev_terms = _sig_term_sets.get(prev, set())
        curr_terms = _sig_term_sets.get(curr, set())
        lost = prev_terms - curr_terms
        _loss_data.append(
            {
                "step": i,
                "prev": prev,
                "curr": curr,
                "n_lost": len(lost),
                "lost_terms": lost,
            }
        )

    if _loss_data:
        max_loss_step = max(_loss_data, key=lambda d: d["n_lost"])
        print(
            f"Largest GO term loss: {max_loss_step['prev']} → {max_loss_step['curr']}: {max_loss_step['n_lost']} terms"
        )

        _lost_df = pd.DataFrame(_loss_data)[["step", "n_lost"]]
        fig_lost, ax_lost = plt.subplots(figsize=(12, 4))
        ax_lost.bar(_lost_df.step, _lost_df.n_lost, color="#d62728")
        ax_lost.set_xlabel("Step index")
        ax_lost.set_ylabel("GO terms lost vs previous step")
        ax_lost.set_title("GO term loss between adjacent harshness steps")
        fig_lost.tight_layout()
        fig_lost.savefig(
            FIGURES_DIR / "go_lost_terms_per_step.svg", bbox_inches="tight"
        )
        fig_lost.savefig(
            FIGURES_DIR / "go_lost_terms_per_step.png", bbox_inches="tight", dpi=200
        )
        plt.show()
else:
    print("Insufficient data for GO term loss analysis.")

In [ ]:
# ── §10.5 Imputation-specific term recovery: supervenn ────────────────────────

try:
    from supervenn import supervenn

    _imp_term_sets = {
        i: set(
            go_results[("A_confirmed_bad", i)].loc[
                go_results[("A_confirmed_bad", i)].p_fdr_bh < 0.05, "GO_ID"
            ]
        )
        for i in ["strict", "knn", "softimpute"]
        if ("A_confirmed_bad", i) in go_results
    }
    _nonempty_imp = {k: v for k, v in _imp_term_sets.items() if v}
    if len(_nonempty_imp) >= 2:
        fig_sv_go, ax_sv_go = plt.subplots(figsize=(10, 5))
        supervenn(list(_nonempty_imp.values()), list(_nonempty_imp.keys()), ax=ax_sv_go)
        ax_sv_go.set_title(
            "GO term recovery by imputation (strat=A_confirmed_bad, FDR<0.05)"
        )
        fig_sv_go.tight_layout()
        fig_sv_go.savefig(FIGURES_DIR / "supervenn_go_by_imp.svg", bbox_inches="tight")
        fig_sv_go.savefig(
            FIGURES_DIR / "supervenn_go_by_imp.png", bbox_inches="tight", dpi=200
        )
        plt.show()
    else:
        print("Not enough GO result sets for supervenn.")

except ImportError:
    print("supervenn not installed.")

---
## §11 — FL-Specific Gene Retention Analysis

Which Follicular Lymphoma marker genes are retained or lost across harmonization strategies? What is the biological cost of aggressive sample removal?

## Gene Set Sources and Citations

| Gene set | Source | DOI / URL |
|---|---|---|
| FL transcriptomic subtypes C1/C2/C3 (FL_2025_MARKERS) | Xochelli et al. 2025, *Leukemia* | https://doi.org/10.1038/s41375-025-02603-9 |
| legacy internal GC B-cell markers (Centroblast/Centrocyte) | unpublished internal panel | superseded by the Dybkaer et al. 2015 panels |
| FL PFS prognostic signature (FL_PROG_SIGNATURES) | Pastore et al. 2019, *J Clin Oncol* | https://doi.org/10.1200/JCO.18.01545 (PMID 29475724) |
| POD24 mutation markers | Huet et al. 2018, *Blood* | https://doi.org/10.1182/blood-2018-03-837443 |
| PROGENy pathway activating genes | Schubert et al. 2018, *Nature Communications* | https://doi.org/10.1038/s41467-017-02391-6 |
| Housekeeping genes (stability reference) | Eisenberg & Levanon 2013, *Trends Genet* | https://doi.org/10.1016/j.tig.2013.05.010 |

In [ ]:
# ── §11.1 Compile FL gene list ────────────────────────────────────────────────

# Source 1: Xochelli et al. 2025 FL C1/C2/C3 subtype markers
FL_2025_MARKERS = [
    "BCL2",
    "BCL6",
    "MKI67",
    "PCNA",
    "AICDA",
    "IRF4",
    "FOXP1",
    "MYC",
    "CD10",
    "CD19",
    "CD20",
    "CD21",
    "CD23",
    "CD38",
    "CD44",
    "CD79A",
    "CXCR4",
    "CXCR5",
    "EZH2",
    "FAS",
    "IGHM",
    "IGHD",
    "IGHG",
    "LMO2",
    "MME",
    "MYBL1",
    "PAX5",
    "PRDM1",
    "SPIB",
    "XBP1",
]

# Source 2: legacy internal GC B-cell markers (Centroblast / Centrocyte)
LEGACY_GC_MARKERS = [
    "AICDA",
    "BCL6",
    "CXCR4",
    "MYBL1",
    "TOP2A",
    "MKI67",
    "PCNA",
    "LMO2",
    "CXCR5",
    "CD83",
    "EBI3",
    "HLA-DRA",
    "CCND2",
    "RGS13",
]

# Source 3: FL PFS prognostic signature (Pastore et al. 2019)
FL_PROG_SIGNATURES = [
    "BCL2L1",
    "CALCRL",
    "CCDC50",
    "DPM2",
    "FAS",
    "GADD45B",
    "GFI1",
    "HVCN1",
    "LBH",
    "MARCKS",
    "PCNA",
    "PER2",
    "PTGDS",
    "RAB27A",
    "S100A8",
    "S1PR1",
    "SGK1",
    "SLC2A6",
    "SPIB",
    "ZBTB7A",
]

# Source 4: POD24 mutation markers (Huet et al. 2018)
POD24_MARKERS = [
    "CREBBP",
    "EP300",
    "EZH2",
    "KMT2D",
    "MEF2B",
    "MYD88",
    "TP53",
    "VHL",
    "ARID1A",
    "CARD11",
    "CDKN2A",
    "FOXO1",
    "GNA13",
    "HIST1H1E",
]

FL_GENES_ALL = sorted(
    set(FL_2025_MARKERS)
    | set(LEGACY_GC_MARKERS)
    | set(FL_PROG_SIGNATURES)
    | set(POD24_MARKERS)
)
FL_GENE_CATEGORIES = {
    g: (
        "FL_2025"
        if g in FL_2025_MARKERS
        else (
            "Normal_B_cells"
            if g in LEGACY_GC_MARKERS
            else "FL_PROG" if g in FL_PROG_SIGNATURES else "POD24"
        )
    )
    for g in FL_GENES_ALL
}

print(f"FL_GENES_ALL: {len(FL_GENES_ALL)} unique genes")
for cat in ["FL_2025", "Normal_B_cells", "FL_PROG", "POD24"]:
    n = sum(1 for g, c in FL_GENE_CATEGORIES.items() if c == cat)
    print(f"  {cat}: {n}")


def validate_gene_aliases(
    gene_list: list[str], reference_genes: set[str]
) -> dict[str, str]:
    """
    Check which genes from a curated list are absent from the expression matrices
    and propose HGNC alias resolution.

    Parameters
    ----------
    gene_list : list of str
        Curated gene symbols to validate.
    reference_genes : set of str
        Union of all gene_sets values (not 25_angel).

    Returns
    -------
    dict[str, str]
        Mapping absent_symbol → suggested_replacement.
    """
    KNOWN_ALIASES = {"CD10": "MME", "CD95": "FAS", "TNFRSF6": "FAS", "CD23": "FCER2"}
    missing = {}
    for g in gene_list:
        if g not in reference_genes:
            replacement = KNOWN_ALIASES.get(g)
            missing[g] = replacement or "not found"
    return missing


if gene_sets_by_strat_imp:
    _ref_all_genes = set.union(*gene_sets_by_strat_imp.values())
    _aliases = validate_gene_aliases(FL_GENES_ALL, _ref_all_genes)
    if _aliases:
        print(f"\nGenes absent from expression matrices ({len(_aliases)}):")
        for g, repl in sorted(_aliases.items()):
            print(f"  {g} → {repl}")
    else:
        print("\nAll FL genes found in expression matrices.")
else:
    _ref_all_genes = set(FL_GENES_ALL)
    print("gene_sets_by_strat_imp empty — skipping alias validation.")

In [ ]:
# ── §11.2 FL gene presence matrix ─────────────────────────────────────────────

if gene_sets_by_strat_imp:
    _fl_presence_rows = []
    for g in FL_GENES_ALL:
        row = {
            f"{s}__{i}": int(g in gene_sets_by_strat_imp.get((s, i), set()))
            for s in ALL_STRATS
            for i in ["strict", "knn", "softimpute"]
            if (s, i) in gene_sets_by_strat_imp
        }
        row["gene"] = g
        row["category"] = FL_GENE_CATEGORIES.get(g, "other")
        _fl_presence_rows.append(row)

    fl_presence = pd.DataFrame(_fl_presence_rows).set_index("gene")
    category_col = fl_presence.pop("category")
    fl_presence.insert(0, "category", category_col)

    attempt_cols = [c for c in fl_presence.columns if c != "category"]
    print(f"fl_presence shape: {fl_presence.shape}")
    print(fl_presence[attempt_cols].sum(axis=0).describe())
else:
    fl_presence = pd.DataFrame()
    print("gene_sets_by_strat_imp empty — skipping §11.2.")

In [ ]:
# ── §11.3 FL gene retention heatmap ──────────────────────────────────────────

if not fl_presence.empty:
    _attempt_cols_sorted = sorted(
        [c for c in fl_presence.columns if c != "category"],
        key=lambda c: (
            (
                ALL_STRATS.index(c.split("__")[0])
                if c.split("__")[0] in ALL_STRATS
                else 99
            ),
            (
                ["strict", "knn", "softimpute"].index(c.split("__")[-1])
                if c.split("__")[-1] in ["strict", "knn", "softimpute"]
                else 9
            ),
        ),
    )
    _gene_order = fl_presence.sort_values("category").index.tolist()

    fig_fl, ax_fl = plt.subplots(
        figsize=(
            max(16, len(_attempt_cols_sorted) * 0.5),
            max(8, len(_gene_order) * 0.3),
        )
    )
    _data_plot = fl_presence.loc[_gene_order, _attempt_cols_sorted].astype(float)
    sns.heatmap(
        _data_plot,
        ax=ax_fl,
        cmap=["#ffcccc", "#006400"],
        linewidths=0.3,
        linecolor="#eeeeee",
        cbar_kws={"label": "Gene present"},
        xticklabels=True,
        yticklabels=True,
    )
    ax_fl.set_xticklabels(ax_fl.get_xticklabels(), rotation=90, fontsize=6)
    ax_fl.set_yticklabels(ax_fl.get_yticklabels(), fontsize=7)

    # Category color bar on the left
    _cat_colors = {
        "FL_2025": "#6a0dad",
        "Normal_B_cells": "#41ab5d",
        "FL_PROG": "#ff7f0e",
        "POD24": "#d62728",
    }
    for ytick in ax_fl.get_yticklabels():
        gene = ytick.get_text()
        cat = FL_GENE_CATEGORIES.get(gene, "other")
        ytick.set_color(_cat_colors.get(cat, "#000000"))

    ax_fl.set_title("FL gene presence across harmonization attempts (dark=present)")
    fig_fl.tight_layout()
    fig_fl.savefig(FIGURES_DIR / "fl_retention_heatmap.svg", bbox_inches="tight")
    fig_fl.savefig(
        FIGURES_DIR / "fl_retention_heatmap.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
# ── §11.4 FL gene progressive retention curves ────────────────────────────────

if HARSHNESS_ORDER and gene_sets_by_strat_imp:
    _cat_list = sorted(set(FL_GENE_CATEGORIES.values()))
    _x11 = list(range(len(HARSHNESS_ORDER)))
    _labels11 = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER]

    # Figure A: one subplot per FL gene category
    fig_a, axes_a = plt.subplots(
        1, len(_cat_list), figsize=(5 * len(_cat_list), 5), sharey=True
    )
    if len(_cat_list) == 1:
        axes_a = [axes_a]
    for ax_cat, cat in zip(axes_a, _cat_list):
        _cat_genes = [g for g, c in FL_GENE_CATEGORIES.items() if c == cat]
        _fracs = [
            sum(1 for g in _cat_genes if g in gene_sets_by_strat_imp.get(si, set()))
            / max(1, len(_cat_genes))
            for si in HARSHNESS_ORDER
        ]
        ax_cat.plot(
            _x11,
            _fracs,
            marker="o",
            markersize=4,
            color=_cat_colors.get(cat, "#AAAAAA"),
        )
        if angel_representative_genes:
            _angel_frac = sum(
                1 for g in _cat_genes if g in angel_representative_genes
            ) / max(1, len(_cat_genes))
            ax_cat.axhline(
                _angel_frac,
                color="red",
                linestyle=":",
                label=f"Angel ({_angel_frac:.2f})",
            )
            ax_cat.legend(fontsize=7)
        ax_cat.set_title(f"{cat} ({len(_cat_genes)} genes)")
        ax_cat.set_xticks(_x11)
        ax_cat.set_xticklabels(_labels11, rotation=45, ha="right", fontsize=7)
        ax_cat.set_ylim(0, 1.05)
        ax_cat.set_ylabel("Fraction of genes retained")
    fig_a.suptitle("FL gene retention by category along harshness axis", fontsize=12)
    fig_a.tight_layout()
    fig_a.savefig(
        FIGURES_DIR / "fl_retention_curves_by_category.svg", bbox_inches="tight"
    )
    fig_a.savefig(
        FIGURES_DIR / "fl_retention_curves_by_category.png",
        bbox_inches="tight",
        dpi=200,
    )
    plt.show()

    # Figure B: overall retention + per-category thin lines
    fig_b, ax_b11 = plt.subplots(figsize=(14, 5))
    _overall_fracs = [
        sum(1 for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set()))
        / max(1, len(FL_GENES_ALL))
        for si in HARSHNESS_ORDER
    ]
    ax_b11.plot(
        _x11, _overall_fracs, linewidth=2.5, color="#000000", label="All FL genes"
    )
    for cat in _cat_list:
        _cat_genes = [g for g, c in FL_GENE_CATEGORIES.items() if c == cat]
        _fracs_cat = [
            sum(1 for g in _cat_genes if g in gene_sets_by_strat_imp.get(si, set()))
            / max(1, len(_cat_genes))
            for si in HARSHNESS_ORDER
        ]
        ax_b11.plot(
            _x11,
            _fracs_cat,
            linewidth=0.8,
            alpha=0.6,
            color=_cat_colors.get(cat, "#AAAAAA"),
            label=cat,
        )
    if angel_representative_genes:
        _af = sum(1 for g in FL_GENES_ALL if g in angel_representative_genes) / max(
            1, len(FL_GENES_ALL)
        )
        ax_b11.axhline(_af, color="red", linestyle=":", label=f"Angel ({_af:.2f})")
    ax_b11.set_xticks(_x11)
    ax_b11.set_xticklabels(_labels11, rotation=45, ha="right", fontsize=7)
    ax_b11.set_ylim(0, 1.05)
    ax_b11.set_ylabel("Fraction of FL genes retained")
    ax_b11.set_title("FL gene retention across harshness axis")
    ax_b11.legend(fontsize=8, loc="lower left")
    fig_b.tight_layout()
    fig_b.savefig(FIGURES_DIR / "fl_retention_curves_overall.svg", bbox_inches="tight")
    fig_b.savefig(
        FIGURES_DIR / "fl_retention_curves_overall.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("HARSHNESS_ORDER empty — run §8.1 and §6 first.")

In [ ]:
# ── §11.5 GO enrichment in FL-specific gene subset ────────────────────────────

if godag is not None and gene_sets_by_strat_imp:
    _fl_go_results: dict[tuple[str, str], pd.DataFrame] = {}
    for si in HARSHNESS_ORDER[:6]:  # top-6 steps for tractability
        strat, imp = si
        study = [g for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set())]
        if len(study) >= 3:
            _cache_fl = GO_CACHE_DIR / f"fl_{strat}_{imp}.parquet"
            if _cache_fl.exists():
                _fl_go_results[si] = pd.read_parquet(_cache_fl)
            else:
                df_fl_enr = run_goatools_enrichment(
                    study, GO_BACKGROUND, full_assoc, godag
                )
                df_fl_enr.to_parquet(_cache_fl, index=False)
                _fl_go_results[si] = df_fl_enr

    if _fl_go_results:
        _fl_go_sig = {
            si: df[df.p_fdr_bh < 0.05].head(20) for si, df in _fl_go_results.items()
        }
        _all_fl_terms = (
            pd.concat(list(_fl_go_sig.values())).drop_duplicates("GO_ID")
            if any(len(v) for v in _fl_go_sig.values())
            else pd.DataFrame()
        )
        if len(_all_fl_terms):
            for ns in ["biological_process", "molecular_function"]:
                _ns_terms = _all_fl_terms[
                    _all_fl_terms.namespace.str.contains(ns[:2], case=False)
                ]
                if not len(_ns_terms):
                    continue
                _term_names = _ns_terms.set_index("GO_ID")["name"].to_dict()
                _dot_data = (
                    pd.DataFrame(
                        {
                            si: _fl_go_results[si]
                            .set_index("GO_ID")
                            .get("p_fdr_bh", pd.Series(dtype=float))
                            for si in list(_fl_go_results.keys())
                        }
                    )
                    .loc[
                        [
                            g
                            for g in _ns_terms.GO_ID
                            if g
                            in pd.concat(list(_fl_go_results.values())).GO_ID.values
                        ]
                    ]
                    .fillna(1.0)
                )

                if not _dot_data.empty:
                    fig_dot, ax_dot = plt.subplots(
                        figsize=(10, max(4, len(_dot_data) * 0.4))
                    )
                    sns.heatmap(
                        -np.log10(_dot_data.astype(float) + 1e-10),
                        ax=ax_dot,
                        cmap="Blues",
                        cbar_kws={"label": "-log10(p_fdr_bh)"},
                        linewidths=0.3,
                    )
                    ax_dot.set_yticks(range(len(_dot_data.index)))
                    ax_dot.set_yticklabels(
                        [_term_names.get(g, g) for g in _dot_data.index], fontsize=7
                    )
                    ax_dot.set_xticklabels(
                        [f"{s[:6]}\n{i[:3]}" for s, i in _dot_data.columns], fontsize=7
                    )
                    ax_dot.set_title(f"FL gene GO enrichment — {ns.replace('_', ' ')}")
                    fig_dot.tight_layout()
                    fig_dot.savefig(
                        FIGURES_DIR / f"fl_go_dot_{ns[:2]}.svg", bbox_inches="tight"
                    )
                    fig_dot.savefig(
                        FIGURES_DIR / f"fl_go_dot_{ns[:2]}.png",
                        bbox_inches="tight",
                        dpi=200,
                    )
                    plt.show()
else:
    print("GO database or gene sets not available — skipping §11.5.")

In [ ]:
# ── §11.6 Progressive FL-genes GO accumulation curve ─────────────────────────

if _fl_go_results:
    _fl_sig_sets = {
        si: set(_fl_go_results[si].loc[_fl_go_results[si].p_fdr_bh < 0.05, "GO_ID"])
        for si in HARSHNESS_ORDER[:6]
        if si in _fl_go_results
    }
    _fl_term_counts = [len(_fl_sig_sets.get(si, set())) for si in HARSHNESS_ORDER[:6]]
    _fl_steps = list(range(len(HARSHNESS_ORDER[:6])))
    _fl_labels = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER[:6]]

    fig_fl_go, ax_fl_go = plt.subplots(figsize=(12, 4))
    ax_fl_go.plot(_fl_steps, _fl_term_counts, marker="o", markersize=5)
    if len(go_results_angel) and "GO_ID" in go_results_angel.columns:
        _n_angel_fl = len(go_results_angel.loc[go_results_angel.p_fdr_bh < 0.05])
        ax_fl_go.axhline(
            _n_angel_fl,
            color="red",
            linestyle=":",
            label=f"Angel ({_n_angel_fl} terms)",
        )
        ax_fl_go.legend(fontsize=8)
    ax_fl_go.set_xticks(_fl_steps)
    ax_fl_go.set_xticklabels(_fl_labels, rotation=45, ha="right", fontsize=7)
    ax_fl_go.set_ylabel("Significant GO terms (FDR<0.05)")
    ax_fl_go.set_title("FL-gene GO enrichment along harshness axis")
    fig_fl_go.tight_layout()
    fig_fl_go.savefig(FIGURES_DIR / "fl_go_accumulation.svg", bbox_inches="tight")
    fig_fl_go.savefig(
        FIGURES_DIR / "fl_go_accumulation.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("FL GO results not available — run §11.5 first.")

In [ ]:
# ── §11.7 Final recommendation table ─────────────────────────────────────────

_rec_rows = []
for _, row in df_ok.sort_values("composite_score", ascending=False).iterrows():
    si = (row["strat"], row["imp"])
    _fl_ret = (
        sum(1 for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set()))
        / max(1, len(FL_GENES_ALL))
        if gene_sets_by_strat_imp
        else None
    )
    _rec_rows.append(
        {
            "strat": row["strat"],
            "imp": row["imp"],
            "method": row["method"],
            "post_rm": row["post_rm"],
            "composite_score": row.get("composite_score"),
            "score_batch_mixing": row.get("score_batch_mixing"),
            "score_bio_preservation": row.get("score_bio_preservation"),
            "r2_RNA_BATCH": row.get("r2_RNA_BATCH"),
            "kbet_acceptance_rate_RNA_BATCH": row.get("kbet_acceptance_rate_RNA_BATCH"),
            "n_genes": row.get("n_genes"),
            "fl_gene_retention": _fl_ret,
            "is_top10": row.name in top_n.index,
        }
    )

rec_df = pd.DataFrame(_rec_rows)
rec_df.insert(0, "rank", range(1, len(rec_df) + 1))

# Display styled
_fmt = {
    "composite_score": "{:.3f}",
    "score_batch_mixing": "{:.3f}",
    "score_bio_preservation": "{:.3f}",
    "r2_RNA_BATCH": "{:.3f}",
    "fl_gene_retention": "{:.2f}",
}
styled = (
    rec_df.head(30)
    .style.background_gradient(subset=["composite_score"], cmap="RdYlGn")
    .background_gradient(subset=["fl_gene_retention"], cmap="Blues")
    .format(_fmt, na_rep="N/A")
)
display(styled)

# Export CSV
rec_df.to_csv("recommendation_table.csv", index=False)
print("Saved: recommendation_table.csv")

# Upload to S3
try:
    _s3_client.put_object(
        Bucket=S3_BUCKET,
        Key=f"{S3_PREFIX}/recommendation_table.csv",
        Body=rec_df.to_csv(index=False).encode(),
    )
    print(f"Uploaded to s3://{S3_BUCKET}/{S3_PREFIX}/recommendation_table.csv")
except Exception as exc:
    print(f"S3 upload failed: {exc}")